# 引入套件

In [1]:
import os
import pandas as pd
import numpy as np
from finlab.dataframe import FinlabDataFrame
from finlab.backtest import sim
from finlab import data
import talib
import finlab
import logging
import openpyxl
from scipy.stats import linregress
from dotenv import load_dotenv
# 載入環境變數
load_dotenv()
# 使用環境變數
finlab.login(os.getenv('FINLAB_API_KEY'))
# 設置 finlab 數據存儲路徑
# data.set_storage(data.FileStorage(path="E:\\pickle"))

輸入成功!


# 下載ETF資料

In [2]:
"""
此程式整合了三種基金與ETF持股資料的爬取功能，並將資料儲存至各自獨立的CSV檔案中。
採用多執行緒 (Multi-threading) 方式並行抓取，以提升執行效率。

1. [每日] 主動式 ETF 持股明細 -> all_etf_holdings.csv (並行 Selenium)
2. [每月] 投信基金前十大投資明細 -> sitca_fund_holdings.csv (並行 Requests)
3. [每季] 投信基金占淨值1%以上投資明細 -> sitca_fund_holdings_over_1_percent_quarterly.csv (並行 Requests)

程式會自動讀取對應的資料檔案，判斷最新時間點，並進行增量更新。
"""

# doc: =============================================================================
# doc: 匯入所有必要的函式庫
# doc: =============================================================================
import pandas as pd
import time
import os
import re
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from io import StringIO
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# [修改] 引入 Selenium 的例外處理
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
# NEW: 匯入 concurrent.futures 模組
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging  # [修改] 增加 logging 模組
# doc: =============================================================================
# doc: 全域設定
# doc: =============================================================================
# SITCA 爬蟲的並行數量 (避免對伺服器造成過大壓力)
MAX_WORKERS_SITCA = 8

# doc: =============================================================================
# doc: (爬蟲核心 1/3) 每日主動式 ETF
# doc: =============================================================================


def get_etf_holdings_selenium(stock_id: str) -> tuple[pd.DataFrame | None, str | None]:
    """
    doc: 使用 Selenium 抓取單一 ETF 的持股明細及更新日期。
    [修改] 增加完整的 try...except...finally 錯誤處理機制。
    
    Args:
        stock_id (str): ETF 的代號。

    Returns:
        tuple[pd.DataFrame | None, str | None]: 包含 (DataFrame, 更新日期字串) 的元組，失敗則返回 (None, None)。
    """
    url = f"https://www.pocket.tw/etf/tw/{stock_id}/fundholding"
    service = Service()
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # [修改] 建議保持 headless 模式
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--log-level=3')
    options.add_experimental_option('excludeSwitches', ['enable-logging'])

    driver = None
    try:
        driver = webdriver.Chrome(service=service, options=options)
        driver.get(url)

        wait = WebDriverWait(driver, 20)  # 等待時間設為 20 秒

        # 1. 等待持股表格出現
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, ".cm-table__table tbody tr")))

        # 2. 等待日期資訊出現
        wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, ".fundholding__date")))

        time.sleep(1)  # 最終等待渲染

        # 3. 抓取日期
        date_elements = driver.find_elements(
            By.CSS_SELECTOR, ".fundholding__date")
        update_date = date_elements[0].text.replace(
            '資料日期：', '').strip() if date_elements else None

        if not update_date:
            logging.warning(f"爬取 {stock_id} ({url}) 時找不到日期元素。")
            return None, None

        # 4. 抓取表格
        tables = pd.read_html(StringIO(driver.page_source))

        fund_holdings_df = None
        for table in tables:
            if len(table.columns) >= 2 and isinstance(table.columns[0], str) and isinstance(table.columns[1], str):
                if '代號' in table.columns[0] and '名稱' in table.columns[1]:
                    fund_holdings_df = table
                    break

        if fund_holdings_df is None:
            logging.warning(f"錯誤：在 {stock_id} 頁面中找不到持股表格。")
            return None, update_date

        df = fund_holdings_df.iloc[:, 0:4].copy()
        df.columns = ['代號', '名稱', '權重', '持有數']

        df['權重'] = pd.to_numeric(df['權重'].astype(str).str.replace(
            '%', '', regex=False), errors='coerce')
        df['持有數'] = pd.to_numeric(df['持有數'].astype(
            str).str.replace(',', '', regex=False), errors='coerce')
        df.dropna(subset=['代號', '名稱', '權重', '持有數'], inplace=True)

        df['持有數'] = (df['持有數'] / 1000).astype(int)
        df['單位'] = '張'

        return df, update_date

    # [修改] 增加更詳細的錯誤捕捉
    except (TimeoutException, NoSuchElementException):
        logging.warning(
            f"爬取 {stock_id} ({url}) 失敗 (Timeout 或 找不到元素)。網頁可能改版或載入過久。")
        return None, None
    except WebDriverException as e:
        logging.error(f"爬取 {stock_id} ({url}) 時 Selenium Driver 發生錯誤: {e}")
        return None, None
    except (IndexError, KeyError, AttributeError) as e:
        logging.warning(
            f"爬取 {stock_id} ({url}) 成功，但解析資料失敗 (Index/Key/AttributeError)。網頁格式可能已變動: {e}")
        return None, None
    except Exception as e:
        logging.error(f"爬取 {stock_id} ({url}) 過程中發生未預期錯誤: {e}")
        return None, None
    finally:
        if driver:
            driver.quit()

# doc: =============================================================================
# doc: (爬蟲核心 2/3) 每月投信前十大持股
# doc: =============================================================================


def crawl_sitca_monthly_top10(year: int, month: int, session: requests.Session) -> pd.DataFrame:
    """
    doc: 爬取 SITCA 網站指定年月的「基金月前十大投資明細」。
    [修改] 增加 requests.exceptions.RequestException 錯誤處理。
    """
    url = "https://www.sitca.org.tw/ROC/Industry/IN2629.aspx?pid=IN22601_04"
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

        # [修改] 增加 timeout
        res_get = session.get(url, headers=headers, timeout=30)
        res_get.raise_for_status()  # 檢查 HTTP 錯誤

        soup = BeautifulSoup(res_get.text, 'html.parser')
        viewstate = soup.find('input', {'id': '__VIEWSTATE'}).get('value')
        eventvalidation = soup.find(
            'input', {'id': '__EVENTVALIDATION'}).get('value')
        viewstategenerator = soup.find(
            'input', {'id': '__VIEWSTATEGENERATOR'}).get('value')

        payload = {
            '__EVENTTARGET': '', '__EVENTARGUMENT': '', '__LASTFOCUS': '',
            '__VIEWSTATE': viewstate, '__VIEWSTATEGENERATOR': viewstategenerator,
            '__EVENTVALIDATION': eventvalidation,
            'ctl00$ContentPlaceHolder1$ddlQ_YM': f"{year}{month:02d}",
            'ctl00$ContentPlaceHolder1$rdo1': 'rbClass',
            'ctl00$ContentPlaceHolder1$ddlQ_Class': 'AA1',
            'ctl00$ContentPlaceHolder1$BtnQuery': '查詢',
        }

        # [修改] 增加 timeout
        res_post = session.post(url, data=payload, headers=headers, timeout=90)
        res_post.raise_for_status()

        soup = BeautifulSoup(res_post.text, 'html.parser')
        header_cell = soup.find('td', class_='DTHeader')
        if not header_cell or '基金名稱' not in header_cell.get_text():
            # [修改] print 改為 logging.info
            logging.info(f"-> [月報] {year} 年 {month} 月查無資料。")
            return pd.DataFrame()

        table = header_cell.find_parent('tr').find_parent('table')
        all_rows_data = []
        current_fund_name = ""
        for row in table.find_all('tr'):
            if row.find('td', class_='DTHeader') or row.find('td', class_='DTsubtotal'):
                continue
            cells = row.find_all('td')
            if not cells:
                continue

            if 'rowspan' in cells[0].attrs:
                current_fund_name = cells[0].text.strip()
                row_data = [current_fund_name] + [cell.text.strip()
                                                  for cell in cells[1:]]
            else:
                row_data = [current_fund_name] + [cell.text.strip()
                                                  for cell in cells]

            if len(row_data) >= 10:
                selected_data = {
                    '基金名稱': row_data[0], '名次': row_data[1], '標的種類': row_data[2],
                    '標的代號': row_data[3], '標的名稱': row_data[4], '金額': row_data[5],
                    '占基金淨資產價值之比例(%)': row_data[9]
                }
                if re.match(r'^\\d{4}$', selected_data['標的代號']):
                    all_rows_data.append(selected_data)

        return pd.DataFrame(all_rows_data)

    # [修改] 增加錯誤處理
    except requests.exceptions.RequestException as e:
        logging.warning(f"-> [月報] {year} 年 {month} 月發生網路錯誤: {e}")
        return pd.DataFrame()
    except (AttributeError, IndexError, KeyError) as e:
        logging.warning(f"-> [月報] {year} 年 {month} 月解析資料失敗 (網頁格式可能變動): {e}")
        return pd.DataFrame()
    except Exception as e:
        logging.error(f"-> [月報] {year} 年 {month} 月發生未預期錯誤: {e}")
        return pd.DataFrame()

# doc: =============================================================================
# doc: (爬蟲核心 3/3) 每季投信1%以上持股
# doc: =============================================================================


def crawl_sitca_quarterly_over_1_percent(year: int, month: int, session: requests.Session) -> pd.DataFrame:
    """
    doc: 爬取 SITCA 網站指定年季的「占基金淨資產價值1%以上投資明細」。
    [修改] 增加 requests.exceptions.RequestException 錯誤處理。
    """
    url = "https://www.sitca.org.tw/ROC/Industry/IN2630.aspx?pid=IN22601_05"
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

        # [修改] 增加 timeout
        res_get = session.get(url, headers=headers, timeout=30)
        res_get.raise_for_status()
        soup = BeautifulSoup(res_get.text, 'html.parser')
        viewstate = soup.find('input', {'id': '__VIEWSTATE'}).get('value')
        eventvalidation = soup.find(
            'input', {'id': '__EVENTVALIDATION'}).get('value')
        viewstategenerator = soup.find(
            'input', {'id': '__VIEWSTATEGENERATOR'}).get('value')

        payload = {
            '__EVENTTARGET': '', '__EVENTARGUMENT': '', '__LASTFOCUS': '',
            '__VIEWSTATE': viewstate, '__VIEWSTATEGENERATOR': viewstategenerator,
            '__EVENTVALIDATION': eventvalidation,
            'ctl00$ContentPlaceHolder1$ddlQ_YM': f"{year}{month:02d}",
            'ctl00$ContentPlaceHolder1$rdo1': 'rbClass',
            'ctl00$ContentPlaceHolder1$ddlQ_Class': 'AA1',
            'ctl00$ContentPlaceHolder1$BtnQuery': '查詢',
        }

        # [修改] 增加 timeout
        res_post = session.post(url, data=payload, headers=headers, timeout=90)
        res_post.raise_for_status()

        soup = BeautifulSoup(res_post.text, 'html.parser')
        header_cell = soup.find('td', class_='DTHeader')
        if not header_cell or '基金名稱' not in header_cell.get_text():
            # [修改] print 改為 logging.info
            logging.info(f"-> [季報] {year} 年 {month} 月查無資料。")
            return pd.DataFrame()

        # ... (以下解析邏輯與您原檔相同) ...
        table = header_cell.find_parent('tr').find_parent('table')
        all_rows_data = []
        current_fund_name = ""
        for row in table.find_all('tr'):
            if row.find('td', class_='DTHeader') or row.find('td', class_='DTsubtotal'):
                continue
            cells = row.find_all('td')
            if not cells:
                continue

            if 'rowspan' in cells[0].attrs:
                current_fund_name = cells[0].text.strip()
                row_data = [current_fund_name] + [cell.text.strip()
                                                  for cell in cells[1:]]
            else:
                row_data = [current_fund_name] + [cell.text.strip()
                                                  for cell in cells]

            stock_id = row_data[2].strip() if len(row_data) > 2 else ""
            if len(row_data) >= 9 and re.match(r'^\\d{4}$', stock_id):
                selected_data = {
                    '基金名稱': row_data[0], '標的種類': row_data[1], '標的代號': stock_id,
                    '標的名稱': row_data[3], '金額': row_data[4],
                    '占基金淨資產價值之比例(%)': row_data[8]
                }
                all_rows_data.append(selected_data)

        return pd.DataFrame(all_rows_data)

    # [修改] 增加錯誤處理
    except requests.exceptions.RequestException as e:
        logging.warning(f"-> [季報] {year} 年 {month} 月發生網路錯誤: {e}")
        return pd.DataFrame()
    except (AttributeError, IndexError, KeyError) as e:
        logging.warning(f"-> [季報] {year} 年 {month} 月解析資料失敗 (網頁格式可能變動): {e}")
        return pd.DataFrame()
    except Exception as e:
        logging.error(f"-> [季報] {year} 年 {month} 月發生未預期錯誤: {e}")
        return pd.DataFrame()

# doc: =============================================================================
# doc: (更新流程 1/3) 每日主動式 ETF
# doc: =============================================================================


def update_daily_etf_data():
    """
    doc: 【已並行化】執行每日主動式 ETF 的檢查與更新流程，存入 all_etf_holdings.csv
    """
    print("\n" + "="*60)
    print("--- (1/3) 開始更新 [每日] 主動式 ETF 持股資料 ---")
    print("="*60)

    etf_list = ['00981A', '00982A', '00980A', '00984A']
    csv_filename = "all_etf_holdings.csv"

    if os.path.exists(csv_filename):
        print(f"找到已存在的檔案: {csv_filename}，將會進行增量更新。")
        existing_df = pd.read_csv(csv_filename)
    else:
        print(f"未找到 {csv_filename}，將會建立新檔案。")
        existing_df = pd.DataFrame()

    newly_scraped_data = []

    # --- [修改] START: 改用 ThreadPoolExecutor ---\n",
    print(f"啟動並行爬蟲，共 {len(etf_list)} 支 ETF...")

    # 使用 ThreadPoolExecutor 來並行抓取
    # max_workers = 4 (len(etf_list))，代表 4 個瀏覽器會同時開啟
    with ThreadPoolExecutor(max_workers=len(etf_list)) as executor:
        # 建立一個 future -> etf_id 的字典，以便稍後取回結果時知道是哪一支
        # executor.submit(fn, *args) 會立即提交任務並返回一個 future 物件
        futures = {
            executor.submit(get_etf_holdings_selenium, etf_id): etf_id
            for etf_id in etf_list
        }

        # as_completed 會在任何一個任務 (future) 完成時立即返回
        for future in as_completed(futures):
            etf_id = futures[future]  # 從字典中找回
            print(f"\n--- 處理 ETF: {etf_id} 的爬取結果 ---")

            try:
                # .result() 會獲取任務的返回值 (df, date)
                holdings_df, data_date = future.result()

                # [修改] 檢查 holdings_df 是否有效 (非 None 且非空)
                if holdings_df is not None and not holdings_df.empty and data_date:
                    is_existing = False
                    if not existing_df.empty:
                        # 檢查同樣ETF和同樣日期的資料是否已存在
                        if not existing_df[(existing_df['etf'] == etf_id) & (existing_df['日期'] == data_date)].empty:
                            is_existing = True

                    if is_existing:
                        print(f"-> 資料已是最新 (日期: {data_date})，跳過 {etf_id}")
                    else:
                        print(f"-> 發現新資料 (日期: {data_date})，準備寫入 {etf_id}")
                        holdings_df['etf'] = etf_id
                        holdings_df['日期'] = data_date
                        newly_scraped_data.append(holdings_df)
                else:
                    # [修改] 將原來的 print 改為 logging.warning
                    logging.warning(f"-> 未能成功抓取到 {etf_id} 的持股資料或日期。")

            except Exception as e:
                # 確保即使單一執行緒出錯，也不會讓整個程式崩潰
                logging.error(f"-> 抓取 {etf_id} 過程中執行緒發生嚴重錯誤: {e}")

    # --- [修改] END: ThreadPoolExecutor 區塊 ---\n",

    if newly_scraped_data:
        print("\n正在合併新舊 ETF 資料...")
        new_df = pd.concat(newly_scraped_data, ignore_index=True)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)

        final_columns = ['日期', 'etf', '代號', '名稱', '權重', '持有數', '單位']

        # [修改] 確保 final_columns 只包含 combined_df 中實際存在的欄位
        final_columns = [
            col for col in final_columns if col in combined_df.columns]
        combined_df = combined_df[final_columns]

        combined_df.drop_duplicates(
            subset=['日期', 'etf', '代號'], keep='last', inplace=True)
        combined_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"[每日] ETF 資料已成功更新並儲存至 {csv_filename}")
    else:
        print("\n本次執行未抓取到任何新的 [每日] ETF 資料。")

# doc: =============================================================================
# doc: (更新流程 2/3) 每月投信前十大持股
# doc: =============================================================================


def update_monthly_data(session: requests.Session):
    """
    doc: 【已並行化】執行每月持股資料的檢查與更新流程，存入 sitca_fund_holdings.csv
    """
    print("\n" + "="*60)
    print("--- (2/3) 開始更新 [每月] 前十大持股資料 ---")
    print("="*60)

    csv_filename = 'sitca_fund_holdings.csv'
    df_existing = pd.DataFrame()

    if os.path.exists(csv_filename):
        print(f"發現已存在檔案 '{csv_filename}'，將從上次結束的地方繼續更新。")
        df_existing = pd.read_csv(csv_filename)
        if not df_existing.empty:
            df_existing['日期'] = pd.to_datetime(df_existing['日期'])
            start_date = df_existing['日期'].max() + pd.DateOffset(months=1)
        else:
            start_date = pd.to_datetime('2015-06-01')
    else:
        print(f"未發現舊檔案，將從 2015年6月 開始全新抓取。")
        start_date = pd.to_datetime('2015-06-01')

    today = pd.Timestamp.now()
    end_date = today - pd.DateOffset(months=1 if today.day > 10 else 2)
    date_range = pd.date_range(start_date, end_date, freq='MS')

    if date_range.empty:
        print("您的 [每月] 資料已經是最新，無需更新！")
        return

    print(
        f"準備爬取從 {date_range[0].strftime('%Y-%m')} 到 {date_range[-1].strftime('%Y-%m')} 的 [每月] 資料...")
    print(f"將使用 {MAX_WORKERS_SITCA} 個並行 worker...")

    all_new_dataframes = [df_existing] if not df_existing.empty else []

    # --- [修改] START: 改用 ThreadPoolExecutor ---\n",
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_SITCA) as executor:
        # 建立 future -> date 的字典
        futures = {
            executor.submit(crawl_sitca_monthly_top10, date.year, date.month, session): date
            for date in date_range
        }

        for future in as_completed(futures):
            date = futures[future]  # 獲取對應的日期
            try:
                df_month = future.result()  # 獲取爬蟲結果
                if not df_month.empty:
                    print(f"-> 成功抓取 [月報] {date.strftime('%Y-%m')} 資料")
                    df_month['日期'] = date
                    all_new_dataframes.append(df_month)
                # (原版的 '查無資料' 訊息已在 crawl_... 函式中印出)

            except Exception as e:
                print(f"-> 抓取 [月報] {date.strftime('%Y-%m')} 過程中執行緒發生嚴重錯誤: {e}")

    # --- [修改] END: ThreadPoolExecutor 區塊 ---\n",

    # 檢查是否有 *新* 資料被加入 (原邏輯是 > 0，若已有舊資料，應 > 1)
    if len(all_new_dataframes) > (1 if not df_existing.empty else 0):
        full_df = pd.concat(all_new_dataframes, ignore_index=True)
        numeric_cols = ['名次', '金額', '占基金淨資產價值之比例(%)']
        for col in numeric_cols:
            full_df[col] = pd.to_numeric(full_df[col].astype(
                str).str.replace(',', ''), errors='coerce')

        full_df['日期'] = pd.to_datetime(full_df['日期'])
        full_df.drop_duplicates(inplace=True)
        full_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"\n[每月] 資料更新完畢，已全部儲存至 {csv_filename}")
    else:
        print("\n本次執行未抓取到新的 [每月] 資料。")


# doc: =============================================================================
# doc: (更新流程 3/3) 每季投信1%以上持股
# doc: =============================================================================
def update_quarterly_data(session: requests.Session):
    """
    doc: 【已並行化】執行每季持股資料的檢查與更新流程，存入 sitca_fund_holdings_over_1_percent_quarterly.csv
    """
    print("\n" + "="*60)
    print("--- (3/3) 開始更新 [每季] 1%以上持股資料 ---")
    print("="*60)

    csv_filename = 'sitca_fund_holdings_over_1_percent_quarterly.csv'
    df_existing = pd.DataFrame()

    if os.path.exists(csv_filename):
        print(f"發現已存在檔案 '{csv_filename}'，將從上次結束的地方繼續更新。")
        df_existing = pd.read_csv(csv_filename)
        if not df_existing.empty:
            df_existing['日期'] = pd.to_datetime(df_existing['日期'])
            start_date = df_existing['日期'].max() + pd.DateOffset(months=3)
        else:
            start_date = pd.to_datetime('2015-06-01')
    else:
        print(f"未發現舊檔案，將從 2015年6月 開始全新抓取。")
        start_date = pd.to_datetime('2015-06-01')

    end_date = pd.Timestamp.now()
    # 修正: 'QE' 是季末 (Quarter End)，SITCA 資料是以 3, 6, 9, 12 月為準
    # 例如 2015-06-01 (start) -> 2015-06-30 (QE)
    all_possible_dates = pd.date_range(start_date, end_date, freq='QE')

    # 季報通常在下一個月 10 號左右公布
    # 直接在季末日期上加上天數 (例如 10 天或 11 天)
    crawlable_dates = [d for d in all_possible_dates if pd.Timestamp.now() > (
        d + pd.DateOffset(days=11))]  # 建議用 11 天 (10號公布，11號抓) 較保險

    if not crawlable_dates:
        print("您的 [每季] 資料已經是最新，或最新一季資料尚未公佈，無需更新！")
        return

    print(
        f"準備爬取從 {crawlable_dates[0].strftime('%Y-%m')} 到 {crawlable_dates[-1].strftime('%Y-%m')} 的 [每季] 資料...")
    print(f"將使用 {MAX_WORKERS_SITCA} 個並行 worker...")

    all_new_dataframes = [df_existing] if not df_existing.empty else []

    # --- [修改] START: 改用 ThreadPoolExecutor ---\n",
    with ThreadPoolExecutor(max_workers=MAX_WORKERS_SITCA) as executor:
        # 建立 future -> date 的字典
        futures = {
            executor.submit(crawl_sitca_quarterly_over_1_percent, date.year, date.month, session): date
            for date in crawlable_dates
        }

        for future in as_completed(futures):
            date = futures[future]  # 獲取對應的日期
            try:
                df_quarter = future.result()  # 獲取爬蟲結果
                if not df_quarter.empty:
                    print(f"-> 成功抓取 [季報] {date.strftime('%Y-%m')} 資料")
                    df_quarter['日期'] = date
                    all_new_dataframes.append(df_quarter)

            except Exception as e:
                print(f"-> 抓取 [季報] {date.strftime('%Y-%m')} 過程中執行緒發生嚴重錯誤: {e}")

    # --- [修改] END: ThreadPoolExecutor 區塊 ---\n",

    if len(all_new_dataframes) > (1 if not df_existing.empty else 0):
        full_df = pd.concat(all_new_dataframes, ignore_index=True)
        numeric_cols = ['金額', '占基金淨資產價值之比例(%)']
        for col in numeric_cols:
            full_df[col] = pd.to_numeric(full_df[col].astype(
                str).str.replace(',', ''), errors='coerce')

        full_df['日期'] = pd.to_datetime(full_df['日期'])
        full_df.drop_duplicates(inplace=True)
        full_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"\n[每季] 資料更新完畢，已全部儲存至 {csv_filename}")
    else:
        print("\n本次執行未抓取到新的 [每季] 資料。")


# doc: =============================================================================
# doc: 主執行函式
# doc: =============================================================================
def main():
    """
    doc: 主執行函式，依序更新每日、每月與每季資料。
    """
    # 執行每日 ETF 更新 (使用 Selenium，已並行化)
    update_daily_etf_data()

    # 針對 SITCA 的更新，使用同一個 session 物件 (thread-safe)
    with requests.Session() as session:
        # 執行每月 SITCA 更新 (使用 Requests，已並行化)
        update_monthly_data(session)
        # 執行每季 SITCA 更新 (使用 Requests，已並行化)
        update_quarterly_data(session)

    print("\n" + "="*60)
    print("--- 所有更新任務已完成 ---")
    print("="*60)


if __name__ == "__main__":
    main()


--- (1/3) 開始更新 [每日] 主動式 ETF 持股資料 ---
找到已存在的檔案: all_etf_holdings.csv，將會進行增量更新。
啟動並行爬蟲，共 4 支 ETF...

--- 處理 ETF: 00981A 的爬取結果 ---
-> 發現新資料 (日期: 2026/01/29)，準備寫入 00981A

--- 處理 ETF: 00980A 的爬取結果 ---
-> 發現新資料 (日期: 2026/01/29)，準備寫入 00980A

--- 處理 ETF: 00984A 的爬取結果 ---
-> 發現新資料 (日期: 2026/01/29)，準備寫入 00984A

--- 處理 ETF: 00982A 的爬取結果 ---
-> 發現新資料 (日期: 2026/01/29)，準備寫入 00982A

正在合併新舊 ETF 資料...
[每日] ETF 資料已成功更新並儲存至 all_etf_holdings.csv

--- (2/3) 開始更新 [每月] 前十大持股資料 ---
發現已存在檔案 'sitca_fund_holdings.csv'，將從上次結束的地方繼續更新。
準備爬取從 2025-10 到 2025-12 的 [每月] 資料...
將使用 8 個並行 worker...

本次執行未抓取到新的 [每月] 資料。

--- (3/3) 開始更新 [每季] 1%以上持股資料 ---
發現已存在檔案 'sitca_fund_holdings_over_1_percent_quarterly.csv'，將從上次結束的地方繼續更新。
準備爬取從 2025-12 到 2025-12 的 [每季] 資料...
將使用 8 個並行 worker...

本次執行未抓取到新的 [每季] 資料。

--- 所有更新任務已完成 ---


# 開始畫圖

In [3]:
import os
import gc
import logging
from datetime import datetime
import textwrap
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib.backends.backend_pdf import PdfPages
from scipy.signal import argrelextrema
from datetime import datetime, timedelta  # [新增]
# [新增] 直接從 finlab.plot 引用 create_treemap_data
from finlab.plot import create_treemap_data
from matplotlib.gridspec import GridSpec  # <--- 就是這一行
from dateutil.relativedelta import relativedelta # [ETF整合] <--- 新增此行

import finlab
from finlab import data
# [新增]
import squarify
import matplotlib.cm as cm
import matplotlib.colors as mcolors
# ===================================================================
# --- Part 0: 因子計算輔助函式 ---
# ===================================================================
# [新增] 波動率計算函式
# finlab程式碼格式

# finlab程式碼格式
# doc: ===================================================================
# doc: --- [新增因子] 實體紅棒強度 ---
# doc: ===================================================================
def calculate_bullish_candle_strength(timeperiod: int = 5):
    """
    doc:
    計算「實體紅棒強度」因子。
    - 定義：(收盤價 - 開盤價) / (最高價 - 最低價)
    - 此因子衡量了上漲K棒的收盤價，在當日總振幅中的相對強弱位置。
    - 數值接近 1 代表收在最高點附近，為強勢紅棒。
    - 數值接近 0 代表十字線或收盤價接近開盤價。
    - 只計算紅棒（收盤價 > 開盤價），黑棒強度計為 0。
    - 最終回傳 N 日的平均強度。
    """
    # 取得所需價量資料
    open_price = data.get("price:開盤價")
    high_price = data.get("price:最高價")
    low_price = data.get("price:最低價")
    close_price = data.get("price:收盤價")

    # 計算實體紅棒長度與當日振幅
    candle_body = close_price - open_price
    candle_range = high_price - low_price

    # 避免分母為零的錯誤
    # 將振幅為 0 的地方設為 NaN，後續 .where 處理時會被忽略
    candle_range = candle_range.replace(0, np.nan)

    # 計算強度比例，只考慮紅棒 (candle_body > 0)
    # 若為黑棒或振幅為0，強度設為 0
    strength_ratio = (candle_body / candle_range).where(candle_body > 0, 0)
    
    # 計算 N 日的移動平均，作為一個更平滑的指標
    avg_strength = strength_ratio.average(timeperiod)
    
    return avg_strength



def etf_get_data_for_month(period_date):
    """
    doc: [ETF整合] 根據指定月份，自動選擇並讀取最合適的ETF資料檔案。
    """
    if period_date.month in [3, 6, 9, 12]:
        file_path = 'sitca_fund_holdings_over_1_percent_quarterly.csv'
    else:
        file_path = 'sitca_fund_holdings.csv'
    try:
        # --- [核心修改] 新增 encoding='cp950' ---
        df = pd.read_csv(file_path, thousands=',', usecols=[
                         '日期', '標的代號', '標的名稱', '金額', '基金名稱'],
                         encoding='utf-8-sig')
        # --- [修改結束] ---

        df['日期'] = pd.to_datetime(df['日期'])
        df.dropna(subset=['標的代號', '基金名稱'], inplace=True)
        df['標的代號'] = df['標的代號'].astype(str).replace(r'\\.0$', '', regex=True)
        df['股票'] = df['標的名稱'].str.strip() + " (" + df['標的代號'].str.strip() + ")"
        month_df = df[df['日期'].dt.to_period(
            'M') == period_date.to_period('M')].copy()
        return month_df
    except FileNotFoundError:
        logging.error(
            f"[ETF分析] 找不到檔案 {file_path}，無法為 {period_date.strftime('%Y-%m')} 產出資料。")
        return pd.DataFrame()


def etf_create_comparison_df(metric_curr, metric_prev, name_curr, name_prev):
    """
    doc: [ETF整合] 建立一個兩期比較的 DataFrame，包含排名與排名變化。
    """
    comp_df = pd.DataFrame({
        name_curr: metric_curr,
        name_prev: metric_prev
    }).fillna(0)
    comp_df['本期排名'] = comp_df[name_curr].rank(method='min', ascending=False).astype(int)
    comp_df['前一期排名'] = comp_df[name_prev].rank(method='min', ascending=False).astype(int)

    def get_rank_change_description(row):
        if row[name_prev] == 0 and row[name_curr] > 0: return '新進榜'
        if row[name_prev] > 0 and row[name_curr] == 0: return '退出榜'
        if row[name_prev] == 0 and row[name_curr] == 0: return '-'
        rank_diff = int(row['前一期排名'] - row['本期排名'])
        if rank_diff > 5: return f"↑{rank_diff} (大進步)"
        if rank_diff > 0: return f"↑{rank_diff} (進步)"
        if rank_diff < -5: return f"↓{abs(rank_diff)} (大退步)"
        if rank_diff < 0: return f"↓{abs(rank_diff)} (退步)"
        return "- (持平)"

    comp_df['排名變化'] = comp_df.apply(get_rank_change_description, axis=1)
    comp_df.reset_index(inplace=True)
    
    # [修正] 在 .extract() 後面加上 [0] 來選取正規表示式抓取到的第一個群組，將其從 DataFrame 轉換為 Series
    comp_df['stock_id'] = comp_df['股票'].str.extract(r'\((\d+)\)')[0].fillna('')
    
    return comp_df.sort_values('本期排名')



# finlab程式碼格式
# 請找到並替換掉原有的 analyze_etf_holdings 函式
def analyze_etf_holdings():
    """
    doc: [v23.1 - Top 30 修改版] 執行 ETF 持股分析。
    - [修改] 將所有 head(20) 修改為 head(30)，以分析前30名焦點股。
    - [修改] 同時處理「季度」與「月度」數據。
    - [新增] 計算並回傳「月度」的共識度與重倉度，用於產製月度四象限分析圖。
    - [新增] 回傳月度重倉股Top30的股票代號列表，供附錄自動生成詳細報告。
    - [新增] 回傳最新的季度與月度日期，供圖表標題使用。
    """
    logging.info("[ETF分析] 開始掃描 ETF/基金持股數據...")
    try:
        primary_file = 'sitca_fund_holdings.csv'
        if not os.path.exists(primary_file):
            logging.warning(f"[ETF分析] 找不到主要資料檔 '{primary_file}'，跳過此分析。")
            return {}, None, None, None, None, None, None, None, None

        date_df = pd.read_csv(primary_file, usecols=['日期'], parse_dates=['日期'], encoding='utf-8-sig')
        latest_date = date_df['日期'].max()

        # --- 決定比較區間 (月度與季度) ---
        m_curr = latest_date
        m_prev = m_curr - relativedelta(months=1)
        
        latest_year, latest_month = latest_date.year, latest_date.month
        if latest_month <= 3: q_curr_month, q_curr_year = 12, latest_year - 1
        elif latest_month <= 6: q_curr_month, q_curr_year = 3, latest_year
        elif latest_month <= 9: q_curr_month, q_curr_year = 6, latest_year
        else: q_curr_month, q_curr_year = 9, latest_year
        q_curr = pd.to_datetime(f'{q_curr_year}-{q_curr_month:02d}-01')
        q_prev = q_curr - relativedelta(months=3)

        logging.info(f"[ETF分析] 最新季度: {q_curr.strftime('%Y-%m')}, 前一季: {q_prev.strftime('%Y-%m')}")
        logging.info(f"[ETF分析] 最新月份: {m_curr.strftime('%Y-%m')}, 前一月份: {m_prev.strftime('%Y-%m')}")

        # --- 讀取各期數據 ---
        df_q_curr = etf_get_data_for_month(q_curr)
        df_q_prev = etf_get_data_for_month(q_prev)
        df_m_curr = etf_get_data_for_month(m_curr)
        df_m_prev = etf_get_data_for_month(m_prev)

        if df_q_curr.empty or df_m_curr.empty:
            logging.error("[ETF分析] 無法讀取最新一季或最新一月的數據，分析中止。")
            return {}, None, None, None, None, None, None, None, None

        # --- 計算各項指標 ---
        # 季度共識度 (維持不變)
        fc_q_curr = df_q_curr.groupby('股票')['基金名稱'].nunique()
        fc_q_prev = df_q_prev.groupby('股票')['基金名稱'].nunique() if not df_q_prev.empty else pd.Series(dtype=int)
        comp_q_consensus = etf_create_comparison_df(fc_q_curr, fc_q_prev, '本期持有基金數', '前期持有基金數')

        # 季度重倉度 (維持不變)
        ti_q_curr = df_q_curr.groupby('股票')['金額'].sum() / 1e8
        ti_q_prev = df_q_prev.groupby('股票')['金額'].sum() / 1e8 if not df_q_prev.empty else pd.Series(dtype=float)
        comp_q_heavyweight = etf_create_comparison_df(ti_q_curr, ti_q_prev, '本期總持股市值(億)', '前期總持股市值(億)')

        # 月度重倉度 (維持不變)
        ti_m_curr = df_m_curr.groupby('股票')['金額'].sum() / 1e8
        ti_m_prev = df_m_prev.groupby('股票')['金額'].sum() / 1e8 if not df_m_prev.empty else pd.Series(dtype=float)
        comp_m_heavyweight = etf_create_comparison_df(ti_m_curr, ti_m_prev, '本期總持股市值(億)', '前期總持股市值(億)')

        # --- [核心修改] 新增計算「月度共識度」 ---
        fc_m_curr = df_m_curr.groupby('股票')['基金名稱'].nunique()
        fc_m_prev = df_m_prev.groupby('股票')['基金名稱'].nunique() if not df_m_prev.empty else pd.Series(dtype=int)
        comp_m_consensus = etf_create_comparison_df(fc_m_curr, fc_m_prev, '本期持有基金數', '前期持有基金數')
        
        # --- 準備四象限圖數據 ---
        # 季度 (原變數名稱加上 _q 以區分)
        quadrant_data_q = pd.merge(
            comp_q_consensus[['股票', '本期持有基金數']],
            comp_q_heavyweight[['股票', '本期總持股市值(億)']],
            on='股票'
        )

        # --- [核心修改] 新增準備「月度」四象限圖數據 ---
        quadrant_data_m = pd.merge(
            comp_m_consensus[['股票', '本期持有基金數']],
            comp_m_heavyweight[['股票', '本期總持股市值(億)']],
            on='股票'
        )
        
        # --- [修改] 將 head(20) 改為 head(30) ---
        top_stocks_set = set(comp_q_consensus.head(30)['stock_id']) | set(comp_q_heavyweight.head(30)['stock_id'])
        top_monthly_heavyweights_ids = comp_m_heavyweight.head(30)['stock_id'].tolist()
        top_stocks_set.update(top_monthly_heavyweights_ids)
        top_stocks_set.discard('')

        # --- [修改] 更新回傳值，擷取前30名 ---
        return (top_stocks_set, comp_q_consensus.head(30), comp_q_heavyweight.head(30), 
                comp_m_heavyweight.head(30), quadrant_data_q, 
                quadrant_data_m, 
                top_monthly_heavyweights_ids,
                q_curr, m_curr)

    except Exception as e:
        logging.error(f"[ETF分析] 執行時發生錯誤: {e}", exc_info=True)
        return {}, None, None, None, None, None, None, None, None






def compute_candle_volatility(timeperiod=20):
    """
    計算K線實體與影線的總路徑長度，作為波動性的代理指標。
    這個指標衡量了價格在一天內的總「移動距離」，而不僅僅是收盤價的變化。
    - bullish_volatility: 上漲 K 線的路徑 (昨收 → 開 → 低 → 高 → 收)
    - bearish_volatility: 下跌 K 線的路徑 (昨收 → 開 → 高 → 低 → 收)
    最後將此波動性對收盤價進行標準化，得到一個百分比形式的指標。
    """
    # 確保 FinlabDataFrame 被正確引入
    from finlab.dataframe import FinlabDataFrame

    close = data.get("price:收盤價")
    high = data.get("price:最高價")
    low = data.get("price:最低價")
    open_ = data.get("price:開盤價")

    bullish_candle = close >= open_
    # 修正：K線實體路徑計算應基於當天的 OHLC
    bullish_volatility = abs(open_ - low) + \
        abs(high - open_) + abs(close - high)
    bearish_volatility = abs(open_ - high) + \
        abs(low - open_) + abs(close - low)

    candle_volatility = FinlabDataFrame(
        np.nan, index=close.index, columns=close.columns)
    candle_volatility[bullish_candle] = bullish_volatility
    candle_volatility[~bullish_candle] = bearish_volatility

    # 計算最近N日的平均波動度，並對收盤價進行標準化
    volatility = candle_volatility.average(
        timeperiod) / close.average(timeperiod) * 100
    return volatility


def volume_ranking_factor(amt):
    """
    交易量排名動量因子。

    根據成交金額的時間序列數據，計算一個動量因子。
    這個因子衡量了成交額排名的變化速度，並對變化量進行了對數化處理以平滑極端值。
    正值通常表示成交額排名正在快速提升，代表市場關注度增加。

    Args:
        amt (pd.DataFrame): 成交金額的時間序列 DataFrame，index 為日期，columns 為股票代碼。

    Returns:
        pd.DataFrame: 計算後的動量因子 DataFrame。
    """
    R0 = amt.rank(axis=1, ascending=False)
    R1 = amt.shift(1).rank(axis=1, ascending=False)
    raw_rii = (R1 - R0)
    # 使用對數分母來平滑，避免排名低（數字大）的股票產生過度影響
    log_denominator = np.log(R1 * R0 + 1)
    log_rii = raw_rii.divide(log_denominator, fill_value=0)
    # 此處僅考慮排名上升（為正值）的情況，並與原始排名變動做差異化處理
    positive_log_rii = log_rii.where(log_rii > 0)
    diff = raw_rii.fillna(0) - positive_log_rii.fillna(0)
    return diff




def preliminary_industry_analysis():
    """
    [doc]
    [v9.5.0 因子整合] 新增「實體紅棒強度」作為輔助篩選條件。
    在主流程開始前，先進行一次全市場的產業掃描，找出符合條件的強勢產業與個股。
    [v9.4.0 修改] 新增均線趨勢條件 (MA60斜率為正、MA5上升)。
    [v9.3.0 修改] 新增基本條件：當日成交金額 > 1,500 萬。
    [v9.2.0 修改] 新增波動率過濾條件 (volatility <= 10)。
    """
    logging.info("[初步分析] 開始進行全市場產業掃描...")
    try:
        # [新增] 從 CONFIG 讀取因子參數 (此函式在 main block 中執行，所以可以直接讀取 CONFIG 變數)
        factor_params = CONFIG.get('FACTOR_PARAMS', {})
        strength_threshold = factor_params.get('BULLISH_STRENGTH_THRESHOLD', 0.5)
        strength_period = factor_params.get('BULLISH_STRENGTH_PERIOD', 5)

        company = data.get('company_basic_info')
        close = data.get('price:收盤價')
        amt = data.get('price:成交金額')

        if len(close) < 61 or len(amt) < 6:
            logging.error(f"FinLab 數據長度不足(需至少61天收盤價, 6天成交額)，無法進行初步產業分析。")
            # [修改] 修正回傳值的數量以符合函式簽名
            return None

        company = company[["stock_id", "公司簡稱", "公司名稱", "產業類別"]]
        company['stock_id'] = company['stock_id'].astype(str)

        valid_stocks = list(set(company['stock_id']) & set(close.columns) & set(amt.columns))
        if not valid_stocks:
            logging.error("無有效的共同股票清單，無法進行初步分析。")
            # [修改] 修正回傳值的數量以符合函式簽名
            return None

        close = close[valid_stocks]
        amt = amt[valid_stocks]
        company = company[company['stock_id'].isin(valid_stocks)]

        # --- 組合分析用 DataFrame ---
        analysis_df = company.copy()
        price_change_5d = (close.iloc[-1] - close.iloc[-6]) / close.iloc[-6] * 100
        analysis_df['漲跌幅_5日(%)'] = analysis_df['stock_id'].map(price_change_5d)
        price_change_10d = (close.iloc[-1] - close.iloc[-11]) / close.iloc[-11] * 100
        analysis_df['漲跌幅_10日(%)'] = analysis_df['stock_id'].map(price_change_10d)
        price_change_20d = (close.iloc[-1] - close.iloc[-21]) / close.iloc[-21] * 100
        analysis_df['漲跌幅_20日(%)'] = analysis_df['stock_id'].map(price_change_20d)
        analysis_df = analysis_df.dropna(
            subset=['產業類別', '漲跌幅_5日(%)', '漲跌幅_10日(%)', '漲跌幅_20日(%)'])

        # --- 第一道過濾：基本流動性 ---
        latest_amt = amt.iloc[-1]
        liquid_stocks = set(latest_amt[latest_amt > 15000000].index)
        original_count = len(analysis_df)
        analysis_df = analysis_df[analysis_df['stock_id'].isin(liquid_stocks)]
        logging.info(
            f"[初步分析] 經過成交額 > 1500萬 過濾後，分析樣本從 {original_count} 檔縮減至 {len(analysis_df)} 檔。")

        # --- 第二道過濾：均線趨勢 ---
        ma5 = close.average(5)
        ma20 = close.average(20)
        ma5_rising = ma5.iloc[-1] > ma5.iloc[-2]
        ma20_rising = ma20.iloc[-1] > ma20.iloc[-2]
        trend_ok_stocks = set(ma5_rising[ma5_rising & ma20_rising].index)

        original_count = len(analysis_df)
        analysis_df = analysis_df[analysis_df['stock_id'].isin(trend_ok_stocks)]
        logging.info(
            f"[初步分析] 經過均線趨勢過濾後，分析樣本從 {original_count} 檔縮減至 {len(analysis_df)} 檔。")

        if analysis_df.empty:
            logging.warning("[初步分析] 無任何股票滿足所有基本條件。")
            # [修改] 修正回傳值的數量以符合函式簽名
            return None

        # --- 後續分析 (基於已過濾的股票) ---
        # ... (後續的產業篩選、選股、波動率過濾邏輯不變) ...
        perf_cols = ['漲跌幅_5日(%)', '漲跌幅_10日(%)', '漲跌幅_20日(%)']
        industry_performance = analysis_df.groupby('產業類別')[perf_cols].mean()
        cond1 = industry_performance['漲跌幅_10日(%)'] > 10
        cond2 = industry_performance['漲跌幅_20日(%)'] < 30
        cond3 = industry_performance['漲跌幅_5日(%)'] > 0
        qualified_strong_industries = industry_performance[cond1 & cond2 & cond3]
        sort_key = '漲跌幅_20日(%)'
        strong_industries = qualified_strong_industries.sort_values(
            by=sort_key, ascending=False)
        if strong_industries.empty:
            logging.warning("[初步分析] 未找到符合所有條件的強勢產業。將採用備用方案：選擇市場相對最強的5個產業。")
            strong_industries = industry_performance.sort_values(
                by=sort_key, ascending=False)
        top_5_industries = strong_industries.head(5).index.tolist()
        logging.info(f"[初步分析] Top 5 強勢產業: {top_5_industries}")

        top_stocks_list = []
        stocks_in_top_industries = analysis_df[analysis_df['產業類別'].isin(
            top_5_industries)].copy()
        positive_stocks = stocks_in_top_industries[
            stocks_in_top_industries['漲跌幅_5日(%)'] > 0]
        grouped = positive_stocks.sort_values(
            '漲跌幅_5日(%)', ascending=False).groupby('產業類別', observed=True)
        for industry_name, group in grouped:
            top_stocks_list.extend(group.head(3)['stock_id'].tolist())
        strong_trend_stock_list = sorted(list(set(top_stocks_list)))

        logging.info(
            f"[初步分析] 從Top5強勢產業中找到 {len(strong_trend_stock_list)} 檔初步領漲股，開始進行波動率過濾...")
        volatility = compute_candle_volatility()
        latest_vol = volatility.iloc[-1]
        filtered_stocks = [
            stock_id for stock_id in strong_trend_stock_list
            if latest_vol.get(stock_id, float('inf')) <= 10
        ]
        logging.info(f"[初步分析] 經過波動率過濾後，剩餘 {len(filtered_stocks)} 檔低波動強勢股。")

        # --- [新增] 第四道過濾：使用「實體紅棒強度」進行二次過濾 ---
        logging.info(f"[初步分析] 開始進行實體紅棒強度 (>{strength_threshold}) 過濾...")
        bullish_strength = calculate_bullish_candle_strength(timeperiod=strength_period)
        latest_strength = bullish_strength.iloc[-1]
        
        final_stocks = [
            stock_id for stock_id in filtered_stocks
            if latest_strength.get(stock_id, 0) > strength_threshold
        ]
        logging.info(f"[初步分析] 經過實體紅棒強度過濾後，最終剩餘 {len(final_stocks)} 檔股票。")
        # --- [新增結束] ---
        
        # [修改] 回傳最終篩選結果，並移除不再需要的 full_industry_df
        return final_stocks

    except Exception as e:
        logging.error(f"[初步分析] 執行時發生錯誤: {e}", exc_info=True)
        # [修改] 修正回傳值的數量以符合函式簽名
        return None



# finlab程式碼格式
def find_rank_jump_stocks():
    """
    [doc]
    [v9.5.0 因子整合] 新增「實體紅棒強度」作為輔助篩選條件。
    掃描全市場，找出成交金額排名發生劇烈變化的股票。
    [v9.4.0 修改] 新增均線趨勢條件 (MA60斜率為正、MA5上升)。
    [v9.3.0 修改] 新增基本條件：當日成交金額 > 1,500 萬。
    [v9.2.0 修改] 新增波動率過濾條件 (volatility <= 10)。
    """
    logging.info("[排名躍升分析] 開始掃描全市場成交金額排名變化...")
    try:
        # [新增] 從 CONFIG 讀取因子參數
        factor_params = CONFIG.get('FACTOR_PARAMS', {})
        strength_threshold = factor_params.get('BULLISH_STRENGTH_THRESHOLD', 0.5)
        strength_period = factor_params.get('BULLISH_STRENGTH_PERIOD', 5)

        amt = data.get('price:成交金額')
        close = data.get('price:收盤價')
        company = data.get('company_basic_info')

        if len(amt) < 2 or len(close) < 61:
            logging.error("[排名躍升分析] 數據長度不足 (需至少2天成交額, 61天收盤價)，無法進行分析。")
            return None

        # --- 第一道過濾：基本流動性 ---
        latest_amt = amt.iloc[-1]
        liquid_stocks_index = set(latest_amt[latest_amt > 15000000].index)

        # --- 第二道過濾：均線趨勢 ---
        ma5 = close.average(5)
        ma20 = close.average(20)
        ma5_rising = ma5.iloc[-1] > ma5.iloc[-2]
        ma20_rising = ma20.iloc[-1] > ma20.iloc[-2]
        trend_ok_stocks = set(ma5_rising[ma5_rising & ma20_rising].index)

        final_pool = liquid_stocks_index.intersection(trend_ok_stocks)

        amt_filtered = amt[list(final_pool)]
        logging.info(
            f"[排名躍升分析] 經過成交額與均線趨勢過濾後，躍升股掃描樣本為 {len(amt_filtered.columns)} 檔。")

        if amt_filtered.empty:
            logging.warning("[排名躍升分析] 無任何股票滿足所有基本條件。")
            return None

        # --- 計算躍升分數 ---
        amt_ranks = amt_filtered.rank(axis=1, ascending=False)
        R0 = amt_ranks.iloc[-1]
        R1 = amt_ranks.iloc[-2]
        rank_df = pd.DataFrame({'R0': R0, 'R1': R1}).dropna()
        rank_df = rank_df[rank_df['R1'] > rank_df['R0']]

        if rank_df.empty:
            logging.warning("[排名躍升分析] 未找到符合條件的排名躍升股。")
            return None

        rank_change = rank_df['R1'] - rank_df['R0']
        log_denominator = np.log(rank_df['R1'] * rank_df['R0'] + 1)
        rank_df['躍升分數'] = rank_change / log_denominator

        # --- 第三道過濾：波動率 ---
        top_5_jumpers = rank_df.nlargest(5, '躍升分數')
        logging.info(
            f"[排名躍升分析] 找到 Top {len(top_5_jumpers)} 排名躍升股，開始進行波動率過濾...")
        volatility = compute_candle_volatility()
        latest_vol = volatility.iloc[-1]
        top_5_jumpers['volatility'] = top_5_jumpers.index.map(
            lambda x: latest_vol.get(x, float('inf')))
        filtered_jumpers = top_5_jumpers[top_5_jumpers['volatility'] <= 10].copy()
        logging.info(f"[排名躍升分析] 經過波動率過濾後，剩餘 {len(filtered_jumpers)} 檔低波動躍升股。")

        if filtered_jumpers.empty:
            logging.warning("[排名躍升分析] 經過波動率過濾後，無剩餘股票。")
            return None

        # --- [新增] 第四道過濾：使用「實體紅棒強度」進行二次過濾 ---
        logging.info(f"[排名躍升分析] 開始進行實體紅棒強度 (>{strength_threshold}) 過濾...")
        bullish_strength = calculate_bullish_candle_strength(timeperiod=strength_period)
        latest_strength = bullish_strength.iloc[-1]
        
        filtered_jumpers['bullish_strength'] = filtered_jumpers.index.map(latest_strength).fillna(0)
        final_jumpers = filtered_jumpers[filtered_jumpers['bullish_strength'] > strength_threshold].copy()
        logging.info(f"[排名躍升分析] 經過實體紅棒強度過濾後，最終剩餘 {len(final_jumpers)} 檔股票。")
        # --- [新增結束] ---

        if final_jumpers.empty:
            logging.warning("[排名躍升分析] 經過實體紅棒強度過濾後，無剩餘股票。")
            return None

        # [修改] 後續的整理與回傳，使用 final_jumpers
        final_jumpers.index.name = 'stock_id'
        final_jumpers = final_jumpers.reset_index()
        company_map = company.set_index('stock_id')['公司簡稱']
        final_jumpers['公司簡稱'] = final_jumpers['stock_id'].map(
            company_map).fillna('未知')
        final_jumpers['排名變化說明'] = final_jumpers.apply(
            lambda row: f"排名由 {int(row['R1'])} → {int(row['R0'])}", axis=1
        )
        return final_jumpers[['stock_id', '公司簡稱', '躍升分數', '排名變化說明']]

    except Exception as e:
        logging.error(f"[排名躍升分析] 執行時發生錯誤: {e}", exc_info=True)
        return None




# finlab程式碼格式
# [ULTRA THINK 因子整合版] - 完整修改示範
# 在全域範圍修改此函式
def analyze_market_breadth():
    """
    [doc]
    [v11.1.0 波動率整合] 新增「波動率 <= 10」作為創高股的輔助篩選條件。
    [v11.0.0 因子整合] 新增「實體紅棒強度」作為創高股的輔助篩選條件。
    [v10.8.0 修改] 新增計算「創20日新高」的個股清單，並回傳。
    [v9.8.0 修改] 數據擴充：新增「成交金額」與「是否放量」。
    """
    logging.info("[市場寬度分析] 開始計算創新高/低家數...")
    try:
        # [新增] 從 CONFIG 讀取因子參數
        factor_params = CONFIG.get('FACTOR_PARAMS', {})
        strength_threshold = factor_params.get('BULLISH_STRENGTH_THRESHOLD', 0.5)
        strength_period = factor_params.get('BULLISH_STRENGTH_PERIOD', 5)

        close = data.get('price:收盤價')
        amt = data.get('price:成交金額')
        company = data.get('company_basic_info')[["stock_id", "公司簡稱", "產業類別"]]
        company['stock_id'] = company['stock_id'].astype(str)

        if len(close) < 200 or len(amt) < 6:
            logging.error("[市場寬度分析] 數據長度不足 (需至少200天收盤價, 6天成交額)，無法進行分析。")
            return None

        # [新增] 提前計算好因子，以傳遞給輔助函式使用，避免重複計算
        bullish_strength = calculate_bullish_candle_strength(timeperiod=strength_period)
        latest_strength = bullish_strength.iloc[-1]
        
        # [您的新需求] 新增計算波動率指標
        logging.info("[市場寬度分析] 正在計算波動率指標...")
        volatility = compute_candle_volatility()
        latest_vol = volatility.iloc[-1]


        latest_amt = amt.iloc[-1]
        liquid_stocks = latest_amt[latest_amt > 15000000].index
        close_filtered = close[liquid_stocks]
        
        logging.info(f"[市場寬度分析] 流動性篩選後，共 {len(liquid_stocks)} 檔股票納入計算。")

        new_high_200_bool = (close_filtered == close_filtered.rolling(200, min_periods=200).max())
        new_high_20_bool = (close_filtered == close_filtered.rolling(20, min_periods=20).max())
        new_low_20_bool = (close_filtered == close_filtered.rolling(20, min_periods=20).min())

        high_counts_200d = new_high_200_bool.tail(200).sum(axis=1)
        high_counts_20d = new_high_20_bool.tail(200).sum(axis=1)
        low_counts = new_low_20_bool.tail(200).sum(axis=1)

        breadth_df = pd.DataFrame({
            'new_high_200': high_counts_200d,
            'new_high_20': high_counts_20d,
            'new_low_20': low_counts,
        })
        
        breadth_df['diff'] = breadth_df['new_high_20'] - breadth_df['new_low_20']

        # [修改] create_high_list_df 的函式簽名，讓它可以接收波動率因子
        def create_high_list_df(high_bool_series, latest_strength_series, latest_vol_series, threshold):
            latest_high_stocks = high_bool_series[high_bool_series].index.tolist()
            if not latest_high_stocks:
                return pd.DataFrame()

            high_info_df = company[company['stock_id'].isin(latest_high_stocks)].copy()
            
            price_change_1d = (close.iloc[-1] - close.iloc[-2]) / close.iloc[-2] * 100
            price_change_5d = (close.iloc[-1] - close.iloc[-6]) / close.iloc[-6] * 100
            amt_ma5 = amt.rolling(5).mean().iloc[-1]
            is_amt_up = latest_amt > amt_ma5

            high_info_df['當日漲跌幅(%)'] = high_info_df['stock_id'].map(price_change_1d)
            high_info_df['近五日漲跌幅(%)'] = high_info_df['stock_id'].map(price_change_5d)
            high_info_df['成交金額'] = high_info_df['stock_id'].map(latest_amt)
            high_info_df['是否放量'] = high_info_df['stock_id'].map(is_amt_up)
            high_info_df.dropna(inplace=True)
            
            # [原有過濾] 在此處進行「實體紅棒強度」過濾
            if not high_info_df.empty:
                strong_stocks_mask = high_info_df['stock_id'].map(latest_strength_series).fillna(0) > threshold
                original_count = len(high_info_df)
                high_info_df = high_info_df[strong_stocks_mask]
                logging.info(f"    ↳ (創高股) 紅棒強度過濾後: {original_count} -> {len(high_info_df)} 檔")

            # [您的新需求] 在此處進行「波動率」過濾
            if not high_info_df.empty:
                low_vol_mask = high_info_df['stock_id'].map(latest_vol_series).fillna(float('inf')) <= 10
                original_count = len(high_info_df)
                high_info_df = high_info_df[low_vol_mask]
                logging.info(f"    ↳ (創高股) 波動率(<10)過濾後: {original_count} -> {len(high_info_df)} 檔")
            
            return high_info_df

        # [修改] 呼叫輔助函式時，傳入新的波動率因子
        new_high_200_info_df = create_high_list_df(new_high_200_bool.iloc[-1], latest_strength, latest_vol, strength_threshold)
        new_high_20_info_df = create_high_list_df(new_high_20_bool.iloc[-1], latest_strength, latest_vol, strength_threshold)

        # [修改] 更新日誌訊息，使其能反映最終篩選後的結果
        logging.info(
            f"[市場寬度分析] 分析完成，今日創200日新高(且符合多重過濾)共 {len(new_high_200_info_df)} 家，創20日新高(且符合多重過濾)共 {len(new_high_20_info_df)} 家。")

        return {
            "breadth_timeseries": breadth_df,
            "latest_new_high_list": new_high_200_info_df,
            "latest_new_high_20_list": new_high_20_info_df
        }

    except Exception as e:
        logging.error(f"[市場寬度分析] 執行時發生錯誤: {e}", exc_info=True)
        return None


# [新增] 產生資券情緒結論的獨立輔助函式
# ===================================================================
# --- 函式 1: 修改後的「融資情緒」結論函式 ---
# ===================================================================
def get_margin_sentiment_conclusion(sentiment_df):
    """
    [doc]
    [v12.8.0 修正] 新增回傳一個數值分數，供熱力圖使用。
    分數定義：多方為正，空方為負。
    """
    # [核心修改] 定義文字與分數的映射
    sentiment_map = {
        "籌碼趨向多方": 2,
        "軋空行情醞釀中": 1,
        "多空交戰激烈": 0,
        "市場人氣退潮": -1,
        "籌碼趨向空方": -2,
        "觀察中": 0,
        "數據不足": np.nan  # 使用 NaN 表示無效分數
    }
    
    if sentiment_df.empty or len(sentiment_df) < 6:
        return "數據不足", np.nan # 回傳文字和分數

    cleaned_df = sentiment_df.dropna(subset=['margin_balance_chg', 'short_balance_chg', 'close_pct_chg_5d'])
    if cleaned_df.empty:
        return "數據不足", np.nan

    latest = cleaned_df.iloc[-1]
    margin_chg, short_chg, price_chg = latest['margin_balance_chg'], latest['short_balance_chg'], latest['close_pct_chg_5d']

    conclusion_text = "觀察中"
    if margin_chg > 0 and short_chg > 0: conclusion_text = "多空交戰激烈"
    elif margin_chg > 0 and short_chg < 0: conclusion_text = "籌碼趨向多方"
    elif margin_chg < 0 and short_chg > 0:
        conclusion_text = "軋空行情醞釀中" if price_chg >= 0 else "籌碼趨向空方"
    elif margin_chg < 0 and short_chg < 0: conclusion_text = "市場人氣退潮"
    
    return conclusion_text, sentiment_map.get(conclusion_text, 0)


def get_shareholder_ratio_conclusion(shareholder_df):
    """
    [doc]
    [v2.2 修正版]
    - 回傳值新增原始的 ratio 數值，供熱力圖使用。
    """
    if shareholder_df.empty or len(shareholder_df) < 2:
        return "N/A", "N/A", np.nan # 回傳3個值

    try:
        latest_data = shareholder_df.iloc[-1]
        latest_date = shareholder_df.index[-1]
        
        date_lag_warning = ""
        if pd.api.types.is_datetime64_any_dtype(latest_date) and (datetime.now() - latest_date).days > 7:
            date_lag_warning = "(上週)"

        latest_ratio = latest_data.get('ratio')
        
        diff_trend_val = shareholder_df['diff1'].diff(1).iloc[-1]

        ratio_text = f"{latest_ratio*100:.1f}%" if pd.notna(latest_ratio) else "N/A"
        
        if pd.notna(diff_trend_val):
            trend_text = "上升▲" if diff_trend_val > 0 else "下降▼" if diff_trend_val < 0 else "持平"
        else:
            trend_text = "N/A"
        
        final_trend_text = f"{trend_text}{date_lag_warning}"
        
        # [核心修改] 將原始數值也回傳出去
        return ratio_text, final_trend_text, latest_ratio

    except (IndexError, KeyError) as e:
        logging.warning(f"計算股東結構結論時發生錯誤: {e}")
        return "N/A", "N/A", np.nan

def get_broker_flow_conclusion(broker_df):
    """
    根據關鍵券商週流向 DataFrame，分析近4週的買賣超情況。
    [v2.1 修改] 回傳值改為 (結論文字, 原始數值) 的元組。
    """
    if broker_df.empty or len(broker_df) < 10:
        return "數據不足", 0 # 回傳文字和數值

    last_4_weeks = broker_df['total'].tail(4)
    net_flow = last_4_weeks.sum()
    
    first_2_weeks_flow = last_4_weeks.head(2).sum()
    
    main_conclusion = ""
    if net_flow > 0:
        status = "轉為買超" if first_2_weeks_flow <= 0 else "持續買超"
        main_conclusion = f"{status} ({net_flow:+.0f}張)"
    elif net_flow < 0:
        status = "轉為賣超" if first_2_weeks_flow >= 0 else "持續賣超"
        main_conclusion = f"{status} ({net_flow:+.0f}張)"
    else:
        main_conclusion = "無明顯進出"

    trend_comment = ""
    latest = broker_df.iloc[-1]
    previous = broker_df.iloc[-2]

    is_golden_cross = latest['ma5'] > latest['ma10'] and previous['ma5'] <= previous['ma10']
    is_death_cross = latest['ma5'] < latest['ma10'] and previous['ma5'] >= previous['ma10']
    
    if is_golden_cross:
        trend_comment = "趨勢黃金交叉"
    elif is_death_cross:
        trend_comment = "趨勢死亡交叉"
    else:
        if latest['ma5'] > previous['ma5']:
            trend_comment = "短期趨勢增強"
        elif latest['ma5'] < previous['ma5']:
            trend_comment = "短期趨勢減弱"

    final_conclusion = f"{main_conclusion}，{trend_comment}" if trend_comment else main_conclusion
    
    # [核心修改] 將最終結論文字和原始買賣超張數一起回傳
    return final_conclusion, net_flow


# [新增] 分析宏觀融資指標的函式
# [修改] 分析宏觀融資指標的函式 (修正數據對齊問題)
def analyze_macro_margin_indicators():
    """
    [doc]
    [v2.1 修正] 確保所有回傳的 DataFrame 索引一致，避免繪圖時的長度不匹配錯誤。
    """
    logging.info("[宏觀分析] 開始計算大盤融資指標...")
    try:
        end_date = datetime.now().date()
        start_date = end_date - timedelta(days=730)
        end = end_date.strftime("%Y-%m-%d")
        start = start_date.strftime("%Y-%m-%d")

        margin_balance = data.get('margin_transactions:融資今日餘額')
        margin_total = data.get('margin_balance:融資券總餘額')
        close = data.get('price:收盤價')
        benchmark = data.get('benchmark_return:發行量加權股價報酬指數').squeeze()

        common_index = margin_balance.index.intersection(margin_total.index)
        margin_total = margin_total.loc[common_index]

        # 使用 .shift() 會產生 NaN，這是長度不一致的源頭
        margin_total['上市融資買賣超'] = (
            margin_total['上市融資交易金額'] - margin_total['上市融資交易金額'].shift()).fillna(0) / 100_000_000
        margin_total['上櫃融資買賣超'] = (
            margin_total['上櫃融資交易金額'] - margin_total['上櫃融資交易金額'].shift()).fillna(0) / 100_000_000

        total_margin_loan = margin_total[['上市融資交易金額', '上櫃融資交易金額']].sum(axis=1)

        valid_stocks = margin_balance.columns.intersection(close.columns)
        margin_market_value = (
            margin_balance[valid_stocks] * close[valid_stocks] * 1000).sum(axis=1)

        maintenance_ratio = (margin_market_value / total_margin_loan)

        # [核心修正] 先篩選出主要的 DataFrame，然後用它的索引去對齊其他數據
        margin_change_df = margin_total.loc[(
            margin_total.index >= start) & (margin_total.index <= end)]
        final_index = margin_change_df.index

        maintenance_df = maintenance_ratio.reindex(
            final_index).to_frame('ratio')
        benchmark_series = benchmark.reindex(final_index)

        logging.info("[宏觀分析] 大盤融資指標數據計算完成。")
        return {
            "start_date": start,
            "end_date": end,
            "maintenance_df": maintenance_df,
            "margin_change_df": margin_change_df,
            "benchmark": benchmark_series
        }
    except Exception as e:
        logging.error(f"[宏觀分析] 計算大盤融資指標時發生錯誤: {e}", exc_info=True)
        return None


# [修改] 尋找散戶凹單股的函式 (修正版)
def find_margin_trap_stocks():
    """
    [doc]
    掃描全市場，找出股價下跌但融資大增的「散戶凹單」個股。
    [v2.0 修正] 新增數據延遲處理機制，確保所有數據在共同的日期基礎上進行計算。
    """
    logging.info("[宏觀分析] 開始掃描散戶凹單股...")
    try:
        # --- [核心修正區域] ---
        # 1. 先加載所有需要的時間序列數據
        margin_balance = data.get('margin_transactions:融資今日餘額')
        adj_close = data.get('etl:adj_close')

        # 2. 找到所有數據共同存在的日期索引
        common_index = margin_balance.index.intersection(adj_close.index)

        if len(common_index) < 6:
            logging.warning(f"[宏觀分析] 融資與股價的共同交易日少于6天，無法計算凹單股。")
            return pd.DataFrame(), "", ""

        # 3. 從共同日期中確定分析的起訖日
        end_date = common_index[-1]
        start_date = common_index[-6]  # 回推5個交易日
        end_str = end_date.strftime("%Y-%m-%d")
        start_str = start_date.strftime("%Y-%m-%d")
        logging.info(
            f"[宏觀分析] 偵測到數據共同最新日期為 {end_str}，分析區間為 {start_str} 至 {end_str}")
        # --- [修正結束] ---

        # [修改] 直接呼叫已引用的 create_treemap_data 函式
        df = create_treemap_data(start_str, end_str, 'return_ratio')
        df = df.set_index(['stock_id'])

        # 後續計算邏輯不變，但現在基於安全的 start_date 和 end_date
        df['margin_balance_change'] = (
            (margin_balance.loc[end_date] / margin_balance.loc[start_date]) - 1).clip(-0.6, 0.6) * 100
        return_ratio = round(
            ((adj_close.loc[end_date] / adj_close.loc[start_date]).dropna().replace(np.inf, 0) - 1) * 100, 2)
        df['return_ratio'] = return_ratio
        df['margin_balance'] = margin_balance.loc[end_date]

        df.dropna(inplace=True)

        with np.errstate(divide='ignore', invalid='ignore'):
            df['trap_ratio'] = df['margin_balance_change'] / df['return_ratio']

        trap_df = df[
            (df['return_ratio'] < -5) &
            (df['margin_balance_change'] > 2) &
            (df['trap_ratio'] < 0) &
            (df['margin_balance'] > 3000)
        ].copy()

        trap_df = trap_df.reset_index()
        logging.info(f"[宏觀分析] 找到 {len(trap_df)} 檔潛在的散戶凹單股。")

        company_info = data.get('company_basic_info')[['stock_id', '公司簡稱']]
        trap_df = pd.merge(trap_df, company_info, on='stock_id', how='left')
        trap_df['stock_id_name'] = trap_df['stock_id'] + \
            ' ' + trap_df['公司簡稱'].fillna('')

        return trap_df, start_str, end_str

    except Exception as e:
        logging.error(f"[宏觀分析] 尋找散戶凹單股時發生錯誤: {e}", exc_info=True)
        return pd.DataFrame(), "", ""


def calculate_it_momentum_signals(all_data):
    """
    [doc]
    根據預加載的數據，計算投信作多動能相關指標。

    Args:
        all_data (dict): 包含多個 DataFrame 的字典，源自 StockAnalyzer._preload_data()。
                         預期包含 'it_buy', 'close', 'volume'。

    Returns:
        pd.DataFrame: 一個包含 stock_id 及對應投信作多指標的 DataFrame。
                      欄位包含:
                      - stock_id: 股票代碼
                      - 投信作多原因: '排名高', '排名竄升', '排名高 & 竄升', 或 ''
                      - 投信作多強度: 投信買賣超佔成交金額比例的市場排名 (0.0 to 1.0)
    """
    logging.info("--- 正在計算投信作多動能指標 ---")
    try:
        it_trade = all_data.get('it_buy')
        close = all_data.get('close')
        # volume 在 _preload_data 中已除以1000，這裡要還原成股數
        volume = all_data.get('volume') * 1000

        # 確保所有需要的 DataFrame 都存在且不為空
        if any(df is None or df.empty for df in [it_trade, close, volume]):
            logging.warning("計算投信動能所需數據(投信買賣超、收盤價、成交股數)不完整，跳過計算。")
            return pd.DataFrame()

        # 計算 投信買賣金額 / 當日總成交金額
        # 使用 .div() 並設定 fill_value=0 來安全地處理分母為0的情況
        amount = close * volume
        it_trade_value = it_trade * close
        it_pct_of_amount = it_trade_value.div(amount).fillna(0)

        # 計算每日排名 (pct=True 會將結果標準化到 0-1 之間)
        it_rank_pct = it_pct_of_amount.rank(axis=1, pct=True)

        if len(it_rank_pct) < 6:
            logging.warning("投信動能指標計算所需天數不足(至少6天)，跳過計算。")
            return pd.DataFrame()

        # 取得最新一日的數據
        latest_rank = it_rank_pct.iloc[-1]

        # --- 判斷條件 ---
        # 條件1: 排名高於市場80%的股票
        is_high_rank = latest_rank > 0.8

        # 條件2: 排名大幅竄升 (今日排名 > 5日平均排名 * 1.2)
        rank_ma5 = it_rank_pct.rolling(5).mean().iloc[-1]
        is_soaring = latest_rank > (rank_ma5 * 1.2)

        # --- 組合最終結果 ---
        results = []
        for stock_id in it_rank_pct.columns:
            reason = ""
            high = is_high_rank.get(stock_id, False)
            soaring = is_soaring.get(stock_id, False)

            if high and soaring:
                reason = "(投信)排名高 & 竄升"
            elif high:
                reason = "(投信)排名高"
            elif soaring:
                reason = "(投信)排名竄升"

            # 只有在需要標記時才加入結果列表
            if reason:
                results.append({
                    'stock_id': stock_id,
                    '投信作多原因': reason,
                    '投信作多強度': latest_rank.get(stock_id, 0.0)
                })

        logging.info(f"計算完成，共找到 {len(results)} 檔符合投信作多訊號的股票。")
        return pd.DataFrame(results)

    except Exception as e:
        logging.error(f"計算投信作多動能指標時發生錯誤: {e}", exc_info=True)
        return pd.DataFrame()



# finlab程式碼格式
# doc: ===================================================================
# doc: --- [v2.0 動能版] 特殊訊號計算函式 (營收 & 成交額) ---
# doc: ===================================================================
def compute_special_signals():
    """
    doc:
    一次性計算全市場股票的兩個關鍵強勢訊號：
    1. [v2.0 動能版] 最近2個月的平均營收，是否創下過去12個月以來的新高。
    2. 最新一日的成交金額創下N日新高 (N為5, 20, 60, 120, 240中的最大值)。

    Returns:
        pd.DataFrame: 一個以 stock_id 為索引的 DataFrame，
                      包含 'is_rev_new_high' (bool) 和 'amt_new_high_period' (int) 兩個欄位。
    """
    logging.info("--- 正在計算「營收動能創高」與「成交額創高」特殊訊號... ---")
    try:
        monthly_revenue = data.get('monthly_revenue:當月營收')
        amt = data.get('price:成交金額')

        if monthly_revenue.empty or amt.empty or len(monthly_revenue) < 13:
            logging.warning("營收或成交額數據不足 (至少需13個月營收)，跳過特殊訊號計算。")
            return pd.DataFrame()

        # doc: --- [v2.0 核心修改] 營收訊號改為計算「2個月營收均線」是否創「12個月以來」的新高 ---
        # 1. 計算2個月營收移動平均 (MA2)\n",
        rev_ma2 = monthly_revenue.rolling(2).mean()
        
        # 2. 找出過去12個月中，MA2的最高點\n",
        rev_ma2_rolling_12m_max = rev_ma2.rolling(12, min_periods=12).max()

        # 3. 比較最新的MA2值是否大於等於過去12個月的最高點\n",
        is_rev_new_high = rev_ma2.iloc[-1] >= rev_ma2_rolling_12m_max.iloc[-1]
        is_rev_new_high.name = 'is_rev_new_high'

        # 訊號2: 成交金額創N日新高 (邏輯維持不變)
        latest_amt = amt.iloc[-1]
        amt_new_high_period = pd.Series(0, index=amt.columns, name='amt_new_high_period')
        periods = [5, 20, 60, 120, 240]

        for p in periods:
            if len(amt) >= p:
                period_max = amt.rolling(p, min_periods=p).max().iloc[-1]
                is_new_high = latest_amt >= period_max
                amt_new_high_period[is_new_high] = p

        special_signals_df = pd.concat([is_rev_new_high, amt_new_high_period], axis=1).dropna(subset=['is_rev_new_high'])
        logging.info("--- 特殊訊號計算完成 ---")
        return special_signals_df

    except Exception as e:
        logging.error(f"計算特殊訊號時發生錯誤: {e}", exc_info=True)
        return pd.DataFrame()


# finlab程式碼格式
# =============================================================================
# --- [Gemini 新增] 處置股與現金增資股 輔助查詢函式 ---
# =============================================================================

# 引入所需函式庫
from finlab import data
import pandas as pd
import numpy as np
from datetime import date, timedelta

def get_current_and_upcoming_disposals(disposal_info_df):
    """
    doc:
    從整理好的處置股資訊中，找出以今天為基準，
    「正在被處置」以及「未來即將被處置」的股票清單。

    Args:
        disposal_info_df (pd.DataFrame): 經過篩選和整理的處置股資訊。

    Returns:
        pd.DataFrame: 包含查詢結果的 DataFrame，並附有目前的狀態欄位。
    """
    # 取得今天的日期，並轉換成 pandas 可以比較的 datetime 格式
    today = pd.to_datetime(date.today())

    # 條件1: 篩選出處置結束時間在今天(含)之後的股票
    relevant_stocks = disposal_info_df[disposal_info_df['處置結束時間'] >= today].copy()

    # 條件2: 排除股票代號不等於4碼的標的(以普通股為主)
    relevant_stocks = relevant_stocks[relevant_stocks['stock_id'].str.len() == 4]

    # 根據今天的日期，為每支股票加上目前的狀態
    conditions = [
        (relevant_stocks['處置開始時間'] <= today),
        (relevant_stocks['處置開始時間'] > today)
    ]
    choices = ['正在處置中', '即將被處置']

    # 使用 numpy.select 根據條件快速賦值
    relevant_stocks['狀態'] = np.select(conditions, choices, default='未知')

    # 重新排列欄位並根據開始時間排序，讓輸出結果更方便閱讀
    relevant_stocks = relevant_stocks[[
        'stock_id', '處置開始時間', '處置結束時間', '狀態'
    ]].sort_values(by='處置開始時間').reset_index(drop=True)

    return relevant_stocks


def get_current_capital_increases_revised():
    """
    doc:
    (修正版) 查詢並回傳市場上「當前正在進行」或「即將進行」現金增資的股票清單。
    此版本使用 '除權交易日' 作為判斷基準。

    Returns:
        pd.DataFrame: 包含正在或即將辦理現金增資的公司詳細資訊。
    """
    # 取得今天的日期，以及一個月前的日期作為篩選起點
    today = pd.to_datetime(date.today())
    start_filter_date = today - timedelta(days=30)

    # 讀取股利相關公告
    df = data.get('dividend_announcement')

    # --- 資料清理與篩選 ---
    # 1. 篩選出明確為「現金增資」的事件
    df.dropna(subset=['現金增資總股數(股)'], inplace=True)
    df = df[df['現金增資總股數(股)'] > 0].copy()

    # 2. 處理關鍵日期欄位 '除權交易日'
    df['除權交易日'] = pd.to_datetime(df['除權交易日'], errors='coerce')
    df.dropna(subset=['除權交易日'], inplace=True)

    # 3. 篩選出最近一個月內及未來的所有事件
    current_increases = df[df['除權交易日'] >= start_filter_date].copy()

    # 4. 排除股票代號不等於4碼的標的
    current_increases = current_increases[current_increases['stock_id'].str.len() == 4]

    # --- 整理輸出結果 ---
    # 增加一個「狀態」欄位，方便判斷事件進程
    conditions = [
        (current_increases['除權交易日'] > today),
        (current_increases['除權交易日'] <= today)
    ]
    choices = ['即將除權', '近期已除權(可能正在繳款)']
    current_increases['狀態'] = np.select(conditions, choices, default='未知')

    # 選取我們關心的欄位
    result = current_increases[[
        'stock_id', '公司名稱', '公告日期', '除權交易日',
        '現金增資認購價(元/股)', '現金增資總股數(股)', '狀態'
    ]].sort_values(by='除權交易日', ascending=False).reset_index(drop=True)

    return result


# finlab程式碼格式

# 1. 將此函式完整新增到您放置輔助函式的儲存格中
def get_current_treasury_stocks_by_finlab():
    """
    doc:
    [v2.0 Gemini data.get 最終版]
    - 完全使用 finlab.data.get() 取得庫藏股資料，不再需要自行爬蟲。
    - 智慧篩選出「正在執行中」與「即將執行」的公司清單。
    """
    print("正在從 FinLab 數據庫撈取庫藏股資料...")
    try:
        # 使用 data.get() 一次性取得所有需要的欄位
        start_dates = data.get('treasury_stock:預定買回期間-起')
        end_dates = data.get('treasury_stock:預定買回期間-迄')
        min_prices = data.get('treasury_stock:買回價格區間-最低')
        max_prices = data.get('treasury_stock:買回價格區間-最高')
        volumes = data.get('treasury_stock:預定買回股數')
        purposes = data.get('treasury_stock:買回目的')

        # 取得公司基本資訊用於顯示名稱
        company_info = data.get('company_basic_info')[
            ['stock_id', '公司簡稱']].set_index('stock_id')

        print("資料撈取完畢，開始整理與篩選...")

        # 將寬格式的 DataFrame 轉換為長格式，方便處理
        df_start = start_dates.unstack().dropna().reset_index()
        df_start.columns = ['stock_id', 'date_declare', '預定買回期間-起']

        df_end = end_dates.unstack().dropna().reset_index()
        df_end.columns = ['stock_id', 'date_declare', '預定買回期間-迄']

        # 合併開始與結束日期
        df_merged = pd.merge(df_start, df_end, on=['stock_id', 'date_declare'])

        # --- 篩選出當前相關的庫藏股 ---
        today = pd.to_datetime(datetime.now().date())
        relevant_stocks = df_merged[df_merged['預定買回期間-迄'] >= today].copy()

        if relevant_stocks.empty:
            return pd.DataFrame()

        # --- 整理其他資訊 ---
        other_info = []
        for _, row in relevant_stocks.iterrows():
            stock_id = row['stock_id']
            declare_date = row['date_declare']
            info = {
                'stock_id': stock_id,
                'date_declare': declare_date,
                '買回價格區間-最低': min_prices.loc[declare_date, stock_id] if declare_date in min_prices.index and stock_id in min_prices.columns else np.nan,
                '買回價格區間-最高': max_prices.loc[declare_date, stock_id] if declare_date in max_prices.index and stock_id in max_prices.columns else np.nan,
                '預定買回股數': volumes.loc[declare_date, stock_id] if declare_date in volumes.index and stock_id in volumes.columns else np.nan,
                '買回目的': purposes.loc[declare_date, stock_id] if declare_date in purposes.index and stock_id in purposes.columns else np.nan,
            }
            other_info.append(info)

        df_other_info = pd.DataFrame(other_info)

        final_df = pd.merge(relevant_stocks, df_other_info,
                            on=['stock_id', 'date_declare'])
        final_df['公司簡稱'] = final_df['stock_id'].map(company_info['公司簡稱'])

        conditions = [
            (final_df['預定買回期間-起'] > today),
            (final_df['預定買回期間-起'] <= today)
        ]
        choices = ['即將執行', '正在執行中']
        final_df['狀態'] = np.select(conditions, choices, default='未知')

        purpose_map = {"1": '轉讓股份予員工', "2": '股權轉換', "3": '維護公司信用及股東權益'}
        final_df['買回目的'] = final_df['買回目的'].astype(
            str).map(purpose_map).fillna('其他')

        result = final_df[[
            'stock_id', '公司簡稱', '狀態', '預定買回期間-起', '預定買回期間-迄',
            '買回價格區間-最低', '買回價格區間-最高', '預定買回股數', '買回目的'
        ]].sort_values(by=['預定買回期間-起', 'stock_id']).reset_index(drop=True)

        result['預定買回期間-起'] = result['預定買回期間-起'].dt.date
        result['預定買回期間-迄'] = result['預定買回期間-迄'].dt.date

        # [修改] 將股數直接格式化為文字，方便後續顯示
        result['預定買回股數_text'] = result['預定買回股數'].apply(
            lambda x: f"{int(x/1000):,} 張" if pd.notna(x) and x > 0 else "未公告"
        )
        return result
    except Exception as e:
        print(f"處理 FinLab 數據時發生錯誤: {e}")
        return pd.DataFrame()



# finlab程式碼格式
def get_recent_investor_conferences(days=7):
    """
    doc:
    [Gemini 新增] 查詢並回傳近期 (預設為過去7天至今) 召開法人說明會的公司清單。
    - 為了方便後續查詢，回傳的 DataFrame 會以 stock_id 為索引。
    """
    try:
        logging.info(f"--- 正在查詢近 {days} 日的法說會資訊 ---")
        # 1. 取得法說會資料
        investor_conference = data.get('investors_conference').reset_index()
        investor_conference['date'] = pd.to_datetime(investor_conference['date'])

        # 2. 定義時間範圍
        today = pd.to_datetime(datetime.now().date())
        start_date = today - timedelta(days=days)

        # 3. 篩選出時間範圍內的法說會
        recent_conferences = investor_conference[investor_conference['date'] >= start_date].copy()

        if recent_conferences.empty:
            logging.info("--- 近期無任何法說會資訊 ---")
            return pd.DataFrame()

        # 4. 整理並回傳結果，以 stock_id 為索引
        recent_companies = recent_conferences.sort_values(
            by='date', ascending=False).drop_duplicates(subset='stock_id')
        
        return recent_companies.set_index('stock_id')

    except Exception as e:
        logging.error(f"查詢法說會資訊時發生錯誤: {e}", exc_info=True)
        return pd.DataFrame()
    
    


class StockAnalyzer:
    """股票分析引擎"""

    def __init__(self, stock_list, params, stock_map, disposal_history_df, cash_increase_history_df):
        self.stock_list = stock_list
        self.params = params
        self.stock_map = stock_map  # <--- [新增] 將傳入的 stock_map 儲存為實例屬性
        self.all_data = {}
        # doc: --- [Gemini v2.0 新增] 儲存完整的歷史事件資料 ---
        self.disposal_history = disposal_history_df.set_index('stock_id') if disposal_history_df is not None and not disposal_history_df.empty else pd.DataFrame()
        self.cash_increase_history = cash_increase_history_df.set_index('stock_id') if cash_increase_history_df is not None and not cash_increase_history_df.empty else pd.DataFrame()
        # doc: --- [新增結束] ---
        
        self._preload_data()


    # finlab程式碼格式
    def _preload_data(self):
        """
        [doc]
        [v9.1.1 當沖佔比修正版]
        - 新增預載入 'intraday_trading:當日沖銷交易成交股數' 數據，解決資料同步問題並提升效率。
        - ... (原註解)
        """
        print("正在預加載所有市場數據...")
        max_days = self.params['max_data_days']
        data_items = {
            "open": "price:開盤價", "high": "price:最高價", "low": "price:最低價", "close": "price:收盤價", "volume": "price:成交股數",
            "foreign_buy": "institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)",
            "it_buy": "institutional_investors_trading_summary:投信買賣超股數",
            "dealer_buy": "institutional_investors_trading_summary:自營商買賣超股數(自行買賣)",
            "margin_balance": "margin_transactions:融資今日餘額",
            "short_balance": "margin_transactions:融券今日餘額",
            "margin_usage": "margin_transactions:融資使用率",
            "short_usage": "margin_transactions:融券使用率",
            "monthly_revenue": "monthly_revenue:當月營收",
            "monthly_revenue_yoy": "monthly_revenue:去年同月增減(%)",
            "top15_buy": "etl:broker_transactions:top15_buy",
            "top15_sell": "etl:broker_transactions:top15_sell",
            "market_cap": "etl:market_value",
            # [新增] 預載入當日沖銷數據，確保資料同步並提升效率
            "day_trade_vol": "intraday_trading:當日沖銷交易成交股數",
            'gross_margin': 'fundamental_features:營業毛利率',
            'operating_margin': 'fundamental_features:營業利益率',
            'roe': 'fundamental_features:ROE稅後',
            'eps': 'financial_statement:每股盈餘'
        }

        loaded_dfs = {}
        for name, dhead in data_items.items():
            print(f"正在加載: {dhead}")
            try:
                full_df = data.get(dhead)
                valid_stocks = [s for s in self.stock_list if s in full_df.columns]
                loaded_dfs[name] = full_df[valid_stocks]
            except Exception as e:
                logging.error(f"加載數據 '{dhead}' 失敗: {e}")
                loaded_dfs[name] = pd.DataFrame()

        if not loaded_dfs:
            logging.error("沒有任何數據成功加載。")
            self.all_data = {}
            return

        common_stocks = set(loaded_dfs.get('close', pd.DataFrame()).columns)
        for name, df in loaded_dfs.items():
            if name == 'close':
                continue
            common_stocks.intersection_update(df.columns)

        common_stocks = sorted(list(common_stocks))
        logging.info(f"經過數據一致性檢查，找到 {len(common_stocks)} 檔在所有表中都有數據的股票。")

        for name, df in loaded_dfs.items():
            self.all_data[name] = df[common_stocks].tail(max_days if "monthly" not in name else 48)

        if 'volume' in self.all_data and not self.all_data['volume'].empty:
            self.all_data['volume'] /= 1000

        print("正在計算新的股權結構指標...")
        try:
            inv = data.get('inventory')
            inv_df = inv.reset_index()
            inv_filtered = inv_df[inv_df['stock_id'].isin(common_stocks)]
            if inv_filtered.empty:
                raise ValueError("在指定的股票清單中，未找到任何對應的股權分級資料。")
            
            self.all_data['inventory_weekly_data'] = inv_filtered
            
            h1_data = inv_filtered[inv_filtered['持股分級'].astype(int) <= 4]
            h2_data = inv_filtered[(inv_filtered['持股分級'].astype(int) >= 11) & (inv_filtered['持股分級'].astype(int) <= 14)]
            h1 = h1_data.groupby(['date', 'stock_id'], observed=True)['持有股數'].sum().unstack()
            h2 = h2_data.groupby(['date', 'stock_id'], observed=True)['持有股數'].sum().unstack()
            if h1.empty or h2.empty:
                 raise ValueError("計算大戶(h2)或散戶(h1)持股時，其中一方資料為空。")
            
            shareholder_ratio = (h2 / (h1 + h2)).reindex(columns=common_stocks)
            shareholder_ratio.index = pd.to_datetime(shareholder_ratio.index)
            self.all_data['shareholder_ratio'] = shareholder_ratio
            self.all_data['shareholder_ratio_diff1'] = shareholder_ratio.diff(6)
            self.all_data['shareholder_ratio_diff2'] = self.all_data['shareholder_ratio_diff1'].diff(6)

            share_level_map = {
                '200-400張': [11], '400-600張': [12], '600-800張': [13],
                '800-1000張': [14], '1000張+': [15]
            }
            dist_data_frames = {}
            for label, levels in share_level_map.items():
                tier_data = inv_filtered[inv_filtered['持股分級'].astype(int).isin(levels)]
                dist_data_frames[label] = tier_data.groupby(['date', 'stock_id'], observed=True)['持有股數'].sum().unstack() / 1000 # 轉為張

            total_shareholders_data = inv_filtered[inv_filtered['持股分級'].astype(int) == 17]
            dist_data_frames['總人數'] = total_shareholders_data.groupby(['date', 'stock_id'], observed=True)['人數'].sum().unstack()
            
            self.all_data['shareholder_distribution'] = pd.concat(dist_data_frames, axis=1)
            self.all_data['shareholder_distribution'].index = pd.to_datetime(self.all_data['shareholder_distribution'].index)

            print("股權結構指標計算完成。")

        except Exception as e:
            logging.error(f"計算股權結構指標時出錯: {e}")
            date_index = self.all_data.get('close', pd.DataFrame()).index
            empty_df = pd.DataFrame(index=date_index, columns=common_stocks)
            self.all_data.update({
                'shareholder_ratio': empty_df, 'shareholder_ratio_diff1': empty_df,
                'shareholder_ratio_diff2': empty_df, 'shareholder_distribution': pd.DataFrame()
            })
            print("股權結構指標計算失敗，已使用空資料替代。")




    def _get_historical_events(self, stock_id: str, days: int) -> dict:
        """
        doc: 
        [v2.0 新增] 從預載入的完整歷史資料中，篩選出指定股票在特定天期內發生的事件。
        """
        historical_events = {
            'disposals': [],
            'cash_increases': []
        }
        end_date = pd.to_datetime(datetime.now())
        start_date = end_date - timedelta(days=days)

        # 1. 篩選處置股歷史
        if not self.disposal_history.empty and stock_id in self.disposal_history.index:
            stock_disposals = self.disposal_history.loc[[stock_id]]
            for _, row in stock_disposals.iterrows():
                # 只要處置期間與圖表顯示期間有重疊，就納入
                if row['處置開始時間'] <= end_date and row['處置結束時間'] >= start_date:
                    historical_events['disposals'].append((row['處置開始時間'], row['處置結束時間']))

        # 2. 篩選現金增資歷史
        if not self.cash_increase_history.empty and stock_id in self.cash_increase_history.index:
            stock_increases = self.cash_increase_history.loc[[stock_id]]
            relevant_increases = stock_increases[
                (stock_increases['除權交易日'] >= start_date) & 
                (stock_increases['除權交易日'] <= end_date)
            ]
            for _, row in relevant_increases.iterrows():
                historical_events['cash_increases'].append(
                    (row['除權交易日'], row['現金增資認購價(元/股)'])
                )
        
        return historical_events




    def analyze(self, stock_id):
        """
        [doc]
        對單一股票執行全面的分析。
        [修改] 新增計算並儲存停損參考價。
        """
        analysis_results = {'stock_id': stock_id}
        close = self.all_data['close'][stock_id].dropna()
        if len(close) < self.params.get('volume_ma_period', 20):
            logging.warning(f"股票 {stock_id} 收盤價數據不足 ({len(close)} 天)，跳過分析。") # 新增警告
            return None

        # 核心修改：調用新版的多因子S/R分析函數
        analysis_results['sr_short'] = self._get_support_resistance(
            stock_id, self.params['plot_days_short'])
        analysis_results['sr_long'] = self._get_support_resistance(
            stock_id, self.params['plot_days_long'])

        # doc: --- [Gemini v2.0 新增] 取得歷史事件資料 ---
        analysis_results['historical_events_short'] = self._get_historical_events(
            stock_id, self.params['plot_days_short'])
        analysis_results['historical_events_long'] = self._get_historical_events(
            stock_id, self.params['plot_days_long'])
        # doc: --- [新增結束] -

        # doc: --- [新增] 計算停損參考價 ---
        # 使用您範例中的參數 X=60, N=20
        analysis_results['stop_loss_price'] = self._calculate_sell_conversion_price(
            stock_id, high_period=60, vol_avg_period=20, multiplier=3.0
        )
        # doc: --- [新增結束] ---


        # 其他分析模組維持不變
        analysis_results['institutional_flow'] = self._analyze_institutional_flow(
            stock_id)
        analysis_results['broker_flow'] = self._analyze_broker_flow(stock_id)
        analysis_results['shareholder_structure'] = self._analyze_shareholder_structure(
            stock_id)
        analysis_results['sentiment'] = self._analyze_sentiment(stock_id)
        analysis_results['fundamentals'] = self._analyze_fundamentals(stock_id)
        return analysis_results

    # ===================================================================
    # --- Part 1: S/R 水平識別模組 (Level Finders) ---
    # ===================================================================

    def _find_swing_levels(self, high: pd.Series, low: pd.Series, n: int):
        """
        [doc]
        使用 argrelextrema 識別價格的轉折高點與轉折低點。
        這是價格行為分析的基礎，用於找出市場的局部極值。
        
        Args:
            high (pd.Series): 最高價序列。
            low (pd.Series): 最低價序列。
            n (int): 識別轉折點所需的鄰近 K 棒數量 (order)。
        
        Returns:
            (pd.Series, pd.Series): 分別包含轉折高點價格和轉折低點價格的 Series。
        """
        high_idx = argrelextrema(high.values, np.greater, order=n)[0]
        low_idx = argrelextrema(low.values, np.less, order=n)[0]
        return high.iloc[high_idx], low.iloc[low_idx]

    def _calculate_pivots(self, df_kline: pd.DataFrame):
        """
        [doc]
        計算標準與斐波那契樞紐點。
        這是一種前瞻性的分析工具，利用前一週期的價位預測當前的潛在 S/R。
        
        Args:
            df_kline (pd.DataFrame): 包含 OHLC 數據的 K 線圖。
        
        Returns:
            dict: 包含各種樞紐點價位的字典。
        """
        # 使用最近的完整一根K棒作為計算基礎
        if len(df_kline) < 1:
            return {}
        
        H = df_kline['high'].iloc[-1]
        L = df_kline['low'].iloc[-1]
        C = df_kline['close'].iloc[-1]

        # Standard Pivots
        PP = (H + L + C) / 3
        R1 = (2 * PP) - L
        S1 = (2 * PP) - H
        R2 = PP + (H - L)
        S2 = PP - (H - L)

        # Fibonacci Pivots
        F_R1 = PP + 0.382 * (H - L)
        F_S1 = PP - 0.382 * (H - L)
        F_R2 = PP + 0.618 * (H - L)
        F_S2 = PP - 0.618 * (H - L)

        return {
            "PIVOT": {"price": PP, "type": "Pivot"},
            "S1": {"price": S1, "type": "S1"}, "R1": {"price": R1, "type": "R1"},
            "S2": {"price": S2, "type": "S2"}, "R2": {"price": R2, "type": "R2"},
            "F_S1": {"price": F_S1, "type": "Fib S1"}, "F_R1": {"price": F_R1, "type": "Fib R1"},
            "F_S2": {"price": F_S2, "type": "Fib S2"}, "F_R2": {"price": F_R2, "type": "Fib R2"},
        }
        
    def _get_volume_profile_levels(self, vp_data: tuple):
        """
        [doc]
        從計算好的成交量分佈圖數據中，提取關鍵水平。
        包括控制點(POC)、價值區高點(VAH)和價值區低點(VAL)。
        
        Args:
            vp_data (tuple): 由 _compute_vp 回傳的元組。
        
        Returns:
            dict: 包含 POC, VAH, VAL 價位的字典。
        """
        if vp_data is None:
            return {}
        
        bin_centers, up_vp, down_vp, _ = vp_data
        total_vp = up_vp + down_vp
        
        if len(total_vp) == 0:
            return {}

        # 1. 找出 POC (Point of Control)
        poc_index = np.argmax(total_vp)
        poc_price = bin_centers[poc_index]

        # 2. 計算價值區 (Value Area)
        total_volume_sum = np.sum(total_vp)
        if total_volume_sum == 0:
            return {"POC": poc_price}

        # 從 POC 開始向外擴展，直到包含70%的成交量
        current_volume = total_vp[poc_index]
        value_area_indices = {poc_index}
        left, right = poc_index - 1, poc_index + 1
        
        while current_volume / total_volume_sum < 0.7:
            if left < 0 and right >= len(total_vp):
                break # 避免無限迴圈
            
            # 比較左右兩側的成交量，將較大的一側納入價值區
            left_vol = total_vp[left] if left >= 0 else -1
            right_vol = total_vp[right] if right < len(total_vp) else -1
            
            if left_vol >= right_vol:
                current_volume += left_vol
                value_area_indices.add(left)
                left -= 1
            else:
                current_volume += right_vol
                value_area_indices.add(right)
                right += 1
                
        val_indices = sorted(list(value_area_indices))
        val_price = bin_centers[val_indices[0]]
        vah_price = bin_centers[val_indices[-1]]
        
        return {"POC": poc_price, "VAH": vah_price, "VAL": val_price}

    # ===================================================================
    # --- Part 2: 核心S/R分析與評分引擎 ---
    # ===================================================================


    def _get_support_resistance(self, s, days):
        """
        [doc]
        多因子支撐壓力分析引擎 (v2.1 - 簡化標籤版)。
        此函數是整個分析的核心，它遵循「匯合原則」：
        1.  **匯總 (Gather)**：從多種獨立來源收集潛在的 S/R 水平。
        2.  **聚類 (Cluster)**：將距離相近的水平合併為一個「S/R 區間」。
        3.  **評分 (Score)**：根據構成每個區間的來源數量和類型，對其進行量化評分。
        4.  **轉換 (Convert)**：應用「角色互換原則」，將被有效突破的壓力轉換為支撐。
        
        Args:
            s (str): 股票代碼。
            days (int): 分析的歷史天數。
        
        Returns:
            dict: 包含分析結果的字典，其中 'levels' 鍵對應了最終帶有評分和標籤的 S/R 水平。
        """
        df_kline = pd.DataFrame({
            'open': self.all_data['open'][s], 'high': self.all_data['high'][s], 
            'low': self.all_data['low'][s], 'close': self.all_data['close'][s], 
            'volume': self.all_data['volume'][s]
        }).tail(days).dropna()

        if len(df_kline) < 61:
            return {'levels': {}, 'vp_short': None, 'vp_long': None, 'trapped_zones': []}

        high, low, close, volume = df_kline['high'], df_kline['low'], df_kline['close'], df_kline['volume']
        current_price = close.iloc[-1]
        
        # --- 1. 匯總 (Gather) 所有潛在的 S/R 水平 ---
        all_raw_levels = []
        
        # 來源 1: 價格行為 (轉折點)
        d_highs, d_lows = self._find_swing_levels(high, low, self.params['sr_n_order_daily'])
        for p in d_highs: all_raw_levels.append({'price': p, 'source': '轉折點', 'weight': 1.0})
        for p in d_lows: all_raw_levels.append({'price': p, 'source': '轉折點', 'weight': 1.0})
        
        try:
            df_resample = df_kline.resample('W-FRI').agg({'high': 'max', 'low': 'min'})
            w_highs, w_lows = self._find_swing_levels(df_resample['high'], df_resample['low'], self.params['sr_n_order_weekly'])
            for p in w_highs: all_raw_levels.append({'price': p, 'source': '週線轉折', 'weight': 1.5}) # 週線權重更高
            for p in w_lows: all_raw_levels.append({'price': p, 'source': '週線轉折', 'weight': 1.5})
        except Exception:
            pass

        # 來源 2: 成交量分佈 (長天期)
        vp_long = self._compute_vp(close, df_kline['open'], high, low, volume, self.params['vp_window_long'], self.params['vp_bins'])
        vp_levels = self._get_volume_profile_levels(vp_long)
        if 'POC' in vp_levels: all_raw_levels.append({'price': vp_levels['POC'], 'source': 'POC', 'weight': self.params['poc_base_score']})
        if 'VAH' in vp_levels: all_raw_levels.append({'price': vp_levels['VAH'], 'source': 'VAH', 'weight': 1.5})
        if 'VAL' in vp_levels: all_raw_levels.append({'price': vp_levels['VAL'], 'source': 'VAL', 'weight': 1.5})

        # 來源 3: 樞紐點 (基於前一日)
        pivot_data = self._calculate_pivots(df_kline.head(len(df_kline)-1))
        for key, val in pivot_data.items():
            all_raw_levels.append({'price': val['price'], 'source': val['type'], 'weight': 1.0})
            
        # --- 2. 聚類 (Cluster) ---
        atr = (high - low).rolling(20).mean().iloc[-1]
        dynamic_eps = atr * self.params['eps_atr_multiplier'] if pd.notna(atr) and atr > 0 else current_price * 0.015
        
        if not all_raw_levels: return {'levels': {}, 'vp_short': None, 'vp_long': None, 'trapped_zones': []}

        sorted_levels = sorted(all_raw_levels, key=lambda x: x['price'])
        clusters = []
        current_cluster = [sorted_levels[0]]
        for i in range(1, len(sorted_levels)):
            if abs(sorted_levels[i]['price'] - current_cluster[-1]['price']) <= dynamic_eps:
                current_cluster.append(sorted_levels[i])
            else:
                clusters.append(current_cluster)
                current_cluster = [sorted_levels[i]]
        clusters.append(current_cluster)

        # --- 3. 評分 (Score) ---
        ma20 = close.rolling(20).mean().iloc[-1]
        ma60 = close.rolling(60).mean().iloc[-1]
        ma240 = close.rolling(240).mean().iloc[-1] if len(close) >= 240 else np.nan
        ma_levels = {'MA20': ma20, 'MA60': ma60, 'MA240': ma240}
        ma_scores = {'MA20': 1.0, 'MA60': 1.5, 'MA240': 2.0}

        enhanced_levels = {}
        for cluster in clusters:
            if not cluster: continue
            
            prices = np.array([item['price'] for item in cluster])
            weights = np.array([item['weight'] for item in cluster])
            center_price = np.average(prices, weights=weights)
            
            level_type = '壓力' if center_price > current_price else '支撐'
            
            total_score = sum(weights)
            
            # 檢查與均線的共振
            for ma_name, ma_value in ma_levels.items():
                if pd.notna(ma_value) and abs(center_price - ma_value) / center_price <= 0.015:
                    total_score += ma_scores[ma_name]
            
            if total_score > 5: strength_prefix = "強"
            elif total_score > 2: strength_prefix = "中繼"
            else: strength_prefix = ""
            
            # <--- 核心修改處：簡化標籤
            final_label = f"[{strength_prefix}{level_type}] [{total_score:.1f}]"
            
            enhanced_levels[center_price] = (final_label, total_score)
        
        # --- 4. 角色轉換 (Role Reversal) 與最終整理 ---
        current_resistance = {k: v for k, v in enhanced_levels.items() if '壓力' in v[0]}
        current_support = {k: v for k, v in enhanced_levels.items() if '支撐' in v[0]}
        # [修改] 改為呼叫新的 _convert_support_zones_ultra 函式，並傳入股票代碼 s
        remaining_resistance, converted_support = self._convert_support_zones_ultra(
            s, current_resistance, close, low, volume, atr)

        final_levels = {**current_support, **remaining_resistance, **converted_support}

        vp_short = self._compute_vp(close, df_kline['open'], high, low, volume, self.params['vp_window_short'], self.params['vp_bins'])
        trapped_zones = self._identify_trapped_zones(vp_short, current_price)
        
        return {'levels': final_levels, 'vp_short': vp_short, 'vp_long': vp_long, 'trapped_zones': trapped_zones}


    
    # finlab程式碼格式
    def _convert_support_zones_ultra(self, s, resistance_zones, close, low, volume, atr):
        """
        doc:
        多因子支撐壓力分析引擎的「壓力轉支撐」模組 (Ultra Thinking 強化版)。
        此版本在原始邏輯基礎上，新增了對「突破品質」與「假突破」的判斷，
        使其對市場行為的解讀更加細膩與強韌。

        Args:
            s (str): 股票代碼，這是本次修正的關鍵，必須從外部傳入。
            resistance_zones (dict): 當前的壓力區間字典。
            close (pd.Series): 收盤價序列。
            low (pd.Series): 最低價序列。
            volume (pd.Series): 成交量序列。
            atr (pd.Series): ATR 指標序列。
        
        Returns:
            (dict, dict): 更新後的壓力區間字典，以及新轉換的支撐區間字典。

        [Ultra Thinking 改進 1] 突破強度量化:
        - 計算突破 K 棒的實體長度佔 ATR 的比例，作為動能指標。
        - 突破強度越高，轉換後的支撐分數加成越多。

        [Ultra Thinking 改進 2] 假突破過濾 (新增確認期):
        - 在滿足基本的「突破持有期」後，增加一個「突破確認期」。
        - 價格必須在確認期內創下新高，才算完成有效的支撐轉換，以過濾牛市陷阱。
        """
        if len(close) < self.params['conversion_breakout_window'] + 15: # 加上確認期的長度
            return resistance_zones, {}

        current_price = close.iloc[-1]
        volume_ma = volume.rolling(self.params['volume_ma_period']).mean()
        
        # [修正] 直接從 self.all_data 中獲取數據，而不是 self.analyzer.all_data
        open_price = self.all_data['open'][s]

        converted_support = {}
        zones_to_remove = []

        for lvl, (label, score) in resistance_zones.items():
            if current_price <= lvl:
                continue
            
            breakout_candidates = close[close > lvl]
            if breakout_candidates.empty:
                continue
                
            first_breakout_date = breakout_candidates.index[0]
            
            # 建立觀測窗口
            obs_window_close = close.loc[first_breakout_date:].head(self.params['conversion_breakout_window'])
            if len(obs_window_close) < self.params['conversion_breakout_window']:
                continue

            # 條件 1: 突破持有期驗證
            is_breakout_sustained = (obs_window_close > lvl).sum() / len(obs_window_close) >= self.params['conversion_hold_ratio']
            
            if is_breakout_sustained:
                new_score = score
                
                # 條件 2: 成交量驗證
                if volume.get(first_breakout_date, 0) > volume_ma.get(first_breakout_date, 0) * self.params['conversion_vol_boost_multiplier']:
                    new_score += 1.0
                
                # 條件 3: 回測驗證
                obs_window_low = low.loc[obs_window_close.index]
                retest_tol = lvl * self.params['conversion_retest_tolerance']
                if ((obs_window_close > lvl) & (obs_window_low < lvl + retest_tol) & (obs_window_low > lvl - retest_tol)).any():
                    new_score += self.params['conversion_retest_bonus']
                    base_label = '[回測支撐]'
                else:
                    base_label = '[轉支撐]'

                # [Ultra Thinking 改進 1: 突破強度量化]
                breakout_candle_open = open_price.get(first_breakout_date)
                breakout_candle_close = close.get(first_breakout_date)
                current_atr_val = atr.get(first_breakout_date)
                
                if pd.notna(breakout_candle_open) and pd.notna(breakout_candle_close) and pd.notna(current_atr_val) and current_atr_val > 0:
                    body_size = abs(breakout_candle_close - breakout_candle_open)
                    strength_ratio = body_size / current_atr_val
                    if strength_ratio > 0.8: # 突破 K 棒實體超過當日 ATR 的 80%，視為強勁突破
                        new_score += 1.5 # 強力突破加分

                # [Ultra Thinking 改進 2: 假突破過濾]
                confirmation_window_size = 10 # 在突破持有期之後，再觀察10天
                confirmation_start_index = obs_window_close.index[-1]
                
                # 確保有足夠的數據進行確認
                if confirmation_start_index in close.index:
                    loc = close.index.get_loc(confirmation_start_index)
                    if loc + 1 < len(close):
                        confirmation_close = close.iloc[loc + 1 : loc + 1 + confirmation_window_size]
                        
                        # 確認條件：在確認期內，股價不僅要維持在支撐之上，還必須創下新高
                        if not confirmation_close.empty and (confirmation_close > lvl).all() and confirmation_close.max() > obs_window_close.max():
                            # 只有通過最終確認的，才加入轉換清單
                            converted_support[lvl] = (f'{base_label} [{new_score:.1f}]', new_score)
                            zones_to_remove.append(lvl)

        # 從原壓力清單中移除已成功轉換的
        for lvl in zones_to_remove:
            if lvl in resistance_zones:
                del resistance_zones[lvl]
                
        return resistance_zones, converted_support
    
    
    

    def _compute_vp(self, close, open_, high, low, volume, window, bins):
        df = pd.DataFrame({'c': close, 'o': open_, 'h': high,
                           'l': low, 'v': volume}).tail(window).dropna()
        if df.empty:
            return None
        price_min, price_max = df['l'].min(), df['h'].max()
        if not (pd.notna(price_min) and pd.notna(price_max)) or price_max == price_min:
            return None
        price_bins = np.linspace(price_min, price_max, bins + 1)
        bin_centers = (price_bins[:-1] + price_bins[1:]) / 2
        up_vp, down_vp = np.zeros(len(bin_centers)), np.zeros(len(bin_centers))
        for _, row in df.iterrows():
            start_idx, end_idx = np.searchsorted(
                price_bins, row['l'], side='right') - 1, np.searchsorted(price_bins, row['h'], side='right') - 1
            num_levels = end_idx - start_idx + 1
            if num_levels <= 0:
                continue
            vol_per_level = row['v'] / num_levels
            target_vp = up_vp if row['c'] >= row['o'] else down_vp
            for i in range(start_idx, end_idx + 1):
                if 0 <= i < len(target_vp):
                    target_vp[i] += vol_per_level
        return bin_centers, up_vp, down_vp, price_bins

    def _identify_trapped_zones(self, vp_results, current_price, max_distance_ratio=0.15):
        if vp_results is None:
            return []
        bin_centers, up_vp, down_vp, price_bins = vp_results
        total_vp = up_vp + down_vp
        if len(total_vp) == 0:
            return []
        threshold = np.percentile(total_vp[total_vp > 0], 70) if len(
            total_vp[total_vp > 0]) > 0 else 0
        zones = [{'lower': price_bins[i], 'upper': price_bins[i+1]} for i in range(len(bin_centers)) if bin_centers[i] >= current_price and total_vp[i] > threshold and abs(
            bin_centers[i] - current_price) / current_price < max_distance_ratio and down_vp[i] > up_vp[i]]
        if not zones:
            return []
        zones.sort(key=lambda x: x['lower'])
        merged = [zones[0]]
        for z in zones[1:]:
            if z['lower'] <= merged[-1]['upper'] * 1.005:
                merged[-1]['upper'] = max(merged[-1]['upper'], z['upper'])
            else:
                merged.append(z)
        return merged

    def _analyze_institutional_flow(self, s):
        df_daily = pd.DataFrame({'total': self.all_data['foreign_buy'][s].fillna(
            0) + self.all_data['it_buy'][s].fillna(0) + self.all_data['dealer_buy'][s].fillna(0)})
        if df_daily.empty:
            return pd.DataFrame()
        df_weekly = df_daily.resample('W-FRI').sum()
        df_weekly['ma5'] = df_weekly['total'].rolling(5).mean()
        df_weekly['ma10'] = df_weekly['total'].rolling(10).mean()
        return df_weekly.tail(self.params.get('plot_weeks_long', 104))

    def _analyze_broker_flow(self, s):
        df_daily = pd.DataFrame({'total': self.all_data['top15_buy'][s].fillna(
            0) - self.all_data['top15_sell'][s].fillna(0)})
        if df_daily.empty:
            return pd.DataFrame()
        df_weekly = df_daily.resample('W-FRI').sum()
        df_weekly['ma5'] = df_weekly['total'].rolling(5).mean()
        df_weekly['ma10'] = df_weekly['total'].rolling(10).mean()
        return df_weekly.tail(self.params.get('plot_weeks_long', 104))

    def _analyze_shareholder_structure(self, s):
        """
        [doc]
        [v9.0.3 錯誤修復]
        - 新增對 DataFrame index 的型別檢查，使其更穩健。
        - 在調用 .resample() 之前，先確認 df.index 是否為 DatetimeIndex。
        - 如果 index 型別不正確 (例如是 RangeIndex)，則記錄一條警告並返回一個空的 DataFrame，
          以防止程式因 TypeError 而崩潰，並允許報告的其餘部分繼續生成。
        """
        try:
            df = pd.DataFrame({
                'ratio': self.all_data['shareholder_ratio'][s],
                'diff1': self.all_data['shareholder_ratio_diff1'][s],
                'diff2': self.all_data['shareholder_ratio_diff2'][s]
            })

            # 如果 DataFrame 在建立後是空的，直接返回空的 DataFrame
            if df.empty:
                return pd.DataFrame()

            # [*** 核心修正點 ***]
            # 在重採樣(resample)前，嚴格檢查索引是否為時間序列類型。
            if not isinstance(df.index, pd.DatetimeIndex):
                logging.warning(
                    f"股票 {s} 的股權結構資料索引類型不正確 (類型為: {type(df.index)})，將跳過此圖表。")
                return pd.DataFrame()  # 返回空 DataFrame 以避免崩潰

            # 索引類型正確，才執行重採樣
            return df.resample('W').last().tail(104).dropna(how='all')

        except KeyError:
            # 捕獲因為股票代碼 s 不在 shareholder_ratio 中的錯誤
            logging.warning(f"股票 {s} 在股權結構資料中不存在，將跳過此圖表。")
            return pd.DataFrame()
        except Exception as e:
            # 捕獲其他可能的錯誤
            logging.error(f"分析股票 {s} 的股權結構時發生預期外的錯誤: {e}")
            return pd.DataFrame()

    def _analyze_sentiment(self, s):
        # [修改] DataFrame 的建構，移除不存在的 'margin_balance_value_k'，
        # 並只選取我們確定需要的欄位。
        df = pd.DataFrame({
            'close': self.all_data['close'][s],
            'margin_usage': self.all_data['margin_usage'][s],
            'short_usage': self.all_data['short_usage'][s],
            'short_balance': self.all_data['short_balance'][s],
            'margin_balance': self.all_data['margin_balance'][s],  # 單位: 股
        }).tail(self.params['plot_days_long']).dropna(subset=['close'])

        # [新增] 新的替代指標計算邏輯：融資擔保品價值指數
        # 1. 計算每日的擔保品總價值
        collateral_value = df['margin_balance'] * df['close']

        # 2. 為了避免指數因股價短期劇烈波動而失真，我們取其20日移動平均來平滑化
        collateral_value_ma = collateral_value.rolling(
            20, min_periods=1).mean()

        # 3. 將這個價值序列轉換成一個基準為100的指數，方便觀察趨勢
        #    找到第一個有效的平滑後價值作為基準點
        first_valid_value = collateral_value_ma.dropna(
        ).iloc[0] if not collateral_value_ma.dropna().empty else 1
        df['collateral_value_index'] = (
            collateral_value_ma / first_valid_value) * 100

        # --- 原有邏輯維持不變 ---
        # 計算融資券與股價的5日變化量
        df['margin_balance_chg'] = df['margin_balance'].diff(5)
        df['short_balance_chg'] = df['short_balance'].diff(5)
        # 使用 pct_change 更能反映股價變動的相對幅度
        df['close_pct_chg_5d'] = df['close'].pct_change(5) * 100

        df['short_to_margin_ratio'] = (
            df['short_balance'] / df['margin_balance']).replace([np.inf, -np.inf], 0).fillna(0)
        df['margin_usage_threshold'] = df['margin_usage'].rolling(
            250, min_periods=50).quantile(0.75)

        return df

    def _analyze_fundamentals(self, s):
        df = pd.DataFrame({'revenue': self.all_data['monthly_revenue'][s],
                           'yoy': self.all_data['monthly_revenue_yoy'][s]}).tail(36).dropna(how='all')
        if df.empty:
            return pd.DataFrame()
        df.index = df.index.to_period('M').to_timestamp('M')
        daily_close = self.all_data['close'][s].dropna()
        if not daily_close.empty:
            monthly_avg_price = daily_close.resample('ME').mean()
            df = df.join(monthly_avg_price.rename('monthly_avg_price'))
        return df




    # finlab程式碼格式
    # doc: ===================================================================
    # doc: --- [修正] 計算賣出轉換價格 (停損價) - 修正 attribute 錯誤 ---
    # doc: ===================================================================
    def _calculate_sell_conversion_price(self, stock_id: str, high_period: int = 60, vol_avg_period: int = 20, multiplier: float = 3.0) -> float | None:
        """
        doc:
        計算指定股票的賣出轉換價格（停損參考價）。
        公式：近 X 日收盤價高點 – (過去 N 天平均 K 線波動幅度的 multiplier 倍)
        [修改] 修正 Attribute 錯誤，將 self.analyzer.all_data 改為 self.all_data。
        [修改] 此版本呼叫全域的 compute_candle_volatility 函式取得波動率。

        Args:
            stock_id (str): 股票代碼。
            high_period (int): 計算最高收盤價的回溯天期 (X)。
            vol_avg_period (int): 計算平均 K 線波動幅度的回溯天期 (N)。
            multiplier (float): K 線波動幅度的乘數，預設為 3。

        Returns:
            float | None: 計算出的停損價格，若數據不足則回傳 None。
        """
        try:
            # 確保有足夠的收盤價數據用於計算最高點和波動率
            required_close_days = max(high_period, vol_avg_period + 1) # 波動率計算需要多一天
            
            # --- [核心修正] ---
            # 將 self.analyzer.all_data 改為 self.all_data
            close_prices = self.all_data['close'][stock_id].dropna() # 使用 analyzer 預載的資料
            # --- [修正結束] ---

            if len(close_prices) < required_close_days:
                logging.warning(f"股票 {stock_id} 收盤價數據不足 ({len(close_prices)} < {required_close_days})，無法計算停損價。")
                return None

            # 1. 計算近 X 日的收盤價最高點 (使用最近 high_period 天)
            highest_close = close_prices.tail(high_period).max()
            if pd.isna(highest_close):
                 logging.warning(f"股票 {stock_id} 無法計算 {high_period} 日最高收盤價。")
                 return None

            # 2. 呼叫 compute_candle_volatility 取得每日波動率序列
            daily_volatility_all = compute_candle_volatility(timeperiod=vol_avg_period) # 這裡的 timeperiod 是內部平均用的
            
            if stock_id not in daily_volatility_all.columns:
                 logging.warning(f"全域 compute_candle_volatility 未能計算股票 {stock_id} 的波動率。")
                 return None

            daily_volatility_stock = daily_volatility_all[stock_id].dropna()

            # 3. 計算過去 N 天的平均波動幅度
            if len(daily_volatility_stock) < vol_avg_period:
                logging.warning(f"股票 {stock_id} 計算出的每日波動率數據不足 ({len(daily_volatility_stock)} < {vol_avg_period})。")
                return None

            # 注意：您的 compute_candle_volatility 內部已經做了 .average(timeperiod)
            avg_volatility = daily_volatility_stock.iloc[-1] # 直接取 compute_candle_volatility 返回的最新值

            if pd.isna(avg_volatility):
                 logging.warning(f"股票 {stock_id} 計算出的平均波動率為 NaN。")
                 return None

            # 4. 計算最終停損價格
            stop_loss_price = highest_close - (avg_volatility * multiplier)

            return stop_loss_price

        except KeyError as e:
            logging.error(f"計算停損價時缺少必要數據欄位 for {stock_id}: {e}")
            return None
        except Exception as e:
            logging.error(f"計算股票 {stock_id} 的停損價時發生未知錯誤: {e}", exc_info=True)
            return None




# ===================================================================
# finlab程式碼格式
# ===================================================================
# --- BrokerChartGenerator Class (v7.2 日期標示強化版) ---
# ===================================================================
class BrokerChartGenerator:
    """
    券商綜合分析圖產生器 (六宮格-最終修正版)。
    - 採用 3x2 網格佈局，左側為每日更新圖表，右側為每週更新圖表。
    - 修正了所有已知的錯誤與警告。
    - [v7.2 使用者需求修改] 在所有六個子圖的標題中，都增加了數據的起始與結束日期。
    """

    def __init__(self, close_price, buy_vol, sell_vol, volume, market_cap, company_info, inventory_weekly_data, shareholder_ratio, margin_balance):
        self.close_price = close_price
        self.buy_vol = buy_vol
        self.sell_vol = sell_vol
        self.volume = volume
        self.market_cap = market_cap
        self.company_info = company_info
        self.inventory_weekly_data = inventory_weekly_data
        self.shareholder_ratio = shareholder_ratio
        self.margin_balance = margin_balance # 這一行現在可以正確運作了

        self.dark_color = '#333333'
        self.share_level_map = {
            '200-400張': [11], 
            '400-600張': [12], 
            '600-800張': [13],
            '800-1000張': [14], 
            '1000張+': [15]
        }
        self.tier_colors = ['#9467bd', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    def _calculate_rolling_mdd(self, series: pd.Series, window: int) -> pd.Series:
        rolling_max = series.rolling(window, min_periods=1).max()
        daily_drawdown = series / rolling_max - 1.0
        return daily_drawdown.rolling(window, min_periods=1).min()
    
    
    
    
    # finlab程式碼格式
    # [ULTRA THINK 新增 1/5] 斜率計算輔助函式
    def _calculate_slope(self, series: pd.Series, window: int = 10) -> float:
        """
        doc: 使用線性迴歸計算一個 Series 最近 N 個點的斜率。
        """
        from scipy.stats import linregress
        
        recent_points = series.dropna().tail(window)
        if len(recent_points) < 3: # 至少需要3個點才能計算有意義的斜率
            return 0.0
            
        # 為了讓斜率可互相比較，對 y 軸進行標準化
        y = recent_points.values
        y_normalized = (y - np.min(y)) / (np.max(y) - np.min(y)) if (np.max(y) - np.min(y)) > 0 else np.zeros_like(y)
        
        x = np.arange(len(y_normalized))
        slope, _, _, _, _ = linregress(x, y_normalized)
        
        return slope



    # [ULTRA THINK 新增 2/5] 布林帶寬繪圖函式
    def _plot_bollinger_bandwidth(self, ax: plt.Axes, upper_band: pd.Series, lower_band: pd.Series, middle_band: pd.Series):
        """
        doc: 繪製布林帶寬指標，並標示出 5% 與 20% 的關鍵門檻。
        """
        bandwidth = ((upper_band - lower_band) / middle_band * 100).dropna()
        ax.set_title("布林帶寬 (擋板高度)", fontsize=16, weight='bold')
        ax.plot(bandwidth.index, bandwidth, color='purple', label='帶寬 (%)')
        
        # 標示出您策略中的關鍵水平線
        ax.axhline(20, color='red', linestyle='--', label='高擋板 (20%)')
        ax.axhline(5, color='green', linestyle='--', label='低擋板 (5%)')
        
        ax.legend()
        ax.yaxis.set_major_formatter(FuncFormatter('{:.1f}%'.format))
        ax.grid(True, linestyle='--', alpha=0.6)
        
        
        

    # [ULTRA THINK 新增 3/5] 通道斜率繪圖函式
    def _plot_bollinger_slope(self, ax: plt.Axes, upper_band: pd.Series, lower_band: pd.Series):
        """
        doc: 繪製布林通道上下軌的斜率趨勢。
        """
        upper_slope = upper_band.rolling(10).apply(self._calculate_slope, raw=False) * 100
        lower_slope = lower_band.rolling(10).apply(self._calculate_slope, raw=False) * 100
        
        ax.set_title("通道斜率 (趨勢強度)", fontsize=16, weight='bold')
        ax.plot(upper_slope.index, upper_slope, color='red', label='上軌斜率')
        ax.plot(lower_slope.index, lower_slope, color='green', label='下軌斜率')
        
        # 標示出您策略中的關鍵水平線
        ax.axhline(1, color='red', linestyle=':', label='轉強門檻 (1%)')
        ax.axhline(-1, color='green', linestyle=':', label='轉弱門檻 (-1%)')
        ax.axhline(0, color='gray', linestyle='--')
        
        ax.legend()
        ax.yaxis.set_major_formatter(FuncFormatter('{:.1f}%'.format))
        ax.grid(True, linestyle='--', alpha=0.6)
 



    '''
    ▌第一步：選對標的，才能夾到寶
    就像夾娃娃要挑選「有手有腳」的，股市選股也是如此。
    我會鎖定有明確技術「扣點」的股票，也就是那些在圖形上能找到明確進場、出場點的標的。
    
    避開「圓滾滾」的娃娃：這類股票就像股市裡的「菜貨」，波動小，獲利機會低。它們通常缺乏明確的技術訊號，你再怎麼努力也難以賺錢，投報率太低。
    
    ▌第二步：看清布林通道，判斷機台擋板
    布林通道的帶寬，就像娃娃機的擋板。它告訴你，接下來的股價會怎麼走。
    
    ➀ 布林帶寬窄（低擋板）：當布林通道帶寬在 5% 以下，股價就像遇到低擋板。這代表股票已經整理很久，準備要「表態」了。這時候，股價很容易大幅上漲或下跌，是等待大行情的關鍵時機。
    
    ➁ 布林帶寬寬（高擋板）：當帶寬超過 20% ，股價就像遇到高擋板，會在一個大區間內震盪。這時候，通常沒有明確趨勢，我會選擇觀望。
    
    ▌第三步：辨識主力，看懂爪子力道
    爪子的力道是夾娃娃的關鍵，對應到股市，就是 #主力籌碼。
    
    ➀ 爪子有力（主力吸貨）：當一檔股票有主力大舉買進時，股價就容易「噴出」。如果你發現主力在股價盤整或低檔時吸貨，這就是跟隨進場的好時機。
    
    ➁ 爪子沒力（主力出貨）：反之，如果主力在大舉賣超，即使技術指標看似良好，我也會避開不碰。有時候主力還會「布局誘多」，把股價拉到布林通道上軌，引誘散戶追高，好讓他們自己順利出貨。
    
    ▌第四步：掌握斜率，抓住「隨機保夾」機會
    日本娃娃機有「隨機保夾」機制，布林通道也有！透過量化「斜率」來判斷趨勢：
    
    ➀ 上通道斜率上翹1%以上：這表示趨勢正在轉強，可以尋找進場機會。
    
    ➁ 下通道斜率下彎1%以上：這表示股價正在加速下跌，則毫不猶豫地「趕緊賣出」。
    
    然而，需要特別留意以下例外情況： 如果布林帶寬是「寬」的（20%以上），且主力籌碼呈現持續買超，那麼股價觸及或接近下軌時，反而是逢低買進的機會。

    '''


    # finlab程式碼格式
    # [ULTRA THINK v9.1 六宮格整合版]
    # finlab程式碼格式
    # doc: =============================================================================
    # doc: --- BrokerChartGenerator Class 內部函式修改 ---
    # doc: =============================================================================

    def _plot_price(self, ax: plt.Axes, price_full: pd.Series, net_buy_full: pd.Series, cvd_full: pd.Series, period: int):
        """
        doc: [v9.2 錯誤修正]
        - [核心修正] 將 ax.axvspan(...) 修改為 ax.fill_between(...) 以修復 'where' 參數的錯誤。
        - (v9.1) 整合「夾娃娃」策略的四個核心要素。
        """
        # --- 1. 資料與指標計算 (維持不變) ---
        price, net_buy, cvd = price_full.tail(period), net_buy_full.tail(period), cvd_full.tail(period)
        
        window, std_multiplier = 20, 1.5
        price_ma = price_full.rolling(window=window).mean()
        price_std = price_full.rolling(window=window).std()
        price_upper = price_ma + (price_std * std_multiplier)
        price_lower = price_ma - (price_std * std_multiplier)

        price_bandwidth = ((price_upper.iloc[-1] - price_lower.iloc[-1]) / price_ma.iloc[-1] * 100)
        
        upper_slope = self._calculate_slope(price_upper, window=10) * 100
        lower_slope = self._calculate_slope(price_lower, window=10) * 100

        net_buy_ma = net_buy_full.rolling(window=window).mean()
        net_buy_std = net_buy_full.rolling(window=window).std()
        net_buy_upper = net_buy_ma + (net_buy_std * std_multiplier)
        net_buy_lower = net_buy_ma - (net_buy_std * std_multiplier)

        # --- 2. 建立標題 (維持不變) ---
        date_title = f'({price.index.min().strftime("%Y-%m-%d")} ~ {price.index.max().strftime("%Y-%m-%d")})'
        bandwidth_title = f"目前帶寬: {price_bandwidth:.1f}%"
        slope_title = f"上軌斜率: {upper_slope:+.1f}% | 下軌斜率: {lower_slope:+.1f}%"
        ax.set_title(f'股價 vs 主力籌碼  {date_title}\n{bandwidth_title} | {slope_title}', fontsize=16, weight='bold')

        # --- 3. 繪製圖表 (維持不變) ---
        ax.plot(price.index, price, label='收盤價', color='darkblue', linewidth=2.5, zorder=10)
        ax.plot(price_ma.tail(period), color='gray', linestyle=':', linewidth=1.5, label=f'{window}日均線')
        ax.plot(price_upper.tail(period), color='red', linestyle='--', linewidth=1.2, label=f'上軌')
        ax.plot(price_lower.tail(period), color='green', linestyle='--', linewidth=1.2, label=f'下軌')
        ax.fill_between(price.index, price_lower.tail(period), price_upper.tail(period), color='skyblue', alpha=0.2)
        
        # doc: =============================================================================
        # doc: --- [核心修正] 將 axvspan 換成 fill_between ---
        # doc: =============================================================================
        bw_series = ((price_upper - price_lower) / price_ma * 100).tail(period)
        buy_condition = (bw_series > 20) & (net_buy_ma.tail(period) > 0) & (price < price_lower.tail(period) * 1.01)
        if buy_condition.any():
            # 使用 fill_between 來根據條件填滿背景區域
            ax.fill_between(price.index, ax.get_ylim()[0], ax.get_ylim()[1], where=buy_condition, 
                        color='springgreen', alpha=0.4, label='例外買點訊號')

        # --- 副圖與座標軸設定 (維持不變) ---
        ax_twin = ax.twinx()
        colors = ['#e63946' if x > 0 else '#4d9d4a' for x in net_buy]
        ax_twin.bar(net_buy.index, net_buy, color=colors, alpha=0.6, label='主力買賣超')
        ax_twin.plot(net_buy_ma.tail(period), color='purple', linestyle=':', linewidth=2, label='主力力道趨勢')
        ax_twin.fill_between(net_buy.index, net_buy_lower.tail(period), net_buy_upper.tail(period), 
                            color='purple', alpha=0.15, label='主力力道通道')
        
        ax.set_ylabel('股\n價', rotation=0, labelpad=20, color='darkblue', fontsize=14, va='center')
        ax_twin.set_ylabel('主\n力\n淨\n買\n賣\n超', rotation=0, labelpad=35, color=self.dark_color, fontsize=14, va='center')
        ax.grid(True, linestyle='--', alpha=0.6)
        ax_twin.axhline(0, color='gray', linestyle='--', linewidth=0.8)
        
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax_twin.get_legend_handles_labels()
        ax.legend(handles=lines + lines2, loc='upper left', fontsize=10, ncol=2)





    def _plot_shareholder_distribution_overlay(self, ax: plt.Axes, stock_id: str, period_weeks: int = 48):
        """
        doc:
        [v3.7 Gemini 雙軸動態縮放]
        - 根據使用者回饋，為右側的「持股百分比」Y軸也新增了動態縮放基準，使其能與左側Y軸一樣，聚焦於數據的變化區間。
        [v3.3 Gemini 最終設計師版]
        - 總股東人數柱狀圖改為動態顏色（增綠減紅）。
        - 折線圖配色改為高對比度的 綠、藍、紅。
        [v3.2 Gemini Y軸動態縮放]
        - 為總股東人數Y軸增加了動態縮放功能。
        """
        import matplotlib.colors as mcolors

        try:
            stock_data = self.inventory_weekly_data[self.inventory_weekly_data['stock_id'] == stock_id].copy()
            if stock_data.empty:
                raise ValueError("查無此股票的集保資料")

            stock_data['date'] = pd.to_datetime(stock_data['date'])
            stock_data['持股分級'] = pd.to_numeric(stock_data['持股分級'], errors='coerce')
            stock_data['人數'] = pd.to_numeric(stock_data['人數'], errors='coerce')
            stock_data['占集保庫存數比例'] = pd.to_numeric(stock_data['占集保庫存數比例'], errors='coerce')

            weekly_summary = {}
            for date, group in stock_data.groupby('date'):
                total_holders_series = group[group['持股分級'] == 17]['人數']
                retail_pct_series = group[group['持股分級'].isin(range(2, 10))]['占集保庫存數比例']
                mid_tier_pct_series = group[group['持股分級'].isin(range(10, 15))]['占集保庫存數比例']
                large_holder_pct_series = group[group['持股分級'] == 15]['占集保庫存數比例']

                weekly_summary[date] = {
                    '總股東人數': total_holders_series.sum(),
                    '1-100張持有百分比': retail_pct_series.sum(),
                    '100-1000張持有百分比': mid_tier_pct_series.sum(),
                    '1000張以上持有百分比': large_holder_pct_series.sum()
                }

            if not weekly_summary:
                raise ValueError("無法彙總每週數據")

            df_plot = pd.DataFrame.from_dict(weekly_summary, orient='index').sort_index().tail(period_weeks)
            if df_plot.empty:
                raise ValueError(f"最近 {period_weeks} 週無有效數據")

            date_title = f'({df_plot.index.min().strftime("%Y-%m-%d")} ~ {df_plot.index.max().strftime("%Y-%m-%d")})'
            ax.set_title(f'股權分散疊圖 (最近 {period_weeks} 週) {date_title}', fontsize=16, weight='bold')

            # --- 左側Y軸 (總股東人數) 動態縮放 ---
            shareholder_count = df_plot['總股東人數']
            if not shareholder_count.empty and shareholder_count.max() > 0 and shareholder_count.min() > 0:
                y_min = shareholder_count.min()
                y_max = shareholder_count.max()
                padding = (y_max - y_min) * 0.15
                ax.set_ylim(bottom=y_min - padding, top=y_max + padding)

            # --- 柱狀圖動態顏色 ---
            change = shareholder_count.diff()
            colors, pos_changes, neg_changes = [], change[change > 0], change[change < 0].abs()
            norm_pos = mcolors.Normalize(vmin=pos_changes.min(), vmax=pos_changes.max()) if not pos_changes.empty else mcolors.Normalize(0, 1)
            norm_neg = mcolors.Normalize(vmin=neg_changes.min(), vmax=neg_changes.max()) if not neg_changes.empty else mcolors.Normalize(0, 1)
            cmap_green = mcolors.LinearSegmentedColormap.from_list("custom_green", ["#a9d6a9", "#1e8449"])
            cmap_red = mcolors.LinearSegmentedColormap.from_list("custom_red", ["#f5b7b1", "#c0392b"])
            bar_colors = []
            for val in change:
                if pd.isna(val): bar_colors.append('#cccccc')
                elif val > 0: bar_colors.append(cmap_green(norm_pos(val)))
                elif val < 0: bar_colors.append(cmap_red(norm_neg(abs(val))))
                else: bar_colors.append('#cccccc')
            
            ax.bar(df_plot.index, df_plot['總股東人數'], color=bar_colors, width=pd.Timedelta(days=5), label='總股東人數')
            ax.set_ylabel('總\n股\n東\n人\n數', rotation=0, labelpad=30, color=self.dark_color, fontsize=14)
            ax.tick_params(axis='y', labelcolor=self.dark_color)
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}k' if x >= 1000 else f'{int(x)}'))
            ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=8, integer=True))

            ax_twin = ax.twinx()
            
            # --- 折線圖繪製 ---
            ax_twin.plot(df_plot.index, df_plot['1-100張持有百分比'], color='#2ca02c', marker='o', markersize=4, lw=2.5, label='1-100張(散戶)')
            ax_twin.plot(df_plot.index, df_plot['100-1000張持有百分比'], color='#1f77b4', marker='o', markersize=4, lw=2.5, label='100-1000張(中實戶)')
            ax_twin.plot(df_plot.index, df_plot['1000張以上持有百分比'], color='#d62728', marker='o', markersize=4, lw=2.5, label='1000張以上(大戶)')
            
            # --- [v3.7 核心修正] 右側Y軸 (持股百分比) 動態縮放 ---
            all_pct_values = pd.concat([
                df_plot['1-100張持有百分比'],
                df_plot['100-1000張持有百分比'],
                df_plot['1000張以上持有百分比']
            ]).dropna()

            if not all_pct_values.empty:
                y_min_pct = all_pct_values.min()
                y_max_pct = all_pct_values.max()
                padding_pct = (y_max_pct - y_min_pct) * 0.15
                # 確保下邊界不會小於0
                ax_twin.set_ylim(bottom=max(0, y_min_pct - padding_pct), top=y_max_pct + padding_pct)
            # --- [修正結束] ---

            ax_twin.set_ylabel('持\n股\n百\n分\n比\n(%)', rotation=0, labelpad=35, color=self.dark_color, fontsize=14)
            ax_twin.yaxis.set_major_formatter(FuncFormatter('{:.1f}%'.format))
            ax_twin.yaxis.set_major_locator(plt.MaxNLocator(nbins=8))

            lines, labels = ax.get_legend_handles_labels()
            lines2, labels2 = ax_twin.get_legend_handles_labels()
            ax_twin.legend(lines + lines2, labels + labels2, loc='upper left', fontsize=10)
            ax.grid(True, linestyle='--', alpha=0.6)

        except Exception as e:
            ax.text(0.5, 0.5, f"無法繪製股權分散疊圖:\n{e}", ha='center', va='center',
                    transform=ax.transAxes, fontsize=14, color='red', wrap=True)
            logging.error(f"為 {stock_id} 繪製股權分散疊圖時出錯: {e}")




    # [v7.2 + Gemini 標題優化+趨勢箭頭最終修正版]
    def _plot_shareholder_flow(self, ax: plt.Axes, stock_id: str, price_series: pd.Series):
        """
        doc:
        [Gemini 趨勢箭頭] 根據使用者需求，新增週趨勢箭頭。
        - 比較最新一週與前一週的持股比例，顯示增加(▲)或減少(▼)。
        [Gemini 最終修正] 修正目標函式，確保在正確的 BrokerChartGenerator class 中進行修改。
        - 在標題中動態計算並加入最新的持股比例。
        """
        try:
            stock_inv = self.inventory_weekly_data[self.inventory_weekly_data['stock_id'] == stock_id].copy(
            )
            stock_inv['date'] = pd.to_datetime(stock_inv['date'])
            stock_inv['持股分級'] = pd.to_numeric(stock_inv['持股分級'], errors='coerce')
            stock_inv['占集保庫存數比例'] = pd.to_numeric(
                stock_inv['占集保庫存數比例'], errors='coerce')

            all_dates = sorted(stock_inv['date'].unique())
            recent_10_weeks = all_dates[-10:]
            start_date = recent_10_weeks[0].strftime('%Y-%m-%d')
            end_date = recent_10_weeks[-1].strftime('%Y-%m-%d')
            date_title = f'({start_date} ~ {end_date})'

            # doc: [趨勢箭頭 1/3] 新增邏輯，抓取並計算最新與前一週的數據。
            title_info = ""
            try:
                # --- 定義計算函式 ---
                def get_ratios_for_date(target_date):
                    data = stock_inv[stock_inv['date'] == target_date]
                    over_200 = data[data['持股分級'].isin(
                        range(11, 16))]['占集保庫存數比例'].sum()
                    under_30 = data[data['持股分級'].isin(
                        range(2, 7))]['占集保庫存數比例'].sum()
                    over_1000_series = data[data['持股分級'] == 15]['占集保庫存數比例']
                    over_1000 = over_1000_series.iloc[0] if not over_1000_series.empty else 0.0
                    return over_200, under_30, over_1000

                # --- 計算最新一週的比例 ---
                latest_date = all_dates[-1]
                over_200_pct, under_30_pct, over_1000_pct = get_ratios_for_date(
                    latest_date)

                # doc: [趨勢箭頭 2/3] 比較最新與前一週的持股比例，產生趨勢符號(▲/▼)。
                trend_200, trend_30, trend_1000 = "", "", ""
                if len(all_dates) > 1:
                    previous_date = all_dates[-2]
                    prev_over_200, prev_under_30, prev_over_1000 = get_ratios_for_date(
                        previous_date)

                    def get_arrow(current, prev):
                        if current > prev:
                            return "▲"
                        if current < prev:
                            return "▼"
                        return ""

                    trend_200 = get_arrow(over_200_pct, prev_over_200)
                    trend_30 = get_arrow(under_30_pct, prev_under_30)
                    trend_1000 = get_arrow(over_1000_pct, prev_over_1000)

                # doc: [趨勢箭頭 3/3] 將趨勢符號加入標題字串中。
                title_info = (f"\n>200張大戶: {over_200_pct:.2f}%{trend_200} | "
                            f"<30張散戶: {under_30_pct:.2f}%{trend_30} | "
                            f"千張大戶: {over_1000_pct:.2f}%{trend_1000}")

            except Exception as e:
                logging.warning(f"為 {stock_id} 計算持股比例趨勢時出錯: {e}")

            ax.set_title(f'大戶籌碼堆疊流向圖 {date_title}{title_info}',
                        fontsize=16, color=self.dark_color, weight='bold')

            # --- 以下繪圖邏輯維持不變 ---
            weekly_changes = {}
            for label, levels in self.share_level_map.items():
                tier_data = stock_inv[stock_inv['持股分級'].astype(int).isin(levels)]
                weekly_sum = tier_data.groupby('date')['持有股數'].sum() / 1000
                weekly_changes[label] = weekly_sum.diff()

            df_change = pd.DataFrame(weekly_changes).loc[recent_10_weeks].fillna(0)
            positive_changes = df_change[df_change > 0]
            negative_changes = df_change[df_change < 0]

            positive_changes.plot(kind='bar', stacked=True, ax=ax,
                                width=0.7, color=self.tier_colors, legend=False, zorder=3)
            negative_changes.plot(kind='bar', stacked=True, ax=ax,
                                width=0.7, color=self.tier_colors, legend=False, zorder=3)

            ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
            ax.set_ylabel("週\n變\n化\n總\n張\n數", rotation=0, labelpad=20,
                        fontsize=14, color=self.dark_color)

            handles = [Patch(color=color, label=label)
                    for color, label in zip(self.tier_colors, df_change.columns)]
            ax.legend(handles=handles, title="大戶分級", loc='upper left')

            ax2 = ax.twinx()
            net_weekly_flow = df_change.sum(axis=1)
            cumulative_flow = net_weekly_flow.cumsum()
            weekly_price = price_series.reindex(df_change.index, method='ffill')

            def min_max_scaler(series):
                min_val, max_val = series.min(), series.max()
                return pd.Series(0.5, index=series.index) if max_val - min_val == 0 else (series - min_val) / (max_val - min_val)

            if not weekly_price.isnull().all() and not cumulative_flow.isnull().all():
                p1, = ax2.plot(ax.get_xticks(), min_max_scaler(
                    cumulative_flow), color='#800080', label='累積淨流向(標準化)', marker='o', markersize=5, lw=2.5)
                p2, = ax2.plot(ax.get_xticks(), min_max_scaler(
                    weekly_price), color='orange', label='股價(標準化)', marker='.', ls='--', lw=2.0)
                ax2.set_ylabel("標\n準\n化\n數\n值", rotation=0,
                            labelpad=20, fontsize=14, color=self.dark_color)
                ax2.legend(handles=[p1, p2], loc='upper right')
                ax2.grid(False)

            plt.setp(ax.get_xticklabels(), visible=False)

        except Exception as e:
            ax.text(0.5, 0.5, f"無法繪製大戶籌碼圖:\n{e}", ha='center', va='center',
                    transform=ax.transAxes, fontsize=14, color='red')
            logging.error(f"為 {stock_id} 繪製大戶籌碼圖時出錯: {e}")



   # finlab程式碼格式
    def _plot_risk(self, ax: plt.Axes, volatility: pd.Series, max_drawdown: pd.Series, margin_balance: pd.Series):
        """
        doc:
        [v8.4 字型符號修正]
        - [核心修正] 將標題中的特殊數學符號 '≤' 替換為 ASCII 字元 '<='，以解決微軟正黑體 (Microsoft JhengHei) 字型缺失該字符而產生 UserWarning 與亂碼的問題。
        - (v8.3) 強化標題資訊，新增相關係數、創高/低訊號、均線趨勢與圖例。
        """
        try:
            # --- 1. 資料準備與計算 ---
            start_date = volatility.index.min().strftime('%Y-%m-%d')
            end_date = volatility.index.max().strftime('%Y-%m-%d')
            date_title = f'({start_date} ~ {end_date})'

            margin_change = margin_balance.diff().fillna(0)
            colors = ['#EF5350' if c > 0 else '#26A69A' for c in margin_change]

            # 1. 計算相關係數
            correlation = volatility.rolling(20).corr(max_drawdown).iloc[-1]
            corr_text = f"Corr(V/MDD): {correlation:+.2f}"

            # 2. 計算融資餘額變化創高/低
            margin_hl_text = ""
            if len(margin_change) >= 20:
                latest_change = margin_change.iloc[-1]
                if latest_change >= margin_change.rolling(20).max().iloc[-1]: margin_hl_text = "創20日新高"
                elif latest_change <= margin_change.rolling(20).min().iloc[-1]: margin_hl_text = "創20日新低"
                elif latest_change >= margin_change.rolling(5).max().iloc[-1]: margin_hl_text = "創5日新高"
                elif latest_change <= margin_change.rolling(5).min().iloc[-1]: margin_hl_text = "創5日新低"
            
            # 3. 計算融資餘額變化均線趨勢
            ma_text = ""
            if len(margin_change) >= 20:
                ma5 = margin_change.rolling(5).mean().iloc[-1]
                ma20 = margin_change.rolling(20).mean().iloc[-1]
                # doc: --- [核心修正] 將 '≤' 換成 '<=' ---
                ma_text = "MA5>20▲" if ma5 > ma20 else "MA5<=20▼"
            
            # --- 2. 繪製圖表 ---
            info_line = f"{corr_text} | 融資變化: {margin_hl_text or '盤整'} ({ma_text})"
            ax.set_title(f'風險指標 vs. 融資餘額變化 {date_title}\n{info_line}', fontsize=16, color=self.dark_color, weight='bold')

            # --- 座標軸層 1 (ax): 左側Y軸 - 波動率 (可見) ---
            line1, = ax.plot(volatility.index, volatility, label='20日滾動波動率', color='#8338ec', linewidth=2.5, zorder=3)
            ax.set_ylabel('二\n十\n日\n滾\n動\n波\n動\n率', rotation=0, labelpad=20, color='#8338ec', fontsize=14)
            ax.tick_params(axis='y', labelcolor='#8338ec')
            ax.yaxis.set_major_formatter(FuncFormatter('{:.2%}'.format))
            ax.grid(True, linestyle='--', alpha=0.6, zorder=0)

            # --- 座標軸層 2 (ax2): 右側Y軸 - 最大回撤 (可見) ---
            ax2 = ax.twinx()
            line2, = ax2.plot(max_drawdown.index, max_drawdown, label='20日滾動最大回撤', color='#ff9f1c', linewidth=2, zorder=4)
            ax2.set_ylabel('二\n十\n日\n滾\n動\n最\n大\n回\n撤', rotation=0, labelpad=30, color='#ff9f1c', fontsize=14)
            ax2.tick_params(axis='y', labelcolor='#ff9f1c')
            ax2.yaxis.set_major_formatter(FuncFormatter('{:.2%}'.format))
            ax2.grid(False)

            # --- 座標軸層 3 (ax_bg): 背景 - 融資餘額變化 (隱藏) ---
            ax_bg = ax.twinx()
            ax_bg.bar(margin_change.index, margin_change, color=colors, width=1.0, alpha=0.9, label='融資餘額變化(張)', zorder=1)
            ax_bg.axhline(0, color='gray', linestyle='--', linewidth=1, zorder=2)
            ax_bg.set_frame_on(False)
            ax_bg.tick_params(left=False, labelleft=False, right=False, labelright=False)
            ax_bg.set_ylabel('')

            # --- 3. 整合圖例 ---
            patch_increase = Patch(color='#EF5350', alpha=0.5, label='融資增加')
            patch_decrease = Patch(color='#26A69A', alpha=0.5, label='融資減少')
            
            handles1 = [line1, patch_increase, patch_decrease]
            handles2 = [line2]
            ax.legend(handles=handles1, loc='upper left', fontsize=12)
            ax2.legend(handles=handles2, loc='upper right', fontsize=12)
        
        except Exception as e:
            ax.set_title(f'風險指標 vs. 融資餘額變化', fontsize=18, color=self.dark_color, weight='bold')
            ax.text(0.5, 0.5, f"無法繪製風險指標圖:\n{e}", ha='center', va='center',
                    transform=ax.transAxes, fontsize=14, color='red', wrap=True)
            # 使用 stock_id，假設它能從某處取得。如果不行，這行需要調整。
            # logging.error(f"為 {stock_id} 繪製風險指標圖時出錯: {e}")
        
        
        

    def _plot_turnover(self, ax: plt.Axes, price: pd.Series, turnover: pd.Series, turnover_ma5: pd.Series, turnover_ma20: pd.Series):
        # --- [v7.2 新增] 獲取日期區間 ---
        start_date = price.index.min().strftime('%Y-%m-%d')
        end_date = price.index.max().strftime('%Y-%m-%d')
        date_title = f'({start_date} ~ {end_date})'
        
        ax.set_title(f'交易熱度分析：周轉率與價量關係 {date_title}', fontsize=18, color=self.dark_color, weight='bold')
        stable_chips_zone = (price > price.rolling(10).mean()) & (turnover_ma5.diff() < 0)
        stable_chips_zone = stable_chips_zone.rolling(3, min_periods=1).mean() >= (2/3)
        blow_off_dates = price[price == price.rolling(20).max()].index.intersection(turnover[turnover > turnover_ma20 * 2].index)
        ax.grid(True, linestyle='--', alpha=0.6, zorder=1)
        ax.plot(price.index, price, color='#0077b6', linewidth=2.5, label='股價', zorder=3)
        ax.set_ylabel('股\n價', rotation=0, labelpad=20, color='#0077b6', fontsize=14, va='center')
        ax2 = ax.twinx()
        ax2.fill_between(turnover_ma5.index, turnover_ma5, 0, color='#f7a072', alpha=0.5, label='5日平均周轉率', zorder=2)
        ax2.set_ylabel('5\n日\n均\n周\n轉\n率\n', rotation=0, labelpad=35, color='#f7a072', fontsize=14, va='center')
        ax2.yaxis.set_major_formatter(FuncFormatter('{:.2%}'.format))
        ymin, ymax = ax.get_ylim()
        ax.fill_between(stable_chips_zone.index, ymin, ymax, where=stable_chips_zone, facecolor="#2b5c02", alpha=0.7, zorder=2, label='籌碼穩定區')
        if not blow_off_dates.empty:
            ax.plot(blow_off_dates, price.loc[blow_off_dates], 'v', color='red', markersize=10, label='高檔爆量警示', zorder=4)
        handles, labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(handles + handles2, labels + labels2, loc='upper left', fontsize=12)
        plt.setp(ax.get_xticklabels(), visible=False)

    def _plot_retail_investor_flow(self, ax: plt.Axes, stock_id: str, price_series: pd.Series):
        try:
            stock_inv = self.inventory_weekly_data[self.inventory_weekly_data['stock_id'] == stock_id].copy()
            if stock_inv.empty: raise ValueError("查無此股票的集保資料")
            stock_inv['date'] = pd.to_datetime(stock_inv['date'])
            recent_10_weeks = sorted(stock_inv['date'].unique())[-10:]

            # --- [v7.2 新增] 獲取日期區間 ---
            start_date = recent_10_weeks[0].strftime('%Y-%m-%d')
            end_date = recent_10_weeks[-1].strftime('%Y-%m-%d')
            date_title = f'({start_date} ~ {end_date})'
            
            ax.set_title(f'散戶籌碼堆疊流向圖 {date_title}', fontsize=18, color=self.dark_color, weight='bold')

            retail_level_map = {'1-10張': list(range(1, 4)), '10-30張': list(range(4, 7)), '30-50張': list(range(7, 9))}
            retail_colors = ['#8da0cb', '#66c2a5', '#b3b3b3']
            weekly_changes = {}
            for label, levels in retail_level_map.items():
                tier_data = stock_inv[stock_inv['持股分級'].astype(int).isin(levels)]
                weekly_sum = tier_data.groupby('date')['持有股數'].sum() / 1000
                weekly_changes[label] = weekly_sum.diff()
            df_change = pd.DataFrame(weekly_changes).loc[recent_10_weeks].fillna(0)
            if df_change.empty: raise ValueError("計算散戶週變化量後無數據")
            df_change.plot(kind='bar', stacked=True, ax=ax, width=0.7, color=retail_colors, legend=False, zorder=3)
            ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
            ax.set_ylabel("週\n變\n化\n總\n張\n數", rotation=0, labelpad=20, fontsize=14, color=self.dark_color)
            handles = [Patch(color=color, label=label) for color, label in zip(retail_colors, df_change.columns)]
            ax.legend(handles=handles, title="散戶分級", loc='upper left')
            ax2 = ax.twinx()
            net_weekly_flow = df_change.sum(axis=1)
            cumulative_flow = net_weekly_flow.cumsum()
            weekly_price = price_series.reindex(df_change.index, method='ffill')
            def min_max_scaler(series):
                min_val, max_val = series.min(), series.max()
                return pd.Series(0.5, index=series.index) if max_val - min_val == 0 else (series - min_val) / (max_val - min_val)
            if not weekly_price.isnull().all() and not cumulative_flow.isnull().all():
                p1, = ax2.plot(ax.get_xticks(), min_max_scaler(cumulative_flow), color='#008080', label='散戶累積淨流向(標準化)', marker='o', markersize=5, lw=2.5)
                p2, = ax2.plot(ax.get_xticks(), min_max_scaler(weekly_price), color='orange', label='股價(標準化)', marker='.', ls='--', lw=2.0)
                ax2.set_ylabel("標\n準\n化\n數\n值", rotation=0, labelpad=20, fontsize=14, color=self.dark_color)
                ax2.legend(handles=[p1, p2], loc='upper right')
            plt.setp(ax.get_xticklabels(), visible=False)
        except Exception as e:
            ax.text(0.5, 0.5, f"無法繪製散戶籌碼圖:\n{e}", ha='center', va='center', transform=ax.transAxes, fontsize=14, color='red')
            logging.error(f"為 {stock_id} 繪製散戶籌碼圖時出錯: {e}")


    # finlab程式碼格式
    def _plot_total_shareholders_chart(self, ax: plt.Axes, stock_id: str, price_series: pd.Series):
        """
        doc:
        [v7.2 使用者需求修改] 在圖表標題中新增日期區間。
        [v2.6 座標軸優化版]
        - [方法論升級] 根據使用者建議，所有籌碼分析的底層邏輯從「人數變化」改為「'占集保庫存數比例'% 的變化」，以資金進行加權，更精確地追蹤籌碼流向。
        - [動能歸因升級] 「主力動能」現在會找出持股比例變化最大的級距，而非人數變化最大的級距。
        - [文字表達升級] 標題文字同步更新，以百分比(%)呈現，更貼近資金流動的事實。
        - [視覺優化] 動態調整總股東人數圖的Y軸上下限，使其能凸顯近期的細微變化，解決因尺度過大而無法觀測的問題。
        """
        import matplotlib.colors as mcolors

        # --- 內部輔助函式：分析資金權重的籌碼變化 (v2.5 升級版) ---
        def get_capital_flow_analysis_text(stock_inv_df: pd.DataFrame) -> str:
            """分析資金權重的籌碼動向，回傳包含1週與6週分析的文字。"""
            try:
                # 定義級距與標籤
                large_tiers = list(range(11, 16))
                retail_tiers = list(range(1, 9))
                tier_labels = {
                    1: "1-999股", 2: "1-5張", 3: "5-10張", 4: "10-15張", 5: "15-20張", 6: "20-30張",
                    7: "30-40張", 8: "40-50張", 9: "50-100張", 10: "100-200張", 11: "200-400張",
                    12: "400-600張", 13: "600-800張", 14: "800-1000張", 15: "1000張以上"
                }

                # --- 數據計算：核心改為使用 '占集保庫存數比例' ---
                pivoted_pct = stock_inv_df.pivot(
                    index='date', columns='持股分級', values='占集保庫存數比例')

                large_holdings_pct = pivoted_pct[large_tiers].sum(axis=1)
                retail_holdings_pct = pivoted_pct[retail_tiers].sum(axis=1)

                # --- 1. 近1週分析 ---
                summary_text = "趨勢分析中..."
                top_large_text = "大戶主力: 分析中..."
                top_retail_text = "散戶主力: 分析中..."

                # 總體趨勢分析
                l_chg_1w = large_holdings_pct.diff(
                    1).iloc[-1] if len(large_holdings_pct) > 1 else 0
                r_chg_1w = retail_holdings_pct.diff(
                    1).iloc[-1] if len(retail_holdings_pct) > 1 else 0

                if l_chg_1w > 0 and r_chg_1w < 0:
                    summary_text = f"大戶增持、散戶減持 ({'大戶主導' if abs(l_chg_1w) > abs(r_chg_1w) else '散戶主導'})"
                elif l_chg_1w < 0 and r_chg_1w > 0:
                    summary_text = f"大戶減持、散戶增持 ({'大戶主導' if abs(l_chg_1w) > abs(r_chg_1w) else '散戶主導'})"
                elif l_chg_1w > 0 and r_chg_1w > 0:
                    summary_text = f"同步增持 (以{'大戶' if abs(l_chg_1w) > abs(r_chg_1w) else '散戶'}為主)"
                elif l_chg_1w < 0 and r_chg_1w < 0:
                    summary_text = f"同步減持 (以{'大戶' if abs(l_chg_1w) > abs(r_chg_1w) else '散戶'}為主)"
                else:
                    summary_text = "趨勢不明"

                # 精準動能歸因
                tier_pct_changes = pivoted_pct.diff().dropna(how='all')
                if not tier_pct_changes.empty:
                    last_pct_changes = tier_pct_changes.iloc[-1]
                    large_chg_subset = last_pct_changes.reindex(
                        large_tiers).dropna()
                    if not large_chg_subset.empty and not all(large_chg_subset == 0):
                        tier = large_chg_subset.abs().idxmax()
                        val = large_chg_subset.loc[tier]
                        top_large_text = f"大戶主力: {tier_labels.get(tier, '')} {'增持' if val > 0 else '減持'} {abs(val):.2f}%"
                    retail_chg_subset = last_pct_changes.reindex(
                        retail_tiers).dropna()
                    if not retail_chg_subset.empty and not all(retail_chg_subset == 0):
                        tier = retail_chg_subset.abs().idxmax()
                        val = retail_chg_subset.loc[tier]
                        top_retail_text = f"散戶主力: {tier_labels.get(tier, '')} {'增持' if val > 0 else '減持'} {abs(val):.2f}%"

                # --- 2. 近6週分析 ---
                l_trend_6w_str = "大戶累計: 分析中..."
                r_trend_6w_str = "散戶累計: 分析中..."
                if len(large_holdings_pct) >= 7:
                    l_chg_6w = large_holdings_pct.diff(6).iloc[-1]
                    l_trend_6w_str = f"大戶累計{'增持' if l_chg_6w > 0 else '減持'} ({l_chg_6w:+.2f}%)"
                if len(retail_holdings_pct) >= 7:
                    r_chg_6w = retail_holdings_pct.diff(6).iloc[-1]
                    r_trend_6w_str = f"散戶累計{'增持' if r_chg_6w > 0 else '減持'} ({r_chg_6w:+.2f}%)"

                return (f"近1週資金流向: {summary_text}\n"
                        f" > {top_large_text} | {top_retail_text}\n"
                        f"近6週資金趨勢: {l_trend_6w_str} | {r_trend_6w_str}")

            except Exception as e:
                return f"資金流向分析失敗: {e}"

        # --- 主函式流程 ---
        reason_text = ""
        try:
            # --- 數據準備 ---
            stock_data_full = self.inventory_weekly_data[self.inventory_weekly_data['stock_id'] == stock_id].copy()
            if stock_data_full.empty:
                raise ValueError("查無此股票的集保資料")
            stock_data_full['持股分級'] = pd.to_numeric(
                stock_data_full['持股分級'], errors='coerce')
            stock_data_full['人數'] = pd.to_numeric(
                stock_data_full['人數'], errors='coerce')
            stock_data_full['占集保庫存數比例'] = pd.to_numeric(
                stock_data_full['占集保庫存數比例'], errors='coerce')  # 確保比例是數值
            stock_data_full['date'] = pd.to_datetime(stock_data_full['date'])
            stock_data_full.dropna(
                subset=['date', '持股分級', '人數', '占集保庫存數比例'], inplace=True)
            if stock_data_full.empty:
                raise ValueError("數據型別轉換後無有效資料")

            reason_text = get_capital_flow_analysis_text(stock_data_full)

            # 總股東人數的計算與繪圖邏輯維持不變...
            total_sh_data = stock_data_full[stock_data_full['持股分級'] == 17]
            if total_sh_data.empty:
                all_tiers_data = stock_data_full[stock_data_full['持股分級'] <= 15]
                if all_tiers_data.empty:
                    raise ValueError("查無任何有效分級(1-15)資料")
                shareholder_count = all_tiers_data.groupby('date')['人數'].sum()
            else:
                shareholder_count = total_sh_data.set_index(
                    'date').sort_index()['人數']

            shareholder_count = shareholder_count.tail(10)
            if shareholder_count.empty:
                raise ValueError("取得最近10週數據後為空")

            # --- [v7.2 新增] 獲取日期區間 ---
            start_date = shareholder_count.index.min().strftime('%Y-%m-%d')
            end_date = shareholder_count.index.max().strftime('%Y-%m-%d')
            date_title = f'({start_date} ~ {end_date})'

            weekly_price = price_series.resample(
                'W-FRI').last().reindex(shareholder_count.index, method='ffill')
            if weekly_price.isnull().all():
                raise ValueError("股價與股東人數日期無法對齊")

            # --- 繪圖區 ---
            ax.set_title(f'總股東人數 vs. 股價趨勢 {date_title}\n{reason_text}', fontsize=14,
                         color=self.dark_color, weight='bold', loc='left', wrap=True)

            change = shareholder_count.diff()
            colors, pos_changes, neg_changes = [], change[change > 0], change[change < 0].abs()
            norm_pos = mcolors.Normalize(vmin=pos_changes.min(), vmax=pos_changes.max(
            )) if not pos_changes.empty else mcolors.Normalize(0, 1)
            norm_neg = mcolors.Normalize(vmin=neg_changes.min(), vmax=neg_changes.max(
            )) if not neg_changes.empty else mcolors.Normalize(0, 1)
            cmap_green = mcolors.LinearSegmentedColormap.from_list(
                "custom_green", ["#a9d6a9", "#1e8449"])
            cmap_red = mcolors.LinearSegmentedColormap.from_list(
                "custom_red", ["#f5b7b1", "#c0392b"])
            for val in change:
                if pd.isna(val):
                    colors.append('#cccccc')
                elif val > 0:
                    colors.append(cmap_green(norm_pos(val)))
                elif val < 0:
                    colors.append(cmap_red(norm_neg(abs(val))))
                else:
                    colors.append('#cccccc')
            x_ticks = np.arange(len(shareholder_count))
            ax.bar(x_ticks, shareholder_count.values,
                   color=colors, width=0.8, zorder=2)

            # --- Y軸與圖例等設定 ---
            ax.set_ylabel('總\n股\n東\n人\n數', color='#006d77',
                          fontsize=14, rotation=0, labelpad=30)
            ax.tick_params(axis='y', labelcolor='#006d77')
            ax.yaxis.set_major_formatter(plt.FuncFormatter(
                lambda x, p: f'{int(x/1000)}k' if x > 1000 else f'{int(x)}'))

            # --- [核心修改] ---
            # 原本的程式碼: ax.set_ylim(top=shareholder_count.max() / 0.75, bottom=0)
            # 新的程式碼:
            if not shareholder_count.empty and shareholder_count.max() > 0 and shareholder_count.min() > 0:
                y_min = shareholder_count.min()
                y_max = shareholder_count.max()
                padding = (y_max - y_min) * 0.1  # 上下增加 10% 的緩衝
                ax.set_ylim(bottom=y_min - padding, top=y_max + padding)
            # --- [修改結束] ---

            ax2 = ax.twinx()
            ax2.plot(x_ticks, weekly_price.values, color='#e76f51',
                     label='股價 (週)', ls='--', lw=2.5, marker='o', markersize=5, zorder=3)
            ax2.set_ylabel('股\n價', color='#e76f51',
                           fontsize=14, rotation=0, labelpad=20)
            ax2.tick_params(axis='y', labelcolor='#e76f51')
            if not weekly_price.empty and weekly_price.min() > 0:
                ax2.set_ylim(bottom=weekly_price.min() * 0.95)
            ax.grid(True, linestyle='--', alpha=0.5, axis='y', zorder=1)
            ax2.legend(loc='upper right')
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(shareholder_count.index.strftime(
                '%Y-%m-%d'), rotation=30, ha='right')
            ax.set_xlabel('日期', fontsize=14, color=self.dark_color)
            plt.setp(ax.get_xticklabels(), visible=True)

        except Exception as e:
            ax.set_title(f'總股東人數 vs. 股價趨勢 (最近10週)', fontsize=18,
                         color=self.dark_color, weight='bold')
            ax.text(0.5, 0.5, f"無法繪製總股東人數圖:\n{e}", ha='center', va='center',
                    transform=ax.transAxes, fontsize=14, color='red', wrap=True)
            logging.error(f"為 {stock_id} 繪製總股東人數圖時出錯: {e}")





    # finlab程式碼格式
# [Gemini v3.9 最終完美版] 請用此版本完整替換掉您現有的 generate_chart 函式

    def generate_chart(self, stock_id: str, period: int = 120) -> plt.figure:
        """
        doc:
        [v3.9 Gemini 最終完美版]
        - 根據使用者回饋，將 subplots_adjust 的 wspace 參數從 0.15 增加至 0.22，
          以拉開左右兩欄子圖的水平間距，使版面更清晰。
        [v3.8 Gemini 標題微調]
        - 將 top 參數微調至 0.95，增加主標題與下方子圖的垂直間距。
        [v3.6 Gemini 手動佈局版]
        - 放棄自動排版，改用手動 fig.subplots_adjust() 精確控制，消除多餘空白。
        """
        # --- 1. 資料準備與計算 (維持不變) ---
        try:
            required_dfs = {
                'close': self.close_price, 'buy_vol': self.buy_vol, 'sell_vol': self.sell_vol,
                'volume': self.volume, 'market_cap': self.market_cap, 'margin_balance': self.margin_balance
            }
            all_indices = [df[stock_id].dropna().index for name, df in required_dfs.items() if stock_id in df.columns and not df[stock_id].dropna().empty]
            if len(all_indices) < len(required_dfs):
                logging.warning(f"股票 {stock_id} 缺少必要的數據欄位，跳過券商圖產生。")
                return None
            
            common_index = all_indices[0]
            for idx in all_indices[1:]:
                common_index = common_index.intersection(idx)

            if len(common_index) < period:
                logging.warning(f"股票 {stock_id} 的共同數據天數 ({len(common_index)}) 少于指定的 {period} 天，跳過券商圖產生。")
                return None
            
            end_date = common_index[-1]
            start_date = common_index[-period - 20] if len(common_index) > (period + 20) else common_index[0]
            price = self.close_price.loc[start_date:end_date, stock_id].dropna()
            
            if price.empty or len(price) < 20:
                logging.warning(f"股票 {stock_id} 在 {start_date.date()} 到 {end_date.date()} 區間內價格數據不足，跳過券商圖產生。")
                return None
            
            final_common_index = price.index
            net_buy = (self.buy_vol.loc[final_common_index, stock_id] - self.sell_vol.loc[final_common_index, stock_id]).dropna()
            cvd = net_buy.cumsum()
            daily_return = price.pct_change()
            volatility = daily_return.rolling(window=20).std().dropna()
            max_drawdown = self._calculate_rolling_mdd(price, window=20).dropna()
            vol = self.volume.loc[final_common_index, stock_id].dropna()
            cap = self.market_cap.loc[final_common_index, stock_id].dropna()
            margin_series = self.margin_balance.loc[final_common_index, stock_id].dropna()

            aligned_vol, aligned_so_price = vol.align(price, join='inner')
            aligned_cap, _ = cap.align(aligned_so_price, join='inner')

            if (aligned_so_price == 0).any():
                turnover = pd.Series(dtype=float)
            else:
                shares_outstanding = (aligned_cap / aligned_so_price)
                if (shares_outstanding == 0).any():
                    turnover = pd.Series(dtype=float)
                else:
                    turnover = (aligned_vol / shares_outstanding * 100).dropna()

            turnover_ma5 = turnover.rolling(5).mean()
            turnover_ma20 = turnover.rolling(20).mean()
            stock_name = self.company_info.loc[stock_id, '公司簡稱'] if stock_id in self.company_info.index else stock_id
            latest_date_str = end_date.strftime('%Y-%m-%d')
        except Exception as e:
            logging.error(f"為 {stock_id} 準備券商圖數據時出錯: {e}", exc_info=True)
            return None

        # --- 2. 手動佈局設定 ---
        plt.style.use('seaborn-v0_8-whitegrid')
        plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
        plt.rcParams['axes.unicode_minus'] = False
        
        fig = plt.figure(figsize=(28, 38))
        fig.suptitle(f'{stock_id} {stock_name} - 綜合分析圖 (資料截至: {latest_date_str})', fontsize=28, weight='bold', color=self.dark_color)
        
        gs = gridspec.GridSpec(7, 2, figure=fig)
        
        # --- 3. 建立子圖 (維持不變) ---
        ax_price = fig.add_subplot(gs[0:2, 0])
        ax_turnover = fig.add_subplot(gs[2:4, 0], sharex=ax_price)
        ax_risk = fig.add_subplot(gs[4:6, 0], sharex=ax_price)
        
        ax_shareholder = fig.add_subplot(gs[0:2, 1])
        ax_retail = fig.add_subplot(gs[2:4, 1], sharex=ax_shareholder)
        ax_total_sh = fig.add_subplot(gs[4:6, 1], sharex=ax_shareholder)

        ax_dist_overlay = fig.add_subplot(gs[6, :])

        # --- 4. 繪製圖表 (維持不變) ---
        self._plot_price(ax_price, price, net_buy.tail(period), cvd.tail(period), period)        
        self._plot_turnover(ax_turnover, price.tail(period), turnover.tail(period), turnover_ma5.tail(period), turnover_ma20.tail(period))
        self._plot_risk(ax_risk, volatility.tail(period), max_drawdown.tail(period), margin_series.tail(period))
        
        self._plot_shareholder_flow(ax_shareholder, stock_id, price)
        self._plot_retail_investor_flow(ax_retail, stock_id, price)
        self._plot_total_shareholders_chart(ax_total_sh, stock_id, price)
        
        self._plot_shareholder_distribution_overlay(ax_dist_overlay, stock_id, period_weeks=48)

        # --- 5. 手動調整間距與邊界 ---
        plt.setp(ax_price.get_xticklabels(), visible=False)
        plt.setp(ax_turnover.get_xticklabels(), visible=False)
        plt.setp(ax_shareholder.get_xticklabels(), visible=False)
        plt.setp(ax_retail.get_xticklabels(), visible=False)
        
        plt.setp(ax_risk.get_xticklabels(), visible=True)
        plt.setp(ax_total_sh.get_xticklabels(), visible=True)
        
        # [v3.9 核心修改] 將 wspace 從 0.15 增加至 0.22，拉開左右子圖間距
        fig.subplots_adjust(left=0.05, right=0.95, bottom=0.04, top=0.95, wspace=0.22, hspace=0.3)
        
        return fig


        



class ReportGenerator:
    """報告生成器"""

 
    
    # finlab程式碼格式
    # [v2.8 錯誤修正版]
    def __init__(self, analyzer, stock_selection_file, stock_topics_file, master_summary_df,
             strong_trend_stocks, rank_jump_stocks_df, market_breadth_data, amt_df,
             macro_margin_data, broker_chart_generator, config, image_dir, disposal_df, cash_increase_df,treasury_stock_df,
              # doc: --- [核心修改 1/3] 在此處新增 short_squeeze_candidate_df 參數 ---
             short_squeeze_candidate_df=None,
             investor_conference_df=None,
             strategy_stocks_set=None,
             etf_consensus_df=None,
             etf_heavyweight_df=None,
             etf_monthly_heavyweight_df=None,
             etf_quadrant_data=None,
             etf_focus_stock_ids=None,
             etf_latest_quarter_date=None,
             etf_latest_month_date=None,
             etf_monthly_quadrant_data=None, 
             margin_trap_df=None, trap_dates=None, 
             daily_etf_holdings_map=None,
             daily_etf_rank_map=None,
             special_signals_df=None,
             price_change_top25_df=None, # [新增] 新增此參數
             
             **kwargs):
        """
        doc:
        初始化報告生成器。
        這個建構函式現在接收所有必要的數據和配置，包括從預處理步驟中得到的數據。

        Args:
            analyzer (StockAnalyzer): 已經預載入所有股票數據的分析器實例。
            stock_selection_file (str): 策略選股清單的 Excel 檔案路徑。
            stock_topics_file (str): 股票主題對應的 Excel 檔案路徑。
            full_industry_df (pd.DataFrame): 包含全市場或目標股票的完整產業數據 DataFrame。
            strong_trend_stocks (list): 從初步分析中篩選出的強勢趨勢股清單。
            rank_jump_stocks_df (pd.DataFrame): 從排名躍升分析中篩選出的股票 DataFrame。
        """

        self.analyzer = analyzer
        self.params = analyzer.params
        self.stock_selection_file = stock_selection_file
        self.stock_topics_file = stock_topics_file
        self.result_df = self._process_stock_list()
        self._setup_plot_style()
        self.industry_df = master_summary_df
        self.master_summary_df = master_summary_df # [新增] 將完整的總表也存起來
        self.strong_trend_stocks = strong_trend_stocks if strong_trend_stocks else []
        self.rank_jump_stocks_df = rank_jump_stocks_df if rank_jump_stocks_df is not None else pd.DataFrame()
        self.market_breadth_data = market_breadth_data
        self.amt_df = amt_df
        self.macro_margin_data = macro_margin_data
        self.margin_trap_df = margin_trap_df if margin_trap_df is not None else pd.DataFrame()
        self.trap_start_date, self.trap_end_date = trap_dates if trap_dates is not None else ("", "")
        self.broker_chart_generator = broker_chart_generator
        self.config = config
        self.image_dir = image_dir
        self.strategy_stocks_set = strategy_stocks_set if strategy_stocks_set is not None else set()
        self.etf_consensus_df = etf_consensus_df
        self.etf_heavyweight_df = etf_heavyweight_df
        self.etf_monthly_heavyweight_df = etf_monthly_heavyweight_df
        self.etf_quadrant_data = etf_quadrant_data
        self.etf_focus_stock_ids = etf_focus_stock_ids if etf_focus_stock_ids is not None else []
        self.etf_latest_quarter_date = etf_latest_quarter_date
        self.etf_latest_month_date = etf_latest_month_date
        self.etf_monthly_quadrant_data = etf_monthly_quadrant_data # [v23.0 新增]
         # --- [v28.3 新增屬性] ---
        self.daily_etf_holdings_map = daily_etf_holdings_map if daily_etf_holdings_map is not None else {}
        self.daily_etf_rank_map = daily_etf_rank_map if daily_etf_rank_map is not None else {}
        
        #doc: [新增] 將傳入的 DataFrame 存為實例屬性
        self.special_signals_df = special_signals_df if special_signals_df is not None else pd.DataFrame()
        self.price_change_top25_df = price_change_top25_df if price_change_top25_df is not None else pd.DataFrame() # [新增] 將傳入的 df 存為實例屬性
        
        # [ULTRA THINK 新增] 初始化一個空字典，用來儲存每日ETF的狀態分析結果
        self.daily_etf_status_map = {}

        self.plotted_stock_ids = set()
        self.etf_focus_stock_ids_set = set(etf_focus_stock_ids) if etf_focus_stock_ids is not None else set()
        self.new_high_200_set = kwargs.get('new_high_200_set', set())
        self.new_high_20_set = kwargs.get('new_high_20_set', set())
        self.strong_trend_set = kwargs.get('strong_trend_set', set())
        self.rank_jump_set = kwargs.get('rank_jump_set', set())
        self.amt_rank_series = kwargs.get('amt_rank_series', pd.Series(dtype=float))
         # doc: --- [Gemini 新增] 儲存處置股與現金增資股資訊 ---
        # 將 DataFrame 的索引設為 stock_id，可以大幅提升後續查詢效率
        self.disposal_stocks_df = disposal_df.set_index('stock_id') if disposal_df is not None and not disposal_df.empty else pd.DataFrame()
        self.cash_increase_df = cash_increase_df.set_index('stock_id') if cash_increase_df is not None and not cash_increase_df.empty else pd.DataFrame()
        # doc: --- [Gemini 新增] 儲存庫藏股資訊 ---
        self.treasury_stock_df = treasury_stock_df.set_index('stock_id') if treasury_stock_df is not None and not treasury_stock_df.empty else pd.DataFrame()
        # doc: --- [新增結束] ---
        # doc: --- [Gemini 新增] 將傳入的法說會 DataFrame 存為實例屬性 ---
        self.investor_conference_df = investor_conference_df if investor_conference_df is not None else pd.DataFrame()
        # doc: --- [核心修改 2/3] 將新的 DataFrame 存為實例屬性，並以 stock_id 為索引 ---
        self.short_squeeze_candidate_df = short_squeeze_candidate_df.set_index('stock_id') if short_squeeze_candidate_df is not None and not short_squeeze_candidate_df.empty else pd.DataFrame()
        # doc: --- [修改結束] ---
        
      
        # doc: =============================================================================
        # doc: --- [核心錯誤修正] 屬性指派 ---
        # doc: 從傳入的 market_breadth_data 字典中，提取創高股清單 DataFrame，
        # doc: 並將它們指派給 self.new_high_stocks_df 和 self.new_high_20_stocks_df。
        # doc: 這樣後續的繪圖函式才能正確地找到這些屬性。
        # doc: =============================================================================
        if self.market_breadth_data and isinstance(self.market_breadth_data, dict) and 'latest_new_high_list' in self.market_breadth_data:
            self.new_high_stocks_df = self.market_breadth_data['latest_new_high_list']
        else:
            self.new_high_stocks_df = pd.DataFrame()

        if self.market_breadth_data and isinstance(self.market_breadth_data, dict) and 'latest_new_high_20_list' in self.market_breadth_data:
            self.new_high_20_stocks_df = self.market_breadth_data['latest_new_high_20_list']
        else:
            self.new_high_20_stocks_df = pd.DataFrame()
        # --- [修正結束] ---
        
        # finlab程式碼格式
        # doc: =============================================================================
        # doc: --- [v2.9 核心錯誤修正] 新增 'rank' 欄位 ---
        # doc: 利用已傳入的 self.amt_rank_series，為創高股 DataFrame 補上成交額排名欄位，
        # doc: 並依成交金額排序，以解決附錄繪圖時發生的 KeyError。
        # doc: =============================================================================
        if not self.new_high_stocks_df.empty and 'stock_id' in self.new_high_stocks_df.columns:
            self.new_high_stocks_df['rank'] = self.new_high_stocks_df['stock_id'].map(self.amt_rank_series).fillna(0).astype(int)
            self.new_high_stocks_df.sort_values(by='成交金額', ascending=False, inplace=True)
        
        if not self.new_high_20_stocks_df.empty and 'stock_id' in self.new_high_20_stocks_df.columns:
            self.new_high_20_stocks_df['rank'] = self.new_high_20_stocks_df['stock_id'].map(self.amt_rank_series).fillna(0).astype(int)
            self.new_high_20_stocks_df.sort_values(by='成交金額', ascending=False, inplace=True)
        # --- [修正結束] ---
        
        
        
    
    # finlab程式碼格式
    # doc: ===================================================================
    # doc: --- [Gemini 新增 v6.0] 判斷「營收動能轉強」輔助函式 ---
    # doc: ===================================================================
    def _has_strong_revenue_momentum(self, stock_id: str) -> bool:
        """
        doc:
        判斷一檔股票是否滿足「近3月平均營收 > 近12月平均營收」的條件。

        Args:
            stock_id (str): 股票代碼。

        Returns:
            bool: 如果滿足營收動能轉強條件，則回傳 True。
        """
        try:
            # 從預加載的數據中獲取月營收資料
            revenue = self.analyzer.all_data.get('monthly_revenue:當月營收', pd.DataFrame()).get(stock_id)
            
            # 確保有足夠的數據進行計算
            if revenue is None or len(revenue.dropna()) < 12:
                return False

            # 計算近3月與近12月的移動平均營收
            mean_3m = revenue.rolling(3).mean().iloc[-1]
            mean_12m = revenue.rolling(12).mean().iloc[-1]

            # 檢查數據是否有效 (非 NaN 或 0)
            if pd.isna(mean_3m) or pd.isna(mean_12m) or mean_12m == 0:
                return False

            return mean_3m > mean_12m
        except Exception:
            # 任何錯誤都視為不滿足條件
            return False    
        
        
    # finlab程式碼格式
    # doc: ===================================================================
    # doc: --- [Gemini 新增 v6.0] 通用型「優先權」篩選器函式 ---
    # doc: ===================================================================
    def _apply_priority_filtering(self, df: pd.DataFrame, priority_check_func: callable, max_regular_plots: int) -> pd.DataFrame:
        """
        doc:
        對 DataFrame 應用優先權篩選邏輯。

        Args:
            df (pd.DataFrame): 包含 'stock_id' 欄位的原始股票清單。
            priority_check_func (callable): 一個函式，接收 stock_id，回傳 bool 值 (True 代表優先)。
            max_regular_plots (int): 對於非優先的股票，最多保留的數量。

        Returns:
            pd.DataFrame: 經過優先權篩選與數量限制後的最終股票清單。
        """
        # 找出所有滿足優先權條件的股票
        priority_indices = [idx for idx, row in df.iterrows() if priority_check_func(row['stock_id'])]
        priority_stocks_df = df.loc[priority_indices]

        # 從原始清單中，排除掉優先股，剩下的就是一般股
        regular_stocks_df = df.drop(priority_indices)

        # 對一般股套用數量上限
        limited_regular_df = regular_stocks_df.head(max_regular_plots)

        # 合併「所有優先股」與「限量的一般股」
        final_df = pd.concat([priority_stocks_df, limited_regular_df])
        
        original_count = len(df)
        final_count = len(final_df)
        priority_count = len(priority_stocks_df)

        if final_count < original_count:
            logging.info(f"    ↳ 篩選與排序: {original_count}檔中, {priority_count}檔因滿足優先條件保留，其餘取前 {max_regular_plots} 檔，最終繪製 {final_count} 檔。")

        return final_df





    # finlab程式碼格式
    # doc: ===================================================================
    # doc: --- [Gemini 新增 v5.0] 判斷「隔日沖主力」警示訊號 ---
    # doc: ===================================================================
    def _get_scalper_warning_signal(self, stock_id: str, buy_threshold: float = 1000, sell_ratio: float = 0.8):
        """
        doc:
        根據昨日主力大買、今日主力大賣且開高走低的模式，判斷是否存在「隔日沖」賣壓。

        Args:
            stock_id (str): 股票代碼。
            buy_threshold (float): 昨日主力買超張數的最低門檻。
            sell_ratio (float): 今日主力賣超張數相對於昨日買超的最低比例。

        Returns:
            tuple[str, str]: 包含 (標籤文字, 標籤顏色) 的元組，若不符合則回傳 ("", "")。
        """
        try:
            top15_buy = self.analyzer.all_data.get('etl:broker_transactions:top15_buy', pd.DataFrame()).get(stock_id)
            top15_sell = self.analyzer.all_data.get('etl:broker_transactions:top15_sell', pd.DataFrame()).get(stock_id)
            open_price = self.analyzer.all_data.get('price:開盤價', pd.DataFrame()).get(stock_id)
            close_price = self.analyzer.all_data.get('price:收盤價', pd.DataFrame()).get(stock_id)
            
            if any(s is None or len(s) < 2 for s in [top15_buy, top15_sell, open_price, close_price]):
                return "", ""

            # 計算近兩日的主力淨買超 (張)
            net_buy = (top15_buy - top15_sell).tail(2) / 1000
            if len(net_buy) < 2:
                return "", ""
                
            yesterday_net_buy = net_buy.iloc[0]
            today_net_buy = net_buy.iloc[1]

            # 條件1: 昨日主力大買
            condition1 = yesterday_net_buy > buy_threshold
            
            # 條件2: 今日主力大賣 (賣超張數 > 昨日買超的 80%)
            condition2 = today_net_buy < 0 and abs(today_net_buy) >= (yesterday_net_buy * sell_ratio)
            
            # 條件3: 今日開高走低收黑K
            condition3 = close_price.iloc[-1] < open_price.iloc[-1]
            
            if condition1 and condition2 and condition3:
                sell_amount_str = f"{abs(today_net_buy):,.0f}"
                signal_text = f"[注意! 疑似隔日沖賣壓 ({sell_amount_str}張)]"
                signal_color = "#07AC2B" # 暗紅色
                return signal_text, signal_color
                
            return "", ""
        except Exception:
            return "", ""







    # finlab程式碼格式
    # doc: 將此新函式完整加入到 ReportGenerator class 內部

    def _plot_price_change_appendix_page(self, pdf):
        """
        doc: [Gemini 新增] 繪製漲幅排行 Top 25 的附錄頁面。
        """
        logging.info("--- 開始繪製【附錄：5日漲幅排行榜】詳細報告 ---\n")
        
        if self.price_change_top25_df.empty:
            logging.info("無漲幅排行資料可供分析，跳過此章節。")
            return

        # 套用通用的附錄篩選器 (例如：篩掉當日跌的股票)
        stocks_to_plot_df = self._apply_appendix_filter(self.price_change_top25_df)
        
        if stocks_to_plot_df.empty:
            logging.info("漲幅排行股經過篩選後無剩餘股票，跳過此章節。")
            return
            
        # 定義來源說明的格式化函式
        def formatter(row):
            rank = row.get('rank', 'N/A')
            change_pct = row.get('漲跌幅_5日(%)', 0.0)
            return [f"來源: 5日漲幅排行 #{rank} ({change_pct:+.2f}%)"]

        # 呼叫通用的附錄產生器
        self._plot_appendix_section(
            pdf=pdf,
            stocks_df=stocks_to_plot_df,
            title_text='5日漲幅排行榜',
            subtitle_text=f"全市場 5 日漲幅最高的前 {len(stocks_to_plot_df)} 檔個股",
            subtitle_color='purple',
            source_topic='5日漲幅排行',
            source_details_formatter=formatter,
            image_section_key='E7_APPENDIX_PRICE_CHANGE' # 對應 CONFIG 的鍵
        )




    # [ULTRA THINK 最終修正版 3/5] 附錄外層函式
    def _plot_daily_etf_appendix_wrapper(self, pdf, daily_etf_appendix_data):
        if not daily_etf_appendix_data or daily_etf_appendix_data.get('top_stocks_df', pd.DataFrame()).empty:
            logging.warning("無每日主動式 ETF 數據可供附錄生成，將跳過 E6 區塊。")
            return
        
        self._plot_daily_etf_appendix_page(
            pdf,
            daily_etf_appendix_data['latest_date_str'],
            daily_etf_appendix_data['top_stocks_df']
        )


        
        
    

    def _save_figure_if_enabled(self, fig, filename, stock_id):
        """
        [doc]
        如果設定檔中的 SAVE_CHARTS_AS_FILES 為 True，且股票代碼存在於策略清單中，
        則將圖表儲存為圖片。
        """
        if self.config.get('SAVE_CHARTS_AS_FILES', False) and self.image_dir and fig and (stock_id in self.strategy_stocks_set):
            safe_filename = "".join(c for c in filename if c.isalnum() or c in ('_','-','.')).rstrip()
            image_path = os.path.join(self.image_dir, f"{safe_filename}.png")
            try:
                fig.savefig(image_path, bbox_inches='tight', dpi=150)
                logging.info(f"    ↳ 已儲存圖片 (僅限策略股): {image_path}")
            except Exception as e:
                logging.error(f"儲存圖片 {image_path} 失敗: {e}")

        
        
        
    def dataframe_to_figure(df, title=""):
        """將 DataFrame 轉換為 Matplotlib Figure 物件"""
        fig, ax = plt.subplots(figsize=(11.69, 8.27)) # A4 landscape
        ax.axis('off')
        ax.axis('tight')
        table = ax.table(cellText=df.values, colLabels=df.columns, cellLoc='center', loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.2)
        ax.set_title(title, fontsize=16, pad=20)
        fig.tight_layout()
        return fig
    
    
    
    # finlab程式碼格式
    # [新增] 步驟一：請將此全新函式加入 ReportGenerator class 中
    def _plot_it_net_buy_volume(self, ax_volume, stock_id, days):
        """
        doc:
        [v31.0 新增] 繪製「投信買賣超」圖表。
        - 買超為紅色柱狀圖，賣超為綠色。
        - 包含 5日 與 20日 移動平均線。
        """
        try:
            it_buy_shares = self.analyzer.all_data.get('it_buy', pd.DataFrame()).get(stock_id)
            if it_buy_shares is None or it_buy_shares.empty:
                raise ValueError("查無投信買賣超數據")

            it_buy_lots = (it_buy_shares / 1000).tail(days) # 轉換為張
            
            # 準備繪圖數據
            dates = it_buy_lots.index
            colors = ['#EF5350' if x > 0 else '#26A69A' for x in it_buy_lots]
            
            # 繪製柱狀圖
            ax_volume.bar(dates, it_buy_lots, width=1.0, color=colors, alpha=0.8, label='投信買賣超(張)')
            
            # 繪製均線
            ma5 = it_buy_lots.rolling(5).mean()
            ma20 = it_buy_lots.rolling(20).mean()
            ax_volume.plot(dates, ma5, color='#3366CC', lw=1.2, label='5日均線')
            ax_volume.plot(dates, ma20, color='#FF6600', lw=1.2, label='20日均線')

            # 設定座標軸與圖例
            ax_volume.set_ylabel('投\n信\n買\n賣\n超', rotation=0, labelpad=25, ha='right', va='center', fontsize=14)
            ax_volume.legend(loc='upper left', fontsize=10, facecolor='white', framealpha=1.0)
            ax_volume.tick_params(axis='x', labelbottom=False)
            ax_volume.yaxis.set_major_formatter(FuncFormatter(self.format_volume))
            ax_volume.grid(True, axis='y', linestyle='--', alpha=0.6)
            ax_volume.axhline(0, color='black', lw=0.5, ls='--')

        except Exception as e:
            ax_volume.text(0.5, 0.5, f"無法繪製投信買賣超圖:\n{e}", ha='center', va='center', transform=ax_volume.transAxes)
            logging.warning(f"為 {stock_id} 繪製投信買賣超圖時出錯: {e}")
    
    
    
    
    
    
    # finlab程式碼格式
    def _plot_it_net_buy_ratio(self, ax_volume, stock_id, days):
        """
        doc:
        [v31.2 動態座標軸優化版]
        - [核心修改] 新增動態調整 Y 軸邏輯。在繪圖後，計算當前畫面中所有數據點 (柱狀圖、均線) 的最大與最小值，
          並加上 15% 的緩衝區間，以自動聚焦於近期變化，解決歷史極端值壓縮座標軸的問題。
        """
        try:
            it_buy_shares = self.analyzer.all_data.get('it_buy', pd.DataFrame()).get(stock_id)
            volume_shares = self.analyzer.all_data.get('volume', pd.DataFrame()).get(stock_id) * 1000
            
            if it_buy_shares is None or volume_shares is None or it_buy_shares.empty or volume_shares.empty:
                raise ValueError("查無投信買賣超或成交量數據")

            it_buy_ratio = (it_buy_shares / volume_shares.replace(0, np.nan) * 100).tail(days)
            it_buy_ratio.dropna(inplace=True)

            if it_buy_ratio.empty:
                raise ValueError("計算投信買賣佔比後無有效數據")
                
            dates = it_buy_ratio.index
            colors = ['#EF5350' if x > 0 else '#26A69A' for x in it_buy_ratio]
            
            ax_volume.bar(dates, it_buy_ratio, width=1.0, color=colors, alpha=0.8, label='投信買賣佔比(%)')
            
            ma5 = it_buy_ratio.rolling(5).mean()
            ma20 = it_buy_ratio.rolling(20).mean()
            ax_volume.plot(dates, ma5, color='#3366CC', lw=1.2, label='5日均線')
            ax_volume.plot(dates, ma20, color='#FF6600', lw=1.2, label='20日均線')

            ax_volume.set_ylabel('投\n信\n買\n賣\n佔\n比', rotation=0, labelpad=25, ha='right', va='center', fontsize=14)
            ax_volume.legend(loc='upper left', fontsize=10, facecolor='white', framealpha=1.0)
            ax_volume.tick_params(axis='x', labelbottom=False)
            ax_volume.yaxis.set_major_formatter(FuncFormatter('{:.2f}%'.format))
            ax_volume.grid(True, axis='y', linestyle='--', alpha=0.6)
            ax_volume.axhline(0, color='black', lw=0.5, ls='--')

            # doc: =============================================================================
            # doc: --- [核心修改] 動態調整 Y 軸，聚焦近期變化 ---
            # doc: =============================================================================
            # 1. 收集當前圖表上所有可見的數據點
            # --- [核心修正開始] ---
            visible_data = pd.concat([it_buy_ratio, ma5, ma20]).dropna()

            if not visible_data.empty:
                min_val = visible_data.min()
                max_val = visible_data.max()

                # 只有在數據有波動時才進行動態縮放
                if max_val > min_val:
                    padding = (max_val - min_val) * 0.15
                    ax_volume.set_ylim(min_val - padding, max_val + padding)
                else:
                    # 如果所有值都一樣（例如都是0），則給定一個小範圍以避免警告
                    ax_volume.set_ylim(min_val - 0.5, max_val + 0.5)
            # --- [核心修正結束] ---
                # doc: --- [修改結束] ---

        except Exception as e:
            ax_volume.text(0.5, 0.5, f"無法繪製投信買賣佔比圖:\n{e}", ha='center', va='center', transform=ax_volume.transAxes)
            logging.warning(f"為 {stock_id} 繪製投信買賣佔比圖時出錯: {e}")
    
    
    
    
    # finlab程式碼格式
    # [ULTRA THINK 新增] 市值趨勢分析輔助函式
    def _calculate_trend_status(self, stock_id, all_etf_data, window, thresholds):
        """
        doc: 使用線性迴歸分析單一股票在最近N筆資料中的總市值趨勢。
        """
        from scipy.stats import linregress
        
        stock_data = all_etf_data[all_etf_data['代號'] == stock_id]
        
        # 按日期匯總該股票的總市值
        market_cap_series = stock_data.groupby('日期')['market_cap'].sum()
        
        # 篩選出最近N個有數據的日期點
        recent_caps = market_cap_series.tail(window)
        
        # 處理特殊情況
        if len(recent_caps) == 0:
            return ""
        if recent_caps.iloc[-1] == 0 and len(recent_caps) > 1:
            return "[已出清]"
        if len(recent_caps) == 1 and market_cap_series.sum() == recent_caps.iloc[-1]:
            return "[新進榜]"
        if len(recent_caps) < 3: # 數據點少於3個，無法有效判斷趨勢，回傳持有
            return "[持有]"

        # 執行線性迴歸
        # x 軸是時間序列 (0, 1, 2, ...), y 軸是市值
        x = np.arange(len(recent_caps))
        y = recent_caps.values
        
        # 為了讓斜率不受市值大小影響，對市值進行標準化 (Normalization)
        y_normalized = (y - np.min(y)) / (np.max(y) - np.min(y)) if (np.max(y) - np.min(y)) > 0 else y - np.min(y)

        slope, _, _, _, _ = linregress(x, y_normalized)

        # 根據斜率決定狀態標籤
        if slope > thresholds.get('STRONG_INCREASE', 0.5):
            return "[強勢加碼]"
        elif slope > 0:
            return "[溫和加碼]"
        elif slope <= thresholds.get('STRONG_DECREASE', -0.5):
            return "[趨勢減碼]"
        elif slope < 0:
            return "[溫和減碼]"
        else:
            return "[趨勢盤整]"
    



    # finlab程式碼格式
    # [ULTRA THINK 市值趨勢分析最終版]
    def _analyze_and_plot_daily_etf_page(self, pdf):
        """
        doc: [ULTRA THINK v29.7 資訊串聯] 
        - [核心修正] 在計算完趨勢狀態後，將結果正確地存入 self.daily_etf_status_map，
          確保後續的 K 線圖標題函式 (_set_main_title) 可以讀取到這些狀態標籤。
        """
        # --- 步驟 1: 從 CONFIG 讀取所有需要的參數 ---
        top_n = self.config.get('DAILY_ETF_TOP_N', 60)
        factor_params = self.config.get('FACTOR_PARAMS', {})
        trend_window = factor_params.get('TREND_ANALYSIS_WINDOW', 5)
        trend_thresholds = factor_params.get('TREND_SLOPE_THRESHOLDS', {})

        logging.info(f"--- (A11) 開始分析【每日主動式ETF Top{top_n}焦點股】---")
        appendix_data_to_return = None
        try:
            # --- 步驟 2: 資料讀取、準備與市值計算 (維持不變) ---
            csv_filename = "all_etf_holdings.csv"
            etf_list_to_analyze = ['00981A', '00982A', '00980A', '00984A']
            if not os.path.exists(csv_filename):
                raise FileNotFoundError(f"找不到 ETF 持股檔案 '{csv_filename}'")
            
            df_etf_raw = pd.read_csv(csv_filename)
            df_etf_raw['代號'] = df_etf_raw['代號'].astype(str).str.strip()
            df_etf_raw['名稱'] = df_etf_raw['名稱'].astype(str).str.strip()
            df_etf_raw['etf'] = df_etf_raw['etf'].astype(str).str.strip()
            df_etf_raw['日期'] = pd.to_datetime(df_etf_raw['日期'])

            df_etf = df_etf_raw[df_etf_raw['etf'].isin(etf_list_to_analyze)].copy()
            if df_etf.empty:
                raise ValueError(f"在 {csv_filename} 中找不到任何屬於 {etf_list_to_analyze} 的資料。")

            unique_dates = sorted(df_etf['日期'].unique(), reverse=True)
            latest_date = unique_dates[0]
            latest_date_str = latest_date.strftime('%Y-%m-%d')
            
            all_stock_ids = df_etf['代號'].unique().tolist()
            close = data.get('price:收盤價')
            aligned_close = close.reindex(unique_dates, method='ffill').iloc[-1]
            
            df_etf['close'] = df_etf['代號'].map(aligned_close)
            df_etf.dropna(subset=['close'], inplace=True)
            df_etf['market_cap'] = (df_etf['持有數'] * 1000) * df_etf['close']

            # --- 步驟 3: 計算最新一日的排名 ---
            df_latest = df_etf[df_etf['日期'] == latest_date].copy()
            stock_total_cap_latest = df_latest.groupby('代號')['market_cap'].sum().sort_values(ascending=False)
            
            top_N_stocks_series = stock_total_cap_latest.head(top_n)
            top_N_stocks_df = top_N_stocks_series.reset_index()
            top_N_stocks_df.columns = ['代號', 'market_cap']
            stock_name_map = df_latest.drop_duplicates(subset=['代號']).set_index('代號')['名稱']
            top_N_stocks_df['名稱'] = top_N_stocks_df['代號'].map(stock_name_map)
            top_N_stocks_df['rank'] = top_N_stocks_df.index + 1
            
            # --- 步驟 4: 判斷是否啟用趨勢分析模式 ---
            is_trend_mode = len(unique_dates) >= trend_window
            if is_trend_mode:
                logging.info(f"資料日期充足 (>{trend_window}筆)，啟用「市值趨勢」分析模式。")
                top_N_stocks_df['status'] = top_N_stocks_df['代號'].apply(
                    lambda sid: self._calculate_trend_status(sid, df_etf, trend_window, trend_thresholds)
                )
                
                # --- [核心修正] 就是這一行！將計算好的狀態存入 class 屬性中 ---
                if 'status' in top_N_stocks_df.columns:
                    self.daily_etf_status_map = top_N_stocks_df.set_index('代號')['status'].to_dict()

            else:
                logging.info(f"資料日期不足 (少於{trend_window}筆)，將使用單日靜態排名模式。")

            # --- 步驟 5: 準備繪圖與回傳 ---
            logging.info(f"已計算出 Top {top_n} 市值焦點股，總市值為 {top_N_stocks_df['market_cap'].sum()/1e8:,.0f} 億元。")
            
            top_symbols = top_N_stocks_df['代號'].tolist()
            df_top_for_pivot = df_latest[df_latest['代號'].isin(top_symbols)].copy()
            pivot_df = df_top_for_pivot.pivot_table(index=['代號', '名稱'], columns='etf', values='market_cap', fill_value=0)
            correct_order_index = pd.MultiIndex.from_frame(top_N_stocks_df[['代號', '名稱']])
            # [Gemini 建議修正] 在 reindex 之前加入對索引唯一性的檢查與修正
            if not correct_order_index.is_unique:
                logging.warning(f"偵測到 pivot_df 的索引中有重複項，將自動移除。")
                # 找出並保留第一個出現的索引
                is_duplicated = correct_order_index.duplicated(keep='first')
                correct_order_index = correct_order_index[~is_duplicated]
                
            pivot_df = pivot_df.reindex(correct_order_index).fillna(0)
            
            self._plot_daily_etf_stacked_bar(pdf, latest_date_str, top_N_stocks_df, pivot_df)
            self._plot_daily_etf_dashboard(pdf, latest_date_str, top_N_stocks_df)
            
            appendix_data_to_return = {'latest_date_str': latest_date_str, 'top_stocks_df': top_N_stocks_df}
            

        except Exception as e:
            logging.error(f"繪製【每日主動式ETF焦點股】時發生錯誤: {e}", exc_info=True)
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, f"無法生成【每日主動式ETF焦點股】頁面\n錯誤: {e}", 
                    ha='center', va='center', fontsize=20, color='red', wrap=True)
            pdf.savefig(fig)
            plt.close(fig)
        
        return appendix_data_to_return




    # finlab程式碼格式
    # [ULTRA THINK 最終邏輯修正版 2/2] 繪圖函式
    # finlab程式碼格式
    # [ULTRA THINK v29.5 倉位變化分析版 2/2] 繪圖函式 (完整無省略)
    def _plot_daily_etf_stacked_bar(self, pdf, latest_date, top_stocks_df, pivot_df):
        """
        doc: [ULTRA THINK v29.5 倉位變化分析] 
        - [核心升級] 智慧判斷：如果傳入的 top_stocks_df 中包含 'status' 欄位，
          則自動在 Y 軸的股票標籤後方附加 [新增]、[加碼] 等狀態，實現視覺化。
        - 其他視覺化設定（顏色、座標軸同步等）維持不變。
        """
        logging.info(f"    ↳ 正在繪製堆疊長條圖 (Top {len(top_stocks_df)})...")
        
        # 建立固定的ETF顏色映射字典
        etf_color_map = {
            '00981A': '#1f77b4', '00982A': '#ff7f0e',
            '00980A': '#2ca02c', '00984A': '#d62728',
        }
        
        # 根據傳入的 top_stocks_df 長度來決定切割點，每頁最多30檔
        split_point = 30
        df_1 = top_stocks_df.iloc[0:split_point]
        df_2 = top_stocks_df.iloc[split_point:]
        
        pivot_df_1 = pivot_df.iloc[0:split_point]
        pivot_df_2 = pivot_df.iloc[split_point:]
        
        # 建立 1x2 的子圖 (一列兩欄)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(28, 24), gridspec_kw={'width_ratios': [1, 1]})
        fig.suptitle(f'{latest_date} 主動式ETF持有總市值前 {len(top_stocks_df)} 名股票分佈圖', fontsize=28, weight='bold', y=0.98)

        # 建立一個統一的標籤生成函式
        def create_y_labels(df_slice, pivot_slice):
            rank_map = df_slice.set_index('代號')['rank'].to_dict()
            
            # 智慧判斷是否存在 'status' 欄位
            if 'status' in df_slice.columns:
                status_map = df_slice.set_index('代號')['status'].to_dict()
                labels = [f"#{int(rank_map.get(s, 0))} {s} {n} [{status_map.get(s, '')}]" for s, n in pivot_slice.index]
            else: # 如果不存在，退回原有的標籤格式
                labels = [f"#{int(rank_map.get(s, 0))} {s} {n}" for s, n in pivot_slice.index]
            return labels

        # --- 繪製左圖 ---
        pivot_df_1.index = create_y_labels(df_1, pivot_df_1)
        colors_1 = [etf_color_map.get(col, '#888888') for col in pivot_df_1.columns]
        pivot_df_1.iloc[::-1].plot(kind='barh', stacked=True, ax=ax1, width=0.8, color=colors_1, legend=False)
        ax1.set_title(f"排名 1-{len(df_1)}", fontsize=20)
        ax1.set_xlabel('ETF持有總市值 (億)', fontsize=16)
        ax1.set_ylabel('股票 (市值排名)', fontsize=16)
        ax1.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e8:,.0f}'))
        ax1.tick_params(axis='x', labelsize=14)
        ax1.tick_params(axis='y', labelsize=12)
        ax1.grid(axis='x', linestyle='--', alpha=0.7)

        # --- 繪製右圖 ---
        if not df_2.empty:
            pivot_df_2.index = create_y_labels(df_2, pivot_df_2)
            colors_2 = [etf_color_map.get(col, '#888888') for col in pivot_df_2.columns]
            pivot_df_2.iloc[::-1].plot(kind='barh', stacked=True, ax=ax2, width=0.8, color=colors_2, legend=False)
            ax2.set_title(f"排名 {split_point + 1}-{len(top_stocks_df)}", fontsize=20)
            ax2.set_xlabel('ETF持有總市值 (億)', fontsize=16)
            ax2.set_ylabel('')
            ax2.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f'{x/1e8:,.0f}'))
            ax2.tick_params(axis='x', labelsize=14)
            ax2.tick_params(axis='y', labelsize=12)
            ax2.grid(axis='x', linestyle='--', alpha=0.7)
        else:
            # 如果沒有右圖資料，關閉右圖座標軸
            ax2.axis('off')

        # --- 同步左右圖表的 X 座標軸範圍 ---
        max_xlim = ax1.get_xlim()[1] 
        ax2.set_xlim(right=max_xlim) 

        # --- 統一處理圖例 ---
        legend_handles = [Patch(color=color, label=name) for name, color in etf_color_map.items()]
        fig.legend(handles=legend_handles, title='ETF', bbox_to_anchor=(0.5, 0.02), loc='lower center', ncol=4, fontsize=14)

        plt.tight_layout(rect=[0.03, 0.05, 0.97, 0.95])

        pdf.savefig(fig)
        plt.close(fig)
            
            
   # [ULTRA THINK 最終修正版 2/5] 儀表板函式 (支援分頁)

    def _plot_daily_etf_dashboard(self, pdf, latest_date, top_stocks_df):
        """
        doc:
        [v29.0 分頁升級] 
        - [核心修改] 新增分頁邏輯，可以將傳入的 top_stocks_df 自動切割，每30檔股票繪製一頁儀表板。
        - 標題會自動加上頁數提示 (例如: 第 1/2 頁)。
        """
        logging.info("    ↳ 正在繪製多因子儀表板...")

        if top_stocks_df.empty:
            return

        STOCKS_PER_PAGE = 30
        num_pages = int(np.ceil(len(top_stocks_df) / STOCKS_PER_PAGE))

        for page_num in range(num_pages):
            start_index = page_num * STOCKS_PER_PAGE
            end_index = start_index + STOCKS_PER_PAGE
            page_data = top_stocks_df.iloc[start_index:end_index]

            fig, ax = plt.subplots(figsize=(28, 24))

            page_info = f" (第 {page_num + 1}/{num_pages} 頁)" if num_pages > 1 else ""

            df_to_plot = pd.merge(
                page_data, self.master_summary_df, left_on='代號', right_on='stock_id', how='left')

            config = {
                'title': f"每日主動式ETF Top {len(top_stocks_df)} 焦點股多因子儀表板 (資料日期: {latest_date}){page_info}",
                'heatmap_cols': ['當日漲跌幅(%)', '漲跌幅_5日(%)', '外資佔市值比_5日', '投信佔市值比_5日',
                                 '外資買賣超_5日', '投信買賣超_5日', '成交金額', '大戶/散戶比_raw',
                                 '情緒結論_raw', '關鍵券商(4週)_raw', '投信今日買賣超(張)', '投信近5日累積買賣超(張)', '投信近20日累積買賣超(張)'],
                'col_defs': {
                    'id_name_industry': {'x': 0.00, 'width': 0.12, 'header': '代碼/名稱/產業', 'type': 'composite_id_name_industry'},
                    'perf_1d':      {'x': 0.12, 'width': 0.05, 'header': '日漲跌', 'type': 'heatmap_perf', 'data_key': '當日漲跌幅(%)'},
                    'perf_5d':      {'x': 0.17, 'width': 0.05, 'header': '5日漲跌', 'type': 'heatmap_perf', 'data_key': '漲跌幅_5日(%)'},
                    'amt':          {'x': 0.22, 'width': 0.07, 'header': '成交額', 'type': 'composite_amt', 'data_key': '成交金額'},
                    'shareholder':  {'x': 0.29, 'width': 0.12, 'header': '大戶籌碼', 'type': 'composite_shareholder', 'data_key': '大戶/散戶比_raw'},
                    'broker':       {'x': 0.41, 'width': 0.20, 'header': '關鍵券商', 'type': 'heatmap_text_broker', 'data_key': '關鍵券商(4週)_raw'},
                    'it_buy_1d':    {'x': 0.61, 'width': 0.07, 'header': '投信現買', 'type': 'heatmap_vol', 'data_key': '投信今日買賣超(張)'},
                    'it_buy_5d':    {'x': 0.68, 'width': 0.07, 'header': '投信5日買', 'type': 'heatmap_vol', 'data_key': '投信近5日累積買賣超(張)'},
                    'it_buy_20d':   {'x': 0.75, 'width': 0.07, 'header': '投信20日買', 'type': 'heatmap_vol', 'data_key': '投信近20日累積買賣超(張)'},
                    'fi_ratio':     {'x': 0.82, 'width': 0.09, 'header': '外資佔比(5日)', 'type': 'heatmap_perf', 'data_key': '外資佔市值比_5日'},
                    'it_ratio':     {'x': 0.91, 'width': 0.09, 'header': '投信佔比(5日)', 'type': 'heatmap_perf', 'data_key': '投信佔市值比_5日'},
                }
            }

            self._draw_dashboard_on_axes(ax, df_to_plot, config)

            plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.93])
            pdf.savefig(fig)
            plt.close(fig)
            gc.collect()




    # finlab程式碼格式
    # [ULTRA THINK 最終修正版 4/5] 附錄繪製函式
    def _plot_daily_etf_appendix_page(self, pdf, latest_date, top_stocks_df):
        logging.info("    ↳ 正在準備附錄詳細報告...")
        
        df_for_appendix = top_stocks_df.rename(columns={'代號': 'stock_id', '名稱': '公司簡稱'})
        
        # [您的新需求] 在此處加入成交金額篩選
        if not self.master_summary_df.empty:
            original_count = len(df_for_appendix)
            df_for_appendix = pd.merge(
                df_for_appendix, 
                self.master_summary_df[['stock_id', '成交金額']], 
                on='stock_id', 
                how='left'
            )
            # 過濾掉成交金額小於門檻或合併後找不到成交金額的股票
            df_for_appendix = df_for_appendix[df_for_appendix['成交金額'] > 15000000].copy()
            logging.info(f"    ↳ 已套用成交金額 > 1500萬篩選，股票數量從 {original_count} 檔過濾至 {len(df_for_appendix)} 檔。")
        
        # [原有的篩選] 在此處加入漲跌幅篩選步驟
        df_for_appendix = self._apply_appendix_filter(df_for_appendix)
        
        def formatter(row):
            # 確保 'rank' 欄位存在，若不存在則給予預設值
            rank_val = f" (排名: #{row.get('rank', 'N/A')})"
            return [f"來源: 每日主動式ETF持有市值 Top {len(df_for_appendix)}{rank_val}"]

        self._plot_appendix_section(
            pdf=pdf,
            stocks_df=df_for_appendix,
            title_text='每日主動式ETF焦點股 詳細報告',
            subtitle_text=f"(基於 {latest_date} 主動式ETF持股市值計算，顯示前 {len(df_for_appendix)} 檔)",
            subtitle_color='darkcyan',
            source_topic='每日主動式ETF焦點股',
            source_details_formatter=formatter,
            image_section_key='E6_APPENDIX_DAILY_ETF'
        )
    
    
   
   
    
    def _plot_industry_volume_flow_page(self, pdf):
        """
        [v3.1 強化版] 繪製全市場成交額前100名股票的產業資金流向分析頁。
        doc:
        - (新) 成分股最多顯示8檔，並自動換行。
        - 柱長: 代表該產業在百強股中的「當日總成交額」。
        - 顏色: 代表資金流向熱度 (5日均量/20日均量)。
        - 內部文字: 直接標示「量能比」、「五日均量」、「二十日均量」的詳細數據。
        - 標籤: 顯示產業成分股、5日排名變化，並對多因子條件加上標記。
        """
        logging.info("--- 開始繪製【(v3.1強化版)產業資金流向(百強股)】分析頁面 ---")
        try:
            # --- 步驟 1: 數據準備 ---
            amt = self.amt_df
            if amt.empty or len(amt) < 21:
                raise ValueError("成交金額數據不足 (需要至少21天) 無法進行分析。")

            all_stock_info = self.industry_df[['stock_id', '公司簡稱', '產業類別']].set_index('stock_id')

            # 輔助函式: 用於計算某一天的產業排名
            def get_industry_rank(date_series):
                top_100 = date_series.nlargest(100)
                top_100_ids = top_100.index.tolist()
                df = all_stock_info.loc[all_stock_info.index.intersection(top_100_ids)].copy()
                df['成交金額'] = df.index.map(top_100)
                summary = df.groupby('產業類別')['成交金額'].sum().sort_values(ascending=False)
                return summary.rank(ascending=False, method='min')

            # 計算今日與五日前排名
            latest_amt = amt.iloc[-1].dropna()
            today_rank = get_industry_rank(latest_amt)
            past_amt = amt.iloc[-6].dropna()
            past_rank = get_industry_rank(past_amt)
            
            # --- 步驟 2: 準備主要顯示數據 (以今日為主) ---
            top_100_stocks = latest_amt.nlargest(100)
            top_100_stock_ids = top_100_stocks.index.tolist()
            top_100_df = all_stock_info.loc[all_stock_info.index.intersection(top_100_stock_ids)].copy()
            top_100_df['成交金額'] = top_100_df.index.map(top_100_stocks)
            amt_ma5 = amt.rolling(5).mean().iloc[-1]
            amt_ma20 = amt.rolling(20).mean().iloc[-1]
            flow_ratio = (amt_ma5 / amt_ma20).replace([np.inf, -np.inf], 1).fillna(1)
            top_100_df['flow_ratio'] = top_100_df.index.map(flow_ratio)
            top_100_df['volume_ma5'] = top_100_df.index.map(amt_ma5)
            top_100_df['volume_ma20'] = top_100_df.index.map(amt_ma20)
            industry_grouped = top_100_df.groupby('產業類別')
            industry_summary = industry_grouped.agg(
                total_volume=('成交金額', 'sum'),
                total_volume_ma5=('volume_ma5', 'sum'),
                total_volume_ma20=('volume_ma20', 'sum'),
                avg_flow_ratio=('flow_ratio', 'mean'),
                stock_count=('成交金額', 'size')
            ).sort_values('total_volume', ascending=False).head(15)

            if industry_summary.empty:
                raise ValueError("無法從前100名股票中匯總出產業資訊。")

            # --- 步驟 3: 產生圖表標籤 (整合多因子標記) ---
            rank_jumper_set = set(self.rank_jump_stocks_df['stock_id']) if not self.rank_jump_stocks_df.empty else set()
            new_high_200_set = set(self.market_breadth_data['latest_new_high_list']['stock_id']) if self.market_breadth_data and not self.market_breadth_data.get('latest_new_high_list', pd.DataFrame()).empty else set()
            new_high_20_set = set(self.market_breadth_data['latest_new_high_20_list']['stock_id']) if self.market_breadth_data and not self.market_breadth_data.get('latest_new_high_20_list', pd.DataFrame()).empty else set()
            strategy_set = set(self.result_df['股票代碼']) if not self.result_df.empty else set()

            y_labels = []
            for i, (industry, row) in enumerate(industry_summary.iterrows()):
                # 處理產業排名變化
                current_r = today_rank.get(industry)
                past_r = past_rank.get(industry, np.nan)
                rank_change_str = "—"
                if pd.notna(past_r) and pd.notna(current_r):
                    change = past_r - current_r
                    if change > 0: rank_change_str = f"▲{int(change)}"
                    elif change < 0: rank_change_str = f"▼{abs(int(change))}"
                rank_info = f"(排名: #{int(current_r)} {rank_change_str})"
                
                # 處理成分股標籤
                stocks_in_industry = top_100_df[top_100_df['產業類別'] == industry].sort_values('成交金額', ascending=False)
                
                # [修改] 增加顯示的股票數量
                STOCKS_TO_SHOW = 15
                top_stocks = stocks_in_industry.head(STOCKS_TO_SHOW)

                stock_details_list = []
                for idx, s_row in top_stocks.iterrows():
                    tags = []
                    if idx in rank_jumper_set:
                        tags.append('▲')
                    if idx in new_high_200_set:
                        tags.append('[200H]')  # 僅添加200日新高標記
                    elif idx in new_high_20_set:  # 只有在非200日新高時，才添加20日新高標記
                        tags.append('[20D]')
                    if idx in strategy_set:
                        tags.append('[策略]')
                    tag_str = ''.join(tags)
                    stock_details_list.append(f"{idx} {s_row['公司簡稱']}{tag_str}")
                
                # [修改] 使用 textwrap 自動換行
                base_string = ", ".join(stock_details_list)
                if len(stocks_in_industry) > STOCKS_TO_SHOW:
                    base_string += "...等"
                stock_details = textwrap.fill(base_string, width=80, subsequent_indent='  ')

                # 組合最終標籤
                label = f"{industry} {rank_info} (共{int(row['stock_count'])}檔)\n> {stock_details}"
                y_labels.append(label)

            # --- 步驟 4: 繪圖 ---
            fig, ax = plt.subplots(figsize=(28, 20), constrained_layout=True)
            fig.suptitle(f"市場焦點：成交額前100名之產業資金流向 (資料日期: {amt.index[-1].strftime('%Y-%m-%d')})", fontsize=24, weight='bold')

            ax.set_title("產業成交額佔比 (Top 100 Stocks)", fontsize=22)
            y_pos = np.arange(len(industry_summary))
            
            colors = ['#e63946' if r > 1.2 else '#f7a072' if r > 1.0 else '#2a9d8f' for r in industry_summary['avg_flow_ratio']]
            
            bars = ax.barh(y_pos, industry_summary['total_volume'], color=colors, align='center', height=0.7, zorder=2)
            ax.set_yticks(y_pos)
            ax.set_yticklabels(y_labels, fontsize=16)
            ax.invert_yaxis()
            ax.set_xlabel("產業總成交金額", fontsize=18)
            ax.xaxis.set_major_formatter(FuncFormatter(self.format_revenue))
            ax.grid(axis='x', linestyle='--', alpha=0.7, zorder=1)
            
            for i, (bar, (idx, row)) in enumerate(zip(bars, industry_summary.iterrows())):
                ma5_text = self.format_revenue(row['total_volume_ma5'], None)
                ma20_text = self.format_revenue(row['total_volume_ma20'], None)
                annotation_text = f"  [量能比: {row['avg_flow_ratio']:.2f}x | 五日均量: {ma5_text} | 二十日均量: {ma20_text}]"
                
                text_color = 'black'
                
                ax.text(ax.get_xlim()[0], bar.get_y() + bar.get_height()/2, annotation_text,
                        ha='left', va='center', color=text_color, fontsize=14, weight='bold', zorder=3,
                        bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.2'))
 
            pdf.savefig(fig)
            plt.close(fig)
            gc.collect()

        except Exception as e:
            logging.error(f"繪製【產業資金流向】頁面時發生錯誤: {e}", exc_info=True)
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, f"無法生成【產業資金流向】頁面\n錯誤: {e}",
                    ha='center', va='center', fontsize=20, color='red', wrap=True)
            pdf.savefig(fig)
            plt.close(fig)
    
    

    def _plot_market_cap_flow(self, pdf):
        """
        [doc]
        繪製市值板塊資金流向分析圖表與摘要表。
        [修改]
        - 摘要表現在會根據個股的5日報酬率，在股票名稱後方加上漲(▲)跌(▼)符號，
        以顯示個股狀況，而非對整個儲存格背景上色。
        """
        logging.info("--- 開始繪製【市值板塊資金流向】分析頁面 ---")

        try:
            # 1. 加載所需數據 (此部分不變)
            market_cap = data.get('etl:market_value')
            close = data.get('price:收盤價')
            all_stock_info = self.industry_df[['stock_id', '公司簡稱']].set_index('stock_id')

            if len(close) < 62 or len(market_cap) < 1:
                raise ValueError("收盤價或市值數據天數不足 (至少需要62天收盤價)。")

            latest_cap = market_cap.iloc[-1].dropna()
            cap_groups = pd.qcut(latest_cap, 10, labels=False, duplicates='drop')
            cap_groups.name = 'cap_decile'

            # 2. 計算報酬率 (此部分不變)
            return_5d = ((close.iloc[-1] - close.iloc[-6]) / close.iloc[-6] * 100).rename('return_5d')
            return_20d = ((close.iloc[-1] - close.iloc[-21]) / close.iloc[-21] * 100).rename('return_20d')
            return_60d = ((close.iloc[-1] - close.iloc[-61]) / close.iloc[-61] * 100).rename('return_60d')

            if return_5d.empty or return_20d.empty or return_60d.empty:
                raise ValueError("報酬率計算結果為空。")

            df = pd.concat([cap_groups, return_5d, return_20d, return_60d], axis=1).dropna()
            grouped_returns = df.groupby('cap_decile')[['return_5d', 'return_20d', 'return_60d']].mean()

            # 3. 準備股票分類集合 (此部分不變)
            strategy_set = set(self.result_df['股票代碼']) if not self.result_df.empty else set()
            new_high_200_set = set(self.market_breadth_data['latest_new_high_list']['stock_id']) if self.market_breadth_data and not self.market_breadth_data.get('latest_new_high_list', pd.DataFrame()).empty else set()
            new_high_20_set = set(self.market_breadth_data['latest_new_high_20_list']['stock_id']) if self.market_breadth_data and not self.market_breadth_data.get('latest_new_high_20_list', pd.DataFrame()).empty else set()

            stock_cap_df = pd.DataFrame({'cap_decile': cap_groups})
            stock_cap_df['is_strategy'] = stock_cap_df.index.isin(strategy_set)
            stock_cap_df['is_new_high_200'] = stock_cap_df.index.isin(new_high_200_set)
            stock_cap_df['is_new_high_20'] = stock_cap_df.index.isin(new_high_20_set)
            cap_stats = stock_cap_df.groupby('cap_decile').agg({'is_strategy': 'sum', 'is_new_high_200': 'sum', 'is_new_high_20': 'sum'}).reindex(grouped_returns.index, fill_value=0)

            # === 圖表頁：市值板塊報酬率長條圖 (此部分不變) ===
            fig, ax = plt.subplots(figsize=(28, 14), constrained_layout=True)
            title_date = close.index[-1].strftime('%Y-%m-%d')
            fig.suptitle(f"市場板塊資金流向分析 (資料日期: {title_date})", fontsize=24, weight='bold')
            ax.set_title("依市值區分的股票群組報酬率", fontsize=18)

            x = np.arange(len(grouped_returns))
            width = 0.25
            rects1 = ax.bar(x - width, grouped_returns['return_5d'], width, label='近5日平均報酬率', color='#e63946')
            rects2 = ax.bar(x, grouped_returns['return_20d'], width, label='近20日平均報酬率', color='#457b9d')
            rects3 = ax.bar(x + width, grouped_returns['return_60d'], width, label='近60日平均報酬率', color='#2a9d8f')

            ax.bar_label(rects1, padding=3, fmt='%.2f%%')
            ax.bar_label(rects2, padding=3, fmt='%.2f%%')
            ax.bar_label(rects3, padding=3, fmt='%.2f%%')

            ax.set_ylabel('平\n均\n報\n酬\n率\n(%)', fontsize=14,rotation=0, labelpad=25)
            ax.set_xlabel('市值分位 (0=最小, 9=最大)', fontsize=14)
            ax.set_xticks(x)
            ax.set_xticklabels(grouped_returns.index)
            ax.legend(fontsize=12)
            ax.axhline(0, color='grey', linewidth=0.8)
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            pdf.savefig(fig)
            plt.close(fig)

            # === 表格頁：摘要表 (修改此處邏輯) ===
            fig, ax = plt.subplots(figsize=(28, 14), constrained_layout=True)
            # [修改] 更新標題，移除背景色相關說明
            ax.set_title("市值分位中股票分類摘要表 (▲/▼ 為個股近5日漲跌)", fontsize=22, weight='bold', pad=20)

            # --- [核心修改] 修改 get_cell_info 函式，以加入漲跌符號 ---
            def get_cell_info(sids):
                def sort_key(sid):
                    is_strategy = sid in strategy_set
                    is_200d = sid in new_high_200_set
                    return 0 if is_strategy else 1 if is_200d else 2

                sorted_sids = sorted(sids, key=sort_key)
                limited_sids = sorted_sids[:15]

                if not limited_sids:
                    return "-"

                # --- 新的個股標籤產生邏輯 ---
                stock_labels = []
                for sid in limited_sids:
                    if sid not in all_stock_info.index:
                        continue
                    name = all_stock_info.loc[sid, '公司簡稱']
                    ret = return_5d.get(sid)
                    marker = ''
                    # 根據個股報酬率決定符號
                    if pd.notna(ret):
                        if ret > 0:
                            marker = '▲'
                        elif ret < 0:
                            marker = '▼'
                    stock_labels.append(f"{sid} {name}{marker}")

                text = ', '.join(stock_labels)
                wrapped_text = textwrap.fill(text, width=60, subsequent_indent='  ')
                
                return wrapped_text

            # --- 建立表格數據 ---
            table_data = [["市值分位", "策略股", "200日新高", "20日新高"]]
            
            for decile in cap_stats.index:
                decile_stocks = stock_cap_df[stock_cap_df['cap_decile'] == decile]
                row_text = [str(decile)]
                
                sids_strategy = decile_stocks[decile_stocks['is_strategy']].index.tolist()
                text = get_cell_info(sids_strategy)
                row_text.append(text)
                
                sids_200h = decile_stocks[decile_stocks['is_new_high_200']].index.tolist()
                text = get_cell_info(sids_200h)
                row_text.append(text)
                
                sids_20h = decile_stocks[decile_stocks['is_new_high_20']].index.tolist()
                text = get_cell_info(sids_20h)
                row_text.append(text)
                
                table_data.append(row_text)

            table = ax.table(cellText=table_data, cellLoc='left', loc='center', colWidths=[0.05, 0.3, 0.3, 0.3])
            table.auto_set_font_size(False)
            table.set_fontsize(15)
            table.scale(1.5, 2.5)

            # --- [核心修改] 移除背景上色邏輯 ---
            for (row, col), cell in table.get_celld().items():
                cell.set_height(0.1)
                if row == 0:
                    cell.set_facecolor("#d9d9d97b")
                    cell.set_text_props(weight='bold')

            ax.axis('off')
            pdf.savefig(fig, bbox_inches='tight', pad_inches=0)
            plt.close(fig)

        except Exception as e:
            logging.error(f"繪製【市值板塊資金流向】頁面時發生錯誤: {e}", exc_info=True)
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, f"無法生成【市值板塊資金流向】頁面\n錯誤: {e}", ha='center', va='center', fontsize=20, color='red', wrap=True)
            pdf.savefig(fig)
            plt.close(fig)







    # finlab程式碼格式
    # [v28.2 版面優化最終版] 請用此版本替換掉您現有的 _plot_etf_summary_page 函式
    def _plot_etf_summary_page(self, pdf):
        """
        doc: [v28.2 版面優化最終版]
        - [核心修改] 根據使用者回饋，不再使用統一的儀表板設定，而是為「市場共識度」與「法人重倉度」建立各自的專屬版面設定。
        - [版面優化] 移除了在各儀表板中沒有數據的空白欄位 (例如共識度中的市值、重倉度中的基金數)。
        - [空間最大化] 將移除空白欄位後多出來的空間，重新分配給現有欄位，使整體表格更寬、更飽滿，大幅減少留白。
        """
        logging.info("--- 開始繪製【(增強版)ETF持股分析摘要】報告頁面 ---\n")

        def process_and_format_df(base_df, period_str):
            merged_df = pd.merge(base_df, self.master_summary_df, left_on='stock_id', right_on='stock_id', how='left')
            
            def format_rank(row):
                curr_rank, prev_rank = row.get(f'本{period_str}排名'), row.get(f'前一{period_str}排名')
                if pd.isna(curr_rank): return "N/A"
                rank_diff = prev_rank - curr_rank
                if pd.isna(prev_rank) or prev_rank == 0: return f"{int(curr_rank)} (新)"
                if rank_diff > 0: return f"{int(curr_rank)} (▲{int(rank_diff)})"
                elif rank_diff < 0: return f"{int(curr_rank)} (▼{int(abs(rank_diff))})"
                else: return f"{int(curr_rank)} (-)"

            processed = merged_df.copy()
            processed[f'排名 (vs 前期)'] = processed.apply(format_rank, axis=1)
            
            raw_cols = ['當日漲跌幅(%)', '漲跌幅_5日(%)', '投信今日買賣超(張)', '投信近5日累積買賣超(張)', 
                        '投信近20日累積買賣超(張)', '成交金額', '大戶/散戶比_raw', '關鍵券商(4週)_raw']
            for col in raw_cols:
                if col in processed.columns:
                    processed[f'{col}_raw'] = processed[col]
            
            return processed

        try:
            required_dfs = {'季度共識度': self.etf_consensus_df, '季度重倉度': self.etf_heavyweight_df, '月度重倉度': self.etf_monthly_heavyweight_df}
            for name, df in required_dfs.items():
                if df is None or df.empty: raise ValueError(f"ETF 分析數據缺失：'{name}' 數據為空。")
            
            q_date_str = self.etf_latest_quarter_date.strftime('%Y-%m')
            m_date_str = self.etf_latest_month_date.strftime('%Y-%m')
            
            # --- [版面優化] 建立「市場共識度」專用設定檔 (無市值欄位) ---
            config_consensus = {
                'heatmap_cols': ['當日漲跌幅(%)', '漲跌幅_5日(%)', '成交金額', '大戶/散戶比_raw', 
                                '關鍵券商(4週)_raw', '投信今日買賣超(張)','投信近5日累積買賣超(張)','投信近20日累積買賣超(張)',
                                '本期持有基金數'],
                'col_defs': {
                    'id_name_industry': {'x': 0.00, 'width': 0.15, 'header': '代碼/名稱/產業', 'type': 'composite_id_name_industry'},
                    'rank':           {'x': 0.15, 'width': 0.09, 'header': '排名 (vs 前期)', 'type': 'colored_text', 'data_key': '排名 (vs 前期)'},
                    'etf_count':      {'x': 0.24, 'width': 0.10, 'header': '持有基金數', 'type': 'heatmap_vol', 'data_key': '本期持有基金數'},
                    'perf_1d':        {'x': 0.34, 'width': 0.05, 'header': '日漲跌', 'type': 'heatmap_perf', 'data_key': '當日漲跌幅(%)'},
                    'perf_5d':        {'x': 0.39, 'width': 0.05, 'header': '5日漲跌', 'type': 'heatmap_perf', 'data_key': '漲跌幅_5日(%)'},
                    'amt':            {'x': 0.44, 'width': 0.08, 'header': '成交額', 'type': 'composite_amt', 'data_key': '成交金額'},
                    'shareholder':    {'x': 0.52, 'width': 0.14, 'header': '大戶籌碼', 'type': 'composite_shareholder', 'data_key': '大戶/散戶比_raw'},
                    'broker':         {'x': 0.66, 'width': 0.12, 'header': '關鍵券商', 'type': 'heatmap_text_broker', 'data_key': '關鍵券商(4週)_raw'},
                    'it_buy_1d':      {'x': 0.78, 'width': 0.07, 'header': '投信現買', 'type': 'heatmap_vol', 'data_key': '投信今日買賣超(張)'},
                    'it_buy_5d':      {'x': 0.85, 'width': 0.07, 'header': '投信5日買', 'type': 'heatmap_vol', 'data_key': '投信近5日累積買賣超(張)'},
                    'it_buy_20d':     {'x': 0.92, 'width': 0.08, 'header': '投信20日買', 'type': 'heatmap_vol', 'data_key': '投信近20日累積買賣超(張)'},
                }
            }
            
            # --- [版面優化] 建立「法人重倉度」專用設定檔 (無基金數欄位) ---
            config_heavyweight = {
                'heatmap_cols': ['當日漲跌幅(%)', '漲跌幅_5日(%)', '成交金額', '大戶/散戶比_raw', 
                                '關鍵券商(4週)_raw', '投信今日買賣超(張)','投信近5日累積買賣超(張)','投信近20日累積買賣超(張)',
                                '本期總持股市值(億)'],
                'col_defs': {
                    'id_name_industry': {'x': 0.00, 'width': 0.15, 'header': '代碼/名稱/產業', 'type': 'composite_id_name_industry'},
                    'rank':           {'x': 0.15, 'width': 0.09, 'header': '排名 (vs 前期)', 'type': 'colored_text', 'data_key': '排名 (vs 前期)'},
                    'etf_value':      {'x': 0.24, 'width': 0.10, 'header': '持有市值(億)', 'type': 'heatmap_vol', 'data_key': '本期總持股市值(億)'},
                    'perf_1d':        {'x': 0.34, 'width': 0.05, 'header': '日漲跌', 'type': 'heatmap_perf', 'data_key': '當日漲跌幅(%)'},
                    'perf_5d':        {'x': 0.39, 'width': 0.05, 'header': '5日漲跌', 'type': 'heatmap_perf', 'data_key': '漲跌幅_5日(%)'},
                    'amt':            {'x': 0.44, 'width': 0.08, 'header': '成交額', 'type': 'composite_amt', 'data_key': '成交金額'},
                    'shareholder':    {'x': 0.52, 'width': 0.14, 'header': '大戶籌碼', 'type': 'composite_shareholder', 'data_key': '大戶/散戶比_raw'},
                    'broker':         {'x': 0.66, 'width': 0.12, 'header': '關鍵券商', 'type': 'heatmap_text_broker', 'data_key': '關鍵券商(4週)_raw'},
                    'it_buy_1d':      {'x': 0.78, 'width': 0.07, 'header': '投信現買', 'type': 'heatmap_vol', 'data_key': '投信今日買賣超(張)'},
                    'it_buy_5d':      {'x': 0.85, 'width': 0.07, 'header': '投信5日買', 'type': 'heatmap_vol', 'data_key': '投信近5日累積買賣超(張)'},
                    'it_buy_20d':     {'x': 0.92, 'width': 0.08, 'header': '投信20日買', 'type': 'heatmap_vol', 'data_key': '投信近20日累積買賣超(張)'},
                }
            }
            
            # --- 繪圖區: 根據不同資料呼叫對應的設定檔 ---
            # 1. 市場共識度儀表板
            df_c_processed = process_and_format_df(self.etf_consensus_df, '期')
            fig_c, ax_c = plt.subplots(figsize=(28, 24))
            config_c = config_consensus.copy()
            config_c['title'] = f"市場共識度 Top {len(df_c_processed)} (最多基金持有) - 季度比較 (資料日期: {q_date_str})"
            self._draw_dashboard_on_axes(ax_c, df_c_processed, config_c)
            pdf.savefig(fig_c); plt.close(fig_c)
            
            # 2. 季度法人重倉度儀表板
            df_hq_processed = process_and_format_df(self.etf_heavyweight_df, '期')
            fig_hq, ax_hq = plt.subplots(figsize=(28, 24))
            config_hq = config_heavyweight.copy()
            config_hq['title'] = f"法人重倉度 Top {len(df_hq_processed)} (總持有市值最高) - 季度比較 (資料日期: {q_date_str})"
            self._draw_dashboard_on_axes(ax_hq, df_hq_processed, config_hq)
            pdf.savefig(fig_hq); plt.close(fig_hq)

            # 3. 月度法人重倉度儀表板
            df_hm_processed = process_and_format_df(self.etf_monthly_heavyweight_df, '期')
            fig_hm, ax_hm = plt.subplots(figsize=(28, 24))
            config_hm = config_heavyweight.copy()
            config_hm['title'] = f"法人重倉度 Top {len(df_hm_processed)} (總持有市值最高) - 月度比較 (資料日期: {m_date_str})"
            self._draw_dashboard_on_axes(ax_hm, df_hm_processed, config_hm)
            pdf.savefig(fig_hm); plt.close(fig_hm)

        except Exception as e:
            logging.error(f"繪製【ETF 持股分析摘要】頁面時出錯: {e}", exc_info=True)
            fig, ax = plt.subplots(figsize=(28, 12)); ax.text(0.5, 0.5, f"無法生成【ETF 持股分析摘要】頁面\n錯誤: {e}", ha='center', va='center', fontsize=20, color='red', wrap=True); pdf.savefig(fig); plt.close(fig)


   
   
   
   
   
   
    # 請找到並替換掉原有的 _plot_etf_quadrant_page 函式
    def _plot_etf_quadrant_page(self, pdf):
        """
        doc: [ETF整合新增 - Top 20 修改版] 繪製 ETF 持股的四象限分析圖。
        - [修改] 每個象限內最多顯示20檔股票。
        - X軸: 市場共識度 (持有基金數量)
        - Y軸: 法人重倉度 (總持股市值)
        - 圖表標題會明確標示資料的季度月份。
        """
        logging.info("--- 開始繪製【ETF 共識度 vs 重倉度四象限分析】報告頁面 ---")
        try:
            if self.etf_quadrant_data is None or self.etf_quadrant_data.empty:
                raise ValueError("用於四象限分析的 ETF 數據為空。")

            quadrant_df = self.etf_quadrant_data
            current_date = self.etf_latest_quarter_date

            # 計算中位數
            median_consensus = quadrant_df['本期持有基金數'].median()
            median_heavyweight = quadrant_df['本期總持股市值(億)'].median()

            # 分配股票到四個象限
            star_stocks = quadrant_df[(quadrant_df['本期持有基金數'] > median_consensus) & (quadrant_df['本期總持股市值(億)'] > median_heavyweight)]
            private_picks = quadrant_df[(quadrant_df['本期持有基金數'] <= median_consensus) & (quadrant_df['本期總持股市值(億)'] > median_heavyweight)]
            potential_stocks = quadrant_df[(quadrant_df['本期持有基金數'] > median_consensus) & (quadrant_df['本期總持股市值(億)'] <= median_heavyweight)]
            
            fig, axes = plt.subplots(2, 2, figsize=(28, 20))
            fig.suptitle(f'ETF 市場共識度 vs 法人重倉度 - 四象限分析 (資料月份: {current_date.strftime("%Y-%m")})', fontsize=28, weight='bold', y=0.97)

            # 繪圖輔助函式
            def plot_quadrant_bars(ax, data, title, sort_col, bar_col, color_col):
                # [修改] 將 head(15) 改為 head(20)
                df_plot = data.sort_values(by=sort_col, ascending=False).head(20).sort_values(by=sort_col, ascending=True)
                if df_plot.empty:
                    ax.text(0.5, 0.5, '此象限無資料', ha='center', va='center', fontsize=18)
                else:
                    norm = mcolors.Normalize(vmin=df_plot[color_col].min(), vmax=df_plot[color_col].max())
                    colors = cm.viridis(norm(df_plot[color_col]))
                    ax.barh(df_plot['股票'], df_plot[bar_col], color=colors)
                    for i, (bar_val, color_val) in enumerate(zip(df_plot[bar_col], df_plot[color_col])):
                        ax.text(bar_val * 0.02, i, f' {bar_val:,.1f} 億', va='center', ha='left', fontsize=12, color='white', weight='bold')
                ax.set_title(title, fontsize=20, weight='bold')
                ax.set_xlabel(f"長條長度: {bar_col}", fontsize=14)
                ax.tick_params(axis='x', labelsize=12)
                ax.tick_params(axis='y', labelsize=12)
            
            # 繪製各象限
            plot_quadrant_bars(axes[0, 1], star_stocks, "【明星象限】高共識 & 高重倉", '本期總持股市值(億)', '本期總持股市值(億)', '本期持有基金數')
            plot_quadrant_bars(axes[0, 0], private_picks, "【私藏象限】低共識 & 高重倉", '本期總持股市值(億)', '本期總持股市值(億)', '本期持有基金數')
            plot_quadrant_bars(axes[1, 1], potential_stocks, "【潛力象限】高共識 & 低重倉", '本期持有基金數', '本期持有基金數', '本期總持股市值(億)')
            axes[1, 0].set_visible(False) 

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            pdf.savefig(fig)
            plt.close(fig)

        except Exception as e:
            logging.error(f"繪製【ETF 四象限分析】頁面時出錯: {e}", exc_info=True)
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, f"無法生成【ETF 四象限分析】頁面\n錯誤: {e}", ha='center', va='center', fontsize=20, color='red', wrap=True)
            pdf.savefig(fig)
            plt.close(fig)
            
            
            
    # finlab程式碼格式
    # finlab程式碼格式
    # 請找到並替換掉原有的 _plot_etf_monthly_quadrant_page 函式
    def _plot_etf_monthly_quadrant_page(self, pdf):
        """
        doc: [v23.0 新增 - Top 20 修改版] 繪製 ETF 持股的「月度」四象限分析圖。
        - [修改] 每個象限內最多顯示20檔股票。
        - X軸: 市場共識度 (持有基金數量)
        - Y軸: 法人重倉度 (總持股市值)
        - 圖表標題會明確標示資料的月度月份。
        """
        logging.info("--- 開始繪製【ETF 月度共識度 vs 重倉度四象限分析】報告頁面 ---")
        try:
            if self.etf_monthly_quadrant_data is None or self.etf_monthly_quadrant_data.empty:
                raise ValueError("用於月度四象限分析的 ETF 數據為空。")

            quadrant_df = self.etf_monthly_quadrant_data
            current_date = self.etf_latest_month_date 

            # 計算中位數
            median_consensus = quadrant_df['本期持有基金數'].median()
            median_heavyweight = quadrant_df['本期總持股市值(億)'].median()

            # 分配股票到四個象限
            star_stocks = quadrant_df[(quadrant_df['本期持有基金數'] > median_consensus) & (quadrant_df['本期總持股市值(億)'] > median_heavyweight)]
            private_picks = quadrant_df[(quadrant_df['本期持有基金數'] <= median_consensus) & (quadrant_df['本期總持股市值(億)'] > median_heavyweight)]
            potential_stocks = quadrant_df[(quadrant_df['本期持有基金數'] > median_consensus) & (quadrant_df['本期總持股市值(億)'] <= median_heavyweight)]
            
            fig, axes = plt.subplots(2, 2, figsize=(28, 20))
            fig.suptitle(f'ETF 市場共識度 vs 法人重倉度 - 四象限分析 (月度資料: {current_date.strftime("%Y-%m")})', fontsize=28, weight='bold', y=0.97)

            # 繪圖輔助函式
            def plot_quadrant_bars(ax, data, title, sort_col, bar_col, color_col):
                # [修改] 將 head(15) 改為 head(20)
                df_plot = data.sort_values(by=sort_col, ascending=False).head(20).sort_values(by=sort_col, ascending=True)
                if df_plot.empty:
                    ax.text(0.5, 0.5, '此象限無資料', ha='center', va='center', fontsize=18)
                else:
                    norm = mcolors.Normalize(vmin=df_plot[color_col].min(), vmax=df_plot[color_col].max())
                    colors = cm.viridis(norm(df_plot[color_col]))
                    ax.barh(df_plot['股票'], df_plot[bar_col], color=colors)
                    for i, (bar_val, color_val) in enumerate(zip(df_plot[bar_col], df_plot[color_col])):
                        ax.text(bar_val * 0.02, i, f' {bar_val:,.1f} 億', va='center', ha='left', fontsize=12, color='white', weight='bold')
                ax.set_title(title, fontsize=20, weight='bold')
                ax.set_xlabel(f"長條長度: {bar_col}", fontsize=14)
                ax.tick_params(axis='x', labelsize=12)
                ax.tick_params(axis='y', labelsize=12)
            
            # 繪製各象限
            plot_quadrant_bars(axes[0, 1], star_stocks, "【明星象限】高共識 & 高重倉", '本期總持股市值(億)', '本期總持股市值(億)', '本期持有基金數')
            plot_quadrant_bars(axes[0, 0], private_picks, "【私藏象限】低共識 & 高重倉", '本期總持股市值(億)', '本期總持股市值(億)', '本期持有基金數')
            plot_quadrant_bars(axes[1, 1], potential_stocks, "【潛力象限】高共識 & 低重倉", '本期持有基金數', '本期持有基金數', '本期總持股市值(億)')
            axes[1, 0].set_visible(False)

            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            pdf.savefig(fig)
            plt.close(fig)

        except Exception as e:
            logging.error(f"繪製【ETF 月度四象限分析】頁面時出錯: {e}", exc_info=True)
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, f"無法生成【ETF 月度四象限分析】頁面\n錯誤: {e}", ha='center', va='center', fontsize=20, color='red', wrap=True)
            pdf.savefig(fig)
            plt.close(fig)


    # 請將此函式新增到 ReportGenerator 類別中
    # finlab程式碼格式
    def _plot_etf_trend_page(self, pdf):
        """
        doc: [v23.0 優化 - Top 20 修改版] 針對月度重倉股 Top 20，回溯過去 6 個月的總持有市值趨勢並繪圖。
        - [修改] 圖例中增加顯示最新排名與排名變化，使資訊更豐富。
        """
        logging.info("--- 開始繪製【ETF 月度關鍵股資金流比較】報告頁面 ---")
        try:
            if self.etf_monthly_heavyweight_df is None or self.etf_monthly_heavyweight_df.empty:
                raise ValueError("月度法人重倉股數據為空，無法生成趨勢圖。")

            # [修改] 將 head(15) 改為 head(20)
            top_stocks_df = self.etf_monthly_heavyweight_df.head(30)
            latest_date = self.etf_latest_month_date
            
            stocks_to_analyze = top_stocks_df['股票'].tolist()
            date_range = pd.date_range(end=latest_date, periods=6, freq='MS')
            trend_data = {stock: [] for stock in stocks_to_analyze}

            for month_date in date_range:
                month_data = etf_get_data_for_month(month_date)
                if not month_data.empty:
                    for stock in stocks_to_analyze:
                        total_investment = month_data[month_data['股票'] == stock]['金額'].sum() / 1e8
                        trend_data[stock].append(total_investment)
                else:
                    for stock in stocks_to_analyze:
                        trend_data[stock].append(0)
            
            trend_df = pd.DataFrame(trend_data, index=date_range)
            
            plt.style.use('seaborn-v0_8-whitegrid')
            plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
            plt.rcParams['axes.unicode_minus'] = False
            
            fig, ax = plt.subplots(figsize=(28, 15))
            
            # [修改] 將 head(15) 改為 head(20)
            top_stocks_df_for_legend = self.etf_monthly_heavyweight_df.head(30)
            
            for _, row in top_stocks_df_for_legend.iterrows():
                stock_name = row['股票']
                rank = row['本期排名']
                rank_change = row['排名變化']
                
                label = f"#{rank:<2} {stock_name:<12} ({rank_change})"
                
                if stock_name in trend_df.columns:
                    ax.plot(trend_df.index, trend_df[stock_name], marker='o', linestyle='-', label=label)
            
            start_month = date_range[0].strftime("%Y-%m")
            end_month = date_range[-1].strftime("%Y-%m")
            # [修改] 標題從 Top 15 改為 Top 20
            ax.set_title(f"月度 Top 30 關鍵股法人總持股趨勢 (億元) (回溯區間: {start_month} ~ {end_month})", fontsize=24, weight='bold')
            ax.set_xlabel("月份", fontsize=16)
            ax.set_ylabel("持\n股\n總\n市\n值\n(億元)",labelpad=20,rotation=0, fontsize=16)
            
            legend = ax.legend(title="股票 (最新排名與變化)", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=14)
            
            ax.grid(True, which='both', linestyle='--', linewidth=0.7)
            ax.tick_params(axis='x', labelsize=12)
            ax.tick_params(axis='y', labelsize=12)
            
            plt.tight_layout(rect=[0, 0, 0.85, 1]) 
            pdf.savefig(fig)
            plt.close(fig)

        except Exception as e:
            logging.error(f"繪製【ETF 月度資金流趨勢】頁面時出錯: {e}", exc_info=True)
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, f"無法生成【ETF 月度資金流趨勢】頁面\n錯誤: {e}", ha='center', va='center', fontsize=20, color='red', wrap=True)
            pdf.savefig(fig)
            plt.close(fig)



    # finlab程式碼格式
    def _plot_etf_stocks_appendix_page(self, pdf):
        """doc: [ETF整合新增] 為月度法人重倉股 Top 30 自動產生詳細的個股分析附錄。"""
        logging.info("--- 開始繪製【附錄：ETF/基金焦點股】詳細報告 ---\n")
        if not self.etf_focus_stock_ids:
            logging.warning("無 ETF 焦點股可供分析，跳過附錄生成。")
            return

        # [您的新需求] 在篩選ID的同時，也篩選成交金額
        stocks_to_plot_df = self.industry_df[
            self.industry_df['stock_id'].isin(self.etf_focus_stock_ids) &
            (self.industry_df['成交金額'] > 15000000)
        ].copy()
        logging.info(f"    ↳ 已套用成交金額 > 1500萬篩選，股票數量從 {len(self.etf_focus_stock_ids)} 檔過濾至 {len(stocks_to_plot_df)} 檔。")

        
        # [原有的篩選] 在此處加入漲跌幅篩選步驟
        stocks_to_plot_df = self._apply_appendix_filter(stocks_to_plot_df)

        if stocks_to_plot_df.empty:
            logging.warning("在主資料表中找不到任何 ETF 焦點股的數據，或經過篩選後無剩餘股票，跳過附錄生成。")
            return
            
        stocks_to_plot_df['stock_id'] = pd.Categorical(stocks_to_plot_df['stock_id'], categories=self.etf_focus_stock_ids, ordered=True)
        stocks_to_plot_df.sort_values('stock_id', inplace=True)

        def formatter(row):
            return [f"來源: ETF 月度法人重倉 Top 30"]

        self._plot_appendix_section(
            pdf=pdf,
            stocks_df=stocks_to_plot_df,
            title_text='ETF/基金 焦點股詳細報告',
            subtitle_text=f"(基於 {self.etf_latest_month_date.strftime('%Y-%m')} 月度法人重倉數據，最多顯示前 {len(stocks_to_plot_df)} 檔)",
            subtitle_color='darkmagenta',
            source_topic='ETF焦點股',
            source_details_formatter=formatter,
            image_section_key='E5_APPENDIX_ETF_STOCKS' 
        )



    def _plot_market_breadth_page(self, pdf):
        """
        [doc]
        [v16.1.0 修正]
        - 擴充儀表板，從顯示前20檔改為最多顯示前50檔。
        - 重構繪圖流程，先繪製上半部圖表，再為下半部儀表板獨立進行分頁繪製，以支援多頁顯示。
        """
        logging.info("--- 開始繪製【(擴充版)創200日新高多因子儀表板】 ---")
        # --- [核心修正開始] ---
        if self.market_breadth_data is None or self.market_breadth_data.get('breadth_timeseries', pd.DataFrame()).empty:
            logging.warning("市場寬度數據為空，無法繪製創200日新高頁面。")
            # 產生一個提示頁面，而不是讓程式繼續執行空的繪圖邏輯
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, "市場寬度數據不足或計算失敗\n無法生成此頁面",
                    ha='center', va='center', fontsize=24, color='gray')
            ax.axis('off')
            pdf.savefig(fig)
            plt.close(fig)
            return
        
        # --- 步驟 1: 獨立繪製並儲存上半部的市場寬度趨勢圖 ---
        breadth_df = self.market_breadth_data['breadth_timeseries']
        fig_top = plt.figure(figsize=(28, 10))
        ax1 = fig_top.add_subplot(1, 1, 1)

        # --- [修改] 更新圖表標題與新增繪圖指令 ---
        ax1.set_title("創新高/低家數趨勢 (200日高 vs 20日高 vs 20日低)", fontsize=20)
        ax1.plot(breadth_df.index, breadth_df['new_high_200'], color='red', lw=2, label='創200日新高家數')
        ax1.plot(breadth_df.index, breadth_df['new_high_20'], color='orange', lw=1.8, ls='--', label='創20日新高家數')
        ax1.plot(breadth_df.index, breadth_df['new_low_20'], color='green', lw=2, label='創20日新低家數')
        # --- 修改結束 ---
        
        ax1.grid(True, linestyle='--', alpha=0.6)
        ax2 = ax1.twinx()
        ax2.bar(breadth_df.index, breadth_df['diff'], color='blue', alpha=0.3, width=1.0, label='新高-新低 (多空溫度計)')
        ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
        fig_top.tight_layout()
        pdf.savefig(fig_top)
        plt.close(fig_top)

        # --- 步驟 2: 準備並繪製下半部的儀表板 (可能分頁) ---
        new_high_stock_ids = list(self.market_breadth_data.get('latest_new_high_list', {}).get('stock_id', []))

        if not new_high_stock_ids:
            logging.info("本日無股票創200日新高，儀表板部分將留空。")
            return
            
        new_high_list_df = self.industry_df[self.industry_df['stock_id'].isin(new_high_stock_ids)].copy()
        df_sorted = new_high_list_df.sort_values(by='漲跌幅_5日(%)', ascending=False).head(50).reset_index(drop=True)

        if df_sorted.empty:
            return

        STOCKS_PER_PAGE = 20
        num_pages = int(np.ceil(len(df_sorted) / STOCKS_PER_PAGE))
        
        for page_num in range(num_pages):
            start_index = page_num * STOCKS_PER_PAGE
            end_index = start_index + STOCKS_PER_PAGE
            page_data = df_sorted.iloc[start_index:end_index]
            
            fig_page, ax_page = plt.subplots(figsize=(28, 24))
            page_info = f" (第 {page_num + 1}/{num_pages} 頁)" if num_pages > 1 else ""
            
             # doc: =============================================================================
            # doc: --- [核心修改] 在 config 中加入 '產業量能比' 欄位 ---
            # doc: =============================================================================
            config = {
                'title': f"創200日新高(趨勢股) 多因子儀表板{page_info}",
                'heatmap_cols': ['當日漲跌幅(%)', '漲跌幅_5日(%)', '外資佔市值比_5日', '投信佔市值比_5日', 
                                    '外資買賣超_5日', '投信買賣超_5日', '成交金額', '大戶/散戶比_raw', 
                                    '情緒結論_raw', '關鍵券商(4週)_raw', 
                                    '營收YoY(%)', '營收動能(3/12)', '產業量能比'], # [新增]
                'col_defs': {
                    'id_name_industry': {'x': 0.00, 'width': 0.12, 'header': '代碼/名稱/產業', 'type': 'composite_id_name_industry'},
                    'perf_1d':      {'x': 0.12, 'width': 0.05, 'header': '日漲跌', 'type': 'heatmap_perf', 'data_key': '當日漲跌幅(%)'},
                    'perf_5d':      {'x': 0.17, 'width': 0.05, 'header': '5日漲跌', 'type': 'heatmap_perf', 'data_key': '漲跌幅_5日(%)'},
                    'yoy':          {'x': 0.22, 'width': 0.05, 'header': '營收YoY', 'type': 'heatmap_perf', 'data_key': '營收YoY(%)'},
                    'rev_mom':      {'x': 0.27, 'width': 0.05, 'header': '營收動能', 'type': 'heatmap_ratio', 'data_key': '營收動能(3/12)', 'display_key':'營收動能(3/12)_ratio'},
                    'industry_flow':{'x': 0.32, 'width': 0.05, 'header': '產業量能', 'type': 'heatmap_ratio', 'data_key': '產業量能比', 'display_key':'產業量能比_ratio'}, # [新增]
                    'amt':          {'x': 0.37, 'width': 0.07, 'header': '成交額', 'type': 'composite_amt', 'data_key': '成交金額'},
                    'sentiment':    {'x': 0.44, 'width': 0.08, 'header': '融資情緒', 'type': 'colored_text', 'data_key': '情緒結論', 'color_map': {'籌碼趨向多方': '#e63946', '軋空行情醞釀中': '#f7a072', '籌碼趨向空方': '#2a9d8f', '多空交戰激烈': '#1d3557', '市場人氣退潮': '#6c757d', '數據不足': 'black'}},
                    'shareholder':  {'x': 0.52, 'width': 0.11, 'header': '大戶籌碼', 'type': 'composite_shareholder', 'data_key': '大戶/散戶比_raw'},
                    'broker':       {'x': 0.63, 'width': 0.11, 'header': '關鍵券商', 'type': 'heatmap_text_broker', 'data_key': '關鍵券商(4週)_raw'},
                    'fi_buy':       {'x': 0.74, 'width': 0.06, 'header': '外資5日超', 'type': 'heatmap_vol', 'data_key': '外資買賣超_5日'},
                    'it_buy':       {'x': 0.80, 'width': 0.06, 'header': '投信5日超', 'type': 'heatmap_vol', 'data_key': '投信買賣超_5日'},
                    'fi_ratio':     {'x': 0.86, 'width': 0.07, 'header': '外資佔比', 'type': 'heatmap_perf', 'data_key': '外資佔市值比_5日'},
                    'it_ratio':     {'x': 0.93, 'width': 0.07, 'header': '投信佔比', 'type': 'heatmap_perf', 'data_key': '投信佔市值比_5日'},
                }
            }
            # doc: --- [修改結束] ---
            
            self._draw_dashboard_on_axes(ax_page, page_data, config)
            
            plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.93])
            pdf.savefig(fig_page)
            plt.close(fig_page)
            gc.collect()

    # ===================================================================
   # 在 ReportGenerator class 中
    # [新增] 輔助函式，用於根據背景色決定文字顏色
    def _get_contrast_color(self, color):
        """
        計算背景色的亮度，並回傳對比度高的文字顏色（黑色或白色）。
        """
        if color is None:
            return 'black'
        # 將顏色轉換為 RGB
        rgb = mcolors.to_rgb(color)
        # 使用 YIQ 公式計算亮度
        yiq = ((rgb[0] * 299) + (rgb[1] * 587) + (rgb[2] * 114)) / 1000
        # 如果亮度大於閾值，則背景為亮色，使用黑色文字；否則使用白色
        return 'black' if yiq >= 0.6 else 'white'





    # finlab程式碼格式
    # [v14.8 語法最終修正] 請用此版本替換掉您現有的 _draw_dashboard_on_axes 函式
    def _draw_dashboard_on_axes(self, ax, data_df, config):
        """
        doc:
        [v14.8 語法最終修正]
        - [核心修正] 修正了先前版本中所有 f-string 和一般字串因包含錯誤的跳脫字元 (\) 而導致的 SyntaxError。
        [v14.7] 重構 'heatmap_perf' 與 'heatmap_vol' 的文字產生邏輯。
        [v14.5] 強化 'colored_text' 的智慧上色功能。
        """
        ax.axis('off')
        ax.set_title(config['title'], fontsize=config.get(
            'title_fontsize', 22), pad=70, weight='bold', color='black')

        # 1. 定義顏色和正規化工具
        pos_cmap = mcolors.LinearSegmentedColormap.from_list("pos_cmap", ['#FFCDD2', '#C62828'])
        neg_cmap = mcolors.LinearSegmentedColormap.from_list("neg_cmap", ['#E8F5E9', '#9CCC65'])
        amt_cmap = plt.get_cmap('Oranges')
        shareholder_cmap = plt.get_cmap('Reds')
        it_cmap = plt.get_cmap('Purples')

        heatmap_cols = config.get('heatmap_cols', [])
        norm_perf, norm_vol, norm_amt, norm_share, norm_senti, norm_broker, norm_it = [None] * 7
        
        if heatmap_cols and not data_df.empty:
            def get_abs_numeric_values(df, col_name):
                return pd.to_numeric(df.get(col_name), errors='coerce').abs().dropna()

            def safe_concat(series_list):
                if not series_list: return []
                non_empty_series = [s for s in series_list if not s.empty]
                if not non_empty_series: return []
                return pd.concat(non_empty_series).tolist()

            perf_values = safe_concat([get_abs_numeric_values(data_df, col) for col in heatmap_cols if '佔比' in col or '漲跌幅' in col or 'ratio' in col or '營收' in col or 'trap_ratio' in col])
            vol_values = safe_concat([get_abs_numeric_values(data_df, col) for col in heatmap_cols if '超' in col and '關鍵券商' not in col or '持有數' in col or '市值' in col])
            amt_values = safe_concat([get_abs_numeric_values(data_df, col) for col in heatmap_cols if '成交金額' in col])
            share_values = safe_concat([get_abs_numeric_values(data_df, col) for col in heatmap_cols if '大戶/散戶比_raw' in col])
            senti_values = safe_concat([get_abs_numeric_values(data_df, col) for col in heatmap_cols if '情緒結論_raw' in col])
            broker_values = safe_concat([get_abs_numeric_values(data_df, col) for col in heatmap_cols if '關鍵券商(4週)_raw' in col])
            it_values = safe_concat([get_abs_numeric_values(data_df, col) for col in heatmap_cols if '投信作多強度' in col])
            
            norm_perf = mcolors.Normalize(vmin=0, vmax=max(perf_values) if perf_values else 1)
            norm_vol = mcolors.Normalize(vmin=0, vmax=max(vol_values) if vol_values else 1)
            norm_amt = mcolors.Normalize(vmin=min(amt_values) if amt_values else 0, vmax=max(amt_values) if amt_values else 1)
            norm_share = mcolors.Normalize(vmin=min(share_values) if share_values else 0, vmax=max(share_values) if share_values else 1)
            norm_senti = mcolors.Normalize(vmin=0, vmax=max(senti_values) if senti_values else 1)
            norm_broker = mcolors.Normalize(vmin=0, vmax=max(broker_values) if broker_values else 1)
            norm_it = mcolors.Normalize(vmin=0, vmax=max(it_values) if it_values else 1)

        # 2. 繪製表頭
        num_rows, col_defs = len(data_df), config['col_defs']
        HEADER_HEIGHT = 1.2
        ax.set_ylim(-1, num_rows)
        for prop in col_defs.values():
            rect = plt.Rectangle((prop['x'], num_rows - 0.5), prop['width'], HEADER_HEIGHT, facecolor='#f0f0f0', edgecolor='black', lw=0.5, zorder=1)
            ax.add_patch(rect)
            ax.text(prop['x'] + prop['width']/2, num_rows + (HEADER_HEIGHT - 1)/2 - 0.5, prop['header'], ha='center', va='center', weight='bold', fontsize=11, color='black', zorder=2)

        # 3. 遍歷每一行數據並繪製內容
        y_positions = np.arange(num_rows)[::-1]
        for i, y_pos in enumerate(y_positions):
            if i >= len(data_df): continue
            row = data_df.iloc[i]
            
            cell_bg_colors = {}
            
            # 步驟 3a: 繪製背景熱力圖與儲存格邊框
            for col_name, prop in col_defs.items():
                background_color = None
                key_for_heatmap, cell_type = prop.get('data_key'), prop.get('type')
                
                if cell_type == 'composite_id_name_industry' and row.get('投信作多原因', ''):
                    key_for_heatmap = '投信作多強度'
                elif key_for_heatmap not in heatmap_cols:
                    key_for_heatmap = None

                if key_for_heatmap:
                    raw_value = pd.to_numeric(row.get(key_for_heatmap), errors='coerce')
                    if pd.notna(raw_value) and raw_value != 0:
                        cmap, norm = None, None
                        if key_for_heatmap == '投信作多強度': cmap, norm = it_cmap, norm_it
                        elif key_for_heatmap == '成交金額': cmap, norm = amt_cmap, norm_amt
                        elif key_for_heatmap == '大戶/散戶比_raw': cmap, norm = shareholder_cmap, norm_share
                        elif key_for_heatmap == '情緒結論_raw': cmap, norm = (pos_cmap, norm_senti) if raw_value > 0 else (neg_cmap, norm_senti)
                        elif key_for_heatmap == '關鍵券商(4週)_raw': cmap, norm = (pos_cmap, norm_broker) if raw_value > 0 else (neg_cmap, norm_broker)
                        elif '超' in key_for_heatmap or '持有數' in key_for_heatmap or '市值' in key_for_heatmap: cmap, norm = (pos_cmap, norm_vol) if raw_value >= 0 else (neg_cmap, norm_vol)
                        else: cmap, norm = (pos_cmap, norm_perf) if raw_value >= 0 else (neg_cmap, norm_perf)
                        if cmap and norm:
                            background_color = cmap(norm(abs(raw_value)))
                
                face_color_to_use = background_color if background_color else 'none'
                ax.add_patch(plt.Rectangle((prop['x'], y_pos - 0.5), prop['width'], 1, 
                                            facecolor=face_color_to_use, edgecolor='black', lw=0.5, zorder=1))
                if background_color:
                    cell_bg_colors[col_name] = background_color

            # 步驟 3b: 繪製前景文字
            for col_name, prop in col_defs.items():
                bg_color = cell_bg_colors.get(col_name)
                text_color = self._get_contrast_color(bg_color)
                cell_type, data_key, value = prop.get('type'), prop.get('data_key'), row.get(prop.get('data_key'))
                
                if cell_type == 'composite_id_name_industry':
                    stock_id, name, industry = row.get('stock_id', ''), row.get('公司簡稱', ''), row.get('產業類別', '')
                    it_reason = row.get('投信作多原因', '')
                    if it_reason:
                        ax.text(prop['x'] + 0.005, y_pos + 0.25, f"{stock_id} {name}", ha='left', va='center', fontsize=12, weight='bold', zorder=2, color=text_color)
                        ax.text(prop['x'] + 0.005, y_pos, f"{industry}", ha='left', va='center', fontsize=10, color=text_color, zorder=2, alpha=0.9)
                        ax.text(prop['x'] + 0.005, y_pos - 0.25, f"[{it_reason}]", ha='left', va='center', fontsize=11, weight='bold', color=text_color, zorder=2)
                    else:
                        ax.text(prop['x'] + 0.005, y_pos + 0.15, f"{stock_id} {name}", ha='left', va='center', fontsize=12, weight='bold', zorder=2, color=text_color)
                        ax.text(prop['x'] + 0.005, y_pos - 0.15, f"{industry}", ha='left', va='center', fontsize=10, color=text_color, zorder=2, alpha=0.9)
                
                elif cell_type == 'composite_id_name':
                    stock_id, name = row.get('stock_id', ''), row.get('公司簡稱', '')
                    ax.text(prop['x'] + prop['width']/2, y_pos, f"{stock_id} {name}", ha='center', va='center', fontsize=12, weight='bold', zorder=2, color=text_color)

                elif cell_type in ['heatmap_perf', 'heatmap_vol']:
                    numeric_value = pd.to_numeric(value, errors='coerce')
                    if pd.notna(numeric_value):
                        text_to_display = ''
                        if '凹單指標' in prop['header']:
                            text_to_display = f"{numeric_value:.2f}"
                        elif cell_type == 'heatmap_vol':
                            text_to_display = f"{numeric_value:,.0f}"
                        else: # 預設為 heatmap_perf
                            text_to_display = f"{numeric_value:+.2f}%"
                        ax.text(prop['x'] + prop['width']/2, y_pos, text_to_display, ha='center', va='center', fontsize=11, weight='bold', zorder=2, color=text_color)
                
                elif cell_type == 'heatmap_ratio':
                    display_key = prop.get('display_key', data_key)
                    numeric_value = pd.to_numeric(row.get(display_key), errors='coerce')
                    if pd.notna(numeric_value):
                        ax.text(prop['x'] + prop['width']/2, y_pos, f"{numeric_value:.2f}", ha='center', va='center', fontsize=11, weight='bold', zorder=2, color=text_color)

                elif cell_type == 'colored_text':
                    final_text_color = 'black'
                    value_str = str(value) if pd.notna(value) else ''
                    if 'color_map' in prop:
                        final_text_color = prop.get('color_map', {}).get(value, 'black')
                    else:
                        if '▲' in value_str or '漲' in value_str or '增' in value_str or '買' in value_str:
                            final_text_color = '#e63946'
                        elif '▼' in value_str or '跌' in value_str or '減' in value_str or '賣' in value_str:
                            final_text_color = '#2a9d8f'
                    ax.text(prop['x'] + prop['width']/2, y_pos, value_str if value_str else 'N/A', 
                            ha='center', va='center', fontsize=12, 
                            color=final_text_color, weight='bold', zorder=2)

                elif cell_type == 'custom_tags':
                    tags_text = str(row.get(data_key, ''))
                    wrapped_text = textwrap.fill(tags_text, width=15)
                    ax.text(prop['x'] + prop['width']/2, y_pos, wrapped_text, 
                            ha='center', va='center', fontsize=10, color='black', weight='bold', zorder=2,
                            linespacing=1.4)
                
                elif cell_type == 'composite_amt':
                    amt_text = self.format_revenue(value, None)
                    vol_text, vol_color = ("▲", 'red') if row.get('是否放量', False) else ("", text_color)
                    ax.text(prop['x'] + prop['width']/2 - 0.01, y_pos, amt_text, ha='center', va='center', fontsize=12, zorder=2, color=text_color)
                    ax.text(prop['x'] + prop['width'] - 0.02, y_pos, vol_text, ha='center', va='center', fontsize=14, color=vol_color, weight='bold', zorder=2)
                
                elif cell_type == 'composite_shareholder':
                    ratio_text, trend_text = row.get('大戶/散戶比', 'N/A'), row.get('籌碼趨勢(6週)', 'N/A')
                    ax.text(prop['x'] + prop['width']/2, y_pos + 0.15, f"大戶比: {ratio_text}", ha='center', va='center', fontsize=11, zorder=2, color=text_color)
                    ax.text(prop['x'] + prop['width']/2, y_pos - 0.15, trend_text, ha='center', va='center', fontsize=11, weight='bold', zorder=2, color=text_color)

                elif cell_type == 'heatmap_text_broker':
                    text = row.get('關鍵券商(4週)', 'N/A')
                    ax.text(prop['x'] + prop['width']/2, y_pos, textwrap.fill(str(text), width=12), ha='center', va='center', fontsize=10, zorder=2, weight='bold', color=text_color)
        
        table_x_start = list(col_defs.values())[0]['x']
        last_col_prop = list(col_defs.values())[-1]
        table_x_end = last_col_prop['x'] + last_col_prop['width']
        table_y_start, table_y_end = -0.5, num_rows - 0.5 + HEADER_HEIGHT
        table_width, table_height = table_x_end - table_x_start, table_y_end - table_y_start

        ax.add_patch(plt.Rectangle((table_x_start, table_y_start), table_width, table_height, 
                                    facecolor='none', edgecolor='black', lw=1.5, zorder=4, clip_on=False))
        




    # ===================================================================
    # --- 函式 2 (輔助繪圖): 繪製熱力圖儲存格 (v12.5.0 修正版) ---
    # ===================================================================
    def _draw_heatmap_cell(self, ax, x, y, width, height, text, value, norm, cmap, text_color=None, fontsize=11):
        """
        [doc]
        [v12.5.0 修正]
        - 簡化函式簽名，只接收一個已經決定好的 cmap。
        - 智慧判斷文字顏色：如果未指定，則根據背景亮度自動選擇黑色或白色。
        """
        if pd.notna(value) and norm is not None and cmap is not None:
            # 使用 abs(value) 來正規化，因為顏色方向 (cmap) 已由外部決定
            facecolor = cmap(norm(abs(value)))
            ax.add_patch(plt.Rectangle((x, y - height/2), width, height,
                                        facecolor=facecolor, edgecolor='white', lw=0.5, zorder=1))

            # 如果未指定文字顏色，則根據背景亮度自動決定
            if text_color is None:
                # RGB 轉換為 YIQ (亮度) 公式: Y = 0.299*R + 0.587*G + 0.114*B
                yiq = facecolor[0] * 0.299 + facecolor[1] * 0.587 + facecolor[2] * 0.114
                final_text_color = 'black' if yiq > 0.6 else 'white'
            else:
                final_text_color = text_color
        else:
            # 如果沒有值，文字預設為黑色
            final_text_color = 'black'

        if text: # 只在 text 有內容時才繪製文字
            ax.text(x + width/2, y, text, ha='center', va='center',
                    fontsize=fontsize, color=final_text_color, weight='bold', zorder=2)
                    
                    

    def _get_color_for_value(self, value):
        """根據數值的正負回傳紅色或綠色，0則回傳黑色"""
        if pd.isna(value):
            return 'gray'
        if value > 0:
            return 'red'
        elif value < 0:
            return 'green'
        else:
            return 'black'




    def _plot_market_margin_page(self, pdf, margin_data):
        """
        [doc]
        使用純 Matplotlib 繪製大盤融資指標。
        """
        logging.info("--- 開始繪製【大盤融資總覽】報告頁面 (Matplotlib) ---")
        if not margin_data:
            logging.warning("無大盤融資數據，跳過此頁面。")
            return

        try:
            df = margin_data['maintenance_df']
            df2 = margin_data['margin_change_df']
            benchmark = margin_data['benchmark']
            date_index = df.index

            fig, axes = plt.subplots(3, 1, figsize=(28, 24), sharex=True)
            fig.suptitle(
                f"台股大盤融資指標 ({margin_data['start_date']} 到 {margin_data['end_date']})", fontsize=28, y=0.95)

            # --- 圖一：融資維持率 ---
            ax1 = axes[0]
            ax1_twin = ax1.twinx()
            ax1.set_title('融資維持率 vs. 加權指數', fontsize=18)

            ax1.plot(date_index, benchmark, color='blue', label='加權指數')
            ax1_twin.plot(date_index, df['ratio'],
                          color='orange', label='融資維持率')
            ax1_twin.axhline(1.4, linestyle='--',
                             color='red', label='追繳壓力線 140%')

            ax1.set_ylabel('指\n數',rotation=0,labelpad=20, fontsize=14, color='blue')
            ax1_twin.set_ylabel('融\n資\n維\n持\n率\n(%)',rotation=0,labelpad=20, fontsize=14, color='orange')
            ax1.legend(loc='upper left')
            ax1_twin.legend(loc='upper right')
            ax1.grid(True, linestyle='--', alpha=0.6)

            # --- 圖二：上市融資餘額 ---
            ax2 = axes[1]
            ax2_twin = ax2.twinx()
            ax2.set_title('上市融資餘額 (億)', fontsize=18)

            ax2.fill_between(
                date_index, df2['上市融資交易金額'] / 100_000_000, color='#efd267', alpha=0.5, label='上市融資餘額')

            colors_tse = ['red' if v > 0 else 'green' for v in df2['上市融資買賣超']]
            ax2_twin.bar(date_index, df2['上市融資買賣超'],
                         color=colors_tse, label='上市融資買賣超')

            ax2.set_ylabel('總\n餘\n額\n(億)',rotation=0,labelpad=20,fontsize=14, color='darkgoldenrod')
            ax2_twin.set_ylabel('買\n賣\n超\n(億)',rotation=0,labelpad=20, fontsize=14)
            ax2.legend(loc='upper left')
            ax2_twin.legend(loc='upper right')

            # --- 圖三：上櫃融資餘額 ---
            ax3 = axes[2]
            ax3_twin = ax3.twinx()
            ax3.set_title('上櫃融資餘額 (億)', fontsize=18)

            ax3.fill_between(
                date_index, df2['上櫃融資交易金額'] / 100_000_000, color='#add8e6', alpha=0.5, label='上櫃融資餘額')

            colors_otc = ['red' if v > 0 else 'green' for v in df2['上櫃融資買賣超']]
            ax3_twin.bar(date_index, df2['上櫃融資買賣超'],
                         color=colors_otc, label='上櫃融資買賣超')

            ax3.set_ylabel('總\n餘\n額\n(億)',rotation=0,labelpad=20, fontsize=14, color='darkblue')
            ax3_twin.set_ylabel('買\n賣\n超\n(億)',rotation=0,labelpad=20, fontsize=14)
            ax3.legend(loc='upper left')
            ax3_twin.legend(loc='upper right')

            plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.93])
            pdf.savefig(fig)
            plt.close(fig)

        except Exception as e:
            logging.error(f"繪製大盤融資總覽頁面時出錯: {e}", exc_info=True)
            plt.close('all')
            
            

    def _plot_margin_trap_page(self, pdf, trap_df, start_str, end_str):
        """
        [doc]
        [v12.2.0 修正]
        - 修復此儀表板的「代碼名稱」欄位消失的問題。
        - 統一使用 _draw_dashboard_on_axes 繪圖引擎，確保風格一致。
        """
        logging.info("--- 開始繪製【(最終版)散戶凹單股儀表板】報告頁面 ---")
        if trap_df.empty:
            logging.info("本日無符合條件的散戶凹單股。")
            fig, ax = plt.subplots(figsize=(28, 24))
            ax.text(0.5, 0.5, "本日無符合條件的散戶凹單股", ha='center',
                    va='center', fontsize=30, color='gray')
            ax.set_title(f"融資凹單股分析 ({start_str} to {end_str})", fontsize=22)
            ax.axis('off')
            pdf.savefig(fig)
            plt.close(fig)
            return

        try:
            # --- 圖一：Treemap (維持不變) ---
            fig1, ax1 = plt.subplots(figsize=(28, 20))
            trap_df_sorted_treemap = trap_df.sort_values(by='market_value', ascending=False)
            sizes = trap_df_sorted_treemap['market_value'].values
            labels = trap_df_sorted_treemap.apply(lambda row: f"{row['stock_id_name']}\n凹單率:{row['trap_ratio']:.2f}", axis=1)
            colors_data = trap_df_sorted_treemap['trap_ratio'].values
            norm = mcolors.Normalize(vmin=min(colors_data), vmax=0)
            cmap = plt.colormaps.get_cmap('Reds_r')
            colors = [cmap(norm(value)) for value in colors_data]
            squarify.plot(sizes=sizes, label=labels, color=colors, alpha=0.8, ax=ax1,
                        text_kwargs={'fontsize': 10, 'color': 'black', 'weight': 'bold'})
            cbar_ax = fig1.add_axes([0.92, 0.25, 0.02, 0.5])
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])
            cbar = fig1.colorbar(sm, cax=cbar_ax)
            cbar.set_label('凹單指標 (數值越小, 散戶攤平力道越強)', rotation=270, labelpad=20)
            ax1.set_title(f'融資凹單股 Treemap (依市值區分) ({start_str} to {end_str})', fontsize=22)
            ax1.axis('off')
            fig1.subplots_adjust(left=0.02, right=0.90, top=0.95, bottom=0.02)
            pdf.savefig(fig1)
            plt.close(fig1)

            # --- 圖二：長條圖 (維持不變) ---
            fig2, ax2 = plt.subplots(figsize=(28, 16))
            short_df = trap_df.sort_values('trap_ratio').head(30)
            ax2.bar(short_df['stock_id_name'], short_df['trap_ratio'], color='darkred')
            ax2.set_title(f'Top 30 融資凹單力道 ({start_str} to {end_str})', fontsize=22)
            ax2.set_ylabel('凹單指標 (負值越大越危險)')
            ax2.tick_params(axis='x', rotation=45, labelsize=12)
            ax2.grid(True, axis='y', linestyle='--')
            fig2.tight_layout()
            pdf.savefig(fig2)
            plt.close(fig2)

            # --- 圖三：凹單股詳細數據儀表板 ---
            logging.info("--- 開始繪製【散戶凹單股】詳細數據儀表板 ---")

            # 1. 數據整合
            summary_cols = ['stock_id', '公司簡稱', '情緒結論', '大戶/散戶比', '籌碼趨勢(6週)', '關鍵券商(4週)', '大戶/散戶比_raw', '關鍵券商(4週)_raw']
            if not self.industry_df.empty:
                # [修正] 確保 公司簡稱 也被合併進來
                merged_trap_df = pd.merge(trap_df, self.industry_df[summary_cols], on='stock_id', how='left')
            else:
                merged_trap_df = trap_df
                for col in summary_cols[1:]:
                    merged_trap_df[col] = 'N/A'
            
            fill_values = {'情緒結論': '數據不足', '大戶/散戶比': 'N/A', '籌碼趨勢(6週)': '', '關鍵券商(4週)': '數據不足'}
            merged_trap_df.fillna(value=fill_values, inplace=True)

            # 2. 分頁與繪圖
            df_sorted = merged_trap_df.sort_values(by='trap_ratio').reset_index(drop=True)
            STOCKS_PER_PAGE = 20
            num_pages = int(np.ceil(len(df_sorted) / STOCKS_PER_PAGE))

            for page_num in range(num_pages):
                start_index = page_num * STOCKS_PER_PAGE
                end_index = start_index + STOCKS_PER_PAGE
                page_data = df_sorted.iloc[start_index:end_index]

                fig3, ax3 = plt.subplots(figsize=(28, 24))

                # 3. 定義儀表板設定
                page_info = f" (第 {page_num + 1}/{num_pages} 頁)" if num_pages > 1 else ""
                
                # [v12.2.0 核心修正]
                config = {
                    'title': f"融資凹單股詳細指標 ({start_str} to {end_str}){page_info}",
                    'heatmap_cols': ['return_ratio', 'margin_balance_change', 'trap_ratio'],
                    'col_defs': {
                        'id_name':    {'x': 0.01, 'width': 0.14, 'header': '代碼 名稱', 'type': 'composite_id_name'}, # 改用 composite_id_name 確保渲染
                        'return':     {'x': 0.15, 'width': 0.08, 'header': '期間跌幅(%)', 'type': 'heatmap_perf', 'data_key': 'return_ratio'},
                        'margin_chg': {'x': 0.23, 'width': 0.08, 'header': '融資增幅(%)', 'type': 'heatmap_perf', 'data_key': 'margin_balance_change'},
                        'trap_ratio': {'x': 0.31, 'width': 0.08, 'header': '凹單指標', 'type': 'heatmap_perf', 'data_key': 'trap_ratio'},
                        'sentiment':  {'x': 0.40, 'width': 0.12, 'header': '融資情緒', 'type': 'colored_text', 'data_key': '情緒結論',
                                    'color_map': {'籌碼趨向多方': 'red', '軋空行情醞釀中': 'darkorange', '籌碼趨向空方': 'green', '多空交戰激烈': 'blue', '市場人氣退潮': 'gray', '數據不足': 'black'}},
                        'shareholder': {'x': 0.53, 'width': 0.18, 'header': '大戶籌碼', 'type': 'composite_shareholder'},
                        'broker': {'x': 0.71, 'width': 0.28, 'header': '關鍵券商', 'type': 'heatmap_text_broker'},
                    }
                }

                # 4. 呼叫通用引擎繪圖
                self._draw_dashboard_on_axes(ax3, page_data, config)

                plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.93])
                pdf.savefig(fig3)
                plt.close(fig3)
                gc.collect()

        except Exception as e:
            logging.error(f"繪製散戶凹單股頁面時出錯: {e}", exc_info=True)
            plt.close('all')



        # finlab程式碼格式
    # doc: [v6.0 修改]
    def _plot_new_high_appendix_page(self, pdf):
        """
        doc:
        [v6.0 修改]
        - 移除原有的 head() 數量限制。
        - 改為調用 _apply_priority_filtering 函式，優先保留滿足營收動能條件的股票。
        """
        if self.new_high_stocks_df.empty:
            return
            
        config = self.config.get('APPENDIX_SETTINGS', {}).get('NEW_HIGH_200', {})
        max_plots = config.get('MAX_PLOTS', 30)
        
        # [v6.0 核心修改] 使用新的優先權篩選器
        stocks_to_plot_df = self._apply_priority_filtering(
            df=self.new_high_stocks_df,
            priority_check_func=self._has_strong_revenue_momentum,
            max_regular_plots=max_plots
        )

        self._plot_appendix_section(
            pdf=pdf,
            stocks_df=stocks_to_plot_df,
            title_text="創 200 日新高",
            subtitle_text=f"今日創下200日以來最高價的個股 (依成交額排序)",
            subtitle_color='#FF4500',
            source_topic="創200日高",
            source_details_formatter=lambda row: f"成交額排名: {row['rank']}",
            image_section_key='E1_APPENDIX_NEW_HIGH_200'
        )
            
        

    # finlab程式碼格式
    # doc: [v6.0 修改]
    def _plot_new_high_20_appendix_page(self, pdf):
        """
        doc:
        [v6.0 修改]
        - 移除原有的 head() 數量限制。
        - 改為調用 _apply_priority_filtering 函式，優先保留滿足營收動能條件的股票。
        """
        if self.new_high_20_stocks_df.empty:
            return
            
        config = self.config.get('APPENDIX_SETTINGS', {}).get('NEW_HIGH_20', {})
        max_plots = config.get('MAX_PLOTS', 30)
        
        # [v6.0 核心修改] 使用新的優先權篩選器
        stocks_to_plot_df = self._apply_priority_filtering(
            df=self.new_high_20_stocks_df,
            priority_check_func=self._has_strong_revenue_momentum,
            max_regular_plots=max_plots
        )

        self._plot_appendix_section(
            pdf=pdf,
            stocks_df=stocks_to_plot_df,
            title_text="創 20 日新高",
            subtitle_text=f"今日創下20日以來最高價的個股 (依成交額排序)",
            subtitle_color='#FFA500',
            source_topic="創20日高",
            source_details_formatter=lambda row: f"成交額排名: {row['rank']}",
            image_section_key='E2_APPENDIX_NEW_HIGH_20'
        )
            
        
        

    # finlab程式碼格式
    # [v12.0 最終修正] 請用此版本替換掉您現有的 _create_multifactor_summary_page 函式
    def _create_multifactor_summary_page(self, pdf):
        """
        [doc]
        [v12.0 最終修正版]
        - [核心修正] 改為呼叫與「創高股儀表板」相同的繪圖引擎 `_draw_dashboard_on_axes`，確保版面配置、字體、高度完全一致。
        - 重新為此儀表板量身打造一組新的欄位佈局設定 `config`。
        - 解決所有已知的排版問題 (擁擠、重疊、字體過小)。
        """
        logging.info("--- 開始建立【多因子強勢股總覽】報告頁面 ---\n")

        if self.industry_df is None or self.industry_df.empty:
            logging.warning("主摘要數據為空，無法生成多因子總覽頁面。")
            return

        industry_perf = self.industry_df.groupby('產業類別')['漲跌幅_20日(%)'].mean()
        strong_industry_list = industry_perf.nlargest(10).index.tolist()

        strong_stocks = self.industry_df[
            self.industry_df['產業類別'].isin(strong_industry_list) &
            self.industry_df['是否創200日新高']
        ].copy()

        if strong_stocks.empty:
            logging.info("未找到同時滿足『強勢產業』與『創200日新高』的股票。")
            # 即使沒有股票，也產生一個空的說明頁面，維持報告結構完整
            fig, ax = plt.subplots(figsize=(28, 24))
            ax.text(0.5, 0.5, "未找到同時滿足『強勢產業』與『創200日新高』的股票", 
                    ha='center', va='center', fontsize=30, color='gray')
            ax.set_title(f"多因子強勢股總覽 (資料日期: {self.industry_df['資料日期'].iloc[0]})", fontsize=22)
            ax.axis('off')
            pdf.savefig(fig)
            plt.close(fig)
            return

        strategy_stocks_set = set(self.result_df['股票代碼'].tolist())
        rank_jump_set = set(self.rank_jump_stocks_df['stock_id'].tolist()) if not self.rank_jump_stocks_df.empty else set()

        conditions = []
        for index, row in strong_stocks.iterrows():
            tags = ['強勢產業', '創200日新高']
            if row['stock_id'] in rank_jump_set:
                tags.append('排名躍升')
            if row['stock_id'] in strategy_stocks_set:
                tags.append('策略選中')
            conditions.append(", ".join(tags))
        strong_stocks['符合條件'] = conditions

        # 1. 準備數據
        # [修改] 排序並限制最多顯示 20 檔，避免表格過長
        summary_df = strong_stocks.sort_values(by='漲跌幅_5日(%)', ascending=False).head(20).reset_index(drop=True)

        # 2. 定義儀表板設定 (參照 _draw_dashboard_on_axes 的格式)
        config = {
            'title': f"多因子強勢股總覽 (資料日期: {self.industry_df['資料日期'].iloc[0]})",
            'heatmap_cols': ['漲跌幅_5日(%)', '大戶/散戶比_raw', '情緒結論_raw', '關鍵券商(4週)_raw'],
            'col_defs': {
                'id_name_industry': {'x': 0.00, 'width': 0.20, 'header': '代碼/名稱/產業', 'type': 'composite_id_name_industry'},
                'perf_5d':      {'x': 0.20, 'width': 0.08, 'header': '5日漲跌', 'type': 'heatmap_perf', 'data_key': '漲跌幅_5日(%)'},
                'sentiment':    {'x': 0.28, 'width': 0.12, 'header': '融資情緒', 'type': 'colored_text', 'data_key': '情緒結論', 
                                'color_map': {'籌碼趨向多方': '#e63946', '軋空行情醞釀中': '#f7a072', '籌碼趨向空方': '#2a9d8f', '多空交戰激烈': '#1d3557', '市場人氣退潮': '#6c757d', '數據不足': 'black'}},
                'shareholder':  {'x': 0.40, 'width': 0.18, 'header': '大戶籌碼', 'type': 'composite_shareholder', 'data_key': '大戶/散戶比_raw'},
                'broker':       {'x': 0.58, 'width': 0.22, 'header': '關鍵券商', 'type': 'heatmap_text_broker', 'data_key': '關鍵券商(4週)_raw'},
                # '符合條件' 欄位較為特殊，我們將手動在繪圖引擎中處理它
                'conditions':   {'x': 0.80, 'width': 0.20, 'header': '符合條件', 'type': 'custom_tags', 'data_key': '符合條件',
                                'color_map': {'策略選中': 'darkorange', '排名躍升': 'purple', '強勢產業': 'darkgreen', '創200日新高': 'darkred'}}
            }
        }

        # 3. 繪圖
        fig, ax = plt.subplots(figsize=(28, 24))
        # [核心修正] 呼叫成功的繪圖引擎
        self._draw_dashboard_on_axes(ax, summary_df, config)
        
        # 使用 tight_layout 確保所有元素都在頁面內
        plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.93])
        pdf.savefig(fig)
        plt.close(fig)




    # finlab程式碼格式
    # [v4.0 穩健性增強版] 請用此版本替換掉您現有的 _plot_appendix_section 函式
    def _plot_appendix_section(self, pdf, stocks_df, title_text, subtitle_text, subtitle_color, source_topic, source_details_formatter, image_section_key):
        """
        doc:
        [v4.0 穩健性增強版]
        - [核心修改] 新增對 _plot_stock_page 回傳的圖表物件 (fig_stock) 的存在性檢查。
        - 此修改可確保只有成功生成的圖表會被寫入 PDF，進一步解決 matplotlib 的 'UserWarning'。
        - 簡化 broker_fig 的日誌記錄，避免重複訊息。

        [v21.0 資訊整合與去重版]
        - 新增通用的附錄區塊繪圖函式。
        - 在繪圖前，會過濾掉 self.plotted_stock_ids 中已存在的股票。
        - 在繪圖後，會將成功繪製的股票加入 self.plotted_stock_ids。
        [GEMINI 修改]
        - 查詢 self.topic_map 取得個股原始主題。
        - 將原始主題與附錄來源 (source_topic) 組合後，顯示於圖表標題。
        """
        if stocks_df.empty:
            logging.info(f"無符合 '{title_text}' 條件的股票可繪製於附錄。")
            return

        # --- 繪圖前過濾 (邏輯不變) ---
        original_count = len(stocks_df)
        stocks_to_plot_df = stocks_df[~stocks_df['stock_id'].isin(self.plotted_stock_ids)].copy()
        filtered_count = len(stocks_to_plot_df)

        if filtered_count < original_count:
            logging.info(f"[{title_text}] 去重: 從 {original_count} 檔過濾掉 {original_count - filtered_count} 檔已繪製過的股票，剩餘 {filtered_count} 檔。")
        
        if stocks_to_plot_df.empty:
            logging.info(f"無符合 '{title_text}' 條件且尚未繪製的股票可於附錄顯示。")
            return
        
        # --- 繪製附錄標題頁 (邏輯不變) ---
        fig_title = plt.figure(figsize=(28, 24))
        fig_title.text(0.5, 0.6, '附錄', ha='center', va='center', fontsize=40, color='gray', alpha=0.5)
        fig_title.text(0.5, 0.5, title_text, ha='center', va='center', fontsize=54, color='black')
        fig_title.text(0.5, 0.4, subtitle_text, ha='center', va='center', fontsize=24, color=subtitle_color)
        pdf.savefig(fig_title)
        plt.close(fig_title)

        save_image_sections = self.config.get('SAVE_IMAGE_SECTIONS', {})

        # --- [v4.0 核心修改] 迴圈繪製個股分析圖 ---
        for _, row in stocks_to_plot_df.iterrows():
            stock_id = str(row['stock_id'])
            stock_name = row['公司簡稱']
            
            if stock_id not in self.analyzer.all_data.get('close', pd.DataFrame()).columns:
                logging.warning(f"[附錄] 跳過 {stock_id} {stock_name}：該股因數據不齊全，在『預加載階段』已被過濾。")
                continue

            try:
                analysis_data = self.analyzer.analyze(stock_id)
                if analysis_data is None:
                    logging.warning(f"[附錄] 跳過 {stock_id} {stock_name}：該股在『個股分析階段』因數據長度不足被過濾。")
                    continue

                # --- 準備標題資訊 (邏輯不變) ---
                strategy_details = source_details_formatter(row)
                try:
                    real_topic = self.topic_map.get(int(stock_id), '無主題')
                    if pd.isna(real_topic): real_topic = '無主題'
                except (ValueError, TypeError):
                    real_topic = '無主題'
                combined_topic = f"{real_topic} / {source_topic}"
                mock_info_row = pd.Series({
                    '股票代碼': stock_id, '股票名稱': stock_name, 
                    '主題': combined_topic, '對應策略': strategy_details
                })
                
                # --- 繪製第一張「六宮格分析圖」 ---
                logging.info(f"正在分析與繪製附錄股 ({source_topic}): {stock_id} {stock_name}")
                fig_stock = self._plot_stock_page(analysis_data, mock_info_row)
                
                # [v4.0 核心修改] 新增對 fig_stock 物件的穩健性檢查
                if fig_stock:
                    pdf.savefig(fig_stock)
                    if save_image_sections.get(image_section_key, False):
                        self._save_figure_if_enabled(fig_stock, f"{stock_id}_{stock_name}_附錄_{source_topic}_六宮格圖", stock_id)
                    plt.close(fig_stock)
                else:
                    logging.warning(f"    ↳ 因數據不足或繪圖失敗，已跳過附錄股 {stock_id} 的六宮格圖。")

                # --- 繪製第二張「券商綜合分析圖」 ---
                logging.info(f"    ↳ 正在生成 {stock_id} 的券商分析圖...")
                broker_fig = self.broker_chart_generator.generate_chart(stock_id)

                # [v4.0 核心修改] 簡化日誌，因為 generate_chart 內部已有警告
                if broker_fig:
                    pdf.savefig(broker_fig)
                    if save_image_sections.get(image_section_key, False):
                        self._save_figure_if_enabled(broker_fig, f"{stock_id}_{stock_name}_附錄_{source_topic}_券商分析圖", stock_id)
                    plt.close(broker_fig)
                
                # 無論是否成功，都將此 stock_id 加入已處理清單，避免重複嘗試
                self.plotted_stock_ids.add(stock_id)
                gc.collect()

            except Exception as e:
                logging.error(f"為附錄股 {stock_id} ({source_topic}) 生成圖表時發生嚴重錯誤: {e}", exc_info=True)
                plt.close('all')
                gc.collect()

        logging.info(f"成功繪製 {len(stocks_to_plot_df)} 檔 '{title_text}' 附錄圖表。")








    def _plot_20day_high_page(self, pdf):
        """
        [doc]
        繪製「創20日新高股儀表板」的頁面。
        [修改] 複製 200 日儀表板的欄位設定，統一設計。
        """
        logging.info("--- 開始繪製【(升級版)創20日新高多因子儀表板】---\n")
        if self.market_breadth_data is None or 'latest_new_high_20_list' not in self.market_breadth_data:
            logging.warning("無創20日新高個股數據，跳過此頁面生成。")
            return

        new_high_20_stock_ids = []
        if not self.market_breadth_data.get('latest_new_high_20_list', pd.DataFrame()).empty:
            new_high_20_stock_ids = self.market_breadth_data['latest_new_high_20_list']['stock_id'].tolist()

        if not new_high_20_stock_ids:
            fig, ax = plt.subplots(figsize=(28, 24))
            ax.text(0.5, 0.5, "本日無股票創20日新高", ha='center',
                    va='center', fontsize=30, color='gray')
            ax.axis('off')
            pdf.savefig(fig)
            plt.close(fig)
            return

        new_high_20_list_df = self.industry_df[self.industry_df['stock_id'].isin(
            new_high_20_stock_ids)].copy()

        MAX_STOCKS_TO_SHOW = 50
        total_initial_stocks = len(new_high_20_list_df)
        is_limited = False
        if total_initial_stocks > MAX_STOCKS_TO_SHOW:
            is_limited = True
            volatility = compute_candle_volatility()
            latest_vol = volatility.iloc[-1]
            new_high_20_list_df['volatility'] = new_high_20_list_df['stock_id'].map(
                latest_vol).fillna(float('inf'))
            new_high_20_list_df['priority'] = np.where(
                new_high_20_list_df['volatility'] <= 10, 1, 2)
            df_for_display = new_high_20_list_df.sort_values(
                by=['priority', '當日漲跌幅(%)'], ascending=[True, False]
            ).head(MAX_STOCKS_TO_SHOW)
        else:
            df_for_display = new_high_20_list_df.sort_values(
                by='當日漲跌幅(%)', ascending=False)

        df_sorted = df_for_display.reset_index(drop=True)

        STOCKS_PER_PAGE = 20
        num_pages = int(np.ceil(len(df_sorted) / STOCKS_PER_PAGE))

        for page_num in range(num_pages):
            start_index = page_num * STOCKS_PER_PAGE
            end_index = start_index + STOCKS_PER_PAGE
            page_data = df_sorted.iloc[start_index:end_index]

            fig, ax = plt.subplots(figsize=(28, 24))
            limit_info = f" (從 {total_initial_stocks} 檔中篩選 Top {len(df_for_display)})" if is_limited else ""
            page_info = f" (第 {page_num + 1}/{num_pages} 頁)" if num_pages > 1 else ""

            # doc: =============================================================================
            # doc: --- [核心修改] 在 config 中加入 '產業量能比' 欄位 ---
            # doc: =============================================================================
            config = {
                'title': f"創20日新高(短期動能股) 多因子儀表板{limit_info}{page_info}",
                'title_fontsize': 20,
                'heatmap_cols': ['當日漲跌幅(%)', '漲跌幅_5日(%)', '外資佔市值比_5日', '投信佔市值比_5日', 
                                '外資買賣超_5日', '投信買賣超_5日', '成交金額', '大戶/散戶比_raw', 
                                '情緒結論_raw', '關鍵券商(4週)_raw', 
                                '營收YoY(%)', '營收動能(3/12)', '產業量能比'], # [新增]
                'col_defs': {
                    'id_name_industry': {'x': 0.00, 'width': 0.12, 'header': '代碼/名稱/產業', 'type': 'composite_id_name_industry'},
                    'perf_1d':      {'x': 0.12, 'width': 0.05, 'header': '日漲跌', 'type': 'heatmap_perf', 'data_key': '當日漲跌幅(%)'},
                    'perf_5d':      {'x': 0.17, 'width': 0.05, 'header': '5日漲跌', 'type': 'heatmap_perf', 'data_key': '漲跌幅_5日(%)'},
                    'yoy':          {'x': 0.22, 'width': 0.05, 'header': '營收YoY', 'type': 'heatmap_perf', 'data_key': '營收YoY(%)'},
                    'rev_mom':      {'x': 0.27, 'width': 0.05, 'header': '營收動能', 'type': 'heatmap_ratio', 'data_key': '營收動能(3/12)', 'display_key':'營收動能(3/12)_ratio'},
                    'industry_flow':{'x': 0.32, 'width': 0.05, 'header': '產業量能', 'type': 'heatmap_ratio', 'data_key': '產業量能比', 'display_key':'產業量能比_ratio'}, # [新增]
                    'amt':          {'x': 0.37, 'width': 0.07, 'header': '成交額', 'type': 'composite_amt', 'data_key': '成交金額'},
                    'sentiment':    {'x': 0.44, 'width': 0.08, 'header': '融資情緒', 'type': 'colored_text', 'data_key': '情緒結論', 'color_map': {'籌碼趨向多方': '#e63946', '軋空行情醞釀中': '#f7a072', '籌碼趨向空方': '#2a9d8f', '多空交戰激烈': '#1d3557', '市場人氣退潮': '#6c757d', '數據不足': 'black'}},
                    'shareholder':  {'x': 0.52, 'width': 0.11, 'header': '大戶籌碼', 'type': 'composite_shareholder', 'data_key': '大戶/散戶比_raw'},
                    'broker':       {'x': 0.63, 'width': 0.11, 'header': '關鍵券商', 'type': 'heatmap_text_broker', 'data_key': '關鍵券商(4週)_raw'},
                    'fi_buy':       {'x': 0.74, 'width': 0.06, 'header': '外資5日超', 'type': 'heatmap_vol', 'data_key': '外資買賣超_5日'},
                    'it_buy':       {'x': 0.80, 'width': 0.06, 'header': '投信5日超', 'type': 'heatmap_vol', 'data_key': '投信買賣超_5日'},
                    'fi_ratio':     {'x': 0.86, 'width': 0.07, 'header': '外資佔比', 'type': 'heatmap_perf', 'data_key': '外資佔市值比_5日'},
                    'it_ratio':     {'x': 0.93, 'width': 0.07, 'header': '投信佔比', 'type': 'heatmap_perf', 'data_key': '投信佔市值比_5日'},
                }
            }
            # doc: --- [修改結束] ---

            self._draw_dashboard_on_axes(ax, page_data, config)
            plt.tight_layout(rect=[0.02, 0.02, 0.98, 0.93])
            pdf.savefig(fig)
            plt.close(fig)
            gc.collect()

    def _get_color_for_value(self, value):
        """根據數值的正負回傳紅色或綠色，0則回傳黑色"""
        if pd.isna(value):
            return 'gray'
        if value > 0:
            return 'red'
        elif value < 0:
            return 'green'
        else:
            return 'black'





    # ===================================================================
    # --- 函式 1: 「策略選股」圖表 (v14.1.0 邏輯修正版) ---
    # ===================================================================
    def _plot_all_stock_performance_page(self, pdf):
        """
        [doc]
        [v14.1.0 修正]
        - 修正核心 Bug：此圖表現在會正確地只篩選 `select_stock.xlsx` 中指定的股票，
        而不再錯誤地包含其他來源（如創高股、躍升股）的股票。
        """
        # [v14.1.0 核心修正] 檢查策略選股結果 (self.result_df)
        if self.result_df.empty:
            logging.warning("策略選股結果 (self.result_df) 為空，無法生成個股漲跌幅比較頁面。")
            # 為了報告結構完整，即使無數據也產生一個說明頁
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, "策略選股清單 (select_stock.xlsx) 中無股票，\n故無法生成此頁面。",
                    ha='center', va='center', fontsize=20, color='gray')
            ax.set_title("策略選出個股漲跌幅與成交金額趨勢比較", fontsize=22)
            pdf.savefig(fig)
            plt.close(fig)
            return

        # [v14.1.0 核心修正] 從 self.industry_df 中，只篩選出來自策略選股的股票
        strategy_stock_ids = self.result_df['股票代碼'].tolist()
        stocks_to_plot_df = self.industry_df[self.industry_df['stock_id'].isin(strategy_stock_ids)].copy()
        
        if stocks_to_plot_df.empty:
            logging.warning("在主資料表中找不到任何策略選股的數據，無法生成個股漲跌幅比較頁面。")
            # ... (此處省略繪製錯誤訊息頁面的代碼) ...
            return

        # --- 後續的繪圖邏輯維持不變，但現在是基於正確的數據源 ---
        
        # 準備 Y 軸標籤
        base_label = '[' + stocks_to_plot_df['產業類別'].astype(str) + '] ' + \
            stocks_to_plot_df['stock_id'] + ' ' + \
            stocks_to_plot_df['公司簡稱']
        if '成交額大於5日均額' in stocks_to_plot_df.columns:
            is_volume_up = stocks_to_plot_df['成交額大於5日均額'].fillna(False)
            vol_tag = np.where(is_volume_up, ' 【放量】', '')
            stocks_to_plot_df['標註'] = base_label + vol_tag
        else:
            stocks_to_plot_df['標註'] = base_label

        stocks_to_plot_df = stocks_to_plot_df.sort_values(
            '漲跌幅_5日(%)', ascending=False)

        total_stocks = len(stocks_to_plot_df)
        for page_start in range(0, total_stocks, 10):
            page_end = min(page_start + 10, total_stocks)
            page_df = stocks_to_plot_df.iloc[page_start:page_end]

            fig = plt.figure(figsize=(28, 14))
            gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1.2], wspace=0.3)
            ax1 = fig.add_subplot(gs[0, 0])
            ax2_container = fig.add_subplot(gs[0, 1])

            title_date = page_df['資料日期'].iloc[0] if not page_df.empty else datetime.now().strftime('%Y-%m-%d')
            fig.suptitle(
                f"策略選出個股漲跌幅與成交金額趨勢比較 (資料日期: {title_date}) 第 {page_start//10+1} 頁",
                fontsize=22, fontweight='bold', y=0.98
            )

            # --- 左圖：價格趨勢（漲跌幅） ---
            y_pos = np.arange(len(page_df))
            bar_height = 0.2
            offsets = [0.2, 0, -0.2]
            price_colors = ['#e63946', '#f7a072', '#a2d2ff']
            price_labels = ['5日漲跌幅', '20日漲跌幅', '60日漲跌幅']
            price_cols = ['漲跌幅_5日(%)', '漲跌幅_20日(%)', '漲跌幅_60日(%)']
            for i in range(len(price_cols)):
                ax1.barh(y_pos + offsets[i], page_df[price_cols[i]],
                        height=bar_height, color=price_colors[i], label=price_labels[i])
            ax1.set_yticks(y_pos, page_df['標註'], fontsize=12)
            ax1.invert_yaxis()
            ax1.set_xlabel('漲跌幅 (%)', fontsize=12)
            ax1.legend(title='期間', fontsize=12)
            ax1.grid(True, axis='x', linestyle='--', alpha=0.7)
            ax1.set_title('個股漲跌幅比較 (依5日漲跌幅排序)', fontsize=18)
            ax1.axvline(0, color='black', linewidth=0.5)

            # --- 右圖：成交金額趨勢圖 ---
            ax2_container.spines['top'].set_visible(False)
            ax2_container.spines['right'].set_visible(False)
            ax2_container.spines['bottom'].set_visible(False)
            ax2_container.spines['left'].set_visible(False)
            ax2_container.set_xticks([])
            ax2_container.set_yticks([])
            ax2_container.set_title('近60日成交金額趨勢', fontsize=18)

            gs2 = gridspec.GridSpecFromSubplotSpec(
                len(page_df), 1, subplot_spec=gs[0, 1], hspace=0.1)

            for i, stock_id in enumerate(page_df['stock_id']):
                ax_mini = fig.add_subplot(gs2[i])
                if stock_id not in self.amt_df.columns:
                    ax_mini.text(0.5, 0.5, '無成交額數據', ha='center',
                                va='center', color='gray')
                    ax_mini.set_xticks([])
                    ax_mini.set_yticks([])
                    continue

                amt_series = self.amt_df[stock_id].tail(60)
                if amt_series.empty:
                    ax_mini.set_visible(False)
                    continue

                amt_ma5 = amt_series.rolling(5).mean()
                amt_ma20 = amt_series.rolling(20).mean()
                is_vol_spike = amt_series > (amt_ma20 * 1.5)
                colors = np.where(is_vol_spike, '#e63946', '#adb5bd')

                ax_mini.bar(amt_series.index, amt_series,
                            color=colors, width=1.0)
                ax_mini.plot(amt_ma5, color='#fca311',
                            linewidth=2.0, label='MA5')
                ax_mini.plot(amt_ma20, color='#1d3557',
                            linewidth=2.0, label='MA20')
                ax_mini.yaxis.set_major_formatter(
                    FuncFormatter(self.format_revenue))
                ax_mini.tick_params(axis="x", labelsize=9, labelrotation=20)
                if i < len(page_df) - 1:
                    ax_mini.set_xticklabels([])
                if i == 0:
                    ax_mini.legend(fontsize='small', loc='upper left')

            fig.subplots_adjust(left=0.2, right=0.98, top=0.92, bottom=0.08)
            pdf.savefig(fig)
            plt.close(fig)
            gc.collect()

    # [新增] 繪製成交金額排名躍升股的獨立報告區塊

    # finlab程式碼格式
    def _plot_rank_jump_stock_pages(self, pdf):
        """[重構] 呼叫通用附錄函式"""
        logging.info("--- 開始繪製【成交金額排名躍升股】詳細報告 ---\n")
        # [核心修正] 在繪圖的最開始，就檢查數據是否存在
        if self.rank_jump_stocks_df.empty:
            logging.info("無排名躍升股可供分析，跳過此章節。")
            # 建立一個提示頁面，而不是產生一個空的圖表導致警告
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, "本日無符合條件的【排名躍升股】",
                    ha='center', va='center', fontsize=24, color='gray')
            ax.axis('off')
            pdf.savefig(fig)
            plt.close(fig)
            return # 直接結束此函式，不往下執行

        
        # [新增] 在此處加入篩選步驟
        df_to_plot = self._apply_appendix_filter(self.rank_jump_stocks_df)
        
        if df_to_plot.empty:
            logging.info("排名躍升股經過篩選後無剩餘股票，跳過此章節。")
            # 同樣建立提示頁面
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, "排名躍升股經過漲跌幅篩選後無剩餘股票",
                    ha='center', va='center', fontsize=24, color='gray')
            ax.axis('off')
            pdf.savefig(fig)
            plt.close(fig)
            return

        def formatter(row):
            return [row['排名變化說明']]

        self._plot_appendix_section(
            pdf=pdf,
            stocks_df=df_to_plot,
            title_text='成交金額排名躍升股分析',
            subtitle_text=f"基於市場傳說「弱者變強」的選股邏輯\n(資料日期: {self.industry_df['資料日期'].iloc[0]})",
            subtitle_color='darkblue',
            source_topic='排名躍升股',
            source_details_formatter=formatter,
            image_section_key='E4_APPENDIX_RANK_JUMP'
        )

 

# ===================================================================
# --- 函式 2: 強勢產業深入分析圖 (v15.2.0 排版修正完整版) ---
# ===================================================================
    def _plot_custom_stock_industry_strength_page(self, pdf):
            """
            [doc]
            [v15.2.0 修正]
            - 子圖比例動態調整：根據最長的Y軸標籤（產業+股票名稱），自動調整左側邊界。
            - 提供完整程式碼，無任何省略。
            [本次修改]
            - 在建立 page_df 切片時，加上 .copy()，以避免 SettingWithCopyWarning。
            """
            if self.industry_df is None or self.industry_df.empty:
                fig, ax = plt.subplots(figsize=(28, 12))
                ax.text(0.5, 0.5, "產業分析數據不足或加載失敗，無法生成自選清單產業強弱分析頁面。",
                        ha='center', va='center', fontsize=20, color='red')
                pdf.savefig(fig)
                plt.close(fig)
                return

            # 只針對策略選出股票進行分析
            custom_industry_df = self.industry_df[
                self.industry_df['stock_id'].isin(self.result_df['股票代碼'].tolist())
            ].copy()
            if custom_industry_df.empty:
                fig, ax = plt.subplots(figsize=(28, 12))
                ax.text(0.5, 0.5, "無策略選出股票產業資料，無法生成自選清單產業強弱分析頁面。",
                        ha='center', va='center', fontsize=20, color='red')
                pdf.savefig(fig)
                plt.close(fig)
                return

            # --- 1. 準備與排序產業資料 ---
            perf_cols = ['漲跌幅_5日(%)']
            industry_performance = custom_industry_df.dropna(
                subset=perf_cols).groupby('產業類別')[perf_cols].mean()
            sorted_industries = industry_performance.sort_values(
                '漲跌幅_5日(%)', ascending=False).index.tolist()
            grouped_by_industry = custom_industry_df.groupby('產業類別', observed=True)

            # --- 2. 智慧型分組/合併邏輯 ---
            plot_batches = []
            temp_batch = []
            temp_stock_count = 0
            for industry_name in sorted_industries:
                current_group = grouped_by_industry.get_group(industry_name)
                current_stock_count = len(current_group)
                if current_stock_count >= 10:
                    if temp_batch:
                        plot_batches.append(temp_batch)
                        temp_batch = []
                        temp_stock_count = 0
                    plot_batches.append([industry_name])
                else:
                    if not temp_batch:
                        temp_batch.append(industry_name)
                        temp_stock_count = current_stock_count
                    elif temp_stock_count + current_stock_count <= 10:
                        temp_batch.append(industry_name)
                        temp_stock_count += current_stock_count
                    else:
                        plot_batches.append(temp_batch)
                        temp_batch = [industry_name]
                        temp_stock_count = current_stock_count
            if temp_batch:
                plot_batches.append(temp_batch)

            # --- 3. 遍歷處理好的批次並繪圖 ---
            for i, batch in enumerate(plot_batches):
                batch_df = custom_industry_df[custom_industry_df['產業類別'].isin(
                    batch)].copy()
                batch_df = batch_df.sort_values('漲跌幅_5日(%)', ascending=False)
                
                for page_start in range(0, len(batch_df), 10):
                    page_end = min(page_start + 10, len(batch_df))
                    
                    # --- [本次修改] ---
                    # 在切割 DataFrame 時，加上 .copy() 來建立一個獨立的複本
                    page_df = batch_df.iloc[page_start:page_end].copy()
                    # --- [修改結束] ---
                    
                    # [v15.2.0 核心修正] 動態計算邊界
                    # 現在可以安全地對 page_df 進行修改
                    page_df['標註'] = '[' + page_df['產業類別'].astype(str) + '] ' + \
                                    page_df['stock_id'] + ' ' + page_df['公司簡稱']
                    max_label_len = page_df['標註'].str.encode('gbk', 'ignore').str.len().max()
                    dynamic_left_margin = 0.05 + max_label_len * 0.0055
                    dynamic_left_margin = min(dynamic_left_margin, 0.3) # 設定上限

                    if len(batch) > 1:
                        title_industry_name = "、".join(batch)
                        main_title_prefix = f"合併產業強弱分析：{title_industry_name}"
                    else:
                        title_industry_name = batch[0]
                        main_title_prefix = f"策略選出股票產業強弱分析：{title_industry_name}"

                    fig = plt.figure(figsize=(28, 12))
                    title_date = page_df['資料日期'].iloc[0] if not page_df.empty else datetime.now().strftime('%Y-%m-%d')
                    fig.suptitle(
                        f"{main_title_prefix} (資料日期: {title_date}) 第 {page_start//10+1} 頁",
                        fontsize=22, fontweight='bold', y=0.97
                    )
                    gs_main = gridspec.GridSpec(
                        1, 2, width_ratios=[1, 1.2], wspace=0.3)

                    # --- 左圖 (ax1)：繪製漲跌幅 ---
                    ax1 = fig.add_subplot(gs_main[0, 0])
                    y_pos = np.arange(len(page_df))
                    bar_height = 0.2
                    ax1.barh(y_pos + bar_height, page_df['漲跌幅_60日(%)'],
                            height=bar_height, color='#a2d2ff', label='60日')
                    ax1.barh(y_pos, page_df['漲跌幅_20日(%)'], 
                            height=bar_height, color='#f7a072', label='20日')
                    ax1.barh(y_pos - bar_height, page_df['漲跌幅_5日(%)'],
                            height=bar_height, color='#e63946', label='5日')
                    ax1.set_yticks(y_pos, page_df['標註'], fontsize=12)
                    ax1.invert_yaxis()
                    ax1.set_xlabel('漲跌幅 (%)', fontsize=12)
                    ax1.legend(title='期間', fontsize=12)
                    ax1.set_title('價格趨勢 (依5日漲跌幅排序)', fontsize=16)
                    ax1.grid(True, axis='x', linestyle='--', alpha=0.7)
                    ax1.axvline(0, color='black', linewidth=0.5)

                    # --- 右圖 (ax2)：繪製成交金額趨勢圖 ---
                    ax2_container = fig.add_subplot(gs_main[0, 1])
                    ax2_container.spines[:].set_visible(False)
                    ax2_container.set_xticks([])
                    ax2_container.set_yticks([])
                    ax2_container.set_title('近60日成交金額趨勢', fontsize=16)

                    gs2 = gridspec.GridSpecFromSubplotSpec(
                        len(page_df), 1, subplot_spec=gs_main[0, 1], hspace=0.1)

                    for idx, stock_id in enumerate(page_df['stock_id']):
                        ax_mini = fig.add_subplot(gs2[idx])
                        if stock_id not in self.amt_df.columns:
                            ax_mini.text(0.5, 0.5, '無成交額數據', ha='center', va='center', color='gray')
                            ax_mini.set_xticks([]); ax_mini.set_yticks([])
                            continue

                        amt_series = self.amt_df[stock_id].tail(60)
                        if amt_series.empty:
                            ax_mini.set_visible(False); continue

                        amt_ma5 = amt_series.rolling(5).mean()
                        amt_ma20 = amt_series.rolling(20).mean()
                        is_vol_spike = amt_series > (amt_ma20 * 1.5)
                        colors = np.where(is_vol_spike, '#e63946', '#adb5bd')

                        ax_mini.bar(amt_series.index, amt_series, color=colors, width=1.0)
                        ax_mini.plot(amt_ma5, color='#fca311', linewidth=1.5, label='MA5')
                        ax_mini.plot(amt_ma20, color='#1d3557', linewidth=1.5, label='MA20')
                        ax_mini.yaxis.set_major_formatter(FuncFormatter(self.format_revenue))
                        ax_mini.tick_params(axis="x", labelsize=8, labelrotation=20)
                        
                        if idx < len(page_df) - 1:
                            ax_mini.set_xticklabels([])
                        if idx == 0:
                            ax_mini.legend(fontsize='x-small', loc='upper left')

                    # [v15.2.0 核心修正] 使用動態計算的左邊界
                    fig.subplots_adjust(left=dynamic_left_margin, right=0.98, top=0.92, bottom=0.08)

                    pdf.savefig(fig)
                    plt.close(fig)
                    gc.collect()



    def _plot_custom_stock_industry_strength_overview_page(self, pdf):
        """
        針對策略選出股票清單，在同一頁面上繪製所有產業的強弱趨勢總覽圖。
        
        doc:
        [v10.4.0 最終修正版]
        - 標示優化 (穩定版)：改為將「任何成分股創新高」的整個產業Y軸標籤，全部設為紅色粗體。
        """
        if self.industry_df is None or self.industry_df.empty:
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, "產業分析數據不足或加載失敗，無法生成自選清單產業強弱分析頁面。",
                    ha='center', va='center', fontsize=20, color='red')
            pdf.savefig(fig)
            plt.close(fig)
            return

        custom_industry_df = self.industry_df.copy()
        custom_industry_df = custom_industry_df[
            custom_industry_df['stock_id'].isin(
                self.result_df['股票代碼'].tolist())
        ]
        if custom_industry_df.empty:
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, "無策略選出股票產業資料，無法生成自選清單產業強弱分析頁面。",
                    ha='center', va='center', fontsize=20, color='red')
            pdf.savefig(fig)
            plt.close(fig)
            return

        # 計算產業平均漲跌幅
        periods = [5, 20, 60]
        perf_cols = [f'漲跌幅_{p}日(%)' for p in periods]
        industry_performance = custom_industry_df.dropna(
            subset=perf_cols).groupby('產業類別', observed=True)[perf_cols].mean()

        industry_performance = industry_performance.sort_values(
            '漲跌幅_5日(%)', ascending=False)

        if industry_performance.empty:
            fig, ax = plt.subplots(figsize=(28, 12))
            ax.text(0.5, 0.5, "計算後無有效產業資料可供繪圖。",
                    ha='center', va='center', fontsize=20, color='orange')
            pdf.savefig(fig)
            plt.close(fig)
            return

        # --- [核心修改] 建立Y軸標籤並根據創高股設定顏色 ---
        new_labels = []
        has_high_flags = []
        # 此處的 '是否創200日新高' 欄位，來自 _prepare_industry_data 方法的計算
        new_high_status_map = custom_industry_df.set_index('stock_id')[
            '是否創200日新高']

        for industry_name in industry_performance.index:
            stocks_in_industry = custom_industry_df[custom_industry_df['產業類別']
                                                    == industry_name]

            # 檢查該產業的成分股中，是否有任何一檔創高
            has_new_high_in_industry = any(new_high_status_map.get(
                sid, False) for sid in stocks_in_industry['stock_id'])
            has_high_flags.append(has_new_high_in_industry)

            # 建立無樣式的純文字標籤
            stock_labels = []
            for _, stock_row in stocks_in_industry.iterrows():
                marker = "[創高] " if new_high_status_map.get(
                    stock_row['stock_id'], False) else ""
                stock_labels.append(
                    f"{marker}{stock_row['stock_id']} {stock_row['公司簡稱']}")

            stock_str = ", ".join(stock_labels)
            wrapped_stock_str = textwrap.fill(f"({stock_str})", width=60)
            new_label = f"{industry_name}\n{wrapped_stock_str}"
            new_labels.append(new_label)

        # --- 繪製產業強弱總覽圖 ---
        fig, ax = plt.subplots(figsize=(28, 14))
        fig.suptitle(
            f"策略選出股票產業強弱總覽 (依5日平均漲跌幅排序)\n資料日期: {custom_industry_df['資料日期'].iloc[0]}",
            fontsize=22, fontweight='bold', y=0.97
        )

        bar_width = 0.25
        r = np.arange(len(industry_performance))
        colors = ['#e63946', '#f7a072', '#a2d2ff']

        for i, col in enumerate(perf_cols):
            ax.barh(r + (1-i)*bar_width - bar_width, industry_performance[col],
                    bar_width, color=colors[i], label=f'{periods[i]}日漲跌幅')

        ax.set_yticks(r, new_labels, fontsize=12)
        # 繪製完標籤後，再根據 flag 決定是否上色
        for i, ticklabel in enumerate(ax.get_yticklabels()):
            if has_high_flags[i]:
                ticklabel.set_color('red')
                ticklabel.set_weight('bold')

        ax.invert_yaxis()
        ax.set_xlabel('平均漲跌幅 (%)', fontsize=12)
        ax.legend(title='期間', fontsize=12)
        ax.grid(axis='x', linestyle='--', alpha=0.7)
        ax.axvline(0, color='black', linewidth=0.5)

        plt.tight_layout(rect=[0.02, 0.02, 1, 0.95])
        pdf.savefig(fig)
        plt.close(fig)
        gc.collect()

    # [修改] 將此方法改為靜態方法
    @staticmethod
    def _prepare_industry_data(stock_list=None, static_data_get=None):
        """
        [doc]
        [v-final 優化] 將籌碼指標改為計算「買賣超金額佔總市值比」。
        [v-final.3 修正] 統一漲跌幅欄位命名，解決 KeyError。
        [本次修改] 新增計算「月營收年增率」與「營收動能(MA3/MA12)」指標。
        """
        print("\n[產業分析模組] 正在準備... [買賣超金額佔總市值比] 與 [營收指標] 數據...")

        data_get_func = static_data_get if static_data_get else data.get

        try:
            # --- 數據加載 ---
            company = data_get_func('company_basic_info')
            close = data_get_func('price:收盤價')
            amt = data_get_func('price:成交金額')
            market_cap = data_get_func('etl:market_value')
            foreign_inv = data_get_func(
                'institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)')
            invest_trust = data_get_func(
                'institutional_investors_trading_summary:投信買賣超股數')
            # [新增] 加載營收相關數據
            monthly_revenue = data_get_func('monthly_revenue:當月營收')
            monthly_revenue_yoy = data_get_func('monthly_revenue:去年同月增減(%)')

            # --- 計算與整合 ---
            if len(close) < 200 or len(amt) < 6:
                logging.error(f"FinLab 數據長度不足，無法進行產業分析。")
                return None

            company = company[["stock_id", "公司簡稱", "公司名稱", "產業類別"]]
            company['stock_id'] = company['stock_id'].astype(str)

            if stock_list is None:
                stock_list = close.columns.tolist()

            # 確保所有股票在核心數據中都存在
            valid_stocks = list(set(stock_list) & set(close.columns) & set(
                amt.columns) & set(market_cap.columns) & set(foreign_inv.columns) & set(invest_trust.columns) &
                set(monthly_revenue.columns) & set(monthly_revenue_yoy.columns)) # [新增] 加入營收數據的交集判斷

            if not valid_stocks:
                logging.error("有效股票清單為空，無法繼續計算。")
                return None

            company = company[company['stock_id'].isin(valid_stocks)]
            final_df = company.copy()
            
            # 對所有 DataFrame 進行過濾，確保股票代碼一致
            close, amt, market_cap, foreign_inv, invest_trust, monthly_revenue, monthly_revenue_yoy = [
                df[valid_stocks] for df in [close, amt, market_cap, foreign_inv, invest_trust, monthly_revenue, monthly_revenue_yoy]
            ]

            # --- [核心修正] ---
            final_df['當日漲跌幅(%)'] = final_df['stock_id'].map(
                (close.iloc[-1] - close.iloc[-2]) / close.iloc[-2] * 100).fillna(0)
            final_df['漲跌幅_5日(%)'] = final_df['stock_id'].map(
                (close.iloc[-1] - close.iloc[-6]) / close.iloc[-6] * 100).fillna(0)
            final_df['漲跌幅_10日(%)'] = final_df['stock_id'].map(
                (close.iloc[-1] - close.iloc[-11]) / close.iloc[-11] * 100).fillna(0)
            final_df['漲跌幅_20日(%)'] = final_df['stock_id'].map(
                (close.iloc[-1] - close.iloc[-21]) / close.iloc[-21] * 100).fillna(0)
            final_df['漲跌幅_60日(%)'] = final_df['stock_id'].map(
                (close.iloc[-1] - close.iloc[-61]) / close.iloc[-61] * 100).fillna(0)

            is_new_high_200 = (close == close.rolling(
                200, min_periods=200).max()).iloc[-1]
            final_df['是否創200日新高'] = final_df['stock_id'].map(
                is_new_high_200).fillna(False)

            latest_amt = amt.iloc[-1]
            amt_ma5_latest = amt.rolling(
                window=5, min_periods=1).mean().iloc[-1]
            final_df['成交額大於5日均額'] = final_df['stock_id'].map(
                latest_amt > amt_ma5_latest).fillna(False)
            final_df['成交金額'] = final_df['stock_id'].map(latest_amt).fillna(0)
            final_df['是否放量'] = final_df['stock_id'].map(
                latest_amt > amt_ma5_latest).fillna(False)

            # --- 法人籌碼數據計算 (佔市值比版本) ---
            latest_market_cap = market_cap.iloc[-1]
            latest_close = close.iloc[-1]
            fi_net_5d_share = foreign_inv.rolling(
                5, min_periods=1).sum().iloc[-1]
            it_net_5d_share = invest_trust.rolling(
                5, min_periods=1).sum().iloc[-1]

            final_df['外資買賣超_5日'] = final_df['stock_id'].map(
                fi_net_5d_share / 1000).fillna(0)
            final_df['投信買賣超_5日'] = final_df['stock_id'].map(
                it_net_5d_share / 1000).fillna(0)
            
            fi_ratio_5d_raw = (fi_net_5d_share *
                               latest_close / latest_market_cap) * 100
            it_ratio_5d_raw = (it_net_5d_share *
                               latest_close / latest_market_cap) * 100
            fi_ratio_5d_clean = fi_ratio_5d_raw.replace(
                [np.inf, -np.inf], np.nan).fillna(0)
            it_ratio_5d_clean = it_ratio_5d_raw.replace(
                [np.inf, -np.inf], np.nan).fillna(0)
            final_df['外資佔市值比_5日'] = final_df['stock_id'].map(
                fi_ratio_5d_clean).fillna(0)
            final_df['投信佔市值比_5日'] = final_df['stock_id'].map(
                it_ratio_5d_clean).fillna(0)

            # --- [本次修改] 新增營收指標計算 ---
            # 1. 最新月營收年增率(YoY)
            latest_yoy = monthly_revenue_yoy.iloc[-1]
            final_df['營收YoY(%)'] = final_df['stock_id'].map(latest_yoy).fillna(0)

            # 2. 營收動能 (MA3/MA12)
            rev_ma3 = monthly_revenue.rolling(3).mean()
            rev_ma12 = monthly_revenue.rolling(12).mean()
            # 使用 .iloc[-1] 取得每支股票的最新均值
            rev_momentum_ratio = (rev_ma3.iloc[-1] / rev_ma12.iloc[-1]).replace([np.inf, -np.inf], np.nan)
            
            # 建立原始比率欄位，用於顯示
            final_df['營收動能(3/12)_ratio'] = final_df['stock_id'].map(rev_momentum_ratio).fillna(1)
            # 建立 (比率-1) 欄位，用於熱力圖上色 (正數呈紅色，負數呈綠色)
            final_df['營收動能(3/12)'] = final_df['營收動能(3/12)_ratio'] - 1

            # doc: =============================================================================
            # doc: --- [新增] 計算產業資金流強度 ---
            # doc: =============================================================================
            # 1. 將每支股票對應到其產業
            stock_to_industry = company.set_index('stock_id')['產業類別']
            
            # 2. 計算成交金額的 5 日和 20 日均線
            amt_ma5 = amt.rolling(5).mean().iloc[-1]
            amt_ma20 = amt.rolling(20).mean().iloc[-1]
            
            # 3. 建立一個包含均量和產業資訊的 DataFrame
            flow_df = pd.DataFrame({'amt_ma5': amt_ma5, 'amt_ma20': amt_ma20})
            flow_df['產業類別'] = flow_df.index.map(stock_to_industry)
            flow_df.dropna(subset=['產業類別'], inplace=True)
            
            # 4. 依產業分組，加總成交金額均量，計算出整個產業的均量
            industry_flow = flow_df.groupby('產業類別')[['amt_ma5', 'amt_ma20']].sum()
            
            # 5. 計算產業量能比，並處理分母為零的情況
            industry_flow['ratio'] = (industry_flow['amt_ma5'] / industry_flow['amt_ma20'].replace(0, np.nan)).fillna(1)
            
            # 6. 將計算出的產業量能比映射回主 DataFrame
            final_df['產業量能比_ratio'] = final_df['產業類別'].map(industry_flow['ratio'])
            final_df['產業量能比'] = final_df['產業量能比_ratio'] - 1 # 用於熱力圖上色
            # --- [新增結束] ---
            
            final_df['資料日期'] = close.index[-1].strftime('%Y-%m-%d')
            print("[產業分析模組] [買賣超金額佔總市值比]、[營收指標] 與 [產業資金流] 數據準備完成。")

            return final_df.dropna(subset=['產業類別'])

        except Exception as e:
            logging.error(f"[產業分析模組] 準備數據時發生錯誤: {e}", exc_info=True)
            return None




    # finlab程式碼格式
    def _process_stock_list(self):
        """
        doc: [v22.2 多主題支援最終修正版]
        - 維持讀取 .xlsx 檔案並清洗亂碼。
        - [本次修改] 根據使用者提供的檔案格式，改用 groupby 方式處理多主題。
        - 舊方法是分割儲存格內的字串，新方法是將同一個股票代碼的多個資料行彙整成一個主題列表。
        - 增加對 stock_topics_df 中 stock_no 欄位的數值型別清洗，使程式更穩健。
        """
        # 步驟 1: 照常讀取 .xlsx 檔案
        stock_selection_df = pd.read_excel(self.stock_selection_file)
        stock_topics_df = pd.read_excel(self.stock_topics_file)
        
        # --- 數據清洗步驟 (維持不變) ---
        for col in stock_topics_df.select_dtypes(include=['object']).columns:
            stock_topics_df[col] = stock_topics_df[col].str.replace('\ufffd', '', regex=False)
        stock_selection_df.columns = [col.replace('\ufffd', '') for col in stock_selection_df.columns]
        
        # 清洗 stock_no 欄位，確保是乾淨的整數型態
        stock_topics_df['stock_no'] = pd.to_numeric(stock_topics_df['stock_no'], errors='coerce')
        stock_topics_df.dropna(subset=['stock_no'], inplace=True)
        stock_topics_df['stock_no'] = stock_topics_df['stock_no'].astype(int)

        # --- [新修改] 建立 stock_map 和 topic_map ---
        # 建立 stock_map (代碼 -> 名稱)，只保留每個代碼的第一個名稱，避免重複
        self.stock_map = stock_topics_df.drop_duplicates(subset=['stock_no']).set_index('stock_no')['stock_name'].to_dict()

        # 建立 topic_map (代碼 -> 主題列表)
        # 使用 groupby 將同一個 stock_no 的所有 topic 彙整成一個 list
        topic_groups = stock_topics_df.groupby('stock_no')['topic'].apply(list)
        self.topic_map = topic_groups.to_dict()
        
        # --- 後續邏輯維持不變 ---
        stock_counts = {}
        for strategy in stock_selection_df.columns:
            cleaned_series = pd.to_numeric(stock_selection_df[strategy], errors='coerce').dropna()
            for stock in cleaned_series:
                stock_id_int = int(stock)
                stock_id_str = str(stock_id_int)
                if stock_id_str not in stock_counts:
                    stock_counts[stock_id_str] = {'count': 0, 'strategies': []}
                stock_counts[stock_id_str]['count'] += 1
                stock_counts[stock_id_str]['strategies'].append(strategy)
                
        sorted_stocks_info = sorted(
            stock_counts.items(), key=lambda x: x[1]['count'], reverse=True)
        
        result_data = []
        for stock_id_str, data_dict in sorted_stocks_info:
            stock_id = int(stock_id_str)
            stock_name = self.stock_map.get(stock_id, "未知")
            # 從 topic_map 中查詢主題列表
            stock_topic_list = self.topic_map.get(stock_id, ["無"])
            # 將主題列表用 '/' 串接成一個字串儲存
            stock_topic_str = "/".join(stock_topic_list)
            result_data.append({'股票代碼': stock_id_str, '股票名稱': stock_name, '主題': stock_topic_str,
                                '被選次數': data_dict['count'], '對應策略': data_dict['strategies']})
                                
        return pd.DataFrame(result_data)
    
    

    def _setup_plot_style(self):
        plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei',
                                           'Heiti TC', 'Arial Unicode MS', 'sans-serif']
        plt.rcParams['axes.unicode_minus'] = False
        plt.rcParams['font.weight'] = 'bold'
        plt.rcParams['axes.labelweight'] = 'bold'
        plt.rcParams['axes.titleweight'] = 'bold'
        plt.rcParams['text.color'] = 'black'
        plt.rcParams['axes.labelcolor'] = 'black'
        plt.rcParams['xtick.color'] = 'black'
        plt.rcParams['ytick.color'] = 'black'
        plt.rcParams['figure.facecolor'] = '#F5F5F5'
        plt.rcParams['axes.facecolor'] = '#E6F0F5'
        plt.rcParams['savefig.facecolor'] = '#F5F5F5'

    # 在 ReportGenerator class 內部

   



    # finlab程式碼格式
    def _plot_strong_trend_stock_pages(self, pdf):
        """[重構] 呼叫通用附錄函式"""
        logging.info("--- 開始繪製【強勢產業趨勢股】詳細報告 ---\n")
        strong_stocks_df = self.industry_df[self.industry_df['stock_id'].isin(self.strong_trend_stocks)]
        original_strategy_stocks = self.result_df['股票代碼'].tolist()
        stocks_to_plot_df = strong_stocks_df[~strong_stocks_df['stock_id'].isin(original_strategy_stocks)]
        
        # [新增] 在此處加入篩選步驟
        stocks_to_plot_df = self._apply_appendix_filter(stocks_to_plot_df)

        if stocks_to_plot_df.empty:
            logging.info("所有強勢趨勢股均已包含在策略報告中，或經過篩選後無剩餘股票，無需生成額外頁面。")
            return

        def formatter(row):
            return ['強勢產業趨勢股']

        self._plot_appendix_section(
            pdf=pdf,
            stocks_df=stocks_to_plot_df,
            title_text='強勢產業趨勢股分析',
            subtitle_text=f"基於市場 Top 5 強勢產業且5日漲幅為正的個股\n(資料日期: {self.industry_df['資料日期'].iloc[0]})",
            subtitle_color='darkred',
            source_topic= '強勢產業趨勢',
            source_details_formatter=formatter,
            image_section_key='E3_APPENDIX_STRONG_TREND'
        )



    def _plot_summary_page_stacked(self, pdf):
        """
        [doc]
        [v9.8.0 修改]
        - 圖表還原：根據使用者回饋，將策略矩陣圖還原為原始的堆疊長條圖樣式。
        - 視覺優化：新增「X軸最小寬度」邏輯。當最大選股次數小於5時，X軸寬度仍會固定為5，
                     解決了低計數時柱狀圖過長、比例失衡的問題，符合使用者「一小格」的視覺預期。
        """
        df_full = self.result_df.copy()
        if df_full.empty:
            logging.warning("策略選股結果為空，跳過繪製 '多策略選股標的與策略分佈' 頁面。")
            return

        df_full['label'] = df_full['股票名稱'] + ' (' + df_full['股票代碼'] + ')'
        s = df_full['對應策略'].explode()
        strategy_dummies = pd.crosstab(s.index, s)
        all_strategies = self.result_df['對應策略'].explode().unique()
        strategy_dummies = strategy_dummies.reindex(
            columns=all_strategies, fill_value=0)
        df_full = df_full.join(strategy_dummies)

        STOCKS_PER_PAGE = 25
        num_pages = int(np.ceil(len(df_full) / STOCKS_PER_PAGE))

        for page_num in range(num_pages):
            start_index = page_num * STOCKS_PER_PAGE
            end_index = start_index + STOCKS_PER_PAGE
            df_page = df_full.iloc[start_index:end_index]

            FIG_SIZE = (28, 24)
            fig, ax = plt.subplots(figsize=FIG_SIZE)

            plot_data = df_page.set_index('label')
            strategy_columns = [
                col for col in all_strategies if col in plot_data.columns]

            # 依被選次數排序 (多 -> 少)，並讓 Y 軸從上到下顯示
            plot_data.sort_values('被選次數', ascending=True, inplace=True)

            # 繪製原始的堆疊長條圖
            plot_data[strategy_columns].plot(
                kind='barh', stacked=True, ax=ax, width=0.8, colormap='tab20', legend=False)

            # 在 bar 中間顯示策略名稱
            for container in ax.containers:
                # 只在 bar 寬度足夠時才顯示標籤，避免文字重疊
                labels = [container.get_label() if v.get_width() >
                          0.3 else '' for v in container]
                ax.bar_label(container, labels=labels, label_type='center',
                             color='black', fontsize=14, weight='bold')

            title = f'多策略選股標的與策略分佈 (頁 {page_num + 1}/{num_pages})'
            ax.set_title(title, fontsize=26, pad=20)
            ax.set_xlabel('被選中次數', fontsize=16)
            ax.set_ylabel('股票', fontsize=16)
            ax.tick_params(axis='y', labelsize=14)

            # --- [核心修改] 調整X軸寬度邏輯 ---
            max_count = plot_data['被選次數'].max()
            MIN_AXIS_WIDTH = 5  # 設定X軸的最小寬度

            if max_count < MIN_AXIS_WIDTH:
                # 如果最大次數小於5，則手動將X軸的右邊界設為5
                ax.set_xlim(right=MIN_AXIS_WIDTH)

            # 讓X軸的刻度為整數
            ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

    # finlab程式碼格式
    def _plot_summary_page_topics(self, pdf):
        """
        [doc]
        [v2.1 版面修正版]
        - [核心修改] 修正了因主題(Topic)字串過長導致右側圖例文字重疊、版面錯亂的問題。
        - 透過 textwrap 對超長的主題名稱進行自動換行處理。
        - 動態調整每則圖例的垂直間距，以容納換行後的文字。
        """
        if self.result_df.empty:
            logging.warning("策略選股結果為空，跳過繪製 '熱門族群分佈' 頁面。")
            return

        # 修正 groupby 的行為，避免潛在的警告
        topic_groups = self.result_df.groupby('主題', observed=True)
        all_top_topics = topic_groups.size().nlargest(len(topic_groups))

        if all_top_topics.empty:
            return

        TOPICS_PER_PAGE = 15
        num_pages = int(np.ceil(len(all_top_topics) / TOPICS_PER_PAGE))

        for page_num in range(num_pages):
            start_index = page_num * TOPICS_PER_PAGE
            end_index = start_index + TOPICS_PER_PAGE
            top_topics_page = all_top_topics.iloc[start_index:end_index]

            FIG_SIZE = (28, 24)
            fig = plt.figure(figsize=FIG_SIZE)
            title = f'熱門族群分佈與成分股清單 (頁 {page_num + 1}/{num_pages})'
            fig.suptitle(title, fontsize=26, y=0.98)

            gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2], wspace=0.1)
            ax_pie = fig.add_subplot(gs[0, 0])
            ax_pie.set_title(
                f'族群佔比 (Top {start_index+1}-{end_index})', fontsize=20, pad=20)

            wedges, texts, autotexts = ax_pie.pie(top_topics_page, autopct='%1.1f%%',
                                                  startangle=90, pctdistance=0.85,
                                                  textprops={
                                                      'fontsize': 14, 'weight': 'bold'},
                                                  wedgeprops={'edgecolor': 'w', 'linewidth': 1})

            # 將圓餅圖的標籤移到圖例區，避免重疊
            for text in texts:
                text.set_visible(False)

            plt.setp(autotexts, size=12, weight="bold", color="white")
            ax_pie.axis('equal')

            # --- [核心修改] 右側圖例繪製邏輯 ---
            y_pos, x_start, line_height = 0.95, 0.55, 0.035

            # 取得圓餅圖的顏色列表
            pie_colors = [wedge.get_facecolor() for wedge in wedges]

            for i, (topic_name, count) in enumerate(top_topics_page.items()):
                if y_pos < 0.1:
                    break

                # 1. 對可能超長的主題標題進行自動換行
                topic_header_text = f"■ {topic_name} ({count}檔)"
                wrapped_header_lines = textwrap.wrap(
                    topic_header_text, width=60)

                # 2. 逐行繪製換行後的主題標題
                for line in wrapped_header_lines:
                    if y_pos < 0.1:
                        break
                    # 使用圓餅圖對應的顏色
                    fig.text(x_start, y_pos, line, fontsize=16, weight='bold',
                             color=pie_colors[i % len(pie_colors)], va='top')
                    y_pos -= line_height * 1.2

                # 3. 繪製成分股清單 (維持原有換行邏輯)
                stocks = topic_groups.get_group(topic_name)
                stock_list_str = "、".join(
                    [f"{row['股票代碼']} {row['股票名稱']}" for _, row in stocks.iterrows()])

                wrapped_list = textwrap.wrap(stock_list_str, width=60)

                for line in wrapped_list:
                    if y_pos < 0.1:
                        break
                    fig.text(x_start + 0.02, y_pos,
                             line, fontsize=14, va='top')
                    y_pos -= line_height

                # 4. 增加與下一個主題的間距
                y_pos -= line_height * 0.8

            pdf.savefig(fig)
            plt.close(fig)
            
            
            
            
    def _plot_summary_page_topics_as_table(self, pdf):
            """
            [doc]
            [v4.0 Ultra Thinking 版] 視覺化與版面配置再進化。
            - [視覺化計數] "成分股數量"旁新增水平長條圖，可立即感知族群規模。
            - [斑馬條紋] 新增交錯的行背景色，大幅提升長表格的可讀性與專業度。
            - [強化排版與字體] "主題"加粗、"成分股"顏色放淡，建立視覺層次感；"數量"置中對齊，版面更均衡。
            - [維持優點] 繼承 v3.3 的動態行高核心邏輯，徹底解決文字與分隔線重疊問題。
            - 顯示最熱門的前 40 個主題。
            """
            logging.info("--- 開始繪製【熱門族群分佈(v4.0 Ultra Thinking版)】報告頁面 ---")
            if self.result_df.empty:
                logging.warning("策略選股結果為空，無法生成熱門族群表格。")
                return

            # 1. 數據彙總
            try:
                topic_groups = self.result_df.groupby('主題', observed=True)
                topic_summary = []
                for name, group in topic_groups:
                    stock_list = "、".join([f"{row['股票代碼']} {row['股票名稱']}" for _, row in group.iterrows()])
                    topic_summary.append({
                        '主題': name,
                        '成分股數量': len(group),
                        '成分股清單': stock_list
                    })
                
                if not topic_summary:
                    raise ValueError("無法從選股結果中彙總出任何主題。")

                summary_df = pd.DataFrame(topic_summary).sort_values(by='成分股數量', ascending=False)
                top_df = summary_df.head(40).reset_index(drop=True)
                # 用於計算長條圖相對長度
                max_count = top_df['成分股數量'].max() if not top_df.empty else 1

            except Exception as e:
                logging.error(f"準備熱門族群表格數據時發生錯誤: {e}", exc_info=True)
                return

            # --- 手動繪製表格 ---
            fig, ax = plt.subplots(figsize=(28, 24))
            ax.axis('off')
            
            page_title = f'熱門族群分佈與成分股清單 (Top {len(top_df)})'
            fig.suptitle(page_title, fontsize=26, y=0.98, weight='bold')

            # 2. 定義表格佈局參數
            y_cursor = 0.93
            x_coords = {'topic': 0.05, 'count_text': 0.26, 'count_bar': 0.28, 'stocks': 0.35}
            col_widths = {'topic': 0.20, 'count': 0.10, 'stocks': 0.60}
            
            line_height_multiplier = 0.023
            row_padding = 0.015
            header_font_size = 16
            cell_font_size = 14

            # 3. 繪製表頭
            headers = {'主題名稱': x_coords['topic'], '成分股數量': x_coords['count_text'], '成分股清單': x_coords['stocks']}
            header_color = '#40466e'
            for header, x_pos in headers.items():
                ha = 'center' if header == '成分股數量' else 'left'
                ax.text(x_pos, y_cursor, header, transform=ax.transAxes, 
                        fontsize=header_font_size, weight='bold', va='bottom', ha=ha, color=header_color)
            
            ax.axhline(y_cursor - 0.02, color='black', lw=1.5, xmin=x_coords['topic'], xmax=0.95)
            
            y_cursor -= 0.04

            # 4. 逐行繪製資料
            for index, row in top_df.iterrows():
                if y_cursor < 0.05:
                    break

                top_of_this_row = y_cursor

                # a. 內容換行
                topic_lines = textwrap.wrap(row['主題'], width=25)
                count_lines = [str(row['成分股數量'])]
                stock_lines = textwrap.wrap(row['成分股清單'], width=85)

                # b. 計算行高
                max_lines = max(len(topic_lines), len(count_lines), len(stock_lines))
                text_block_height = max_lines * line_height_multiplier
                
                # d. 計算該列的底部 Y 座標
                bottom_of_this_row = top_of_this_row - text_block_height - row_padding

                # [ULTRA優化 1: 斑馬條紋]
                if index % 2 == 1:
                    rect = plt.Rectangle((x_coords['topic'], bottom_of_this_row), 
                                        0.9, top_of_this_row - bottom_of_this_row,
                                        facecolor='#f5f5f5', edgecolor='none', zorder=-1)
                    ax.add_patch(rect)

                # c. 繪製文字 (優化字體樣式)
                ax.text(x_coords['topic'], top_of_this_row, "\n".join(topic_lines), transform=ax.transAxes, 
                        va='top', ha='left', fontsize=cell_font_size, linespacing=1.4, weight='bold')
                ax.text(x_coords['count_text'], top_of_this_row, "\n".join(count_lines), transform=ax.transAxes, 
                        va='top', ha='center', fontsize=cell_font_size, linespacing=1.4, weight='bold')
                ax.text(x_coords['stocks'], top_of_this_row, "\n".join(stock_lines), transform=ax.transAxes, 
                        va='top', ha='left', fontsize=cell_font_size, linespacing=1.4, color='#555555')
                
                # [ULTRA優化 2: 視覺化計數長條圖]
                bar_height = line_height_multiplier * 0.6
                bar_max_width = 0.06
                bar_width = (row['成分股數量'] / max_count) * bar_max_width
                bar_y_pos = top_of_this_row - (text_block_height / 2) # 垂直置中
                
                rect_bar = plt.Rectangle((x_coords['count_bar'], bar_y_pos - bar_height / 2),
                                        bar_width, bar_height,
                                        facecolor='#457b9d', alpha=0.7, edgecolor='none')
                ax.add_patch(rect_bar)

                # e. 更新 Y 軸指標
                y_cursor = bottom_of_this_row

                # f. 繪製分隔線
                ax.axhline(y_cursor + (row_padding / 2), color='grey', lw=0.5, ls='--', xmin=x_coords['topic'], xmax=0.95)

            pdf.savefig(fig)
            plt.close(fig)


    
    
    
    

    def _plot_industry_strength_page(self, pdf, output_dir, today_str):
        """
        [doc]
        繪製全市場的產業強弱度分析圖表，包含一個總覽頁面和多個鑽取頁面。
        此版本包含了您指定的顏色修改。

        [v15.4.0 最終排版修正]
        - 啟用 Matplotlib 的 constrained_layout 自動排版引擎，從根本上解決文字標籤溢出和子圖比例問題。
        - 移除所有手動計算邊界的程式碼，讓排版更智慧、更穩健。
        """
        logging.info("開始生成全市場產業強弱度分析圖表...")
        if self.industry_df is None or self.industry_df.empty:
            logging.warning("產業分析數據為空，跳過全市場產業強弱度頁面生成。")
            return

        final_df = self.industry_df
        periods_to_analyze = [5, 20, 60]
        perf_cols = [f'漲跌幅_{p}日(%)' for p in periods_to_analyze]
        industry_performance = final_df.dropna(
            subset=['當日漲跌幅(%)'] + perf_cols).groupby('產業類別')[['當日漲跌幅(%)'] + perf_cols].mean()

        sort_key = '漲跌幅_20日(%)'
        top_n = 10
        strong_industries = industry_performance.sort_values(by=sort_key, ascending=False).head(top_n)
        weak_industries = industry_performance.sort_values(by=sort_key, ascending=True).head(top_n)

        def format_top_bottom_stocks(industry_name, full_df):
            industry_stocks = full_df[full_df['產業類別'] == industry_name]
            if industry_stocks.empty: return "", False
            
            sorted_stocks = industry_stocks.sort_values('漲跌幅_5日(%)', ascending=False).copy()
            
            def create_label(row):
                new_high_marker = "[創高] " if row.get('是否創200日新高', False) else ""
                return f"{new_high_marker}{row['stock_id']} {row['公司簡稱']}({row['漲跌幅_5日(%)']:.1f}%)"
            
            sorted_stocks['label'] = sorted_stocks.apply(create_label, axis=1)
            top_n_count = min(3, len(sorted_stocks))
            top_stocks = sorted_stocks.head(top_n_count)
            has_new_high = any(row.get('是否創200日新高', False) for _, row in top_stocks.iterrows())
            
            top_str = "▲ " + ", ".join(top_stocks['label'])
            
            full_stock_info = top_str
            if len(sorted_stocks) > top_n_count:
                bottom_stocks = sorted_stocks.tail(top_n_count).iloc[::-1]
                bottom_str = "▼ " + ", ".join(bottom_stocks['label'])
                full_stock_info = f"{top_str}\n  {bottom_str}"
            
            return textwrap.fill(full_stock_info, width=80, subsequent_indent='  '), has_new_high

        # --- 繪製產業總覽圖 ---
        fig_overview, axes = plt.subplots(nrows=1, ncols=2, figsize=(28, 16), constrained_layout=True)
        fig_overview.suptitle(f"全市場產業強弱勢總覽圖 (資料日期: {final_df['資料日期'].iloc[0]})", fontsize=22, fontweight='bold')

        new_labels_strong = [f"{name}\n{format_top_bottom_stocks(name, final_df)[0]}" for name in strong_industries.index]
        new_labels_weak = [f"{name}\n{format_top_bottom_stocks(name, final_df)[0]}" for name in weak_industries.index]
        
        ax1 = axes[0]
        y_pos_strong = np.arange(len(strong_industries))
        bar_height = 0.25
        ax1.barh(y_pos_strong + bar_height, strong_industries['漲跌幅_60日(%)'], height=bar_height, color='#a2d2ff', label='60日')
        ax1.barh(y_pos_strong, strong_industries['漲跌幅_20日(%)'], height=bar_height, color='#f7a072', label='20日')
        ax1.barh(y_pos_strong - bar_height, strong_industries['漲跌幅_5日(%)'], height=bar_height, color='#e63946', label='5日')
        ax1.set_yticks(y_pos_strong, new_labels_strong, fontsize=12)
        ax1.invert_yaxis()
        ax1.set_title('符合策略的強勢產業', fontsize=18, fontweight='bold')
        
        ax2 = axes[1]
        y_pos_weak = np.arange(len(weak_industries))
        ax2.barh(y_pos_weak + bar_height, weak_industries['漲跌幅_60日(%)'], height=bar_height, color='#a2d2ff', label='60日')
        ax2.barh(y_pos_weak, weak_industries['漲跌幅_20日(%)'], height=bar_height, color='#f7a072', label='20日')
        ax2.barh(y_pos_weak - bar_height, weak_industries['漲跌幅_5日(%)'], height=bar_height, color='#2a9d8f', label='5日')
        ax2.set_yticks(y_pos_weak, new_labels_weak, fontsize=12)
        ax2.invert_yaxis()
        ax2.set_title('弱勢產業 (依20日排序)', fontsize=18, fontweight='bold')

        for ax in [ax1, ax2]:
            ax.set_xlabel('平均漲跌幅 (%)', fontsize=12)
            ax.legend(title='期間')
            ax.grid(axis='x', linestyle='--', alpha=0.7)
            ax.axvline(0, color='black', linewidth=0.5)

        pdf.savefig(fig_overview)
        plt.close(fig_overview)
        gc.collect()

        # --- 繪製強勢產業鑽取圖 ---
        top_industries_to_drill = strong_industries.head(5).index.tolist()
        logging.info("--- 正在顯示：符合策略之 Top 5 強勢產業的成分股趨勢圖 ---")
        
        for i, industry_name in enumerate(top_industries_to_drill):
            industry_stocks = final_df[final_df['產業類別'] == industry_name].copy()
            stocks_to_plot = industry_stocks.sort_values(by='漲跌幅_20日(%)', ascending=False).head(8)
            if stocks_to_plot.empty: continue

            stocks_to_plot['顯示名稱'] = stocks_to_plot.apply(lambda row: f"{row['公司簡稱']} ({row['stock_id']})", axis=1)
            
            max_label_len_drill = stocks_to_plot['顯示名稱'].str.encode('gbk', 'ignore').str.len().max()
            dynamic_left_margin_drill = 0.05 + max_label_len_drill * 0.0055
            dynamic_left_margin_drill = min(dynamic_left_margin_drill, 0.3)

            fig_drill = plt.figure(figsize=(28, 12))
            fig_drill.suptitle(f'強勢產業 Top {i+1}: {industry_name} - 價格與成交金額趨勢', fontsize=18, fontweight='bold')
            gs_main = gridspec.GridSpec(1, 2, width_ratios=[1, 1.2], wspace=0.3)
            
            # 左圖：價格趨勢
            ax1_drill = fig_drill.add_subplot(gs_main[0, 0])
            y_pos_stocks = np.arange(len(stocks_to_plot))
            bar_height_stocks = 0.2
            offsets = [-0.3, -0.1, 0.1, 0.3]
            price_labels = ['1日', '5日', '20日', '60日']
            price_cols = ['當日漲跌幅(%)', '漲跌幅_5日(%)', '漲跌幅_20日(%)', '漲跌幅_60日(%)']
            
            # #############################################################
            # ###                       這裡就是修改的地方                      ###
            # #############################################################
            # price_colors = ['#c9184a', '#ff4d6d', '#ff8fa3', '#ffb3c1'] # 原本的顏色 (全為紅色系)
            price_colors = ['#e63946', '#f7a072', '#457b9d', '#588157'] # <<< 修改後的顏色 (紅、橘、藍、綠)
            
            for j, col in enumerate(price_cols):
                ax1_drill.barh(y_pos_stocks + offsets[j], stocks_to_plot[col].fillna(0),
                            height=bar_height_stocks, color=price_colors[j], label=price_labels[j])
            
            ax1_drill.set_title('價格趨勢 (依20日漲跌幅排序)', fontsize=14)
            ax1_drill.set_xlabel('個股漲跌幅 (%)')
            ax1_drill.legend(title='價格期間')
            ax1_drill.grid(axis='x', color='grey', linestyle='--', linewidth=0.8, alpha=0.8)
            ax1_drill.set_yticks(y_pos_stocks, stocks_to_plot['顯示名稱'], fontsize=12)
            ax1_drill.invert_yaxis()
            ax1_drill.axvline(0, color='black', linewidth=0.5)
            
            # 右圖：成交金額趨勢圖
            ax2_container = fig_drill.add_subplot(gs_main[0, 1])
            ax2_container.spines[:].set_visible(False)
            ax2_container.set_xticks([])
            ax2_container.set_yticks([])
            ax2_container.set_title('近60日成交金額趨勢', fontsize=14)
            
            gs2 = gridspec.GridSpecFromSubplotSpec(len(stocks_to_plot), 1, subplot_spec=gs_main[0, 1], hspace=0.1)
            
            for idx, stock_id in enumerate(stocks_to_plot['stock_id']):
                ax_mini = fig_drill.add_subplot(gs2[idx])
                if stock_id not in self.amt_df.columns:
                    ax_mini.text(0.5, 0.5, '無成交額數據', ha='center', va='center', color='gray')
                    ax_mini.set_xticks([]); ax_mini.set_yticks([])
                    continue
                
                amt_series = self.amt_df[stock_id].tail(60)
                if amt_series.empty:
                    ax_mini.set_visible(False); continue
                
                amt_ma5 = amt_series.rolling(5).mean()
                amt_ma20 = amt_series.rolling(20).mean()
                is_vol_spike = amt_series > (amt_ma20 * 1.5)
                colors = np.where(is_vol_spike, '#e63946', '#adb5bd')
                
                ax_mini.bar(amt_series.index, amt_series, color=colors, width=1.0)
                ax_mini.plot(amt_ma5, color='#fca311', linewidth=1.5, label='MA5')
                ax_mini.plot(amt_ma20, color='#1d3557', linewidth=1.5, label='MA20')
                ax_mini.yaxis.set_major_formatter(FuncFormatter(self.format_revenue))
                ax_mini.tick_params(axis="x", labelsize=8, labelrotation=20)
                
                if idx < len(stocks_to_plot) - 1:
                    ax_mini.set_xticklabels([])
                if idx == 0:
                    ax_mini.legend(fontsize='x-small', loc='upper left')

            fig_drill.subplots_adjust(left=dynamic_left_margin_drill, right=0.98, top=0.92, bottom=0.08)
            pdf.savefig(fig_drill)
            plt.close(fig_drill)
            gc.collect()
            
        logging.info("全市場產業強弱度分析圖表生成完畢。")



    # finlab程式碼格式
    # [ULTRA THINK v21.6 sr_data 參數修正版]
    def _plot_stock_page(self, analysis_data, stock_info_row):
        """
        doc:
        [v21.6] 修正了呼叫 _plot_primary_chart 時遺漏 'sr_data' 參數的錯誤。
        [v21.5] 語意化修正：清理 _plot_primary_chart 的函式呼叫。
        [v21.4] 修正版面配置，將 K 線圖區域的 subgridspec 從 3x1 改為 2x1。
        [Gemini 修正] 將版面配置還原為 3x1，以符合 _plot_primary_chart 函式的要求。
        """
        
        s = analysis_data['stock_id']
        # --- [Gemini 建議修正] ---
        # 在建立圖表前，先檢查最核心的 K線數據是否存在且足夠
        df_check = pd.DataFrame({
            'open': self.analyzer.all_data['open'].get(s), 
            'high': self.analyzer.all_data['high'].get(s),
            'low': self.analyzer.all_data['low'].get(s), 
            'close': self.analyzer.all_data['close'].get(s)
        }).dropna()
        
        # 如果數據不足以繪製有意義的圖表 (例如少於一個月的數據)，則記錄警告並直接回傳 None
        if df_check.empty or len(df_check) < 22:
            logging.warning(f"股票 {s} 核心 K 線數據不足 ({len(df_check)} 天)，無法生成六宮格圖，已跳過。")
            return None # 回傳 None 以通知上層函式跳過儲存
        # --- [修正結束] ---
        fig = plt.figure(figsize=(28, 26), constrained_layout=True)
        
        # doc: =============================================================================
        # doc: --- [Gemini 核心修正] 調整標題區與圖表區的高度比例 ---
        # doc: 將 height_ratios 從 [1, 15] 調整為 [2, 9]，給予標題區更多垂直空間以解決重疊問題。
        # doc: =============================================================================
        gs_page = gridspec.GridSpec(2, 1, height_ratios=[2, 14], figure=fig)

        ax_title = fig.add_subplot(gs_page[0, 0])
        ax_title.axis('off')
        
        # doc: --- [核心修改] 將 analysis_data 傳遞下去 ---
        self._set_main_title(ax_title, s, stock_info_row, analysis_data)
        # doc: --- [修改結束] ---

        gs_main = gs_page[1, 0].subgridspec(1, 2, width_ratios=[1.5, 1.2], wspace=0.2)
        
        gs_left = gs_main[0, 0].subgridspec(2, 1, height_ratios=[2.5, 2.2])
        
        # doc: =============================================================================
        # doc: --- 短線圖表修正 ---
        # doc: =============================================================================
        # doc: [*** 修正 ***] 將 subgridspec 從 (2, 1) 改回 (3, 1)，以建立三個圖表區域。
        gs_short = gs_left[0, 0].subgridspec(3, 1, height_ratios=[3, 1, 1], hspace=0)
        ax_main_short = fig.add_subplot(gs_short[0, 0])
        # doc: [*** 新增 ***] 建立中層的成交量圖表區域 (ax_volume_short)。
        ax_volume_short = fig.add_subplot(gs_short[1, 0], sharex=ax_main_short)
        ax_fundamental_short = fig.add_subplot(gs_short[2, 0], sharex=ax_main_short)
        
        # doc: --- [核心修改 1/2] 呼叫短線圖時，傳入 chart_type='short' 與歷史事件 ---
        self._plot_primary_chart(ax_main_short, ax_volume_short, ax_fundamental_short, s,
                            self.params['plot_days_short'], 
                            analysis_data.get('sr_short'), 
                            chart_type='short',
                            historical_events=analysis_data.get('historical_events_short')) # [Gemini v2.0 新增]
        ax_main_short.set_title(f"短線結構 ({self.params['plot_days_short']}日)", fontsize=16)

        # doc: =============================================================================
        # doc: --- 長線圖表修正 ---
        # doc: =============================================================================
        # doc: [*** 修正 ***] 將 subgridspec 從 (2, 1) 改回 (3, 1)，以建立三個圖表區域。
        gs_long = gs_left[1, 0].subgridspec(3, 1, height_ratios=[3, 1, 1], hspace=0)
        ax_main_long = fig.add_subplot(gs_long[0, 0])
        # doc: [*** 新增 ***] 建立中層的成交量圖表區域 (ax_volume_long)。
        ax_volume_long = fig.add_subplot(gs_long[1, 0], sharex=ax_main_long)
        ax_fundamental_long = fig.add_subplot(gs_long[2, 0], sharex=ax_main_long)

        # doc: --- [核心修改 2/2] 呼叫長線圖時，傳入 chart_type='long' 與歷史事件 ---
        self._plot_primary_chart(ax_main_long, ax_volume_long, ax_fundamental_long, s,
                                self.params['plot_days_long'], 
                                analysis_data.get('sr_long'), 
                                chart_type='long',
                                historical_events=analysis_data.get('historical_events_long')) # [Gemini v2.0 新增]
        ax_main_long.set_title(f"長線結構 ({self.params['plot_days_long']}日)", fontsize=16)
        
        # --- 繪製右側 5 個指標圖 (維持不變) ---
        gs_right = gs_main[0, 1].subgridspec(5, 1, hspace=0.25)
        axes = [fig.add_subplot(gs_right[i, 0]) for i in range(5)]
        for ax in axes[:-1]: 
            ax.tick_params(axis='x', labelbottom=False)

        self._plot_institutional_flow(axes[0], analysis_data.get('institutional_flow'))
        self._plot_broker_flow(axes[1], analysis_data.get('broker_flow'))
        self._plot_shareholder_structure(axes[2], analysis_data.get('shareholder_structure'))
        self._plot_sentiment(axes[3], analysis_data.get('sentiment'))
        self._plot_fundamentals(axes[4], analysis_data.get('fundamentals'))
        
     
        
        return fig
    
    
    
    
    # finlab程式碼格式
    # doc: ===================================================================
    # doc: --- [Gemini 新增 v3.0] 判斷「底部墊高」趨勢輔助函式 ---
    # doc: ===================================================================
    def _has_higher_lows(self, price_series: pd.Series, order: int = 10, num_lows: int = 3) -> bool:
        """
        doc:
        使用 scipy.signal.argrelextrema 找出價格序列的轉折低點，並判斷最近 N 個低點是否呈現墊高趨勢。

        Args:
            price_series (pd.Series): 待分析的價格序列 (通常是收盤價或最低價)。
            order (int): argrelextrema 的 order 參數，代表尋找轉折點時，左右需參考的天數。
                         數值越大，找到的轉折點越少，但越顯著。
            num_lows (int): 要驗證的最近轉折低點數量，預設為 3 (即比較最近3個低點)。

        Returns:
            bool: 如果最近 N 個轉折低點的價格呈現嚴格遞增，則回傳 True。
        """
        from scipy.signal import argrelextrema
        import numpy as np

        if len(price_series) < order * 2 + 1:
            return False

        # 找出所有轉折低點的索引
        low_indices = argrelextrema(price_series.values, np.less, order=order)[0]
        
        # 如果找不到足夠的轉折低點，則視為不符合條件
        if len(low_indices) < num_lows:
            return False
            
        # 選取最近的 N 個轉折低點的價格
        last_n_low_prices = price_series.iloc[low_indices].tail(num_lows)
        
        # 確保選出的低點數量足夠
        if len(last_n_low_prices) < num_lows:
            return False

        # 檢查這些低點的價格是否「嚴格遞增」
        # np.diff 會計算序列中相鄰元素的差值，如果所有差值都 > 0，代表序列是遞增的
        return np.all(np.diff(last_n_low_prices.values) > 0)


    # finlab程式碼格式
    # doc: ===================================================================
    # doc: --- [Gemini 升級 v3.0] 「低基期強勢整理」判斷函式 ---
    # doc: ===================================================================
    def _is_low_base_consolidation(self, stock_id: str, lookback_period: int = 240, 
                                 quantile_threshold: float = 0.3, bbw_threshold: float = 10.0,
                                 # [新增] v3.0 新參數
                                 check_higher_lows: bool = True, swing_order: int = 10, num_swing_lows: int = 3):
        """
        doc:
        [v3.0 升級]
        判斷一檔股票是否處於「低基期強勢整理」狀態。

        核心邏輯：
        1. 價位區間位階：計算目前股價在過去 N 日高低點範圍內的百分位 (e.g., < 30%)。
        2. 波動率收縮：計算布林帶寬度 (BBW)，確認處於盤整 (e.g., < 10%)。
        3. [新增] 底部墊高趨勢：利用 scipy 找出近期 N 個轉折低點，確認它們呈現依序墊高的多頭整理型態。
        
        Args:
            (前略...)
            check_higher_lows (bool): 是否啟用「底部墊高」檢查，預設為 True。
            swing_order (int): 尋找轉折點的 order 參數。
            num_swing_lows (int): 驗證底部墊高的轉折點數量。

        Returns:
            bool: 如果同時滿足所有條件，則回傳 True。
        """
        try:
            close_price = self.analyzer.all_data.get('close', pd.DataFrame()).get(stock_id)
            if close_price is None or len(close_price) < lookback_period:
                return False

            recent_prices = close_price.tail(lookback_period)
            
            # --- 1. 價位區間位階判斷 (邏輯不變) ---
            min_price = recent_prices.min()
            max_price = recent_prices.max()
            price_range = max_price - min_price
            if price_range == 0:
                is_low_position = True
            else:
                current_position = (recent_prices.iloc[-1] - min_price) / price_range
                is_low_position = current_position <= quantile_threshold

            # --- 2. 波動率收縮判斷 (邏輯不變) ---
            ma20 = recent_prices.rolling(20).mean()
            std20 = recent_prices.rolling(20).std()
            upper_band = ma20 + (std20 * 2)
            lower_band = ma20 - (std20 * 2)
            latest_bbw = ((upper_band.iloc[-1] - lower_band.iloc[-1]) / ma20.iloc[-1]) * 100
            is_low_volatility = latest_bbw <= bbw_threshold
            
            # --- [修改] v3.0 組合最終條件 ---
            base_conditions_met = is_low_position and is_low_volatility

            # 如果不啟用底部墊高檢查，或基礎條件已不滿足，則直接回傳結果
            if not check_higher_lows or not base_conditions_met:
                return base_conditions_met
            
            # [新增] 3. 底部墊高趨勢判斷
            is_trending_up = self._has_higher_lows(
                price_series=recent_prices, 
                order=swing_order, 
                num_lows=num_swing_lows
            )
            
            # 最終回傳：必須同時滿足基礎條件與趨勢條件
            return base_conditions_met and is_trending_up

        except Exception:
            return False



    # finlab程式碼格式
    # doc: ===================================================================
    # doc: --- [Gemini 修改 v2.0] 使用進階版低基期判斷 ---
    # doc: ===================================================================
    def _get_it_mom_buy_signal(self, stock_id: str):
        """
        doc:
        [v2.0 修改]
        - 低基期的判斷邏輯，改為調用更精準的 _is_low_base_consolidation 函式，
          同時考量「價位在240日區間的底部30%」與「20日布林帶寬小於10%」。
        
        核心邏輯：
        1. 低基期盤整：調用新函式判斷。
        2. 連續買超：最近 2 天投信皆為買超。
        3. 高投量比：最近 2 天的平均投量比 > 10%。
        4. 雙重呵護：在滿足上述條件時，若最近一日主力也為買超，則為最優先訊號。
        """
        try:
            # --- 1. 取得所需數據 (此部分不變) ---
            it_buy_shares = self.analyzer.all_data.get('it_buy', pd.DataFrame()).get(stock_id)
            volume_shares = self.analyzer.all_data.get('volume', pd.DataFrame()).get(stock_id) * 1000
            close_price = self.analyzer.all_data.get('close', pd.DataFrame()).get(stock_id)
            top15_buy = self.analyzer.all_data.get('top15_buy', pd.DataFrame()).get(stock_id)
            top15_sell = self.analyzer.all_data.get('top15_sell', pd.DataFrame()).get(stock_id)
            
            if any(s is None or len(s) < 240 for s in [it_buy_shares, volume_shares, close_price, top15_buy, top15_sell]):
                return "", ""

            # --- 2. 計算各項指標 ---
            
            # doc: --- [核心修改] ---
            # doc: 移除舊的 MA240 判斷，改為直接調用新的輔助函式
            is_low_base = self._is_low_base_consolidation(stock_id)
            # doc: --- [修改結束] ---
            
            # (以下指標計算不變)
            it_buy_lots = it_buy_shares / 1000
            is_continuous_buy = (it_buy_lots.tail(2) > 0).all()

            volume_lots = volume_shares / 1000
            safe_volume_lots = volume_lots.replace(0, pd.NA)
            it_ratio = (it_buy_lots / safe_volume_lots) * 100
            avg_it_ratio_2d = it_ratio.tail(2).mean()
            is_high_ratio = avg_it_ratio_2d > 10

            net_main_force_buy = (top15_buy - top15_sell).iloc[-1]
            is_main_force_buy = net_main_force_buy > 0

            # --- 3. 組合條件並回傳訊號 (此部分不變) ---
            if is_low_base and is_continuous_buy and is_high_ratio and is_main_force_buy:
                signal_text = f"[雙重呵護(投+主)] (投量比: {avg_it_ratio_2d:.1f}%)"
                signal_color = '#FF4500' 
                return signal_text, signal_color
            
            if is_low_base and is_continuous_buy and is_high_ratio:
                signal_text = f"[專心栽培(投)] (投量比: {avg_it_ratio_2d:.1f}%)"
                signal_color = '#DC143C'
                return signal_text, signal_color
                
            return "", ""

        except Exception as e:
            logging.warning(f"為 {stock_id} 計算投信媽媽訊號時發生錯誤: {e}")
            return "", ""




    # finlab程式碼格式
    # doc: ===================================================================
    # doc: --- [Gemini 新增 v5.0] 判斷「主力連續買超」趨勢訊號 ---
    # doc: ===================================================================
    def _get_main_force_trend_signal(self, stock_id: str, consecutive_days: int = 3):
        """
        doc:
        判斷 top15 主力是否連續 N 天淨買超。

        Args:
            stock_id (str): 股票代碼。
            consecutive_days (int): 連續買超的天數門檻。

        Returns:
            tuple[str, str]: 包含 (標籤文字, 標籤顏色) 的元組，若不符合則回傳 ("", "")。
        """
        try:
            top15_buy = self.analyzer.all_data.get('etl:broker_transactions:top15_buy', pd.DataFrame()).get(stock_id)
            top15_sell = self.analyzer.all_data.get('etl:broker_transactions:top15_sell', pd.DataFrame()).get(stock_id)

            if top15_buy is None or len(top15_buy) < consecutive_days:
                return "", ""

            net_buy = (top15_buy - top15_sell).tail(consecutive_days)
            if len(net_buy) < consecutive_days:
                return "", ""

            # 檢查是否所有天的淨買超都大於 0
            if (net_buy > 0).all():
                signal_text = f"[主力連買({len(net_buy)}日)]"
                signal_color = "#DF8C0F" # 鋼藍色
                return signal_text, signal_color
                
            return "", ""
        except Exception:
            return "", ""




    
    def _set_main_title(self, ax_title, s, stock_info_row, analysis_data=None): # 新增 analysis_data=None
        """
        doc:
        [Gemini] 新增處置股與現金增資股的警告標題與標籤。
        [v30.2] 新增對「模式一」(獨立ETF分析) 的相容性修正。
        [v30.1] 採用動態 Y 軸游標 (y_cursor) 進行佈局。
        """
        import textwrap
        import pandas as pd
        import logging

        stock_id = s
        stock_name = stock_info_row['股票名稱']

        # --- 步驟 1: 準備所有文字內容 ---
        topic_list = self.topic_map.get(int(stock_id), ['無主題'])
        topic_to_display = "/".join(topic_list[:8])

        industry = ""
        if self.industry_df is not None and not self.industry_df.empty and 'stock_id' in self.industry_df.columns:
            industry_series = self.industry_df[self.industry_df['stock_id'] == stock_id]['產業類別']
            if not industry_series.empty:
                industry = industry_series.iloc[0]
        if not industry:
            try:
                # 假設 finlab 的 data 模組已經匯入
                from finlab import data
                company_info = data.get('company_basic_info')
                industry_series = company_info[company_info['stock_id'] == stock_id]['產業類別']
                if not industry_series.empty:
                    industry = industry_series.iloc[0]
            except Exception as e:
                logging.warning(f"從 company_basic_info 查詢 {stock_id} 的產業類別失敗: {e}")
                industry = "未知產業"

        strategies_val = stock_info_row['對應策略']
        strategies = ", ".join(sorted(list(set(strategies_val)), key=strategies_val.index)) if isinstance(
            strategies_val, list) else str(strategies_val)
        close_series = self.analyzer.all_data['close'][stock_id].dropna()
        volume_series = self.analyzer.all_data['volume'][stock_id].dropna()

        title_line1 = f"{stock_name} ({s}) - [{industry}] {topic_to_display}"
        consolidated_info_line = f"策略: {strategies}"
        stop_loss_text = "" # 初始化停損價文字
        
        
        consolidated_info_line = f"策略: {strategies}"
        if not close_series.empty and not volume_series.empty:
            last_close, pct_change, last_volume = close_series.iloc[-1], close_series.pct_change(
            ).iloc[-1], volume_series.iloc[-1]
            five_day_str = f" | 5日漲跌: {close_series.pct_change(periods=5).iloc[-1]:+.2%}" if len(
                close_series) > 5 else ""
            day_trade_percentage = 0.0
            day_trade_vol_all = self.analyzer.all_data.get('day_trade_vol')
            if last_volume > 0 and (day_trade_vol_all is not None and not day_trade_vol_all.empty):
                last_date = volume_series.index[-1]
                if last_date in day_trade_vol_all.index and s in day_trade_vol_all.columns:
                    day_trade_val = day_trade_vol_all.loc[last_date, s]
                    if pd.notna(day_trade_val):
                        day_trade_volume_lots = (day_trade_val / 2 / 1000)
                        day_trade_percentage = (
                            day_trade_volume_lots / last_volume * 100)
            info_part2 = f"收盤: {last_close:.2f} (漲跌: {pct_change:+.2%}{five_day_str})"
            info_part3 = f"總量: {last_volume:,.0f} 張 (當沖佔比: {day_trade_percentage:.1f}%)"
            
            # doc: --- [核心修改] 取得並格式化停損價 ---
            if analysis_data and 'stop_loss_price' in analysis_data:
                stop_loss_val = analysis_data['stop_loss_price']
                if stop_loss_val is not None and pd.notna(stop_loss_val):
                    stop_loss_text = f" | 停損參考: {stop_loss_val:.2f}" # 加入文字
                else:
                     stop_loss_text = " | 停損參考: N/A" # 數據不足時顯示 N/A
            
            # 將停損價加入顯示
            consolidated_info_line = f"{consolidated_info_line}|{info_part2}|{info_part3}{stop_loss_text}"
            # doc: --- [修改結束] ---

        # --- 準備 TAGS ---
        tags = []
        tag_colors = {'成交額': '#0077b6', 'ETF焦點股': '#9400D3', '創200日新高': '#FF4500',
                    '創20日新高': '#FFA500', '強勢產業趨勢': '#228B22', '排名躍升': '#00BFFF',
                    '主動式ETF': '#6A0DAD', '增資股': '#B3446C', '庫藏股': '#20c997'}
        
        
        
        # --- [Gemini 新增] 判斷量能濾網 (5日均量 > 10日均量) ---
        try:
            # 1. 從 analyzer 獲取成交量數據 (單位: 張)
            volume_series = self.analyzer.all_data.get('volume', pd.DataFrame()).get(stock_id)

            # 2. 確保有足夠天數的資料來計算均線
            if volume_series is not None and len(volume_series.dropna()) >= 10:
                
                # 3. 計算 5日 與 10日 均量
                vol_ma5 = volume_series.rolling(5).mean().iloc[-1]
                vol_ma10 = volume_series.rolling(10).mean().iloc[-1]

                # 4. 判斷條件並加上標籤
                if pd.notna(vol_ma5) and pd.notna(vol_ma10) and vol_ma5 > vol_ma10:
                    tags.append(('[量能增溫]', '#FF8C00')) # 使用橘色標籤

        except Exception as e:
            logging.warning(f"為 {stock_id} 計算量能濾網時發生錯誤: {e}")
        # --- [新增結束] ---
        
        
        # doc: =============================================================================
        # doc: --- [Gemini 新增] 取得「投信媽媽」訊號標籤 ---
        # doc: 在此處調用我們剛剛建立的新函式
        # doc: =============================================================================
        try:
            
             # doc: --- [Gemini v5.0 修改] 在此處加入對新函式的調用 ---
            main_force_tag, main_force_color = self._get_main_force_trend_signal(stock_id)
            if main_force_tag:
                tags.append((main_force_tag, main_force_color))
            
            scalper_tag, scalper_color = self._get_scalper_warning_signal(stock_id)
            if scalper_tag:
                tags.append((scalper_tag, scalper_color))
                
            it_mom_tag, it_mom_color = self._get_it_mom_buy_signal(stock_id)
            if it_mom_tag:
                # 如果有回傳訊號，則將其加入 tags 列表
                tags.append((it_mom_tag, it_mom_color))
                
                
        except Exception as e:
            logging.warning(f"調用 _get_it_mom_buy_signal 失敗: {e}")
        # --- [新增結束] ---
        
        

        if not self.cash_increase_df.empty and stock_id in self.cash_increase_df.index:
            tags.append(('[小心下跌:增資]', tag_colors['增資股']))

        if not self.amt_rank_series.empty and stock_id in self.amt_rank_series.index:
            rank = self.amt_rank_series.loc[stock_id]
            if pd.notna(rank) and rank <= 100:
                tags.append((f'[成交額 #{int(rank)}]', tag_colors['成交額']))
                
        if stock_id in self.etf_focus_stock_ids_set:
            tags.append(('[ETF焦點股]', tag_colors['ETF焦點股']))
        if stock_id in self.daily_etf_rank_map:
            rank = self.daily_etf_rank_map.get(stock_id)
            status = self.daily_etf_status_map.get(stock_id, '')
            status_str = f" {status.replace('[', '').replace(']', '')}" if status else ""
            tags.append(
                (f"[主動式ETF Top{self.config.get('DAILY_ETF_TOP_N', 60)} #{int(rank)}{status_str}]", tag_colors['主動式ETF']))
        if stock_id in self.daily_etf_holdings_map:
            held_by_etfs = self.daily_etf_holdings_map.get(stock_id, [])
            if held_by_etfs:
                etf_names_str = ", ".join(
                    [etf.replace('A', '') for etf in held_by_etfs])
                tags.append(
                    (f"[持有: {etf_names_str}]", tag_colors.get('主動式ETF', '#6A0DAD')))
        if stock_id in self.new_high_200_set:
            tags.append(('[創200日新高]', tag_colors['創200日新高']))
        elif stock_id in self.new_high_20_set:
            tags.append(('[創20日新高]', tag_colors['創20日新高']))
        if stock_id in self.strong_trend_set:
            tags.append(('[強勢產業趨勢]', tag_colors['強勢產業趨勢']))
        if stock_id in self.rank_jump_set:
            tags.append(('[排名躍升]', tag_colors['排名躍升']))
            
        

        # doc: =============================================================================
        # doc: --- [核心修改] 「融資使用率」標籤詳細資訊化 ---
        # doc: =============================================================================
        try:
            margin_usage_series = self.analyzer.all_data['margin_usage'][stock_id].dropna()
            if len(margin_usage_series) >= 20:
                latest_mu = margin_usage_series.iloc[-1]
                mu_ma5 = margin_usage_series.rolling(5).mean().iloc[-1]
                mu_ma20 = margin_usage_series.rolling(20).mean().iloc[-1]
                
                # 1. 產生詳細的比較字串
                comp1 = ">" if latest_mu > mu_ma5 else "<" if latest_mu < mu_ma5 else "="
                comp2 = ">" if mu_ma5 > mu_ma20 else "<" if mu_ma5 < mu_ma20 else "="
                full_comp_str = f"當日 {comp1} 5日均 {comp2} 20日均"
                
                # 2. 組合新的標籤文字
                mu_text = f"[融資使用率: {latest_mu:.2f}% | {full_comp_str}]"
                
                # 3. 根據「當日值」與「5日均」的關係決定顏色，更即時反映短期強弱
                mu_trend_color = 'red' if latest_mu > mu_ma5 else 'green' if latest_mu < mu_ma5 else 'gray'
                
                tags.append((mu_text, mu_trend_color))
                
        except (KeyError, IndexError):
            # 找不到資料或資料長度不足，靜默處理
            pass
        
        # doc: =============================================================================
        # doc: --- [核心修改 2/3] 新增「成交量放大」標籤 ---
        # doc: =============================================================================
        try:
            volume_series = self.analyzer.all_data['volume'][stock_id].dropna()
            if len(volume_series) >= 21: # 確保有足夠數據計算20日均線
                latest_vol = volume_series.iloc[-1]
                vol_ma20 = volume_series.rolling(20).mean().iloc[-1]

                if latest_vol > (vol_ma20 * 2.5):
                    ratio = latest_vol / vol_ma20 if vol_ma20 > 0 else float('inf')
                    tags.append((f"[爆量 {ratio:.1f}x]", '#e63946')) # 紅色
        except (KeyError, IndexError): pass
        # doc: --- [修改結束] ---

        # doc: =============================================================================
        # doc: --- [核心修改 3/3] 新增「投信買超佔比」標籤 ---
        # doc: =============================================================================
        try:
            it_buy_series = self.analyzer.all_data['it_buy'][stock_id].dropna() / 1000 # 換算為張
            volume_series = self.analyzer.all_data['volume'][stock_id].dropna()
            it_buy_series, volume_series = it_buy_series.align(volume_series, join='inner')

            it_tag_added = False
            # 檢查近一日
            if len(volume_series) >= 1:
                latest_it = it_buy_series.iloc[-1]
                latest_vol = volume_series.iloc[-1]
                if latest_it > 0 and latest_vol > 0 and (latest_it / latest_vol) >= 0.15:
                    ratio = latest_it / latest_vol * 100
                    tags.append((f"[投信急買 {ratio:.0f}%]", '#8A2BE2')) # 藍紫色
                    it_tag_added = True
            
            # 如果一日不滿足，再檢查三日
            if not it_tag_added and len(volume_series) >= 3:
                it_sum_3d = it_buy_series.tail(3).sum()
                vol_sum_3d = volume_series.tail(3).sum()
                if it_sum_3d > 0 and vol_sum_3d > 0 and (it_sum_3d / vol_sum_3d) >= 0.15:
                    ratio = it_sum_3d / vol_sum_3d * 100
                    tags.append((f"[投信連買 {ratio:.0f}%]", '#9400D3')) # 深紫色
        except (KeyError, IndexError): pass
        # doc: --- [修改結束] ---
    
    
    

        # --- 動態佈局邏輯 ---
        y_cursor = 0.95
        has_special_tag = False
        
        # doc: =============================================================================
        # doc: 微調垂直間距，解決重疊問題 ---
        # doc: 將此值從 0.3 調回至 0.35，在「緊湊」與「足夠空間」之間取得平衡。
        # doc: =============================================================================
        y_warning_step = 0.25
        
        
        # doc: =============================================================================
        # doc: --- [新增] 動能品質分析標籤 ---
        # doc: =============================================================================
        try:
            # 條件1: 股票必須是創高股
            is_new_high = stock_id in self.new_high_200_set or stock_id in self.new_high_20_set

            if is_new_high and self.master_summary_df is not None and not self.master_summary_df.empty:
                stock_data = self.master_summary_df[self.master_summary_df['stock_id'] == stock_id]

                if not stock_data.empty:
                    industry_ratio = stock_data['產業量能比_ratio'].iloc[0]

                    # 條件2: 根據產業量能比決定標籤內容與顏色
                    if industry_ratio > 1.1:
                        # 高品質動能
                        warning_text = f"高品質動能：價漲且產業量增 (產業量能比: {industry_ratio:.2f}x)"
                        bbox_props = dict(
                            facecolor='gold', edgecolor='darkorange', boxstyle='round,pad=0.5', lw=2)
                        text_color = 'black'
                    elif industry_ratio < 0.9:
                        # 潛在動能陷阱
                        warning_text = f"動能陷阱警示：價漲但產業量縮 (產業量能比: {industry_ratio:.2f}x)"
                        bbox_props = dict(
                            facecolor='#4682B4', edgecolor='darkblue', boxstyle='round,pad=0.5', lw=2)
                        text_color = 'white'
                    else:
                        warning_text = None  # 在中間區間，不顯示標籤

                    if warning_text:
                        ax_title.text(0.5, y_cursor, warning_text, ha='center', va='top', fontsize=18, weight='bold', color=text_color,
                                      bbox=bbox_props,
                                      transform=ax_title.transAxes)
                        y_cursor -= y_warning_step  # 往下移動，為下一個標籤騰出空間
        except Exception as e:
            logging.warning(f"為 {stock_id} 繪製動能品質標籤時出錯: {e}")
        # --- [新增結束] ---
        
        
    
        
        # doc: =============================================================================
        # doc: --- [核心修改] 新增「法說會押寶」警告標籤 ---
        # doc: =============================================================================
        try:
            # 條件 1: 檢查近期是否有法說會
            if not self.investor_conference_df.empty and stock_id in self.investor_conference_df.index:
                info = self.investor_conference_df.loc[stock_id]
                conf_date = pd.to_datetime(info['date']).strftime('%Y-%m-%d')
                conf_location = info.get('地點', '未提供')
                
                # 條件 2: 檢查近5日漲幅是否小於7% (股價尚未發動)
                price_change_5d = close_series.pct_change(periods=5).iloc[-1]
                is_not_grown = price_change_5d < 0.07

                # 條件 3: 檢查營收基本面 (YoY > 0 或 營收動能 > 1)
                # 我們需要從 master_summary_df (self.industry_df) 獲取這些預先計算好的指標
                has_good_revenue = False
                if self.industry_df is not None and not self.industry_df.empty:
                    stock_summary = self.industry_df[self.industry_df['stock_id'] == stock_id]
                    if not stock_summary.empty:
                        yoy = stock_summary['營收YoY(%)'].iloc[0]
                        rev_mom = stock_summary['營收動能(3/12)_ratio'].iloc[0]
                        if yoy > 0 or rev_mom > 1:
                            has_good_revenue = True

                # 如果所有條件都滿足，則顯示警告標籤
                if is_not_grown and has_good_revenue:
                    warning_text = f"注意：法說會押寶 | 日期: {conf_date} ({conf_location}) | 條件: 股價尚未發動 & 營收成長"
                    ax_title.text(0.5, y_cursor, warning_text, ha='center', va='top', fontsize=16, weight='bold', color='white',
                                bbox=dict(facecolor='#8A2BE2', edgecolor='darkviolet', boxstyle='round,pad=0.5', lw=2),
                                transform=ax_title.transAxes)
                    y_cursor -= y_warning_step
        except Exception as e:
            logging.warning(f"為 {stock_id} 繪製法說會押寶警告標籤時出錯: {e}")
            
            
            
        
        # doc: =============================================================================
        # doc: --- [核心修改 1/3] 細緻化「潛在軋空」警告標籤 ---
        # doc: =============================================================================
        try:
            if not self.short_squeeze_candidate_df.empty and stock_id in self.short_squeeze_candidate_df.index:
                info_series = self.short_squeeze_candidate_df.loc[stock_id]
                info = info_series.iloc[0] if isinstance(info_series, pd.DataFrame) else info_series
                
                buy_in_date = pd.to_datetime(info['最後回補日'])
                days_until = (buy_in_date - datetime.now()).days + 1 # 加1使其包含當天
                ratio = pd.to_numeric(info['券資比'], errors='coerce')
                reason = info['原因']

                warning_text = ""
                # 條件1: 細緻化 - 中期高券資比軋空
                if 7 < days_until <= 14 and pd.notna(ratio) and ratio > 20:
                    warning_text = f"注意：中期軋空 ({reason}) | {days_until}天後回補 (券資比: {ratio:,.1f}%)"
                # 條件2: 原有的通用警告
                else:
                    warning_text = f"注意：潛在軋空 ({reason}) | 最後回補日: {info['最後回補日']} | 最新券資比: {ratio:,.1f}"
                
                ax_title.text(0.5, y_cursor, warning_text, ha='center', va='top', fontsize=16, weight='bold', color='black',
                            bbox=dict(facecolor='gold', edgecolor='darkorange', boxstyle='round,pad=0.5', lw=2),
                            transform=ax_title.transAxes)
                y_cursor -= y_warning_step
        except Exception as e:
            logging.warning(f"為 {stock_id} 繪製潛在軋空警告標籤時出錯: {e}")
        # doc: --- [修改結束] ---
        

        try:
            if not self.treasury_stock_df.empty and stock_id in self.treasury_stock_df.index:
                info_series = self.treasury_stock_df.loc[stock_id]
                info = info_series.iloc[0] if isinstance(info_series, pd.DataFrame) else info_series
                start_date = info['預定買回期間-起'].strftime('%Y-%m-%d')
                end_date = info['預定買回期間-迄'].strftime('%Y-%m-%d')
                price_low = info['買回價格區間-最低']
                price_high = info['買回價格區間-最高']
                shares_text = info['預定買回股數_text']
                warning_text = f"注意：庫藏股 ({info['狀態']}) | 期間: {start_date} ~ {end_date} | 價格區間: {price_low} ~ {price_high} 元 | 預定買回: {shares_text}"
                ax_title.text(0.5, y_cursor, warning_text, ha='center', va='top', fontsize=16, weight='bold', color='white',
                            bbox=dict(facecolor='#008080', edgecolor='darkgreen', boxstyle='round,pad=0.5', lw=2),
                            transform=ax_title.transAxes, linespacing=1.4)
                y_cursor -= y_warning_step
                has_special_tag = True
        except Exception as e:
            logging.warning(f"為 {stock_id} 繪製庫藏股警告標籤時出錯: {e}")

        try:
            if not self.disposal_stocks_df.empty and stock_id in self.disposal_stocks_df.index:
                info_series = self.disposal_stocks_df.loc[stock_id]
                if isinstance(info_series, pd.DataFrame):
                    info = info_series.iloc[0]
                else:
                    info = info_series
                start_date_str = info['處置開始時間'].strftime('%Y-%m-%d')
                end_date_str = info['處置結束時間'].strftime('%Y-%m-%d')
                warning_text = f"注意：處置股 ({info['狀態']}) | 期間: {start_date_str} ~ {end_date_str}"
                ax_title.text(0.5, y_cursor, warning_text, ha='center', va='top', fontsize=18, weight='bold', color='white',
                            bbox=dict(facecolor='#FF6347', edgecolor='darkred', boxstyle='round,pad=0.5', lw=2),
                            transform=ax_title.transAxes)
                y_cursor -= y_warning_step
                has_special_tag = True

            if not self.cash_increase_df.empty and stock_id in self.cash_increase_df.index:
                info_series = self.cash_increase_df.loc[stock_id]
                if isinstance(info_series, pd.DataFrame):
                    info_filtered = info_series[info_series['現金增資認購價(元/股)'] > 0].sort_values('公告日期')
                    info = info_filtered.iloc[-1] if not info_filtered.empty else info_series.iloc[-1]
                else:
                    info = info_series
                price = info['現金增資認購價(元/股)']
                ex_date = info['除權交易日'].strftime('%Y-%m-%d')
                warning_text = f"注意：現金增資 ({info['狀態']}) | 認購價: {price} | 除權日: {ex_date}"
                ax_title.text(0.5, y_cursor, warning_text, ha='center', va='top', fontsize=18, weight='bold', color='white',
                            bbox=dict(facecolor='#4682B4', edgecolor='darkblue', boxstyle='round,pad=0.5', lw=2),
                            transform=ax_title.transAxes)
                y_cursor -= y_warning_step
                has_special_tag = True
        except Exception as e:
            logging.warning(f"為 {stock_id} 繪製處置/增資警告標籤時出錯: {e}")

        try:
            if not self.special_signals_df.empty and stock_id in self.special_signals_df.index:
                signals = self.special_signals_df.loc[stock_id]
                rank = self.amt_rank_series.get(stock_id)
                if pd.notna(rank):
                    total_stocks = len(self.amt_rank_series.dropna())
                    is_top_10_percent = (rank <= total_stocks * 0.1)
                    if signals.get('is_rev_new_high', False):
                        has_special_tag = True
                        special_text_parts = ["營收創新高!"]
                        if is_top_10_percent:
                            high_period = int(signals.get('amt_new_high_period', 0))
                            if high_period > 0:
                                special_text_parts.append(f"& 成交額創{high_period}日高")
                            latest_amt_val = self.amt_df[stock_id].iloc[-1]
                            special_text_parts.append(f"| 市場排名: #{int(rank)}")
                            special_text_parts.append(f"| 當日金額: {latest_amt_val/1e8:.1f}億")
                        special_text = " ".join(special_text_parts)
                        ax_title.text(0.5, y_cursor, special_text, ha='center', va='top', fontsize=18, weight='bold', color='black',
                                    bbox=dict(facecolor='gold', edgecolor='red', boxstyle='round,pad=0.5', lw=2),
                                    transform=ax_title.transAxes)
                        y_cursor -= y_warning_step
        except Exception as e:
            logging.warning(f"為 {stock_id} 繪製特殊訊號標籤時出錯: {e}")
            
        main_title_y = y_cursor
        final_title_text = f"{title_line1}\n{consolidated_info_line}"
        ax_title.text(0.05, main_title_y, final_title_text, ha='left',
                    va='top', fontsize=18, linespacing=1.5, transform=ax_title.transAxes)
        
        # doc: =============================================================================
        # doc: --- [v4.0 核心修正] 具備自動換行功能的標籤佈局引擎 ---
        # doc: =============================================================================
        y_cursor = main_title_y - 0.4  # 從主標題下方開始排列
        x_pos = 0.05                  # 起始 X 座標
        line_height = 0.18            # 每一行標籤的高度

        for tag_text, tag_color in tags:
            # 估算標籤寬度 (這是一個簡化估算，但已足夠)
            # 'gbk' 編碼能較好地處理中英文字符寬度
            est_width_in_fig_coords = len(tag_text.encode('gbk', 'ignore')) * 0.007
            
            # 如果加入這個標籤會超出右邊界 (0.98)，則換行
            if x_pos + est_width_in_fig_coords > 0.98:
                y_cursor -= line_height  # Y 座標往下移一行
                x_pos = 0.05             # X 座標重置到最左邊
            
            # 在計算好的位置上繪製標籤
            ax_title.text(x_pos, y_cursor, tag_text, ha='left', va='top', fontsize=14, weight='bold', color='white',
                                    bbox=dict(facecolor=tag_color,
                                            edgecolor='none', boxstyle='round,pad=0.3'),
                                    transform=ax_title.transAxes)
            
            # 更新下一個標籤的 X 座標
            x_pos += (est_width_in_fig_coords + 0.01)
            
 
            

     
    def _find_non_overlapping_y(self, y, occupied, y_range):
        min_sep = 0.04 * y_range
        y_final = y
        # 增加一個計數器，防止無限迴圈
        attempts = 0
        while any(abs(y_final - pos) < min_sep for pos in occupied) and attempts < 100:
            y_final += min_sep * 0.5  # 每次移動半個最小間距，更精細
            attempts += 1
        return y_final
    
    
    # finlab程式碼格式
    def _plot_volume_with_daytrade_split(self, ax_volume, dates, stable_vol, day_trade_shares):
        """
        [doc]
        繪製區分「安定籌碼」與「當沖籌碼」的堆疊成交量圖。
        """
        # 繪製底層的安定成交量
        ax_volume.bar(dates, stable_vol, width=1.0, color='#26A69A', alpha=0.8, label='安定成交量')
        
        # 在安定成交量之上，疊加當沖成交量
        ax_volume.bar(dates, day_trade_shares, bottom=stable_vol, width=1.0, color='#FF8C00', alpha=0.7, label='當沖成交量')

        # 均線計算基礎應為總成交量
        total_vol = stable_vol + day_trade_shares
        ax_volume.plot(dates, total_vol.rolling(5).mean(), color='#3366CC', lw=1.2, label='5日均量')
        ax_volume.plot(dates, total_vol.rolling(20).mean(), color='#FF6600', lw=1.2, label='20日均量')





    def _plot_primary_chart(self, ax_main, ax_volume_upper, ax_volume_lower, s, days, sr_data, chart_type='short', historical_events=None):
        """
        doc:
        [v31.0] 新增 chart_type 參數，智慧選擇繪製「安定成交量」或「投信買賣超」。
        """
        df = pd.DataFrame({
            'open': self.analyzer.all_data['open'][s], 'high': self.analyzer.all_data['high'][s],
            'low': self.analyzer.all_data['low'][s], 'close': self.analyzer.all_data['close'][s],
            'volume': self.analyzer.all_data['volume'][s]
        }).tail(days).dropna()
        if df.empty:
            ax_main.text(0.5, 0.5, "無K線數據", ha='center', va='center')
            return

        # --- 1. 繪製主 K 線圖 (此區塊無錯誤，維持不變) ---
        dates, up = df.index, df['close'] >= df['open']
        up_color, down_color = '#EF5350', '#26A69A'
        ax_main.bar(dates[up], df['high'][up] - df['low'][up], 1.0, bottom=df['low'][up], color=up_color, ec=up_color, lw=0.5, zorder=3, alpha=0.1)
        ax_main.bar(dates[~up], df['high'][~up] - df['low'][~up], 1.0, bottom=df['low'][~up], color=down_color, ec=down_color, lw=0.5, zorder=3, alpha=0.1)
        ax_main.bar(dates[up], df['close'][up] - df['open'][up], 1.0, bottom=df['open'][up], color=up_color, ec=up_color, lw=0.5, zorder=3)
        ax_main.bar(dates[~up], df['open'][~up] - df['close'][~up], 1.0, bottom=df['close'][~up], color=down_color, ec=down_color, lw=0.5, zorder=3)
        ma_colors = {'5': '#FFC300', '20': '#00BFFF', '60': '#9966CC'}
        for ma in [5, 20, 60]:
            if len(df) >= ma:
                ax_main.plot(df['close'].rolling(ma).mean(), label=f'{ma}MA', lw=1.5, zorder=4, color=ma_colors.get(str(ma)))
                
        # doc: =============================================================================
        # doc: --- [Gemini v2.0 + v3.0 文字標籤修改] 繪製歷史事件浮水印與文字 ---
        # doc: =============================================================================
        disposal_label_added = False
        increase_label_added = False
        if historical_events:
            # 繪製歷史處置期間 (灰色半透明背景)
            for start, end in historical_events.get('disposals', []):
                label = '處置期間' if not disposal_label_added else ""
                ax_main.axvspan(start, end, color='gray', alpha=0.15, zorder=0, label=label)
                disposal_label_added = True

                # doc: --- [核心修改] 在背景區域頂部加上文字說明 ---
                # 使用 blended transform，讓 Y 軸座標相對於圖表高度 (0=底, 1=頂)
                trans = ax_main.get_yaxis_transform()
                ax_main.text(start, 0.98, f"處置 {start.strftime('%m/%d')}~{end.strftime('%m/%d')}",
                             transform=trans,
                             ha='left',
                             va='top',
                             color='#555555',
                             fontsize=10,
                             bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1))
                # doc: --- [修改結束] ---


            # 繪製歷史現增除權日 (紫色垂直虛線)
            for event_date, price in historical_events.get('cash_increases', []):
                label = '現金增資' if not increase_label_added else ""
                ax_main.axvline(x=event_date, color='purple', linestyle=':', linewidth=1.5, alpha=0.6, zorder=2, label=label)
                increase_label_added = True
        # doc: --- [修改結束] ---
        
        # --- 價量分佈、壓力支撐、圖例等 (邏輯不變) ---     
                
                
        ax_vp = ax_main.twiny()
        ax_vp.set_xticks([]); ax_vp.set_xlabel('價量分佈', fontsize=12)
        for vp_data, color, alpha in [(sr_data.get('vp_long'), 'gray', 0.2), (sr_data.get('vp_short'), 'orange', 0.4)]:
            if vp_data:
                bc, up_vp, down_vp, price_bins = vp_data
                height = (price_bins[1] - price_bins[0]) * 0.9 if len(price_bins) > 1 else 0.1
                if height > 0: ax_vp.barh(bc, up_vp + down_vp, height=height, color=color, alpha=alpha, zorder=1, edgecolor=None)
        if sr_data.get('trapped_zones'):
            for i, zone in enumerate(sr_data['trapped_zones']): ax_main.axhspan(zone['lower'], zone['upper'], facecolor='cyan', alpha=0.2, zorder=0, label='套牢區' if i == 0 else "")
        y_low, y_high = df['low'].min(), df['high'].max()
        if pd.notna(y_low) and pd.notna(y_high) and (price_range := y_high - y_low) > 0: ax_main.set_ylim(y_low - price_range * 0.1, y_high + price_range * 0.1)
        if sr_data and sr_data.get('levels'):
            top_levels = sorted(sr_data['levels'].items(), key=lambda item: item[1][1], reverse=True)[:5]
            sorted_levels_by_price = sorted(top_levels, key=lambda item: item[0])
            y_min_lim, y_max_lim = ax_main.get_ylim(); occupied_y = []
            for lvl, (label, score) in sorted_levels_by_price:
                if not (y_min_lim <= lvl <= y_max_lim): continue
                if score > 5: lw, ls = 2.0, '-'
                elif score > 2: lw, ls = 1.5, '-'
                else: lw, ls = 1.2, '--'
                c = 'darkred' if '壓力' in label else 'darkgreen'
                if '回測' in label or '轉支' in label: c = 'darkorange'
                ax_main.axhline(y=lvl, c=c, ls=ls, lw=lw, alpha=0.9, zorder=2)
                safe_y = self._find_non_overlapping_y(lvl, occupied_y, y_max_lim - y_min_lim); occupied_y.append(safe_y)
                ax_main.text(1.01, safe_y, f' {label}', c=c, fontsize=11, va='center', transform=ax_main.get_yaxis_transform(), bbox=dict(fc='w', ec=c, lw=0.5, alpha=0.7))
                if abs(safe_y - lvl) > (y_max_lim - y_min_lim) * 0.005:
                    trans = ax_main.get_yaxis_transform()
                    ax_main.plot([1.005, 1.01], [lvl, safe_y], transform=trans, color=c, linestyle=':', linewidth=0.8, clip_on=False)
        legend_handles = []
        for ma, color in ma_colors.items(): legend_handles.append(plt.Line2D([0], [0], color=color, lw=2, label=f'{ma}MA'))
        if sr_data and sr_data.get('trapped_zones'): legend_handles.append(plt.Rectangle((0, 0), 1, 1, facecolor='cyan', alpha=0.3, label='套牢區'))
        ax_main.legend(handles=legend_handles, loc='upper left', fontsize=10, facecolor='white', framealpha=1.0)

        # --- 2. 繪製成交量圖 (智慧選擇，此區塊無錯誤，維持不變) ---
        ax_volume_upper.clear()
        if chart_type == 'long':
            # 如果是長線圖 (480日)，呼叫我們在步驟一新增的「投信買賣佔比」函式
        # 如果是長線圖 (480日)，呼叫我們在步驟一新增的「投信買賣佔比」函式
            self._plot_it_net_buy_ratio(ax_volume_upper, s, days)
            # self._plot_it_position_heatmap(ax_volume_upper, s, days)
            # self._plot_it_net_buy_ratio(ax_volume_upper, s, days) # <--- 將此行原本的 _plot_it_net_buy_volume 換成 _plot_it_net_buy_ratio
        else:
            day_trade_vol_all = self.analyzer.all_data.get('day_trade_vol', pd.DataFrame())
            day_trade_shares = pd.Series(0, index=df.index)
            if s in day_trade_vol_all.columns:
                day_trade_series = day_trade_vol_all[s].reindex(df.index).fillna(0)
                day_trade_shares = (day_trade_series / 2 / 1000)
            total_volume = df['volume']
            stable_volume = (total_volume - day_trade_shares).clip(lower=0)
            ax_volume_upper.bar(df.index[up], stable_volume[up], width=1.0, color=up_color, alpha=0.7)
            ax_volume_upper.bar(df.index[~up], stable_volume[~up], width=1.0, color=down_color, alpha=0.7)
            ax_volume_upper.plot(df.index, stable_volume.rolling(5).mean(), color='#3366CC', lw=1.2, label='5日均量')
            ax_volume_upper.plot(df.index, stable_volume.rolling(20).mean(), color='#FF6600', lw=1.2, label='20日均量')
            ax_volume_upper.set_ylabel('安\n定\n成\n交\n量', rotation=0, labelpad=25, ha='right', va='center', fontsize=14)
            ax_volume_upper.legend(loc='upper left', fontsize=10, facecolor='white', framealpha=1.0)
            ax_volume_upper.tick_params(axis='x', labelbottom=False)
            ax_volume_upper.yaxis.set_major_formatter(FuncFormatter(self.format_volume))
            ax_volume_upper.grid(True, axis='y', linestyle='--', alpha=0.6)

        # doc: --- [核心修正區] ---
        # doc: 以下是「基本面」圖的繪圖邏輯，已移除所有不必要的反斜線 `\`
        # doc: --------------------
        ax_volume_lower.clear()
        ax_volume_lower.set_facecolor('white')
        ax_volume_lower.grid(True, linestyle='--', alpha=0.6)
        def convert_quarterly_index(quarterly_series):
            if quarterly_series.empty: return quarterly_series
            date_str_index = quarterly_series.index.astype(str)
            date_str_index = date_str_index.str.replace('/Q1', '-03-31').str.replace('-Q1', '-03-31')
            date_str_index = date_str_index.str.replace('/Q2', '-06-30').str.replace('-Q2', '-06-30')
            date_str_index = date_str_index.str.replace('/Q3', '-09-30').str.replace('-Q3', '-09-30')
            date_str_index = date_str_index.str.replace('/Q4', '-12-31').str.replace('-Q4', '-12-31')
            quarterly_series.index = pd.to_datetime(date_str_index, format='%Y-%m-%d')
            return quarterly_series
        try:
            if days == self.params['plot_days_short']:
                ax_volume_lower.set_title("季度財務比率 (近8季)", fontsize=12)
                ax2_fundamental = ax_volume_lower.twinx()
                roe = convert_quarterly_index(self.analyzer.all_data['roe'][s].dropna().tail(8)); gm = convert_quarterly_index(self.analyzer.all_data['gross_margin'][s].dropna().tail(8)); om = convert_quarterly_index(self.analyzer.all_data['operating_margin'][s].dropna().tail(8))
                roe_aligned = roe.reindex(df.index, method='ffill'); gm_aligned = gm.reindex(df.index, method='ffill'); om_aligned = om.reindex(df.index, method='ffill')
                roe_colors = ['#e63946' if x >= 0 else '#f28482' for x in roe_aligned]
                bars = ax_volume_lower.bar(roe_aligned.index, roe_aligned, label='ROE(%)', color=roe_colors, width=2.0, alpha=0.7)
                ax_volume_lower.set_ylabel("ROE\n(%)", rotation=0, labelpad=25, ha='left', va='center', fontsize=12, color='#e63946'); ax_volume_lower.tick_params(axis='y', labelcolor='#e63946')
                p1, = ax2_fundamental.plot(gm_aligned.index, gm_aligned, label='毛利率(%)', color='#1d3557', lw=1.8, ls='--', marker='o', markersize=4)
                p2, = ax2_fundamental.plot(om_aligned.index, om_aligned, label='利益率(%)', color='#457b9d', lw=1.8, ls=':', marker='^', markersize=4)
                ax2_fundamental.set_ylabel("毛\n利\n率\n/\n利\n益\n率\n(%)", rotation=0, labelpad=35, ha='right', va='center', fontsize=12, color='#1d3557'); ax2_fundamental.tick_params(axis='y', labelcolor='#1d3557'); ax2_fundamental.grid(False)
                legend_elements = [p1, p2, Patch(facecolor='#e63946', alpha=0.7, label='ROE(%)')]
                ax_volume_lower.legend(handles=legend_elements, loc='upper left', fontsize=10, facecolor='white', framealpha=1.0)
            elif days == self.params['plot_days_long']:
                ax_volume_lower.set_title("季度每股盈餘 (EPS) (近16季)", fontsize=12)
                eps = convert_quarterly_index(self.analyzer.all_data['eps'][s].dropna().tail(16))
                eps_aligned = eps.reindex(df.index, method='ffill')
                colors = ['#f4a261' if x >= 0 else '#2a9d8f' for x in eps_aligned]
                ax_volume_lower.bar(eps_aligned.index, eps_aligned, color=colors, width=2.0, alpha=0.8)
                ax_volume_lower.set_ylabel("EPS\n(元)", rotation=0, labelpad=25, ha='right', va='center', fontsize=12)
                ax_volume_lower.axhline(0, color='black', lw=0.5, ls='--')
        except Exception as e:
            ax_volume_lower.text(0.5, 0.5, f"無法繪製基本面圖:\n{e}", ha='center', va='center'); logging.warning(f"為 {s} 繪製基本面圖時出錯: {e}")
        finally:
            if 'ax2_fundamental' in locals():
                min_left, max_left = ax_volume_lower.get_ylim(); min_right, max_right = ax2_fundamental.get_ylim()
                if min_left > 0 and min_right < 0: ax_volume_lower.set_ylim(bottom=0)
                if min_right > 0 and min_left < 0: ax2_fundamental.set_ylim(bottom=0)
            plt.setp(ax_volume_lower.get_xticklabels(), rotation=30, ha='right')



       
        

    def _plot_institutional_flow(self, ax, flow_data):
        ax.set_title("三大法人流向 (週合計)", fontsize=16, pad=12)
        if flow_data.empty:
            ax.text(0.5, 0.5, "無數據", ha='center', va='center',
                    transform=ax.transAxes, fontsize=16)
            return
        positive_flow = flow_data['total'] >= 0
        ax.bar(flow_data.index[positive_flow], flow_data['total']
               [positive_flow], width=5, color='#EF5350', alpha=0.8, label='週買超')
        ax.bar(flow_data.index[~positive_flow], flow_data['total']
               [~positive_flow], width=5, color='#26A69A', alpha=0.8, label='週賣超')
        ax2 = ax.twinx()
        ax2.plot(flow_data.index,
                 flow_data['ma5'], color='orange', lw=1.8, ls='-', label='5週均線')
        ax2.plot(flow_data.index, flow_data['ma10'],
                 color='blue', lw=1.8, ls='--', label='10週均線')
        ax.set_ylabel('買\n賣\n超', rotation=0, labelpad=30,
                      ha='right', va='center', fontsize=14)
        ax2.set_ylabel('均\n線', rotation=0, labelpad=30,
                       ha='right', va='center', fontsize=14)
        ax.axhline(0, color='black', lw=0.5, ls='--')
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines + lines2, labels + labels2, fontsize=11,
                   loc='center left', bbox_to_anchor=(1.15, 0.5))
        ax.grid(True, axis='y', linestyle='--', alpha=0.6)

    def _plot_broker_flow(self, ax, flow_data):
        ax.set_title("關鍵券商流向 (週合計)", fontsize=16, pad=12)
        if flow_data.empty:
            ax.text(0.5, 0.5, "無數據", ha='center', va='center',
                    transform=ax.transAxes, fontsize=16)
            return
        positive_flow = flow_data['total'] >= 0
        ax.bar(flow_data.index[positive_flow], flow_data['total']
               [positive_flow], width=5, color='#EF5350', alpha=0.8, label='週買超')
        ax.bar(flow_data.index[~positive_flow], flow_data['total']
               [~positive_flow], width=5, color='#26A69A', alpha=0.8, label='週賣超')
        ax2 = ax.twinx()
        ax2.plot(flow_data.index,
                 flow_data['ma5'], color='magenta', lw=1.8, ls='-', label='5週均線')
        ax2.plot(flow_data.index, flow_data['ma10'],
                 color='black', lw=1.8, ls='--', label='10週均線')
        ax.set_ylabel('買\n賣\n超', rotation=0, labelpad=30,
                      ha='right', va='center', fontsize=14)
        ax2.set_ylabel('均\n線', rotation=0, labelpad=30,
                       ha='right', va='center', fontsize=14)
        ax.axhline(0, color='black', lw=0.5, ls='--')
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax2.legend(lines + lines2, labels + labels2, fontsize=11,
                   loc='center left', bbox_to_anchor=(1.15, 0.5))
        ax.grid(True, axis='y', linestyle='--', alpha=0.6)

    def _plot_shareholder_structure(self, ax, data):
        ax.set_title("大戶/散戶持股比例變化 (週)", fontsize=16, pad=12)
        if data.empty:
            ax.text(0.5, 0.5, "無數據", ha='center', va='center',
                    transform=ax.transAxes, fontsize=16)
            return
        ax.plot(data.index, data['ratio'] * 100,
                color='darkblue', lw=2, label='大戶/散戶比')
        ax.set_ylabel("大\n戶\n/\n散\n戶\n比", rotation=0, labelpad=30,
                      ha='right', va='center', color='darkblue', fontsize=14)
        ax.tick_params(axis='y', labelcolor='darkblue', labelsize=12)
        ax2 = ax.twinx()
        ax2.bar(data.index, data['diff1'], width=10,
                color='orange', alpha=0.6, label='籌碼集中速度')
        ax2.bar(data.index, data['diff2'], width=10,
                color='grey', alpha=0.5, label='籌碼集中加速度')
        ax2.axhline(0, color='black', lw=0.5, ls='--')
        ax2.set_ylabel("趨\n勢\n變\n化", rotation=0, labelpad=30,
                       ha='right', va='center', fontsize=14)
        lines, labels = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines + lines2, labels + labels2, fontsize=11,
                  loc='center left', bbox_to_anchor=(1.15, 0.5))
        ax2.tick_params(axis='y', labelsize=12)




    def _get_sentiment_scenario(self, data):
        """
        [新增] 根據情緒分析數據，判斷當前股票的籌碼情境。
        [修正] 新增分母檢查，避免 RuntimeWarning。
        """
        # --- 數據準備 ---
        if len(data) < 11:
            return "數據不足"

        # [修正] 在進行除法前檢查分母是否為零
        price_denom_5d = data['close'].iloc[-6]
        price_chg_5d = (data['close'].iloc[-1] / price_denom_5d - 1) * 100 if price_denom_5d > 0 else 0

        margin_denom_5d = data['margin_balance'].iloc[-6]
        margin_chg_5d = (data['margin_balance'].iloc[-1] / margin_denom_5d - 1) * 100 if margin_denom_5d > 0 else 0

        price_denom_10d = data['close'].iloc[-11]
        price_chg_10d = (data['close'].iloc[-1] / price_denom_10d - 1) * 100 if price_denom_10d > 0 else 0
        
        margin_denom_10d = data['margin_balance'].iloc[-11]
        margin_chg_10d = (data['margin_balance'].iloc[-1] / margin_denom_10d - 1) * 100 if margin_denom_10d > 0 else 0

        # 價值指數的變化
        value_index_chg_10d = 0
        if 'collateral_value_index' in data.columns and data['collateral_value_index'].notna().all():
            value_denom_10d = data['collateral_value_index'].iloc[-11]
            if value_denom_10d > 0:
                value_index_chg_10d = (data['collateral_value_index'].iloc[-1] / value_denom_10d - 1) * 100

        # --- 情境規則判斷 (可依您的經驗調整閾值) ---
        # 規則的優先級很重要，最危險的放最前面

        # 情境2: 危險凹單 (價跌資增)
        if price_chg_5d < -3 and margin_chg_5d > 1:
            return "情境：危險凹單 (價跌資增)"

        # 情境5: 末跌段趕底 (融資斷頭)
        if price_chg_10d < -8 and margin_chg_10d < -5:
            return "情境：末跌段趕底 (融資斷頭)"

        # 情境4: 法人主導 (價漲資減/平)
        if price_chg_10d > 5 and margin_chg_10d < 1:
            return "情境：法人主導 (籌碼安定)"

        # 情境1: 主升段行情 (價漲資增)
        if price_chg_10d > 5 and margin_chg_10d > 3 and value_index_chg_10d > price_chg_10d:
            return "情境：主升段行情 (價量齊揚)"

        # 情境3: 洗盤換手 (價穩資減)
        if price_chg_10d > -5 and margin_chg_10d < -2:
            return "情境：洗盤換手 (籌碼沉澱)"

        # 預設情況
        return "情境：觀察中"



    def _plot_sentiment(self, ax, data):
        # [修改] 改為呼叫全域函式，並根據回傳的 "分數" 決定顏色
        conclusion, _ = get_margin_sentiment_conclusion(data) # 分數在此處暫不使用，但保留以符合函式回傳
        scenario_text = self._get_sentiment_scenario(data)

        # [修改] 建立一個結論與顏色的映射字典
        sentiment_color_map = {
            "籌碼趨向多方": "#e63946",
            "軋空行情醞釀中": "#f7a072",
            "籌碼趨向空方": "#2a9d8f",
            "多空交戰激烈": "#1d3557",
            "市場人氣退潮": "#6c757d",
            "觀察中": "black",
            "數據不足": "black"
        }
        conclusion_color = sentiment_color_map.get(conclusion, "black")
        
        ax.set_title(f"進階情緒分析\n{scenario_text}\n結論：{conclusion}",
                    fontsize=16, pad=20, color=conclusion_color)

        if data.empty:
            ax.text(0.5, 0.5, "無數據", ha='center', va='center',
                    transform=ax.transAxes, fontsize=16)
            return

        # --- 後續繪圖邏輯維持不變 ---
        p1, = ax.plot(data.index, data['margin_usage'],
                    color='orange', lw=1.5, label='融資使用率(%)')
        ax.fill_between(data.index, data['margin_usage'], data['margin_usage_threshold'], where=data['margin_usage']
                        > data['margin_usage_threshold'], color='orange', alpha=0.3, interpolate=True, label='高融資警示區')
        p2, = ax.plot(data.index, data['short_usage'],
                    color='blueviolet', lw=1.5, label='融券使用率(%)', alpha=0.7)
        ax.set_ylabel("使\n用\n率\n(%)", rotation=0, labelpad=30,
                    ha='right', va='center', fontsize=14)
        ax.tick_params(axis='y', labelcolor='black', labelsize=12)

        ax2 = ax.twinx()
        p3, = ax2.plot(data.index, data['short_to_margin_ratio'],
                    color='deepskyblue', lw=1.5, ls='--', label='券資比')

        if 'collateral_value_index' in data.columns and data['collateral_value_index'].notna().any():
            p4, = ax2.plot(data.index, data['collateral_value_index'],
                        color='green', lw=2.0, ls=':', label='融資品價值指數')
            ax2.axhline(100, color='gray', lw=1, ls='-.')
        else:
            p4 = None

        ax2.set_ylabel("指\n數\n/\n比\n率", rotation=0, labelpad=30, ha='right',
                    va='center', color='black', fontsize=14)
        ax2.tick_params(axis='y', labelcolor='black', labelsize=12)

        handles = [p1, p2, p3]
        labels = ['融資使用率(%)', '融券使用率(%)', '券資比']
        if p4:
            handles.append(p4)
            labels.append('融資品價值指數')

        ax.legend(handles=handles, labels=labels,
                fontsize=11, loc='upper left')

        ax2.grid(True, axis='y', linestyle='--', alpha=0.6)

    def format_volume(self, x, pos):
        """格式化成交量為千或萬單位"""
        if abs(x) >= 10000:
            return f'{x/10000:,.1f}萬'
        if abs(x) >= 1000:
            return f'{x/1000:,.1f}千'
        return f'{x:,.0f}'

    def format_revenue(self, x, pos):
        """格式化營收為萬或億單位"""
        if abs(x) >= 1e8:
            return f'{x/1e8:,.1f}億'
        if abs(x) >= 1e4:
            return f'{x/1e4:,.0f}萬'
        return f'{x:,.0f}'

    # finlab程式碼格式
    def _plot_fundamentals(self, ax, data_df):
        """
        doc:
        繪製企業營運基本面圖表，包含月營收與月均價。
        [v2.0 Gemini 修正版]
        - 修正了月均價 (monthly_avg_price) 與營收月份 (x_axis_dates) 因存在 NaN 值而導致繪圖長度不匹配的問題。
        - 透過將日期與價格配對後再移除空值，確保 X, Y 軸資料長度一致。
        """
        try:
            
            # 在所有操作開始前，先移除 data_df 中可能存在的重複索引，並保留最後一筆
            data_df = data_df[~data_df.index.duplicated(keep='last')].copy()
            # --- [修正結束] ---
        
            if data_df.empty or 'revenue' not in data_df.columns or data_df['revenue'].dropna().empty:
                ax.set_title("企業營運基本面 (月)", fontsize=16, pad=12)
                raise ValueError("基本面數據為空或無營收資料")
            
            rev = data_df['revenue'].dropna().tail(24)
            if rev.empty:
                ax.set_title("企業營運基本面 (月)", fontsize=16, pad=12)
                raise ValueError("月營收數據在移除NaN後為空")

            latest_revenue_month = rev.index[-1] - pd.DateOffset(months=1)
            title_date_str = latest_revenue_month.strftime("%Y年%m月")
            dynamic_title = f"企業營運基本面 (最新營收: {title_date_str})"
            ax.set_title(dynamic_title, fontsize=16, pad=12)

            x_axis_dates = rev.index - pd.DateOffset(months=1)
            
            yoy = data_df['yoy'].reindex(rev.index).fillna(0)
            rev_ma2 = rev.rolling(window=2, min_periods=1).mean()
            high_revenue = (rev_ma2 == rev_ma2.rolling(9, min_periods=6).max()).fillna(False)

            def get_color(yoy_val, is_high):
                if is_high: return '#FFD700'
                if not np.isfinite(yoy_val): return '#BBBBBB'
                if yoy_val > 0:
                    return '#FF0000' if yoy_val > 20 else '#FF3333' if yoy_val > 10 else '#FF6666'
                else:
                    return '#006400' if yoy_val < -20 else '#008000' if yoy_val < -10 else '#228B22'
            
            colors = [get_color(y, h) for y, h in zip(yoy.values, high_revenue.values)]
            
            ax.bar(x_axis_dates, rev.values, color=colors, alpha=0.7, width=20, zorder=2)
            
            ma3, ma12 = rev.rolling(3, min_periods=1).mean(), rev.rolling(12, min_periods=1).mean()

            ax.plot(x_axis_dates, ma3.values, color='#1E90FF', label='營收MA(3)', linewidth=2, zorder=3)
            ax.plot(x_axis_dates, ma12.values, color='#FF8C00', label='營收MA(12)', linewidth=2.5, zorder=3, linestyle='--')
            
            ax.set_ylabel("月\n營\n收", rotation=0, labelpad=35, ha='right', va='center', color='steelblue', fontsize=14)
            ax.tick_params(axis='y', labelcolor='steelblue', labelsize=12)
            ax.yaxis.set_major_formatter(FuncFormatter(self.format_revenue))
            ax.grid(True, linestyle='--', alpha=0.6, axis='y')
            
            has_monthly_price = False
            ax_price = ax.twinx()
            try:
                if 'monthly_avg_price' not in data_df.columns or data_df['monthly_avg_price'].dropna().empty:
                    raise ValueError("無月均價數據")
                
                # doc: =============================================================================
                # doc: --- [Gemini 核心修正] --
                # doc: 1. 將營收日期(x_axis_dates)與月均價(monthly_avg_price)對齊
                # doc: 2. 建立一個暫時的 DataFrame，將兩者配對
                # doc: 3. 使用 .dropna() 同時移除日期和價格中的無效配對
                # doc: 4. 使用清理過的數據進行繪圖，確保 X, Y 軸長度一致
                # doc: =============================================================================
                monthly_price_series = data_df['monthly_avg_price'].reindex(rev.index)
                aligned_plot_df = pd.DataFrame({'date': x_axis_dates, 'price': monthly_price_series}).dropna()

                if aligned_plot_df.empty:
                    raise ValueError("對齊營收日期後無月均價數據")

                ax_price.plot(aligned_plot_df['date'], aligned_plot_df['price'], color='black', linewidth=2.5, marker='s', markersize=7, label='月均價', zorder=4)
                has_monthly_price = True
                
            except Exception as e_price:
                logging.warning(f"無法繪製月均價圖: {e_price}")
                ax_price.text(0.5, 0.5, "月均價資料不足\n無法繪製", ha='center', va='center', transform=ax_price.transAxes, fontsize=14, color='red')
            
            ax_price.set_ylabel("月\n均\n價", rotation=0, labelpad=30, ha='right', va='center', color='black', fontsize=14)
            ax_price.tick_params(axis='y', labelcolor='black', labelsize=12)
            ax_price.grid(False)
            
            lines1, labels1 = ax.get_legend_handles_labels()
            all_lines, all_labels = lines1, labels1
            if has_monthly_price:
                lines2, labels2 = ax_price.get_legend_handles_labels()
                all_lines += lines2
                all_labels += labels2
            
            ax.legend(all_lines, all_labels, fontsize=11, loc='center left', bbox_to_anchor=(1.18, 0.5))
            
            for i in range(-6, 0):
                if i + len(rev) >= 0:
                    date_loc, bar_height, yoy_val = x_axis_dates[i], rev.iloc[i], yoy.iloc[i]
                    if pd.notna(bar_height) and pd.notna(yoy_val):
                        ax.text(date_loc, bar_height, f'{yoy_val:.0f}%', ha='center', va='bottom', fontsize=10, rotation=45, color='black', zorder=5)
        
        except Exception as e:
            ax.text(0.5, 0.5, f'無法繪製基本面圖\n{e}', ha='center', va='center', transform=ax.transAxes, fontsize=16)
        
        finally:
            ax.tick_params(axis='x', rotation=30, labelsize=12)
            
            

    # 請將此新函式加入 ReportGenerator class 中
    def _apply_appendix_filter(self, df_to_filter, price_change_col='當日漲跌幅(%)'):
        """
        doc: 根據 CONFIG 設定，對傳入的附錄用 DataFrame 進行漲跌幅篩選。
        """
        filter_config = self.config.get('APPENDIX_PRICE_CHANGE_FILTER', {})
        # 如果功能未啟用或傳入的 DataFrame 是空的，直接返回原樣
        if not filter_config.get('ENABLED', False) or df_to_filter.empty:
            return df_to_filter

        threshold = filter_config.get('THRESHOLD_PCT', 5.0)

        # 檢查 DataFrame 中是否已經有漲跌幅欄位
        if price_change_col not in df_to_filter.columns:
            # 若沒有，則嘗試從主資料表 (master_summary_df) 中合併進來
            id_col = 'stock_id' if 'stock_id' in df_to_filter.columns else '代號'
            if id_col in df_to_filter.columns and not self.master_summary_df.empty:
                # 執行合併
                df_with_change = pd.merge(
                    df_to_filter,
                    self.master_summary_df[['stock_id', price_change_col]],
                    left_on=id_col,
                    right_on='stock_id',
                    how='left'
                )
                # 如果合併後產生重複的 stock_id 欄位，清理一下
                if 'stock_id_y' in df_with_change.columns:
                     df_with_change = df_with_change.drop(columns=['stock_id_y']).rename(columns={'stock_id_x': 'stock_id'})
                df_to_filter = df_with_change
            else:
                logging.warning(f"無法套用附錄漲跌幅過濾：在 DataFrame 中找不到 '{price_change_col}' 欄位且無法自動合併。")
                return df_to_filter
        
        # 執行篩選
        original_count = len(df_to_filter)
        filtered_df = df_to_filter[df_to_filter[price_change_col] > threshold].copy()
        logging.info(f"    ↳ 已套用附錄篩選 (當日漲幅 > {threshold}%)，股票數量從 {original_count} 檔過濾至 {len(filtered_df)} 檔。")
        
        return filtered_df
            
       
   # finlab程式碼格式

    def create_report(self):
        """
        doc:
        [v4.0 穩健性增強版]
        - [核心修改] 在儲存圖檔前，新增對圖表物件 (fig, broker_fig) 的存在性檢查。
        - 此修改可確保只有成功生成的圖表會被寫入 PDF，從而完全解決 matplotlib 的 'UserWarning: constrained_layout' 警告。
        - 新增日誌記錄，當圖表因故生成失敗而被跳過時，會在 console 中明確提示。
        - 增加結構化註解，使報告生成的 A, B, C, D, E 各區塊流程更加清晰。

        [v28.0 修改] 調整主動式 ETF 附錄的生成流程。
        - 在 A 區塊呼叫分析函式後，會用一個變數 `daily_etf_appendix_data` 來儲存回傳的附錄資料。
        - 在 E 區塊的最後，新增對 `_plot_daily_etf_appendix_wrapper` 的呼叫，並傳入先前儲存的資料。
        """
        if not self.config.get('GENERATE_PDF_REPORT', False):
            logging.info("--- 已跳過【PDF 報告】生成 ---")
            return

        today_str = datetime.today().strftime("%Y%m%d")
        output_dir = os.path.join('output', datetime.today().strftime(
            '%Y'), datetime.today().strftime('%m'))
        os.makedirs(output_dir, exist_ok=True)
        final_pdf_path = os.path.join(
            output_dir, f'tomstrategypro{today_str}.pdf')
        logging.info(f"開始生成 PDF 報告，將保存至: {final_pdf_path}")

        run_sections = self.config.get('RUN_SECTIONS', {})
        daily_etf_appendix_data = None

        with PdfPages(final_pdf_path) as pdf:
            # doc: =============================================================================
            # doc: --- Section A: 宏觀與市場掃描 ---
            # doc: =============================================================================
            if run_sections.get('A1_MACRO_MARGIN', False):
                self._plot_market_margin_page(pdf, self.macro_margin_data)
            if run_sections.get('A2_MARGIN_TRAP', False) and self.margin_trap_df is not None and not self.margin_trap_df.empty:
                self._plot_margin_trap_page(
                    pdf, self.margin_trap_df, self.trap_start_date, self.trap_end_date)
            if run_sections.get('A3_INDUSTRY_FLOW', False):
                self._plot_industry_volume_flow_page(pdf)
            if run_sections.get('A4_MARKET_CAP_FLOW', False):
                self._plot_market_cap_flow(pdf)
            if run_sections.get('A5_MARKET_BREADTH_200D', False):
                self._plot_market_breadth_page(pdf)
            if run_sections.get('A6_MARKET_BREADTH_20D', False):
                self._plot_20day_high_page(pdf)
            if run_sections.get('A7_ETF_SUMMARY', False):
                self._plot_etf_summary_page(pdf)
            if run_sections.get('A8_ETF_QUADRANT', False):
                self._plot_etf_quadrant_page(pdf)
            if run_sections.get('A9_ETF_TREND', False):
                self._plot_etf_trend_page(pdf)
            if run_sections.get('A10_ETF_MONTHLY_QUADRANT', False):
                self._plot_etf_monthly_quadrant_page(pdf)
            if run_sections.get('A11_DAILY_ETF_ANALYSIS', False):
                daily_etf_appendix_data = self._analyze_and_plot_daily_etf_page(
                    pdf)

            # doc: =============================================================================
            # doc: --- Section B & C: 策略綜合分析 & 全市場產業分析 ---
            # doc: =============================================================================
            if run_sections.get('B1_MULTIFACTOR_SUMMARY', False):
                self._create_multifactor_summary_page(pdf)
            if run_sections.get('B2_STRATEGY_STACKED_CHART', False):
                self._plot_summary_page_stacked(pdf)
            if run_sections.get('B3_STRATEGY_TOPICS_PIE', False):
                self._plot_summary_page_topics_as_table(pdf)
            if run_sections.get('B4_STRATEGY_PERFORMANCE', False):
                self._plot_all_stock_performance_page(pdf)
            if run_sections.get('B5_STRATEGY_INDUSTRY_OVERVIEW', False):
                self._plot_custom_stock_industry_strength_overview_page(pdf)
            if run_sections.get('B6_STRATEGY_INDUSTRY_DRILLDOWN', False):
                self._plot_custom_stock_industry_strength_page(pdf)
            if run_sections.get('C1_OVERALL_INDUSTRY_STRENGTH', False):
                self._plot_industry_strength_page(pdf, output_dir, today_str)

            # doc: =============================================================================
            # doc: --- Section D: 個股詳細報告 (核心區塊) ---
            # doc: =============================================================================
            if run_sections.get('D1_DETAILED_STRATEGY_STOCKS', False):

                df_for_d_section = self.result_df.copy()

                filter_config = self.config.get(
                    'STRATEGY_STOCK_PRICE_CHANGE_FILTER', {})
                if filter_config.get('ENABLED', False) and not df_for_d_section.empty:
                    threshold = filter_config.get('THRESHOLD_PCT', 5.0)
                    logging.info(
                        f"--- 正在為【策略訊號股(D區)】套用當日漲幅 > {threshold}% 篩選 ---")
                    df_merged = pd.merge(df_for_d_section, self.master_summary_df[['stock_id', '當日漲跌幅(%)']],
                                         left_on='股票代碼', right_on='stock_id', how='left')
                    original_count = len(df_merged)
                    df_filtered = df_merged[df_merged['當日漲跌幅(%)'] > threshold].copy(
                    )
                    logging.info(
                        f"    ↳ 篩選結果: 股票數量從 {original_count} 檔過濾至 {len(df_filtered)} 檔。")
                    df_for_d_section = df_filtered

                logging.info("--- 開始繪製【策略訊號股】詳細報告 ---\n")
                save_image_sections = self.config.get(
                    'SAVE_IMAGE_SECTIONS', {})

                for _, row in df_for_d_section.iterrows():
                    stock_id = str(row['股票代碼'])
                    if stock_id not in self.analyzer.all_data.get('close', pd.DataFrame()).columns:
                        logging.warning(
                            f"股票 {stock_id} {row['股票名稱']} 因資料不齊全，在預加載時已被過濾，將跳過分析。")
                        continue
                    try:
                        logging.info(f"正在分析與繪製股票: {stock_id} {row['股票名稱']}")
                        analysis_data = self.analyzer.analyze(stock_id)
                        if analysis_data is None:
                            logging.warning(f"股票 {stock_id} 數據不足，已跳過。")
                            continue

                        # doc: --- [v4.0 核心修改] 新增對 fig 物件的穩健性檢查 ---
                        fig = self._plot_stock_page(analysis_data, row)
                        if fig:
                            pdf.savefig(fig)
                            if save_image_sections.get('D1_DETAILED_STRATEGY_STOCKS', False):
                                self._save_figure_if_enabled(
                                    fig, f"{stock_id}_{row['股票名稱']}_六宮格分析圖", stock_id)
                            plt.close(fig)
                        else:
                            logging.warning(
                                f"    ↳ 因數據不足或繪圖失敗，已跳過 {stock_id} 的六宮格圖。")

                        logging.info(f"    ↳ 正在生成 {stock_id} 的券商分析圖...")
                        broker_fig = self.broker_chart_generator.generate_chart(
                            stock_id)

                        if broker_fig:
                            pdf.savefig(broker_fig)
                            if save_image_sections.get('D1_DETAILED_STRATEGY_STOCKS', False):
                                self._save_figure_if_enabled(
                                    broker_fig, f"{stock_id}_{row['股票名稱']}_券商綜合分析圖", stock_id)
                            plt.close(broker_fig)
                        # doc: broker_fig 為 None 的情況，其內部已有日誌記錄，故此處不需額外 else 處理。
                        # doc: --- [修改結束] ---

                        self.plotted_stock_ids.add(stock_id)
                        gc.collect()
                    except Exception as e:
                        logging.error(
                            f"為股票 {stock_id} 生成圖表時發生嚴重錯誤: {e}", exc_info=True)
                        plt.close('all')
                        gc.collect()

            # doc: =============================================================================
            # doc: --- Section E: 附錄 ---
            # doc: =============================================================================
            appendix_plot_map = {
                'E1_APPENDIX_NEW_HIGH_200': self._plot_new_high_appendix_page,
                'E2_APPENDIX_NEW_HIGH_20': self._plot_new_high_20_appendix_page,
                'E3_APPENDIX_STRONG_TREND': self._plot_strong_trend_stock_pages,
                'E4_APPENDIX_RANK_JUMP': self._plot_rank_jump_stock_pages,
                'E5_APPENDIX_ETF_STOCKS': self._plot_etf_stocks_appendix_page,
                'E6_APPENDIX_DAILY_ETF': lambda p: self._plot_daily_etf_appendix_wrapper(p, daily_etf_appendix_data),
                'E7_APPENDIX_PRICE_CHANGE': self._plot_price_change_appendix_page, # [新增] 將新函式加入地圖中

            }
            appendix_order = self.config.get('APPENDIX_ORDER', [])
            run_sections = self.config.get('RUN_SECTIONS', {})
            logging.info("--- 開始繪製【附錄】(依自訂順序) ---")
            for section_key in appendix_order:
                if run_sections.get(section_key, False):
                    plot_function = appendix_plot_map.get(section_key)
                    if plot_function:
                        plot_function(pdf)

        logging.info(f"PDF 報告生成完成！已保存於 {final_pdf_path}")

# 執行

In [4]:
# ===================================================================
# --- (v9 - 雙模式同時執行最終版) ---
# ===================================================================
# 載入所有需要的函式庫
from matplotlib.backends.backend_pdf import PdfPages
import gc

# 輔助函式：取得 ETF 最新成分股
def get_latest_holdings_from_etfs(target_etfs: list, file_path: str = 'all_etf_holdings.csv') -> dict:
    """從指定的CSV檔案中，讀取目標ETF的最新成分股資訊。"""
    try:
        logging.info(f"正在從 {file_path} 讀取 ETF 持股資料...")
        df = pd.read_csv(file_path, parse_dates=['日期'], date_format='%Y/%m/%d')
        df['代號'] = df['代號'].astype(str)
        df_filtered = df[df['etf'].isin(target_etfs)].copy()
        if df_filtered.empty: return {}
        latest_dates = df_filtered.groupby('etf')['日期'].max().reset_index()
        latest_dates.rename(columns={'日期': '最新日期'}, inplace=True)
        df_merged = pd.merge(df_filtered, latest_dates, on='etf')
        final_holdings_df = df_merged[df_merged['日期'] == df_merged['最新日期']]
        if final_holdings_df.empty: return {}
        holdings_dict = {}
        for _, row in final_holdings_df.iterrows():
            stock_id = row['代號']
            etf_info = {'etf': row['etf'], 'weight': row['權重'], 'shares': row['持有數']}
            if stock_id not in holdings_dict: holdings_dict[stock_id] = []
            holdings_dict[stock_id].append(etf_info)
        logging.info(f"已成功篩選出 {len(holdings_dict)} 支不重複的成分股。")
        return holdings_dict
    except Exception as e:
        logging.error(f"讀取 ETF 持股檔案時發生錯誤: {e}", exc_info=True)
        return {}


# finlab程式碼格式
def run_focused_etf_analysis(CONFIG):
    """
    doc:
    [v2.7 Gemini 最終整合版]
    - 移除獨立的「市場事件總覽」頁面，將資訊呈現回歸到每一張個股的 K 線分析圖中。
    - 保留了查詢「處置股」、「現金增資」、「庫藏股」、「法說會」等市場事件的資料獲取邏輯，
      因為個股分析圖表的標題與圖示會需要用到這些資料。
    - (v2.5) 修正了 StockAnalyzer 初始化時缺少 'disposal_history_df' 和 'cash_increase_history_df' 參數的錯誤。
    """
    logging.info("\n======================================================")
    logging.info("--- 開始執行【模式一：獨立 ETF 成分股 PDF 分析】---")
    logging.info("======================================================")
    conf = CONFIG['FOCUSED_ETF_PDF_ANALYSIS']
    target_stock_ids_info = get_latest_holdings_from_etfs(conf['TARGET_ETFS'])
    target_stock_ids = list(target_stock_ids_info.keys())

    if not target_stock_ids:
        logging.warning("在獨立 ETF 分析模式中找不到任何成分股，流程結束。")
        return

    try:
        logging.info("--- (模式一) 正在預計算全市場訊號以供標籤使用 ---")
        special_signals_df = compute_special_signals()
        amt_df = data.get('price:成交金額')
        latest_amt = amt_df.iloc[-1]
        amt_rank_series = latest_amt.rank(method='min', ascending=False)
        logging.info("--- (模式一) 市場訊號預計算完成 ---")

        PARAMS = {'plot_days_short': 240, 'plot_days_long': 480, 'max_data_days': 550, 'sr_n_order_daily': 5, 'sr_n_order_weekly': 3, 'eps_atr_multiplier': 0.5, 'volume_ma_period': 20, 'poc_base_score': 2.0, 'volume_ratio_cap': 2.0, 'conversion_breakout_window': 10,
                  'conversion_hold_ratio': 0.6, 'conversion_retest_tolerance': 0.01, 'conversion_vol_boost_multiplier': 1.5, 'conversion_retest_bonus': 2.0, 'vp_window_short': 60, 'vp_window_long': 240, 'vp_bins': 80, 'plot_weeks_long': 104}

        stock_map_df = pd.read_excel('tw_stock_topics.xlsx')
        stock_map = stock_map_df.drop_duplicates(
            subset=['stock_no']).set_index('stock_no')['stock_name'].to_dict()

        logging.info("--- (模式一) 正在載入歷史與當前市場事件資料 ---")
        # 1. 處置股 (歷史+當前)
        disposal_history_df = data.get('disposal_information').sort_index()
        disposal_history_df = disposal_history_df[~disposal_history_df["分時交易"].isna(
        )].dropna(how='all')
        disposal_history_df = disposal_history_df.reset_index()[
            ["stock_id", "date", "處置結束時間"]]
        disposal_history_df.columns = ["stock_id", "處置開始時間", "處置結束時間"]
        disposal_history_df['處置開始時間'] = pd.to_datetime(
            disposal_history_df['處置開始時間'])
        disposal_history_df['處置結束時間'] = pd.to_datetime(
            disposal_history_df['處置結束時間'])
        disposal_df = get_current_and_upcoming_disposals(disposal_history_df)
        logging.info(f"查詢完成，目前市場上有 {len(disposal_df)} 筆處置股事件。")

        # 2. 現金增資 (歷史+當前)
        cash_increase_history_df = data.get('dividend_announcement')
        cash_increase_history_df.dropna(subset=['現金增資總股數(股)'], inplace=True)
        cash_increase_history_df = cash_increase_history_df[cash_increase_history_df['現金增資總股數(股)'] > 0].copy(
        )
        cash_increase_history_df['除權交易日'] = pd.to_datetime(
            cash_increase_history_df['除權交易日'], errors='coerce')
        cash_increase_history_df.dropna(subset=['除權交易日'], inplace=True)
        cash_increase_df = get_current_capital_increases_revised()
        logging.info(f"查詢完成，近期市場上有 {len(cash_increase_df)} 筆現金增資事件。")

        # 3. 庫藏股
        treasury_stock_df = get_current_treasury_stocks_by_finlab()
        logging.info(f"查詢完成，目前市場上有 {len(treasury_stock_df)} 筆庫藏股事件。")
        # 4. 法說會
        investor_conference_df = get_recent_investor_conferences()
        logging.info(f"查詢完成，近期市場上有 {len(investor_conference_df)} 筆法說會事件。")
        logging.info("--- (模式一) 市場事件資料載入完成 ---")
        
        # doc: --- [核心修改] 將產生潛在軋空候選清單的完整邏輯放入 ---
        logging.info("--- (模式一) 正在產生潛在軋空候選清單... ---")

        # 取得所有停券（強制回補）的歷史資料
        suspension_df = data.get('margin_short_sale_suspension')

        # 將日期欄位轉換為 datetime 物件，方便比較
        suspension_df['停券迄日'] = pd.to_datetime(suspension_df['停券迄日'])
        suspension_df['停券起日(最後回補日)'] = pd.to_datetime(
            suspension_df['停券起日(最後回補日)'])

        # 取得今天的日期（不含時間）
        today = pd.to_datetime(datetime.now().date())

        # 篩選條件
        upcoming_suspensions_df = suspension_df[
            (suspension_df['停券迄日'] >= today) &
            (suspension_df['stock_id'].str.len() == 4) &
            (~suspension_df['原因'].isin(['除息', '除權收益']))
        ].copy()
        upcoming_suspensions_df.sort_values(by='停券迄日', inplace=True)

        # 計算所有股票「最新一日」的券資比
        short_balance = data.get('margin_transactions:融券今日餘額')
        margin_balance = data.get('margin_transactions:融資今日餘額')
        short_margin_ratio = margin_balance / \
            short_balance.replace(0, float('nan'))
        latest_short_margin_ratio = short_margin_ratio.iloc[-1]

        # 合併資訊並美化表格
        upcoming_suspensions_df['券資比'] = upcoming_suspensions_df['stock_id'].map(
            latest_short_margin_ratio)
        company_info = data.get('company_basic_info')
        stock_name_map = company_info.set_index('stock_id')['公司簡稱']
        upcoming_suspensions_df['公司簡稱'] = upcoming_suspensions_df['stock_id'].map(
            stock_name_map)

        merged_df = upcoming_suspensions_df[[
            'stock_id', '公司簡稱', '停券起日(最後回補日)', '停券迄日', '原因', '券資比'
        ]].sort_values(by='券資比', ascending=False).reset_index(drop=True)

        merged_df.rename(columns={'停券起日(最後回補日)': '最後回補日'}, inplace=True)
        merged_df['最後回補日'] = merged_df['最後回補日'].dt.strftime('%Y-%m-%d')
        merged_df['停券迄日'] = merged_df['停券迄日'].dt.strftime('%Y-%m-%d')

        # 將最終結果賦值給 report generator 需要的變數
        short_squeeze_candidate_df = merged_df

        logging.info(
            f"--- (模式一) 已產生 {len(short_squeeze_candidate_df)} 筆潛在軋空候選股 ---")
        # doc: --- [整合結束] ---

        analyzer = StockAnalyzer(
            stock_list=target_stock_ids,
            params=PARAMS,
            stock_map=stock_map,
            disposal_history_df=disposal_history_df,
            cash_increase_history_df=cash_increase_history_df
        )

        stocks_loaded_successfully = analyzer.all_data.get(
            'close', pd.DataFrame()).columns.tolist()
        stocks_failed_to_load = set(
            target_stock_ids) - set(stocks_loaded_successfully)
        if stocks_failed_to_load:
            logging.warning(
                f"以下 {len(stocks_failed_to_load)} 支股票因數據不足被過濾: {list(stocks_failed_to_load)}")

        final_analysis_list = [
            sid for sid in target_stock_ids if sid in stocks_loaded_successfully]
        if not final_analysis_list:
            raise ValueError("所有目標成分股都因數據問題被過濾。")

        company_info_raw = data.get('company_basic_info')
        company_info_df = company_info_raw[[
            "stock_id", "公司簡稱"]].set_index('stock_id')

        broker_chart_gen = BrokerChartGenerator(analyzer.all_data['close'], analyzer.all_data['top15_buy'], analyzer.all_data['top15_sell'], analyzer.all_data['volume'],
                                                analyzer.all_data['market_cap'], company_info_df, analyzer.all_data.get('inventory_weekly_data', pd.DataFrame()), analyzer.all_data.get('shareholder_ratio', pd.DataFrame()), analyzer.all_data['margin_balance'])

        mock_report_generator = ReportGenerator(
            analyzer=analyzer,
            config=CONFIG,
            stock_selection_file='select_stock.xlsx',
            stock_topics_file='tw_stock_topics.xlsx',
            master_summary_df=pd.DataFrame(),
            strong_trend_stocks=[],
            rank_jump_stocks_df=pd.DataFrame(),
            market_breadth_data=None,
            amt_df=amt_df,
            macro_margin_data=None,
            broker_chart_generator=broker_chart_gen,
            image_dir=None,
            disposal_df=disposal_df,
            cash_increase_df=cash_increase_df,
            treasury_stock_df=treasury_stock_df,
            investor_conference_df=investor_conference_df,
            strategy_stocks_set=set(target_stock_ids),
            special_signals_df=special_signals_df,
            # doc: --- [新增] 將軋空候選清單傳入 ---
            short_squeeze_candidate_df=short_squeeze_candidate_df,
            # doc: --- [新增結束] ---
            **{
                'amt_rank_series': amt_rank_series,
                'etf_focus_stock_ids_set': set(target_stock_ids)
            }
        )

        today = datetime.now()
        year_str = today.strftime('%Y')
        month_str = today.strftime('%m')
        output_dir_monthly = os.path.join('output', year_str, month_str)
        os.makedirs(output_dir_monthly, exist_ok=True)
        output_path = os.path.join(output_dir_monthly, conf['OUTPUT_FILENAME'])
        logging.info(f"--- 開始產生獨立 ETF 分析報告，將儲存至: {output_path} ---")
        with PdfPages(output_path) as pdf:

            # doc: --- [核心修改] 移除獨立的總覽頁面，因為資訊將直接呈現在下方的個股圖表中 ---
            # mock_report_generator._plot_market_events_summary_page(
            #     pdf,
            #     disposal_df,
            #     cash_increase_df,
            #     treasury_stock_df,
            #     investor_conference_df
            # )

            for stock_id in final_analysis_list:
                stock_name = company_info_df.loc[stock_id,
                                                 '公司簡稱'] if stock_id in company_info_df.index else stock_id
                logging.info(f"正在處理 (模式一): {stock_id} {stock_name}...")
                try:
                    analysis_data = analyzer.analyze(stock_id)
                    if not analysis_data:
                        continue
                    etf_details_list = target_stock_ids_info.get(stock_id, [])
                    strategy_details = [
                        f"來自 {info['etf']} (權重: {info['weight']}% | 持有: {info['shares']:,.0f}張)" for info in etf_details_list]
                    mock_info_row = pd.Series(
                        {'股票代碼': stock_id, '股票名稱': stock_name, '主題': "ETF 成分股", '對應策略': strategy_details})

                    if CONFIG['RUN_SECTIONS'].get('F1_ETF_STOCK_PAGE', False):
                        fig1 = mock_report_generator._plot_stock_page(
                            analysis_data, mock_info_row)
                        pdf.savefig(fig1)
                        plt.close(fig1)

                    if CONFIG['RUN_SECTIONS'].get('F2_ETF_BROKER_CHART', False):
                        fig2 = broker_chart_gen.generate_chart(stock_id)
                        if fig2:
                            pdf.savefig(fig2)
                            plt.close(fig2)

                    gc.collect()
                except Exception as e_inner:
                    logging.error(
                        f"為 {stock_id} 繪製圖表時發生錯誤: {e_inner}", exc_info=True)

        logging.info(f"【模式一】報告完成！已儲存至: {output_path}")

    except Exception as e:
        logging.error(f"執行【模式一】時發生嚴重錯誤: {e}", exc_info=True)


# finlab程式碼格式
def run_full_market_report(CONFIG):
    """執行模式二：原有的完整 PDF 報告流程"""
    logging.info("\n======================================================")
    logging.info("--- 開始執行【模式二：完整市場分析報告】---")
    logging.info("======================================================")
    try:
        STOCK_SELECTION_FILE, STOCK_TOPICS_FILE = 'select_stock.xlsx', 'tw_stock_topics.xlsx'
        if not os.path.exists(STOCK_SELECTION_FILE) or not os.path.exists(STOCK_TOPICS_FILE):
            raise FileNotFoundError(
                f"請確認 '{STOCK_SELECTION_FILE}' 和 '{STOCK_TOPICS_FILE}' 檔案存在於當前目錄。")

        # doc: =============================================================================
        # doc: --- 【錯誤修正核心】第 1 步：提前執行所有「步驟 1」的資料收集 ---
        # doc: 將原先分散在各處的股票清單收集程式碼全部集中在此，確保 all_stocks_to_analyze 在被使用前已定義。
        # doc: =============================================================================
        logging.info("--- 步驟 1-1：從各來源收集股票代碼 ---")
        RUN_MARGIN_TRAP_ANALYSIS = CONFIG['RUN_SECTIONS'].get(
            'A2_MARGIN_TRAP', False)
        amt_df = data.get('price:成交金額')
        latest_amt = amt_df.iloc[-1]
        amt_rank_series = latest_amt.rank(method='min', ascending=False)
        logging.info(
            f"計算完成，今日全市場共有 {len(amt_rank_series.dropna())} 檔股票有成交額排名。")

        temp_df = pd.read_excel(STOCK_SELECTION_FILE)
        # doc: =============================================================================
        # doc: --- 【核心修正】更穩健地處理 Excel 中的非數字字元 ---
        # doc: 使用 pd.to_numeric(errors='coerce') 先將無效值轉為 NaN，再移除。
        # doc: =============================================================================
        strategy_stocks_set = set()
        for col in temp_df.columns:
            # 1. 嘗試轉換為數字，無法轉換的變成 NaN
            numeric_series = pd.to_numeric(temp_df[col], errors='coerce')
            # 2. 移除 NaN 值
            valid_numbers = numeric_series.dropna()
            # 3. 將有效的數字轉換為整數，再轉換為字串，加入集合
            strategy_stocks_set.update(valid_numbers.astype(int).astype(str))
        # doc: --- [修正結束] ---

        logging.info(f"從策略檔案讀取到 {len(strategy_stocks_set)} 檔股票。")

        strong_trend_stocks = preliminary_industry_analysis()
        strong_trend_stocks_set = set(
            strong_trend_stocks) if strong_trend_stocks else set()
        logging.info(f"從初步分析找到 {len(strong_trend_stocks_set)} 檔強勢股。")

        rank_jump_stocks_df = find_rank_jump_stocks()
        rank_jump_stocks_set = set(rank_jump_stocks_df['stock_id'].tolist(
        )) if rank_jump_stocks_df is not None and not rank_jump_stocks_df.empty else set()
        logging.info(f"從排名分析找到 {len(rank_jump_stocks_set)} 檔躍升股。")

        market_breadth_data = analyze_market_breadth()
        new_high_200_stocks_set = set(market_breadth_data['latest_new_high_list']['stock_id'].tolist(
        )) if market_breadth_data and not market_breadth_data.get('latest_new_high_list', pd.DataFrame()).empty else set()
        logging.info(f"從市場寬度找到 {len(new_high_200_stocks_set)} 檔創200日新高股。")

        new_high_20_stocks_set = set(market_breadth_data['latest_new_high_20_list']['stock_id'].tolist(
        )) if market_breadth_data and not market_breadth_data.get('latest_new_high_20_list', pd.DataFrame()).empty else set()
        logging.info(f"從市場寬度找到 {len(new_high_20_stocks_set)} 檔創20日新高股。")

        macro_margin_data = analyze_macro_margin_indicators()

        (etf_stocks_set, etf_q_consensus_df, etf_q_heavyweight_df,
         etf_m_heavyweight_df, etf_quadrant_data_q, etf_quadrant_data_m,
         etf_focus_ids, etf_q_date, etf_m_date) = analyze_etf_holdings()
        logging.info(f"[ETF分析] 找到 {len(etf_stocks_set)} 檔 ETF 焦點股。")

        special_signals_df = compute_special_signals()

        daily_etf_holdings_map, daily_etf_rank_map, daily_etf_stock_ids = {}, {}, set()
        try:
            logging.info("--- 步驟 1-1b: [新增] 提前分析每日主動式ETF持股以供預加載 ---")
            df_etf_daily = pd.read_csv("all_etf_holdings.csv")
            df_etf_daily['代號'] = df_etf_daily['代號'].astype(str)
            latest_date_daily = pd.to_datetime(df_etf_daily['日期']).max()
            df_latest_daily = df_etf_daily[pd.to_datetime(
                df_etf_daily['日期']) == latest_date_daily].copy()
            daily_etf_holdings_map = df_latest_daily.groupby(
                '代號')['etf'].apply(list).to_dict()
            close = data.get('price:收盤價')
            latest_close = close.iloc[-1].reindex(
                df_latest_daily['代號'].unique()).dropna()
            df_latest_daily['close'] = df_latest_daily['代號'].map(latest_close)
            df_latest_daily.dropna(subset=['close'], inplace=True)
            df_latest_daily['market_cap'] = (
                df_latest_daily['持有數'] * 1000) * df_latest_daily['close']
            stock_total_cap_daily = df_latest_daily.groupby(
                '代號')['market_cap'].sum().sort_values(ascending=False)
            top_N_daily = CONFIG.get('DAILY_ETF_TOP_N', 60)
            top_stocks_daily = stock_total_cap_daily.head(top_N_daily)
            daily_etf_rank_map = {stock_id: rank + 1 for rank,
                                  stock_id in enumerate(top_stocks_daily.index)}
            daily_etf_stock_ids = set(df_latest_daily['代號'].unique())
            logging.info(
                f"從每日ETF資料中找到 {len(daily_etf_stock_ids)} 檔獨特持股以進行預加載。")
        except Exception as e:
            logging.error(f"提前分析每日ETF資料時發生錯誤: {e}")

        margin_trap_df, trap_start, trap_end, margin_trap_stocks_set = pd.DataFrame(), "", "", set()
        if RUN_MARGIN_TRAP_ANALYSIS:
            margin_trap_df, trap_start, trap_end = find_margin_trap_stocks()
            margin_trap_stocks_set = set(
                margin_trap_df['stock_id'].tolist()) if not margin_trap_df.empty else set()
            logging.info(f"從宏觀分析找到 {len(margin_trap_stocks_set)} 檔凹單股。")
        else:
            logging.info("--- 已跳過【融資凹單股】分析 ---")

        logging.info("--- 步驟 1-1e: [進階版] 正在載入多因子篩選所需數據... ---")
        close = data.get('price:收盤價')
        rev = data.get('monthly_revenue:當月營收')
        amt = data.get('price:成交金額')
        monthly_revenue_yoy = data.get('monthly_revenue:去年同月增減(%)')
        volatility = compute_candle_volatility()
        conditions_met = (
            (close > close.average(20)) &
            (close > close.average(60)) & 
            (close > close.average(120)) &
            (rev.average(3) > rev.average(12)) &
            (monthly_revenue_yoy > 0) &
            ( amt > 20000000) &
            (volatility <= 8)
        )
        latest_conditions = conditions_met.iloc[-1]
        filtered_stock_ids = latest_conditions[latest_conditions].index.tolist(
        )
        logging.info(
            f"--- 步驟 1-1e: [進階版] 共有 {len(filtered_stock_ids)} 檔股票滿足您的多因子篩選條件。---")

        temp_master_summary = ReportGenerator._prepare_industry_data(
            stock_list=filtered_stock_ids)
        price_change_top25_df = pd.DataFrame()
        price_change_top25_set = set()
        if temp_master_summary is not None and not temp_master_summary.empty:
            price_change_top25_df = temp_master_summary.sort_values(
                by='漲跌幅_5日(%)', ascending=False).head(25).copy()
            price_change_top25_df['rank'] = range(
                1, len(price_change_top25_df) + 1)
            price_change_top25_set = set(
                price_change_top25_df['stock_id'].tolist())
        logging.info(f"最終選入附錄的股票共 {len(price_change_top25_set)} 檔。")

        logging.info("--- 步驟 1-1c: 正在查詢市場事件... ---")
        disposal_history_df = data.get('disposal_information').sort_index()
        disposal_history_df = disposal_history_df[~disposal_history_df["分時交易"].isna(
        )].dropna(how='all')
        disposal_history_df = disposal_history_df.reset_index()[
            ["stock_id", "date", "處置結束時間"]]
        disposal_history_df.columns = ["stock_id", "處置開始時間", "處置結束時間"]
        disposal_history_df['處置開始時間'] = pd.to_datetime(
            disposal_history_df['處置開始時間'])
        disposal_history_df['處置結束時間'] = pd.to_datetime(
            disposal_history_df['處置結束時間'])
        disposal_df = get_current_and_upcoming_disposals(disposal_history_df)

        cash_increase_history_df = data.get('dividend_announcement')
        cash_increase_history_df.dropna(subset=['現金增資總股數(股)'], inplace=True)
        cash_increase_history_df = cash_increase_history_df[cash_increase_history_df['現金增資總股數(股)'] > 0].copy(
        )
        cash_increase_history_df['除權交易日'] = pd.to_datetime(
            cash_increase_history_df['除權交易日'], errors='coerce')
        cash_increase_history_df.dropna(subset=['除權交易日'], inplace=True)
        cash_increase_df = get_current_capital_increases_revised()

        treasury_stock_df = get_current_treasury_stocks_by_finlab()
        investor_conference_df = get_recent_investor_conferences()

        suspension_df = data.get('margin_short_sale_suspension')
        suspension_df['停券迄日'] = pd.to_datetime(suspension_df['停券迄日'])
        suspension_df['停券起日(最後回補日)'] = pd.to_datetime(
            suspension_df['停券起日(最後回補日)'])
        today = pd.to_datetime(datetime.now().date())
        upcoming_suspensions_df = suspension_df[
            (suspension_df['停券迄日'] >= today) & (suspension_df['stock_id'].str.len() == 4) &
            (~suspension_df['原因'].isin(['除息', '除權收益']))
        ].copy()
        short_balance = data.get('margin_transactions:融券今日餘額')
        margin_balance = data.get('margin_transactions:融資今日餘額')
        short_margin_ratio = margin_balance / \
            short_balance.replace(0, float('nan'))
        upcoming_suspensions_df['券資比'] = upcoming_suspensions_df['stock_id'].map(
            short_margin_ratio.iloc[-1])
        company_info_map = data.get(
            'company_basic_info').set_index('stock_id')['公司簡稱']
        upcoming_suspensions_df['公司簡稱'] = upcoming_suspensions_df['stock_id'].map(
            company_info_map)
        short_squeeze_candidate_df = upcoming_suspensions_df.rename(
            columns={'停券起日(最後回補日)': '最後回補日'})

        logging.info("--- 步驟 1-2：合併所有股票代碼建立統一分析範疇 ---")
        all_stocks_set = set.union(
            strategy_stocks_set, strong_trend_stocks_set, rank_jump_stocks_set,
            margin_trap_stocks_set, new_high_200_stocks_set, new_high_20_stocks_set,
            etf_stocks_set, daily_etf_stock_ids, price_change_top25_set
        )
        all_stocks_to_analyze = list(all_stocks_set)
        logging.info(f"合併後，總共需要分析 {len(all_stocks_to_analyze)} 檔獨一無二的股票。")

        # ===================================================================
        # --- 步驟 2：核心分析引擎初始化與計算 ---
        # ===================================================================
        logging.info("--- 步驟 2-1：建立基礎資訊總表 (名稱/產業/漲跌幅) ---")
        base_summary_df = ReportGenerator._prepare_industry_data(
            stock_list=all_stocks_to_analyze)

        PARAMS = {
            'plot_days_short': 240, 'plot_days_long': 480, 'max_data_days': 550, 'sr_n_order_daily': 5,
            'sr_n_order_weekly': 3, 'eps_atr_multiplier': 0.5, 'volume_ma_period': 20, 'poc_base_score': 2.0,
            'volume_ratio_cap': 2.0, 'conversion_breakout_window': 10, 'conversion_hold_ratio': 0.6,
            'conversion_retest_tolerance': 0.01, 'conversion_vol_boost_multiplier': 1.5,
            'conversion_retest_bonus': 2.0, 'vp_window_short': 60, 'vp_window_long': 240,
            'vp_bins': 80, 'plot_weeks_long': 104,
        }

        stock_topics_df = pd.read_excel(STOCK_TOPICS_FILE)
        stock_topics_df['stock_no'] = pd.to_numeric(
            stock_topics_df['stock_no'], errors='coerce').dropna().astype(int)
        stock_map = stock_topics_df.set_index(
            'stock_no')['stock_name'].to_dict()

        logging.info("--- 步驟 2-2：初始化分析引擎並預載入所有數據 ---")
        analyzer = StockAnalyzer(
            stock_list=all_stocks_to_analyze,
            params=PARAMS,
            stock_map=stock_map,
            disposal_history_df=disposal_history_df,
            cash_increase_history_df=cash_increase_history_df
        )

        summary_data_list = []
        for stock_id in analyzer.all_data.get('close', pd.DataFrame()).columns:
            stock_name = analyzer.stock_map.get(int(stock_id), "未知")
            logging.info(f"正在分析 {stock_id} {stock_name} 的摘要數據...")
            analysis_data = analyzer.analyze(stock_id)
            if analysis_data is None:
                continue
            sentiment_conclusion, sentiment_score = get_margin_sentiment_conclusion(
                analysis_data['sentiment'])
            shareholder_ratio_text, shareholder_trend, shareholder_ratio_raw = get_shareholder_ratio_conclusion(
                analysis_data['shareholder_structure'])
            broker_conclusion_text, broker_flow_raw = get_broker_flow_conclusion(
                analysis_data['broker_flow'])
            summary_data_list.append({
                'stock_id': stock_id, '情緒結論': sentiment_conclusion, '情緒結論_raw': sentiment_score,
                '大戶/散戶比': shareholder_ratio_text, '籌碼趨勢(6週)': shareholder_trend,
                '大戶/散戶比_raw': shareholder_ratio_raw, '關鍵券商(4週)': broker_conclusion_text, '關鍵券商(4週)_raw': broker_flow_raw,
            })
        extra_summary_df = pd.DataFrame(summary_data_list)

        logging.info("--- 步驟 2-3：計算投信作多進階指標 ---")
        it_momentum_df = calculate_it_momentum_signals(analyzer.all_data)

        logging.info("--- 步驟 2-4：合併基礎與進階數據，建立最終全功能總表 ---")
        master_summary_df = pd.merge(base_summary_df, extra_summary_df, on='stock_id',
                                     how='left') if not extra_summary_df.empty else base_summary_df
        if not it_momentum_df.empty:
            master_summary_df = pd.merge(
                master_summary_df, it_momentum_df, on='stock_id', how='left')

        it_buy_shares = analyzer.all_data.get('it_buy')
        if it_buy_shares is not None and not it_buy_shares.empty and len(it_buy_shares) >= 20:
            it_buy_lots = it_buy_shares / 1000
            it_extra_df = pd.DataFrame({
                '投信今日買賣超(張)': it_buy_lots.iloc[-1],
                '投信近5日累積買賣超(張)': it_buy_lots.rolling(5, min_periods=1).sum().iloc[-1],
                '投信近20日累積買賣超(張)': it_buy_lots.rolling(20, min_periods=1).sum().iloc[-1]
            })
            it_extra_df['stock_id'] = it_extra_df.index.astype(str)
            master_summary_df = pd.merge(
                master_summary_df, it_extra_df, on='stock_id', how='left')

        master_summary_df.fillna({
            '情緒結論': '數據不足', '大戶/散戶比': 'N/A', '籌碼趨勢(6週)': '', '關鍵券商(4週)': '數據不足',
            '投信作多原因': '', '投信作多強度': 0.0, '投信今日買賣超(張)': 0.0,
            '投信近5日累積買賣超(張)': 0.0, '投信近20日累積買賣超(張)': 0.0
        }, inplace=True)

        logging.info("--- 全功能分析總表建立完成 ---")

        # ===================================================================
        # --- 步驟 3：初始化報告生成器並產出報告 ---
        # ===================================================================
        company_info = data.get('company_basic_info')[
            ["stock_id", "公司簡稱"]].set_index('stock_id')
        broker_chart_gen = BrokerChartGenerator(
            close_price=analyzer.all_data['close'], buy_vol=analyzer.all_data['top15_buy'],
            sell_vol=analyzer.all_data['top15_sell'], volume=analyzer.all_data['volume'],
            market_cap=analyzer.all_data['market_cap'], company_info=company_info,
            inventory_weekly_data=analyzer.all_data.get(
                'inventory_weekly_data', pd.DataFrame()),
            shareholder_ratio=analyzer.all_data.get(
                'shareholder_ratio', pd.DataFrame()),
            margin_balance=analyzer.all_data['margin_balance']
        )

        today_str = datetime.today().strftime("%Y%m%d")
        output_dir = os.path.join('output', datetime.today().strftime(
            '%Y'), datetime.today().strftime('%m'))
        image_dir = os.path.join(
            output_dir, f'charts_{today_str}') if CONFIG['SAVE_CHARTS_AS_FILES'] else None
        if image_dir:
            os.makedirs(image_dir, exist_ok=True)

        logging.info("--- 正在初始化【主報告】產生器 ---")
        report_generator = ReportGenerator(
            analyzer=analyzer, config=CONFIG, stock_selection_file=STOCK_SELECTION_FILE,
            stock_topics_file=STOCK_TOPICS_FILE, master_summary_df=master_summary_df,
            strong_trend_stocks=strong_trend_stocks, rank_jump_stocks_df=rank_jump_stocks_df,
            market_breadth_data=market_breadth_data, amt_df=amt_df,
            macro_margin_data=macro_margin_data, broker_chart_generator=broker_chart_gen,
            image_dir=image_dir, disposal_df=disposal_df, cash_increase_df=cash_increase_df,
            treasury_stock_df=treasury_stock_df, investor_conference_df=investor_conference_df,
            short_squeeze_candidate_df=short_squeeze_candidate_df,
            strategy_stocks_set=strategy_stocks_set, etf_consensus_df=etf_q_consensus_df,
            etf_heavyweight_df=etf_q_heavyweight_df, etf_monthly_heavyweight_df=etf_m_heavyweight_df,
            etf_quadrant_data=etf_quadrant_data_q, etf_focus_stock_ids=etf_focus_ids,
            etf_latest_quarter_date=etf_q_date, etf_latest_month_date=etf_m_date,
            etf_monthly_quadrant_data=etf_quadrant_data_m, margin_trap_df=margin_trap_df,
            trap_dates=(
                trap_start, trap_end), daily_etf_holdings_map=daily_etf_holdings_map,
            daily_etf_rank_map=daily_etf_rank_map, special_signals_df=special_signals_df,
            price_change_top25_df=price_change_top25_df,
            **{
                'new_high_200_set': new_high_200_stocks_set, 'new_high_20_set': new_high_20_stocks_set,
                'strong_trend_set': strong_trend_stocks_set, 'rank_jump_set': rank_jump_stocks_set,
                'amt_rank_series': amt_rank_series, 'etf_focus_stock_ids_set': etf_stocks_set
            }
        )

        logging.info("--- 主報告產生器初始化完成，開始生成報告... ---")
        report_generator.create_report()

    except Exception as e:
        logging.error(f"程式主流程執行失敗: {e}", exc_info=True)
        # 即使出錯，如果 report_generator 已成功初始化，仍嘗試產生報告
        if 'report_generator' in locals():
            logging.info("--- 嘗試在發生錯誤後繼續生成報告... ---")
            report_generator.create_report()





# ===================================================================
# --- 程式啟動點 ---
# ===================================================================
if __name__ == "__main__":
    
    # doc: ===============================================================
    # doc: --- 【總開關】請在此處修改您想執行的模式 ---
    # doc: 1: 只執行【模式一：ETF 焦點分析】
    # doc: 2: 執行【模式二：完整市場報告】(會根據底下 RUN_SECTIONS 設定)
    # doc: ===============================================================
    '''
    想只畫 ETF 成分股報告？
    將 MODE_TO_RUN 設為 1。
    
    
    想畫完整的市場宏觀報告 + 個股詳細報告？
    將 MODE_TO_RUN 設為 2。
    並將 RUN_SECTIONS 中 A, B, C, D, E 區塊的開關都設為 True。
    
    
    想只畫策略清單 (select_stock.xlsx) 的個股詳細報告？
    將 MODE_TO_RUN 設為 2。
    並使用我上面提供的 RUN_SECTIONS 設定（即 A, B, C, E 區塊都設為 False，只有 D 區塊為 True）。
    '''
    MODE_TO_RUN = 2  # <--- 在這裡切換模式 (1 或 2)
    
    
    
    CONFIG = {
        
        # ===================================================================
        # --- 【[新增] 我的excel區核心報告專用篩選器】 ---
        # ===================================================================
        'STRATEGY_STOCK_PRICE_CHANGE_FILTER': {
            'ENABLED': False,           # 設為 True 來啟用此功能，False 則停用
            'THRESHOLD_PCT': 5.0       # 設定漲跌幅門檻，例如 5.0 代表只顯示漲幅 > 5% 的股票
        },
        
        
        # ===================================================================
        # --- 【[新增] 附錄專用篩選器】 ---
        # ===================================================================
        'APPENDIX_PRICE_CHANGE_FILTER': {
            'ENABLED': True,           # 設定為 True 來啟用此功能，False 則停用
            'THRESHOLD_PCT': 0       # 設定漲跌幅門檻，例如 5.0 代表只顯示漲幅 > 5% 的股票
        },
        
        
        # ===================================================================
        # --- 模式一：指定 ETF 成分股，產出獨立 PDF 報告 ---
        # ===================================================================
        'FOCUSED_ETF_PDF_ANALYSIS': {

            # doc: 總開關。設為 True 時，程式只會執行這個獨立分析模式。
            # doc: 設為 False 時，則會執行底下您原有的「完整市場分析報告」流程。
            'ENABLED': True,

            # doc: 【請在此處修改】您想獨立分析的 ETF 清單'00982A','00980A','00984A'。
            'TARGET_ETFS': ['00981A'],

            # doc: 在獨立分析模式下，輸出的 PDF 檔案名稱。
            'OUTPUT_FILENAME': f'00981A_report{datetime.now().strftime("%Y%m%d")}.pdf',
        },

        # ===================================================================
        # --- 模式二 (原有功能) 的全域設定 ---
        # ===================================================================

        # doc: (此設定僅對「完整報告模式」有效)
        # doc: 是否將報告中「策略選股」的圖表另外儲存成獨立的 .png 圖片檔。
        'SAVE_CHARTS_AS_FILES': False,

        # doc: (此設定僅對「完整報告模式」有效)
        # doc: 是否要產生最終的完整市場分析報告 PDF 檔案。
        'GENERATE_PDF_REPORT': True,

        # doc: 各類進階因子計算時所需的參數設定。
        'FACTOR_PARAMS': {
            # doc: 「實體紅棒強度」因子的參數。
            'BULLISH_STRENGTH_THRESHOLD': 0.5,  # 紅棒強度門檻值 (0.0 ~ 1.0)。
            'BULLISH_STRENGTH_PERIOD': 1,      # 計算紅棒強度的平均天期。

            # doc: 「市值趨勢」分析的參數。
            'TREND_ANALYSIS_WINDOW': 5,        # 回溯多少筆資料來計算趨勢 (例如最近5筆)。
            'TREND_SLOPE_THRESHOLDS': {        # 趨勢斜率的門檻，用來區分強弱。
                'STRONG_INCREASE': 0.5,        # 斜率 > 0.5, 視為 [強勢加碼]。
                'STRONG_DECREASE': -0.5        # 斜率 < -0.5, 視為 [趨勢減碼]。
            }
        },

        # doc: (此設定僅對「完整報告模式」有效)
        # doc: 在分析「每日主動式ETF」時，要取市值前幾名的成分股來顯示。
        'DAILY_ETF_TOP_N': 60,

        # doc: (此設定僅對「完整報告模式」有效)
        # doc: 控制完整報告中「附錄(E區)」各個章節的出現順序。
        'APPENDIX_ORDER': [
            'E6_APPENDIX_DAILY_ETF',
            'E5_APPENDIX_ETF_STOCKS',
            'E7_APPENDIX_PRICE_CHANGE',      # <--- 新增的漲跌幅篩選附錄
            'E1_APPENDIX_NEW_HIGH_200',
            'E2_APPENDIX_NEW_HIGH_20',
            'E3_APPENDIX_STRONG_TREND',
            'E4_APPENDIX_RANK_JUMP',
        ],

        # ===================================================================
        # --- 報告區塊生成總開關 (最重要的設定區) ---
        # ===================================================================
        'RUN_SECTIONS': {
            # --- A. 宏觀與市場掃描 ---
            'A1_MACRO_MARGIN': True,          # 市場融資總覽
            'A2_MARGIN_TRAP': False,         # 融資凹單股分析
            'A3_INDUSTRY_FLOW': True,        # 產業資金流向 (百強股)
            'A4_MARKET_CAP_FLOW': True,      # 市值板塊資金流向
            'A5_MARKET_BREADTH_200D': True,   # 市場寬度指標 (創200日新高)
            'A6_MARKET_BREADTH_20D': True,    # 市場寬度指標 (創20日新高)
            'A7_ETF_SUMMARY': True,          # 每月/季 ETF 持股摘要
            'A8_ETF_QUADRANT': True,         # 每月/季 ETF 四象限分析
            'A9_ETF_TREND': True,            # 每月/季 ETF 關鍵股資金流比較
            'A10_ETF_MONTHLY_QUADRANT': True,  # 月度 ETF 四象限分析
            'A11_DAILY_ETF_ANALYSIS': True,  # 每日主動式 ETF 焦點股分析

            # --- B. 策略綜合分析 ---
            'B1_MULTIFACTOR_SUMMARY': True,    # 多因子強勢股總覽
            'B2_STRATEGY_STACKED_CHART': True,  # 策略選股分佈圖
            'B3_STRATEGY_TOPICS_PIE': True,    # 熱門族群分佈圖
            'B4_STRATEGY_PERFORMANCE': False,   # (已停用) 策略績效回測
            'B5_STRATEGY_INDUSTRY_OVERVIEW': False,  # (已停用) 策略選股產業總覽
            'B6_STRATEGY_INDUSTRY_DRILLDOWN': False,  # (已停用) 策略選股產業深入分析

            # --- C. 全市場產業分析 ---
            'C1_OVERALL_INDUSTRY_STRENGTH': True,  # 全市場產業強弱勢總覽

            # --- D. 個股詳細報告 (核心區塊) ---
            'D1_DETAILED_STRATEGY_STOCKS': True,  # 為 B1 區塊選出的股票，產生詳細的兩頁分析報告

            # --- E. 附錄 (提供額外參考清單的詳細報告) ---
            'E7_APPENDIX_PRICE_CHANGE': True,  # [新增] 新增漲幅排行附錄的開關
            'E1_APPENDIX_NEW_HIGH_200': True,
            'E2_APPENDIX_NEW_HIGH_20': True,
            'E3_APPENDIX_STRONG_TREND': True,
            'E4_APPENDIX_RANK_JUMP': False,
            'E5_APPENDIX_ETF_STOCKS': False,
            'E6_APPENDIX_DAILY_ETF': False,

            # --- F. 獨立 ETF 分析模式專用開關 (僅在 ENABLED:True 時有效) ---
            'F1_ETF_STOCK_PAGE': True,       # 產生「六宮格分析圖」
            'F2_ETF_BROKER_CHART': True,     # 產生「券商綜合分析圖」
        },

        # ===================================================================
        # --- 獨立圖片儲存設定 (僅在 SAVE_CHARTS_AS_FILES:True 時有效) ---
        # ===================================================================
        'SAVE_IMAGE_SECTIONS': {
            # doc: 將此處從 False 改為 True，代表您「要」儲存策略股 (D1區塊) 的詳細圖檔。
            'D1_DETAILED_STRATEGY_STOCKS': True,
            'E1_APPENDIX_NEW_HIGH_200': False,
            'E2_APPENDIX_NEW_HIGH_20': False,
            'E3_APPENDIX_STRONG_TREND': False,
            'E4_APPENDIX_RANK_JUMP': False,
            'E5_APPENDIX_ETF_STOCKS': False,
            'E6_APPENDIX_DAILY_ETF': False,
        }
    }

    # --- 初始化 ---
    load_dotenv()
    finlab.login(os.getenv('FINLAB_API_KEY'))
    
    # --- 根據 MODE 選擇執行的函式 ---
    if MODE_TO_RUN == 1:
        run_focused_etf_analysis(CONFIG)
    elif MODE_TO_RUN == 2:
        run_full_market_report(CONFIG)

    logging.info("\n--- 所有已啟用的分析任務執行完畢 ---")
    
    
    
    
    
    
    
    
    

輸入成功!


Use "pip install finlab==1.5.7" to update the latest version.



[產業分析模組] 正在準備... [買賣超金額佔總市值比] 與 [營收指標] 數據...
Daily usage: 2420.6 / 5000 MB - institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)
Daily usage: 2429.9 / 5000 MB - institutional_investors_trading_summary:投信買賣超股數
[產業分析模組] [買賣超金額佔總市值比]、[營收指標] 與 [產業資金流] 數據準備完成。
Daily usage: 2430.5 / 5000 MB - disposal_information
Daily usage: 2433.5 / 5000 MB - dividend_announcement
正在從 FinLab 數據庫撈取庫藏股資料...
資料撈取完畢，開始整理與篩選...
Daily usage: 2440.1 / 5000 MB - investors_conference
Daily usage: 2441.2 / 5000 MB - margin_short_sale_suspension

[產業分析模組] 正在準備... [買賣超金額佔總市值比] 與 [營收指標] 數據...
[產業分析模組] [買賣超金額佔總市值比]、[營收指標] 與 [產業資金流] 數據準備完成。
正在預加載所有市場數據...
正在加載: price:開盤價
正在加載: price:最高價
正在加載: price:最低價
正在加載: price:收盤價
正在加載: price:成交股數
正在加載: institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)
正在加載: institutional_investors_trading_summary:投信買賣超股數
正在加載: institutional_investors_trading_summary:自營商買賣超股數(自行買賣)
Daily usage: 2457.4 / 5000 MB - institutional_investors_trading_summary:自營商買賣超股數(自行買賣)
正在加載: margin_trans

c:\Users\user\anaconda3\envs\FINLAB\lib\site-packages\finlab\data\data.py:381: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, short_df])\


Daily usage: 2475.5 / 5000 MB - etl:broker_transactions:top15_buy
正在加載: etl:broker_transactions:top15_sell


c:\Users\user\anaconda3\envs\FINLAB\lib\site-packages\finlab\data\data.py:381: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, short_df])\


Daily usage: 2490.6 / 5000 MB - etl:broker_transactions:top15_sell
正在加載: etl:market_value
正在加載: intraday_trading:當日沖銷交易成交股數
正在加載: fundamental_features:營業毛利率
正在加載: fundamental_features:營業利益率
正在加載: fundamental_features:ROE稅後
正在加載: financial_statement:每股盈餘
正在計算新的股權結構指標...
股權結構指標計算完成。


C:\Users\user\AppData\Local\Temp\ipykernel_18620\1454881114.py:8138: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  pdf.savefig(fig)
C:\Users\user\AppData\Local\Temp\ipykernel_18620\1454881114.py:8138: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  pdf.savefig(fig)
C:\Users\user\AppData\Local\Temp\ipykernel_18620\1454881114.py:8138: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  pdf.savefig(fig)
C:\Users\user\AppData\Local\Temp\ipykernel_18620\1454881114.py:8138: UserWarning: constrained_layout not applied because axes sizes collapsed to zero.  Try making figure larger or Axes decorations smaller.
  pdf.savefig(fig)
C:\Users\user\AppData\Local\Temp\ipykernel_18620\1454881114.py:8138: UserWarning: constrained_layout